# Stage 2 rev12 — captions that hold every valid change the pipeline measured

**Why rev12** (from the rev11 trace on the 82 sample2 pairs, and the RSCC reports):
- The system caption was Qwen's blind draft **copied verbatim**, with corrections **appended at the end** in the pipeline's own wording ("Collapsed-building rubble across the scene has become bare ground").
- Only **9/82** captions held every detected change: flood 89%, earthquake 61%, volcano 58%. The checker could add at most 2 measured changes and send at most 4 edits, and it never used texture change, RemoteCLIP's "cloud" label on clear ground, AnyChange or SAM2.
- On TONGA 000024 the dark deposit behind the volcano verdict never reached Qwen. On turkey_earthquake5_336 the caption never named the earthquake, and an appended sentence created a new contradiction ("surrounding intact buildings").
- `_ZONE_WORDS` held 24 literal backspace characters where `\b` was meant, so single directions (north, western, centre…) never matched and the location tests were effectively off.

**The system caption is now EVIDENCE → LOOK → WRITE → VERIFY** (`CFG.caption_mode = "look"`):

| Step | What happens | Who |
|---|---|---|
| 1 · Evidence | `caption_evidence`: every valid change the pipeline measured, **uncapped** — located physical changes (incl. texture), typed darkening, RemoteCLIP changes (incl. a pale veil RemoteCLIP calls "cloud" on clear ground), confirmed surfaces, damaged buildings, AnyChange regions, SAM2 object changes. A change in two separate places is two items. Exclusions are logged with the reason (below 2% of the tile, shadow, unreliable "cloud" BEFORE class, unconfirmed water, contradicted by a confident absence) | code |
| 2 · Look | Items are grouped by place (≤ `caption_max_regions` close-ups; extra items widen a crop, none dropped). Qwen sees a BEFORE/AFTER close-up of each region and says what changed — **no evidence words in the prompt**. An item it misses gets one guided look with a plain cue ("parts of it look darker"); an item it still cannot see is logged as *not visible* | Qwen |
| 3 · Write | One caption **from scratch**: the full images, the verdict and Qwen's **own notes** for every region. The blind draft is never passed, so nothing is kept or appended to | Qwen |
| 4 · Verify | Every visible change must be in the caption (its kind, or its note, at its place), the event named, AFTER clouds mentioned, no confident measurement contradicted. Otherwise the caption is **rewritten from scratch** with the omitted notes flagged (≤ `caption_max_rewrites`); the best attempt is kept | code |

Every item's status (visible / not visible / excluded + reason / covered / missing) is in `r.look`, in the §22 reports and in §21 G.

| | Change | Switch |
|---|---|---|
| §2 | `caption_mode="look"` default; size floor, regions, crops, guided look, confirmation, rewrites, coverage thresholds | `caption_*`, `look_max_new_tokens` |
| §5 | Deposit and burn carry where they are (3×3 shares) | — |
| §11 | Zone-word bug fixed; every measured change carries its 3×3 location; `measured_facts` adds AnyChange regions, SAM2 object changes and all RemoteCLIP changes; the look-then-write code; `QwenCaptioner.look` / `.compose` | — |
| §12 | `predict()` runs look-then-write; `r.look` (record) and `r.look_crops` (close-ups) | `caption_mode` |
| §21 | C1 = rev12; C2 = same looks without the verdict; **C5 = the rev9–rev11 revise caption** for comparison; G reports coverage by kind of change; I uses a verdict flip on the same looks and a full grey-image rerun. Outputs `*_rev12.*` | `CC_RUN_REVISE` |
| §22–§23 | Reports show the evidence table, each close-up with Qwen's notes, every attempt with its check and the coverage; summaries print coverage by kind. Output folders `evidence_trace_rev12*` | `trace_folder(caption_mode=...)` |

**Unchanged:** classification (DKG, zero-shot prior, pooling, calibration), RVG, all perception modules. **Cost:** about 4–10 short close-up calls plus 1–3 writing calls per pair, roughly 1.5–2× the rev11 caption time.

**Still to validate on Kaggle:** coverage and METEOR/ROUGE in §21 against C3 and C5; how often Qwen cannot see a measured change (`not_visible`, by kind) — a high rate for one kind means that detector over-fires or its close-up is too small.

---

# Stage 2 rev11 — dark deposits vs burns vs shadows; water bodies must smooth; open-class §21

**Why rev11** (from the rev10 run on 52 pairs: flood 13/13, volcano 27/39, down from 29):
- All 12 volcano misses had **little or no DKG evidence**, so the zero-shot prior decided. That prior is right on only 9 of 39 volcano pairs, and it called dark scenes wildfire, cyclone or earthquake.
- Dark deposits (TONGA 000024, 000076) had no route to any rule. Physical priors use whole-tile means, RemoteCLIP has no "dark deposit" phrase, and the "ash" that rev9 saw in 000024 was a cloud, now masked.
- On 7 volcano pairs the DKG still said flood, because RemoteCLIP calls ash-darkened ground "water" and rev10 let a connected body of such cells count.

**Changes, all defined by surface physics, none by an event:**

| | Change | Switch |
|---|---|---|
| §3 | New maps: chromaticity shift, colour loss, smoothing, post texture (outside the composite) | — |
| §5 | **Darkening is split by what it does to different surfaces,** per 3×3 zone. *Deposit*: open ground (roads, bare or paved ground; building footprints and confirmed water excluded) darkened and changed colour at least half as much as vegetation. It feeds ashfall. *Burn*: vegetation only, open ground untouched; it feeds burned. *Shadow*: darker but the same colours; ignored. A zone with no open ground is judged against the rest of the tile | `deposit_*` |
| §5 | **A RemoteCLIP water body counts only if it became smoother or is very flat.** Water smooths a surface; ash-darkened fields keep their structure | `water_body_smooth`, `water_body_flat` |
| §11 | The checker passes the typed darkening to Qwen ("a dark layer now covers roads and open ground…", "vegetation … has been blackened") instead of only "darker ground" | — |
| §22a | The trace lists deposit, burn, ambiguous and shadow cover with locations, plus a per-zone breakdown | — |
| §21 | **Classes are open.** Any disaster word in a pair's name (wildfire, hurricane, landslide…) gives its class, and every table follows the classes present. Regression reference: rev9 (best), with rev10 shown alongside | — |
| §22b | Fixed `refs_file`: the edited line `globals().get("<path>")` always returned None, so reports lost their RSCC references. Output now goes to `evidence_trace_rev11` | — |

**Tested here:**
- On all 11 Tonga report pairs: every one is a deposit, with no burn.
- On synthetic scenes: a burn is called burn, a deposit deposit, a shadow shadow.
- The flood case is protected: the volcano rule is vetoed by inundation.

**Still to validate on Kaggle:**
- **Wildfire pairs:** a burn must give `burned`, not ash. RSCC includes xBD wildfire events, and §21 now accepts them.
- **The Turkey earthquake pairs.**

---

**rev10 — clouds, alignment and class-neutral water; captions that use the measurements**

Run order: §1 install → **restart the kernel** → §2–§13 → §21 (the 57-pair comparison: validates every change on all three classes) and/or §22–§23 (evidence traces).

**Why rev10:** a review of the evidence traces for 11 Tonga pairs (000024, 000076, 000093, 000149, 000222, 000238, 000394, 000509, 000596, 000891, 000918) found:

- Clouds and haze in either image were read as ground change by every module. A BEFORE cloud over forest became "new water", which made 000076 a flood. An AFTER cloud became "grey ash" and "cleared forest" (000024).
- A few pixels of misalignment between BEFORE and AFTER outlined every road and roof as change.
- The water evidence was biased toward flood in three ways. The semantic group "flood" had five phrases whose probabilities were summed. The override applied to the AFTER image only, and could turn ash into water. "Water widened" fired at a 1.5% increase.
- My rev9 water rule could never count brown floodwater.
- Strong local changes were invisible, because physical channels were judged by their whole-image mean. The checker never passed them, or cloud information, to the caption.

**Every change below is defined by physics, not by an event**, so it acts the same way on every class:

| | Change | Switch |
|---|---|---|
| §3, §12 | **Cloud and haze mask.** Bright, colourless, smooth, large, soft-edged (unlike white roofs) and transient regions, confirmed by RemoteCLIP. Masked pixels and clouded grid cells are excluded from the physical channels, RemoteCLIP coverage and transitions, the water override and AnyChange. Physical means are over the observable ground | `cloud_mask` |
| §3, §12 | **AFTER aligned onto BEFORE** by phase correlation on gradients, only when that measurably improves the match | `align_images` |
| §4 | **RemoteCLIP groups scored by their mean phrase probability** (renormalised), not the sum, so "flood" (5 phrases) has no head start | `clip_group_score` |
| §3, §4 | **Water override:** skips clouded cells, never converts ash, and new water must not be rougher than before | `water_override_from`, `water_smooth_test` |
| §5, §12 | **Water counts when confirmed:** by the pixel test in that cell, or as a connected body of ≥ 3 RemoteCLIP water cells, so brown floods count again. "Water widened" needs a 5% increase of *confirmed* water, otherwise it is dropped | `water_min_body_cells`, `water_derived_min` |
| §3 | **Vegetation index bounded to [−1, 1].** It exploded on bluish and dark pixels | — |
| §8 | **Bright + colourless over vegetation counts as the neutral `surface_change`,** no longer earthquake `debris_dust`. Over built or bare ground it stays dust | — |
| §11 | **The checker uses what was measured.** It corrects cloud misreads, adds the strongest measured changes the draft misses (located physical regions, RemoteCLIP changes; at most 2, in neutral words), and rewrites "unchanged" claims that a measurement contradicts. Mention tests are location-aware | `caption_max_added_changes` |
| §22a | **Trace fixes:** true areas (they were halved), word-boundary mention tests, the head column, a cloud overlay, and located physical regions in the change inventory | — |
| §21 | Regression baseline is now the rev9 run (43/57). New log, calibration and summary file names, because the channels changed | — |

**Deliberately not changed, to avoid biasing toward volcano:** no new volcano evidence; RemoteCLIP's "cloud" label on observable ground (often ash in Tonga) still feeds no rule; DKG rule weights unchanged.

**Validate before reporting.** Run §21 on the 57 pairs (13 flood, 5 earthquake, 39 volcano). Every switch above can be turned off on its own to see its effect.

---

**rev9 — draft → check → revise captioning, calibrated classification**

Run top to bottom: §1 install → **restart the kernel** → §2 onward → §21. The only paths to set are the two at the top of §21.

**What changed from rev8** (from the rev8 run on 57 RSCC pairs: Tonga 39, Pakistan 13, Turkey 5):

| | Change | Why (rev8 run) |
|---|---|---|
| §2, §11, §12 | **System caption = DRAFT → CHECK → REVISE** (`caption_mode="revise"`). Qwen drafts from the images alone; code compares every claim with the measurements (disaster verdict, water, ash, fire, smoke, lava, buildings), negation-aware; Qwen then fixes **only** the flagged claims and keeps the rest. Every edit is logged with its measurement and re-checked (`resolved`) | the rev8 checklist was copied: copy rate 0.147 (rev7 0.15), 34% of sentences restated a checklist line, flood captions of different pairs more alike (0.39) than the references (0.17) |
| §11 | The disaster label is one of the **checks**, always applied | when draft and pipeline disagreed (37/57) the pipeline was right 27×, the draft 1× |
| §11 | Building claims: ChangeOS used where it **detects** damage; collapse it does not see is logged as unverifiable (`caption_check_buildings="add_only"`) | ChangeOS saw no damage on 5/5 Turkey tiles whose references all describe collapse |
| §5 | **Water counts only where the semantic and pixel tests agree** (`water_requires_agreement`) | 10 of the 19 misclassifications were small semantic-only water patches |
| §19, §21 | **DKG physical priors calibrated** before classifying (`fast_physical_baselines`: per-concept median, no labels, seconds per pair) | 17/19 misclassifications were "flood", the bias the load warning describes |
| §21 | Conditions: `C3_baseline` (= the draft), `C1_system` (revised), `C2_no_label` (revised without the disaster check), `C4_checklist` (rev8, comparison) | — |
| §21 H | METEOR / ROUGE per class + overall for **all pairs** and for **pairs whose verdict is right** (CSV `subset` column) | separates captioning from classification errors |
| §21 I | **Grounding tests**: corrupt the measurements (does the caption follow them?) and grey out the images (does it depend on them?) | proves "used, not copied" |
| §21 | Visual-check sheets where ChangeOS / water disagree with the reference; BLEU and CIDEr-D via pycocoevalcap; pair bootstrap CI (event bootstrap only from 5 events); consistency audit uses the negation-aware checker and is labelled "not ground truth"; detector firing table; posterior margins | review items 15, 16, 21–25 |
| §13, §19 | `DM3_TO_CLASS` guarded (loader removed in rev7); §19 caption harness marked superseded; log says `head(not pooled)` for the untrained head | review items 20, 31 |

**§22–§23 (debug):** evidence trace — per-pair HTML reports showing every change each module detects, where it goes (verdict / caption), the DKG's reasoning and every caption step; one earthquake, one volcanic eruption and one flood report shown inline. §22b runs it on the RSCC folder, §23 on any folder without caption ground truth. Changes nothing in the pipeline.

Still open: the visual check itself (look at `visual_check/*.png`), calibrating on pairs outside the test set (`CC_CAL_DIR`), more events for an event-level CI, human verification of a sample of edits.

---

**rev8 — what changed from rev7** (approved items 1–4 from the rev7 caption review):

| | Change | Why (rev7 run, 15 RSCC pairs) |
|---|---|---|
| §2, §11, §12 | **C1 is the system caption**: `label_in_prompt=True` by default; the checklist is headed by the pipeline's own posterior. C2 (no label) stays as the echo/leakage ablation | C1 passes the pipeline's prediction, not the ground truth — that is what a deployed system does |
| §11 | **Every condition is asked to open by naming the disaster**; C1's checklist uses that event type's vocabulary (`DISASTER_VOCAB`) | 67% of the RSCC references open with a disaster name; rev7 pipeline captions 0%, and the verdict line is stripped before scoring |
| §5, §11, §12 | **Evidence is a checklist** (`thing: change \| extent \| where`), and every item carries **where** it was measured (grid cells, physical maps, building centroids, strongest heat-map / AnyChange regions) | rev7 C2 captions copied 17% of their word pairs from the evidence (baseline 1%); 0/15 pipeline captions located anything, vs 5/15 references |
| §11 | **"Confirmed absent" only for confident absences** (semantic negative at a relaxed threshold *and* no physical cue fired) | rev7 forbade smoke on TONGA 000509, where reference and baseline both see thick smoke |
| §10 | RVG does not score `caption_support` for a label-primed caption | C1 states the posterior by construction; scoring it would inflate confidence |
| §21 | Conditions renamed `C3_baseline` / `C1_system` / `C2_no_label`; C3 and C1 come straight from `predict()`; new section **G** (copy rate, disaster opener, locations) | so the evaluated caption is the one the pipeline returns, and change 3 is measurable |
| §1, §21 | **Standard METEOR and ROUGE** per class (volcanic eruption, earthquake, flood) and overall — section **H** + CSV: PTB tokens, METEOR 1.5 (Java), COCO ROUGE-L, ROUGE-1/2 (rouge-score, stemmed). The custom ROUGE-L is removed | numbers comparable with published RSICC results |

Not changed here (review items still open): BLEU / CIDEr-D implementations, the event-bootstrap CI, the circular grounding audit (§21 E), detector positive controls, the ChangeOS-vs-reference building disagreement on Tonga.

---

**rev7 — what changed from the two-reasoner notebook** (all from the perception diagnostic on the 43 pairs):

| | Fix | Evidence |
|---|---|---|
| §3 | AnyChange clears its embedding cache per pair | 53 calls returned 1 identical result |
| §3 | pixel newly-water and edge maps | flood AUROC 0.90 and 0.91 |
| §4 | floodwater phrasings + water override | `flood_extent` was 0.000 on all 43 pairs |
| §5 | muddy water counts as water, not ash | sediment water 0.000 on Guatemala |
| §5 | buildings counted with xBD levels | RSCC quotes these in 35–40% of references |
| §6 | physical phrases state no cause | 'darker, burned…' primed wildfire |
| §10 | caption label read word-by-word, markdown-safe | 'char' matched 'characterized' |
| §11 | evidence names only what is present, in words | 'particulate cover 0%' reached captions |
| §11 | RSCC-style prompts, shared by baseline and pipeline | old prompt forced 4–6 sentences, figures, an uncertainty section |

The single-pair demos (§14–§17) and the DisasterM3 loader are removed; §19's trailing demos are commented `[rev7] demo disabled`.

# Disaster Change Captioning — Stage 2: Semantics, Learned Classification, Calibrated Fusion

```
                            ┌─────────────── SCENE BRANCH ───────────────┐
                            │  AnyChange (zero-shot, class-agnostic)     │
pre ──┐                     │  Physical channels (VARI / albedo /        │
      ├──── full res ──────►│      saturation / texture, RGB-native)     │
post ─┘                     │  RemoteCLIP MULTI-SCALE patch grids 8,16   │
                            │      → SemanticTransition objects          │
                            │      → SceneEvidence (20 aggregates)       │
                            └────────────────────┬───────────────────────┘
                                                 │
                            ┌─────────────── OBJECT BRANCH ──────────────┐
                            │  ChangeOS (xBD, 4-level damage grades)     │
                            │  SAM2 instance refinement                  │
                            │  RemoteCLIP PRE-crop vs POST-crop          │
                            │      → object transitions + xBD grade      │
                            │      → ObjectEvidence (14 aggregates)      │
                            └────────────────────┬───────────────────────┘
                                                 │
                                MULTI-SCALE FUSION  →  SDA v2
                                                 │
                     EVIDENCE TOKENISER  →  CROSS-BRANCH ATTENTION  (new)
                        scene / object / transition / physical tokens
                        training-free cosine attention over RemoteCLIP embeddings
                                                 │
                 ┌───────────────────────────────┼───────────────────────────────┐
                 │                               │                               │
          DKG v3 (symbolic)           LEARNED HEAD (new)              ZERO-SHOT PRIOR
       4 evidence streams, 7          frozen features +               RemoteCLIP text
       hypotheses, supporting         text-init residual MLP          prototypes on the
       + conflicting evidence         over fused embedding            POST image
                 │                               │                               │
                 └────────► LOG-LINEAR OPINION POOL (calibrated) ◄────────────────┘
                                                 │
                                     DisasterPosterior (probs, top-k,
                                      margin, entropy, per-source)
                                                 │
                                RVG v3 — 9 weighted checks + BLIND-CAPTION test
                                                 │
                             Qwen2.5-VL-7B with a STRUCTURED EVIDENCE PACKAGE
                                                 │
                          Draft → measurement check → revised caption + edit log
```

**The branches are still computed independently and only meet at `MultiScaleFusion` and the
cross-branch attention.** Nothing in the scene branch reads the object branch, and vice versa.

## What Stage 2 adds

| # | Part | What it is | Where |
|---|---|---|---|
| 1 | Semantic understanding | `SemanticTransition` with PRE label, POST label, both confidences, patch box, embedding similarity and a marginalised transition probability; multi-scale grids; derived morphological transitions (`water_expanded` / `water_receded`) | §4 |
| 2 | DKG v3 | Four evidence streams (physical / transition / **scene** / **object**), 7 hypotheses, per-hypothesis supporting **and conflicting** evidence with source provenance | §8 |
| 3 | Learned classifier | Frozen foundation features → text-initialised residual MLP head → probabilities, embedding, top-k; pooled with the DKG by **log-linear opinion pooling** (two sources: DKG + the prior while untrained, DKG + the head once trained) | §9 |
| 4 | RVG v3 | 9 weighted consistency checks across scene / object / transitions / classifier / DKG / caption, plus a **blind-caption test** (Qwen pass 1 sees only the images) → weakest source and a stated reason for uncertainty | §10 |
| 5 | Qwen prompting | A structured `EvidencePackage`, not a bare label; the caption must cover scene damage, object damage, disaster type, supporting evidence and uncertainty | §11 |
| 6 | Tokenisation + fusion | `EvidenceTokeniser` — scene and object tokens carry RemoteCLIP **image** embeddings, transition and physical tokens carry **text** embeddings — and training-free `CrossBranchAttention` over them; no randomly initialised layers | §6 |
| 7 | Visualization | Transition map with labels, attention heatmaps, per-source probability bars, RVG check bars, DKG graph, evidence package, blind vs grounded captions | §15 |
| 8 | Evaluation | Confusion matrix, macro-F1, top-k accuracy, caption metrics, per-image scene/object contribution probe, and a 10-row ablation over every switch | §19 |

## Honest note on the learned head

The notebook ships **no labelled data**, so a randomly initialised classifier would be noise.
Instead the head is

```
logits = s · (z · Pᵀ)  +  MLP([t ; z])          last layer of the MLP zero-initialised
```

where `P` are frozen RemoteCLIP text prototypes for the seven disaster descriptions. At
construction the head is *exactly* a calibrated zero-shot classifier; training learns a residual
on top of it. So the pipeline runs correctly with zero labels, and `train_head()` (§18) improves it
the moment you supply a manifest. `posterior.weights` switches automatically on `is_trained`.

## Run order

§1 (deps) → **restart the kernel** → run §2–§13 → set the two paths at the top of §21
(`CC_IMG_DIR`, `CC_RSCC_JSONL`) → run everything else. No single image path is needed;
`pipeline.predict(pre_path, post_path)` still captions one pair by hand.


## Tier 2, worked through (this notebook)

Built on the rev5 you uploaded, with the earthquake pair kept as the focus. Tier 1 (#2
leakage, #3 correlated sources) was already closed in rev5. Every edit below is marked
`ISSUE n` in a comment at the line it affects.

Your earthquake run changed the picture, and some of it contradicts what the one-image
flood audit concluded. That is recorded honestly below rather than quietly fixed.

| Tier-2 # | Issue | What this notebook does | Settled? |
|---|---|---|---|
| 4 | Object branch gated behind building damage | Scene branch runs first; its change map seeds SAM2 alongside ChangeOS severity (`sam2_seed="object+scene"`) | **Code closed.** Was never universal — your earthquake pair already produced 11 instances. It is fatal only on disasters that damage no buildings. |
| 5 | Per-image percentile normalisation | `phys_scale="absolute"` divides each channel by a fixed physical range; `measure_change_floor` builds (pre, pre) controls | **Needs the control run.** The evidence it is real: scene_coverage was 0.655 / 0.581 / 0.643 on flood / volcano / earthquake — three different disasters, the same number. |
| 6 | Four of eight contributions change nothing | SAM2 prompt sampling made deterministic; `run_ablation_multi` reports per-switch effects over a pair set with event-level bootstrap CIs | **Not a code fix.** It is an experiment-design fix, and it needs the sweep to run. |
| 7 | Coverage fields are prompt shares, not areas | `group_coverage="mask"` counts cells clearing `group_area_thr` — a real area fraction, exactly zero when nothing matches | **Closed.** |
| 8 | DKG has no zero floor | `concept_floor` gates `required` and `veto`; `dkg_accumulate="noisy_or"` stops one source double counting; `calibrate_physical_baselines` measures the priors | **Floor closed, bias open.** The inundation bias needs the calibrator to run. |
| 9 | Class list mismatch | `assert_taxonomy_consistent()` checks DKG rules, CLIP prototypes, caption keywords, caption aliases and `DM3_TO_CLASS` against one list; `Config.eval_classes` pins the scored taxonomy and metrics now report which classes have support | **Closed, with a caveat.** The premise has moved: DisasterM3 Bench carries all seven classes, not four. The live problem is that landslide has one event, so it has no training fold under leave-event-out. |

### Two findings from your earthquake run

**The ablation sweep was not reproducible.** Instance counts alternated between 11 and 12
across rows that should have been identical, because `_spread_points` drew from the
module-level RNG whose state advances with every call. A 0.01 posterior delta was
therefore not attributable to the switch being tested. Fixed: the generator is seeded
from the config and the mask, so it does not depend on call order.

**Concept mass was not on the scale the code claimed.** `structural_damage` reached
1.064 on the earthquake pair, because `damaged_buildings_frac`, `destroyed_buildings_frac`
and `damaged_area_frac` are three views of the same buildings and were summed. The
docstring's "mass on the same [0,1] scale as the coverage fractions" was false. Noisy-OR
accumulation brings that object route from 0.849 to 0.632 and bounds each source at 1.

**Also fixed:** overriding the focus paths without the label left the notebook printing
`ground truth: volcanic_eruption | predicted: earthquake WRONG` on a pair it had got
right. `focus_truth` is now set beside the paths and validated.

### Before quoting any number

```python
floor = measure_change_floor(pipeline, dm3_pairs, n=20)          # issue 5
base  = calibrate_physical_baselines(pipeline, dm3_pairs, n=30)  # issue 8
refs  = calibrate_rvg_denominators(pipeline, dm3_pairs, n=30)
abl   = run_ablation_multi(pipeline, dm3_pairs)                  # issue 6
cv    = cross_validate_leave_event_out(pipeline, dm3_pairs, n_folds=5)
```


## What changed in rev5 (issues 2 and 3)

rev4 moved both fixes in the right direction but each left a working bug. rev5 closes them and
touches nothing else — the DKG prior bias, the dead object branch and the per-image normalisation
are still open and are *not* fixed here.

**Issue 2 — label leakage into the Qwen prompt.**
`label_in_prompt=False` correctly removed the classifier top-3, the DKG hypothesis and the fused
posterior. It did not remove the hypotheses carried by the *measurement names*: rev4's C2 prompt
still read `volcanic ash coverage 1%`, `burn scars 0%, flood extent 0%`, `landslide scars`,
`ashfall mass 0.03`, `inundation mass 0.11`, and transitions like `vegetation -> ash`. Four of the
seven classes were named in all but the word, and the rev4 caption reasoned directly off them
("No burn scars or flood extent are observed, ruling out wildfires or floods"). rev5 adds
`neutral_measurement_names` (default True), which rewrites every such field, concept and
transition group as the physical measurement it is.

The grounded prompt also gained a mandatory `Most likely disaster: <type>` closing line without a
token budget to reach it — the rev4 grounded caption was cut off mid-sentence, so the C1-vs-C2
experiment's headline metric would have been read from the keyword fallback on most pairs.
`max_new_tokens` 320 -> 512, `blind_max_new_tokens` 160 -> 224.

**Issue 3 — classifier and CLIP prior pooled as independent sources.**
The pool is now correctly two sources and `source_dependence` reports the duplication
(rank correlation 0.964 on the rev4 pair). Two consequences were not followed through:

1. `caption_support` still scored by counting disaster keywords in the caption body. Under C1 the
   caption always echoed the label so this never showed; under C2 the caption can disagree, and on
   the rev4 run a caption that concluded *cyclone* scored 1.00 against a *wildfire* posterior
   because the words "wildfires" and "burn" appeared in a sentence ruling them out. It raised final
   confidence from 0.483 to 0.566. rev5 scores the class the caption *states*.
2. `classifier_graph_agreement` (weight 0.14, the joint-heaviest check) still audited the learned
   head, which is no longer pooled while untrained. It scored 0.081 and was the main drag on
   confidence. rev5 drops the check when the head is not a pooled source and renormalises.

Also added: `use_zeroshot` so the CLIP prior can be ablated (there was no switch for it, which made
`-learned classifier` a guaranteed no-op row), a `-CLIP prior` ablation row, and an abstention is
recorded as `undetermined` in cross-validation instead of `argmax` of a uniform vector silently
scoring as class 0.


## 1 · Dependencies

Run this cell, then **restart the kernel** once. Kaggle: enable **GPU T4 x2** and **Internet**.
Stage 2 adds `pandas` and `scikit-learn` (ablation tables, confusion matrices, the training split)
and uses `huggingface_hub` to list and fetch DisasterM3 one file at a time.
Everything downloads from public sources — there is no manual checkpoint step anywhere in this
notebook.


In [ ]:
import subprocess, sys
from pathlib import Path

# ISSUE 19 -- reproducible, checked installs. Three rules, each learned the hard way:
#
#  1. A failed install must stop the notebook HERE, not three cells later as an unrelated
#     ImportError. `sh()` raises.
#  2. A pin is only worth having if it is a version that EXISTS and was RUN. rev6's first
#     attempt pinned `changeos==0.0.3`, which was never published (PyPI has 0.1 and 0.2),
#     and `huggingface_hub==0.26.2`, which `transformers==4.51.3` forbids (it needs >=0.30).
#     So every pin below was looked up, and anything that was not RUN is opt-in.
#  3. Pins prove nothing about what you actually ran. The last cell therefore writes the
#     RESOLVED versions to requirements_resolved.txt -- that file, not this cell, is what
#     goes in the paper's reproducibility appendix.
FAILED_INSTALLS: list = []
FELL_BACK: list = []
PY = sys.executable


def sh(cmd, required: bool = True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (r.stdout or "")[-1200:]
    if tail.strip():
        print(tail)
    if r.returncode != 0:
        print("STDERR:", (r.stderr or "")[-2000:])
        if required:
            FAILED_INSTALLS.append(cmd)
            raise RuntimeError(f"install failed (exit {r.returncode}): {cmd}")
    return r.returncode


def install(spec: str, fallback: str = None):
    """pip-install `spec`. If a PIN cannot be satisfied and a `fallback` is given, retry
    the unpinned form and say so loudly -- a bad pin must not take the notebook down, but
    it must not pass silently either."""
    if fallback is None:
        sh(f"{PY} -m pip install -q {spec}")
        return
    if sh(f"{PY} -m pip install -q {spec}", required=False) != 0:
        print(f"\n!! PIN FAILED, retrying unpinned: {fallback}\n")
        FELL_BACK.append(spec)
        sh(f"{PY} -m pip install -q {fallback}")


# Set True to install the exact ML-stack versions below. They EXIST on PyPI and are
# mutually consistent, but they have NOT been run with this notebook -- your last working
# run installed them unpinned and did not record what resolved. Leave False for the first
# run, read requirements_resolved.txt, and pin to THAT.
PIN_ML_STACK = False

# --- change detection (these two pins are VERIFIED) --------------------------------
# changeos 0.2: the only release whose weights URL resolves. 0.1 fetches
# .../releases/download/v1.0/changeos_r101.pt, which is a 404; 0.2 fetches v0.2, which is
# the file your working run downloaded.
install("changeos==0.2")                                      # ChangeOS (xBD damage)
install("torchange==0.0.4", fallback="torchange")             # AnyChange; the wrapper in
                                                              # section 3 documents THIS contract

# SAM / SAM2 at a commit, not a branch. Both are the current HEAD of their repos (SAM
# 2024-09-18, SAM2 2024-12-15) and neither has moved since, so this equals what an
# unpinned install gets today while staying fixed if `main` later changes.
install("'git+https://github.com/facebookresearch/segment-anything.git@dca509f'")
install("'git+https://github.com/facebookresearch/sam2.git@2b90b9f'")

# --- semantics / captioning (the ML stack) ------------------------------------------
if PIN_ML_STACK:
    # huggingface_hub is deliberately NOT pinned: transformers 4.51.3 requires >=0.30,<1.0
    install("open_clip_torch==2.24.0 huggingface_hub",
            fallback="open_clip_torch huggingface_hub")
    install("-U 'transformers==4.51.3' accelerate==1.6.0 qwen-vl-utils==0.0.11 "
            "bitsandbytes==0.45.3",
            fallback="-U 'transformers>=4.51.0' accelerate qwen-vl-utils bitsandbytes")
else:
    install("open_clip_torch huggingface_hub")
    install("-U 'transformers>=4.51.0' accelerate qwen-vl-utils bitsandbytes")

# ISSUE 13: the SBERT caption-similarity fix imports sentence_transformers. If it is not
# installed the code falls back to the CLIP text tower -- the exact thing the fix removes --
# with only a log warning. Install it so the fix actually takes effect.
install("sentence-transformers")

# --- metrics: pinned, because they change the NUMBERS (tokenisation, scoring) -------
install("sacrebleu==2.4.3 rouge-score==0.1.2 bert-score==0.3.13",
        fallback="sacrebleu rouge-score bert-score")
# rev8: METEOR 1.5 and the PTB tokenizer as the COCO caption evaluation runs them. Both are
# Java programs shipped inside pycocoevalcap; nltk is the labelled METEOR fallback.
install("pycocoevalcap==1.2 nltk", fallback="pycocoevalcap nltk")
import shutil as _sh
if _sh.which("java") is None:
    sh("apt-get update -qq && apt-get install -y -qq default-jre-headless", required=False)
print("java:", _sh.which("java") or "NOT FOUND -- §21 will use NLTK METEOR and label it")

# --- general stack: LEFT AS KAGGLE SHIPS IT ----------------------------------------
# networkx, scikit-image, matplotlib, pandas, scikit-learn are preinstalled and do not
# affect results. Re-pinning them downgrades working copies and risks numpy ABI breaks.
install("networkx scikit-image matplotlib pandas scikit-learn")

# --- record what ACTUALLY installed -------------------------------------------------
import importlib, importlib.metadata as md
importlib.invalidate_caches()
KEY_PACKAGES = ["changeos", "torchange", "segment-anything", "SAM-2", "open_clip_torch",
                "transformers", "huggingface_hub", "accelerate", "qwen-vl-utils",
                "bitsandbytes", "sentence-transformers", "torch", "torchvision", "numpy",
                "sacrebleu", "rouge-score", "bert-score", "pycocoevalcap", "nltk",
                "scikit-learn", "pandas",
                "networkx", "scikit-image", "matplotlib"]
RESOLVED = {}
for _p in KEY_PACKAGES:
    try:
        RESOLVED[_p] = md.version(_p)
    except Exception:
        RESOLVED[_p] = "NOT FOUND"
_out = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")) \
    / "requirements_resolved.txt"
_out.write_text("\n".join(f"{k}=={v}" for k, v in RESOLVED.items() if v != "NOT FOUND") + "\n")

print("\n" + "=" * 70)
print("RESOLVED VERSIONS  ->", _out)
print("=" * 70)
for k, v in RESOLVED.items():
    print(f"  {k:<22}{v}")
if FELL_BACK:
    print(f"\n!! {len(FELL_BACK)} pin(s) failed and fell back to unpinned: {FELL_BACK}")
missing = [k for k, v in RESOLVED.items() if v == "NOT FOUND"]
if missing:
    raise RuntimeError(f"installed but not importable by metadata: {missing}")
print("\nDependencies installed. RESTART THE KERNEL, then continue from section 2.")
print("Copy requirements_resolved.txt into your paper's reproducibility appendix.")

## 2 · Configuration

Nothing here requires editing for a first run. Every weight path is resolved automatically.

Stage 2 adds three groups of settings to `Config`:

* **Semantics** — `patch_grids=(8,16)`, `transition_min_change`, `transition_min_conf`,
  `transition_max_embed_sim` (a transition must show *embedding motion*, not just a label flip),
  `water_change_thr` for derived water transitions.
* **Reasoning** — `clf_hidden`, `clf_dropout`, `zeroshot_logit_scale`, `dkg_temperature`, and the two
  pooling weight sets `pool_weights_trained` / `pool_weights_untrained` (two sources each, never three),
  and `label_in_prompt` (False = measurements-only Qwen prompt) plus
  `neutral_measurement_names` (True = no measurement name implies a class either).
* **DisasterM3**: `dm3_repo`, `dm3_root`, `dm3_local_dir`, `dm3_eval_per_class`. The dataset is listed over
  the Hugging Face API and fetched file by file — the 30 GB image zip is never downloaded.
* **Ablation switches** — `use_changeos`, `use_anychange`, `use_transitions`, `use_sda`, `use_dkg`,
  `use_classifier`, `use_rvg`, `use_attention`, `use_blind_pass`, `use_zeroshot`.

The switches gate *the pipeline*, not model loading: every model is loaded unconditionally in §13 so
that an ablation changes exactly one thing and nothing else. `use_transitions=False` disables the
whole RemoteCLIP patch-semantics module — both the per-patch labels and the transitions built from
them — which is why it also zeroes the scene evidence fields derived from those labels.


In [ ]:
import os, sys, re, json, math, time, logging, warnings, urllib.request, itertools
from dataclasses import dataclass, asdict, field, replace
from collections import Counter, defaultdict
from typing import Optional, List, Dict, Tuple, Any, Sequence
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("capstone")
for noisy in ("httpx", "httpcore", "urllib3", "filelock", "open_clip"):
    logging.getLogger(noisy).setLevel(logging.WARNING)


class ModelLoadError(RuntimeError):
    """Raised when a required model cannot be loaded. Never swallowed, never substituted."""
    def __init__(self, model, why, fix):
        super().__init__(
            f"\n\n=== MODEL LOAD FAILED: {model} ===\n"
            f"WHY : {why}\n"
            f"FIX : {fix}\n"
            f"This pipeline uses real models only and will not silently substitute another.\n")


WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

SAM_CKPT_URLS = {
    "vit_h": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",  # 2.4 GB
    "vit_l": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth",  # 1.2 GB
    "vit_b": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",  # 0.4 GB
}

# The seven target types. This tuple is the single source of truth: DKG rules, the
# classifier head, the zero-shot prompts and every evaluation table are ordered by it.
DISASTER_CLASSES: Tuple[str, ...] = (
    "volcanic_eruption", "flood", "wildfire", "earthquake",
    "cyclone", "landslide", "tsunami")

# ISSUE 9: the taxonomy was asserted in five places that could silently disagree --
# DISASTER_CLASSES, the DKG rules, the CLIP prototypes, the caption keywords and the
# dataset label map. `assert_taxonomy_consistent()` (called at the end of §13) fails
# loudly if any of them drifts, and `Config.eval_classes` records which subset a given
# evaluation actually covers so a reported macro-F1 cannot quietly average over
# classes the test set has no support for.

# Canonical intermediate concepts. DKG accumulates mass on these; the classifier
# consumes the same vector, so the two reasoners see identical evidence.
CONCEPTS: Tuple[str, ...] = (
    "ashfall", "lava_flow", "burned", "inundation", "water_expansion", "sediment_load",
    "vegetation_loss", "terrain_change", "surface_change", "mass_movement",
    "structural_damage", "debris_scatter", "infrastructure_damage", "plume", "obscuration")


@dataclass
class Config:
    device: str = "cuda"
    dtype: torch.dtype = torch.float16
    seed: int = 42

    # ISSUE 9: None = every class in DISASTER_CLASSES. Set it to the subset the data
    # actually covers; the evaluation harness reports against this list and refuses to
    # average F1 over a class with no support.
    eval_classes: Optional[Tuple[str, ...]] = None

    # ---- pipeline mode -----------------------------------------------------
    # "dual"        : scene branch + object branch + fusion       (Stage 1/2 pipeline)
    # "object_only" : ChangeOS -> SAM2 -> POST-only labels        (reproduces the OLD pipeline)
    mode: str = "dual"

    # ---- ABLATION SWITCHES (Part 8) ---------------------------------------
    # Every switch is honoured by the pipeline itself, so an ablation is a config
    # change and not a second code path.
    use_changeos: bool = True          # object branch detector
    use_anychange: bool = True         # scene branch zero-shot detector
    use_transitions: bool = True       # RemoteCLIP PRE->POST semantic transitions
    use_sda: bool = True               # Subtle Damage Attention
    use_dkg: bool = True               # knowledge-graph reasoning
    use_classifier: bool = True        # learned disaster head
    use_rvg: bool = True               # reliability verification gate
    use_attention: bool = True         # cross-branch cosine attention
    use_zeroshot: bool = True          # CLIP prototype prior as a pooled source
    use_blind_pass: bool = True        # RVG blind-caption verification (2nd Qwen call)

    # ---- resolutions -------------------------------------------------------
    work_size: int = 512          # resolution every map/mask lives at
    native_max: int = 1024        # ChangeOS is fed at this res, NOT an upsampled 512
    # ISSUE 15: resize((n, n)) anisotropically distorted any non-square input, which
    # corrupts every area fraction the pipeline reports. "letterbox" pads to square and
    # keeps the aspect ratio; "stretch" reproduces the old behaviour.
    resize_mode: str = "letterbox"            # letterbox | stretch

    # ---- OBJECT branch: ChangeOS ------------------------------------------
    changeos_variant: str = "changeos_r101"   # r18 | r34 | r50 | r101
    changeos_size: int = 1024

    # ---- SCENE branch: AnyChange ------------------------------------------
    anychange_sam: str = "vit_b"              # vit_b (0.4GB, T4-safe) | vit_l | vit_h
    anychange_points_per_side: int = 16       # 32 = paper setting, ~4x slower
    anychange_stability: float = 0.95
    anychange_conf: float = 145.0             # change_confidence_threshold (author default)
    anychange_angle_span: float = 30.0        # FIXED reference for [0,1] scaling, in DEGREES
    anychange_min_regions: int = 1
    # ISSUE 11: verify_scene_branch() only checked that the scene map was not identical
    # to the physical composite. It passed while AnyChange dominated 0.27% of pixels and
    # moved the posterior by zero. A branch must be MATERIAL, not merely present.
    anychange_material_coverage: float = 0.02
    sam_ckpt_dir: str = f"{WORK}/checkpoints"

    # ---- SCENE branch: physical RGB channels -------------------------------
    phys_smooth: int = 5
    phys_texture_win: int = 9
    # ISSUE 5: robust_norm() stretches every channel to [0,1] per image, so a channel
    # mean measures contrast WITHIN one pair, not how much changed. scene_coverage came
    # out at 0.65 on every pair, disaster or not. "absolute" divides by a fixed physical
    # range instead, so the same ground change gives the same number on any image and a
    # no-change pair reads near zero. "percentile" reproduces the old behaviour.
    phys_scale: str = "absolute"              # absolute | percentile
    # The composite was max() over the five channels, so ONE channel firing marked a
    # pixel as changed. texture_change has the smallest range (0.10), and off-nadir
    # viewing geometry perturbs texture everywhere, so the composite covered 87-93% of
    # every image -- disaster or not. "second" takes the SECOND-highest channel, i.e. a
    # pixel counts as changed only when two independent physical signatures agree.
    phys_composite: str = "second"            # second | max
    # Ash/rubble correction. Measured over this dataset, the OLD ashfall prior
    # (bright x desaturated) scored Turkey earthquake 0.0260 against Tonga 0.0041 and
    # Guatemala 0.0055 -- it fires 5-6x harder on rubble than on real ash, because pale
    # concrete dust IS bright and colourless. Volcanic ash instead DARKENS a surface and
    # leaves it neutral. These set the new priors.
    phys_neutral_sat: float = 0.30            # saturation at which POST counts as colourless
    phys_veg_thr: float = 0.05                # VARI above this = vegetated in PRE
    phys_burn_selectivity: float = 1.0        # 0 = old burn prior, 1 = fully gated on
                                              # vegetation-selective darkening
    phys_burn_sel_ref: float = 0.05           # selectivity giving a full-strength burn
    # Guatemala-style lahar is BROWN, not grey, so the neutrality gate alone misses it
    # (Tonga 0.176 but Guatemala 0.020). Also accepting vegetation burial lifts Guatemala
    # to 0.058 and Tonga to 0.212, at the cost of raising earthquake 0.087 -> 0.104.
    # True = cover both volcanic modes; False = grey ashfall only.
    phys_ash_vegloss_path: bool = True
    phys_abs_range: Dict[str, float] = field(default_factory=lambda: {
        "vegetation_loss": 0.25, "brightening": 0.20, "darkening": 0.20,
        "desaturation": 0.25, "texture_change": 0.10})

    # ISSUE 7: a "coverage" field was the MEAN SOFTMAX PROBABILITY of a prompt over the
    # patch grid -- a soft share with a non-zero floor, reported to Qwen as a percentage
    # of ground ("volcanic ash coverage 2%" on a flood image). "mask" instead counts the
    # cells whose group probability clears `group_area_thr`, which is a real area
    # fraction and is zero when nothing matches. "soft" reproduces the old behaviour.
    group_coverage: str = "mask"              # mask | soft
    group_area_thr: float = 0.50

    # ---- SCENE branch: semantic transitions (Part 1) -----------------------
    patch_grids: Tuple[int, ...] = (8, 16)    # multi-scale patch grids for PRE/POST matching
    semantic_grid: int = 16                   # the grid coverage statistics are read from
    transition_min_conf: float = 0.18         # joint CLIP confidence x change gate
    transition_min_change: float = 0.10       # ...or where the scene barely changed
    transition_max_embed_sim: float = 0.985   # PRE/POST patches this similar are not a transition
    transition_min_prob: float = 0.04         # P(src)*P(dst) floor, group-marginalised
    water_change_thr: float = 0.015           # |d water area| that counts as river/flood morphology

    # ---- MULTI-SCALE FUSION ------------------------------------------------
    fuse_w_object: float = 0.45
    fuse_w_anychange: float = 0.35
    fuse_w_physical: float = 0.20

    # ---- SDA v2 (continuous) ----------------------------------------------
    sda_mu: float = 0.30
    sda_sigma: float = 0.16
    sda_gain: float = 0.85
    sda_coherence_k: int = 9

    # ---- SAM2 --------------------------------------------------------------
    # ISSUE 4: SAM2 used to be seeded from ChangeOS severity alone, whose only values
    # are {0,.10,.45,.75,1.0}; thresholding at 0.40 means "xBD grade >= 2". On any
    # disaster that does not damage buildings that is empty, and the whole object
    # branch -- instances, object transitions, object tokens, cross-branch attention --
    # collapsed with it (0 instances on both evaluated pairs). "object+scene" seeds from
    # max(object severity, scene change), so the branch segments whatever changed.
    sam2_seed: str = "object+scene"           # object+scene | object
    sam2_scene_seed_thr: float = 0.45         # scene-change floor for a SAM2 prompt
    sam2_id: str = "facebook/sam2.1-hiera-small"
    sam2_max_prompts: int = 12
    sam2_point_pool: int = 20000
    change_threshold: float = 0.40
    min_instance_area: int = 48

    # ---- RemoteCLIP --------------------------------------------------------
    remoteclip_repo: str = "chendelong/RemoteCLIP"
    remoteclip_arch: str = "ViT-L-14"
    remoteclip_max_missing: int = 8
    clip_batch: int = 32

    # ---- Evidence tokenisation + cross-branch attention (Part 6) -----------
    max_scene_tokens: int = 24
    max_object_tokens: int = 16
    max_transition_tokens: int = 16
    attention_temperature: float = 0.05       # softmax temperature over COSINE similarity
    attention_residual: float = 0.5           # weight of the cross-branch context residual
    emb_w_global: float = 0.40                # z = w_g*global + w_s*scene + w_o*object
    emb_w_scene: float = 0.40
    emb_w_object: float = 0.20

    # ---- Learned disaster classifier (Part 3) ------------------------------
    # ISSUE 10: the tabular vector `t` carried the DKG concept-mass block, so a TRAINED
    # head reads the graph it is supposed to be an independent opinion against. False
    # drops those 15 features (TAB_DIM 58 -> 43) and keeps the two pooled sources
    # genuinely separate. True reproduces the old layout.
    clf_use_concepts: bool = False
    clf_hidden: int = 128
    clf_dropout: float = 0.10
    clf_epochs: int = 300
    clf_lr: float = 3e-4
    clf_weight_decay: float = 1e-4
    clf_batch: int = 16
    clf_val_frac: float = 0.2
    clf_ckpt: str = f"{WORK}/outputs/disaster_head.pt"
    zeroshot_logit_scale: float = 50.0
    zeroshot_mode: str = "both"           # "post" | "diff" | "both"  (see zero_shot)
    zeroshot_diff_weight: float = 0.5     # weight of the bi-temporal term in "both"
    zeroshot_min_diff_norm: float = 0.05  # below this PRE~POST, so fall back to "post"

    # ---- Calibrated evidence fusion (Part 3) -------------------------------
    dkg_temperature: float = 0.15             # softmax temperature over DKG rule scores
    # TWO sources are pooled, never three. The untrained head is initialised from the same
    # RemoteCLIP prototypes the zero-shot prior reads (its last layer is zero-initialised,
    # so its logits ARE the prototype logits), and the trained head's input z contains
    # 40% of the same whole-image POST embedding. Pooling both would count one CLIP
    # computation twice. A source with no weight here is computed and logged (see
    # `source_dependence`) but never enters the product.
    #   untrained head -> symbolic DKG + CLIP prior   (head is a duplicate of the prior)
    #   trained head   -> symbolic DKG + trained head (head subsumes the prior)
    pool_weights_trained: Dict[str, float] = field(
        default_factory=lambda: {"classifier": 0.60, "dkg": 0.40})
    pool_weights_untrained: Dict[str, float] = field(
        default_factory=lambda: {"dkg": 0.65, "zeroshot": 0.35})

    # ---- Qwen prompt condition ----------------------------------------------
    # rev8 (change 1): C1 is the SYSTEM caption.
    # True (default) [C1]: the evidence checklist is headed by the pipeline's OWN fused
    #   verdict (the posterior -- never the ground truth) and worded in that event type's
    #   vocabulary. This is what a deployed system does: the posterior is the system's
    #   output, and the caption reports it.
    # False [C2]: the same checklist with no class name or synonym anywhere -- the
    #   echo/leakage ablation. C1 - C2 measures what the label adds; the echo rate (how
    #   often the caption repeats the posterior when the posterior is wrong) measures
    #   what it costs.
    # The blind pass (images only) is the baseline, condition C3.
    # rev9: in caption_mode "revise" the label is not a heading for the writer; it is one
    # of the CHECKS -- if Qwen's own draft names a different disaster, that is corrected.
    # When the draft and the pipeline disagreed in the rev8 run (37/57 pairs) the pipeline
    # was right 27 times and the draft once, at every confidence level, so the check is
    # always applied; wrong verdicts are fixed in classification, not by hiding them.
    label_in_prompt: bool = True

    # How the system caption is written.
    #   "look"      rev12 (default): EVIDENCE (every valid measured change, uncapped) ->
    #               LOOK (Qwen describes a close-up of each region, no evidence words) ->
    #               WRITE (one caption from scratch: images + verdict + its own notes) ->
    #               VERIFY (every visible item covered, else rewrite from scratch).
    #   "revise"    rev9-rev11: DRAFT (Qwen, images only) -> CHECK -> REVISE (Qwen fixes
    #               only the flagged claims, at most caption_max_edits). Comparison C5.
    #   "checklist" rev8: Qwen writes from the evidence checklist (copies it; kept as a
    #               comparison condition, C4 in §21).
    caption_mode: str = "look"
    # rev12 look-then-write
    caption_min_area: float = 0.02          # smallest change (share of the tile) that is caption material
    caption_max_regions: int = 5            # close-ups per pair; more items share (larger) crops, none dropped
    caption_crop_margin: float = 0.08       # margin around a region's cells, as a share of the tile side
    caption_crop_min_side: int = 448        # close-ups are upscaled to at least this many pixels
    look_max_new_tokens: int = 96           # one or two sentences per close-up
    caption_guided_look: bool = True        # second look, with a plain cue, for items the first missed
    caption_require_visual_confirmation: bool = True   # an item Qwen cannot see is logged, not captioned
    caption_max_rewrites: int = 2           # from-scratch rewrites when verification finds a problem
    caption_max_sentences: int = 6          # sentences grow with the number of regions, up to this
    caption_cover_overlap: float = 0.25     # share of a note's content words a sentence must carry
    caption_cover_sim: float = 0.60         # or this sentence similarity to the note (sbert)
    caption_max_edits: int = 4          # corrections sent per caption, in priority order
    # Building claims. ChangeOS found no damage on all 5 Turkey tiles and most Tonga tiles
    # whose RSCC references describe collapse, so its "no damage" is not yet trusted:
    #   "add_only" -- use ChangeOS only where it DETECTS damage (add / correct "intact");
    #                 a collapse claim it does not see is logged as unverifiable, kept;
    #   "full"     -- also rewrite collapse claims ChangeOS does not see;
    #   "off"      -- no building checks.
    caption_check_buildings: str = "add_only"
    # rev9: standing water counts only where the semantic grid AND the pixel water test
    # both see it (flood_extent = min of the two). With max(), a 5-12% semantic-only patch
    # made 9 Tonga tiles and 1 Turkey tile "flood" in the rev8 run.
    water_requires_agreement: bool = True

    # ---- rev10: image quality -- what is NOT ground ------------------------
    # Clouds and haze in either image were read as ground change by every module: a
    # cloud in PRE over forest became "new water" (TONGA 000076 -> flood), a cloud in
    # POST became "grey ash" and "cleared forest" (000024). A cloud is detected from
    # physics only -- bright, colourless, smooth, soft-edged, large, and not the same
    # bright surface in the other image -- then confirmed by RemoteCLIP on the region's
    # crop. Its pixels are then excluded from every measurement. Edge softness is what
    # separates clouds from large white roofs: on 11 report pairs every cloud/haze region
    # scored <= 0.200 and every roof >= 0.238 (Sobel mean on the region's rim, at 256 px).
    cloud_mask: bool = True
    cloud_res: int = 256                 # analysis resolution of the cloud test
    cloud_lum: float = 0.62              # smoothed luminance at least this bright
    cloud_sat: float = 0.20              # ... and at most this saturated
    cloud_std: float = 0.045             # ... and at most this textured
    cloud_min_frac: float = 0.01         # region at least 1% of the tile
    cloud_edge_max: float = 0.22         # soft rim (roofs have sharp rims)
    cloud_persist_max: float = 0.40      # not the same bright surface in the other image
    cloud_confirm_clip: bool = True      # RemoteCLIP must call the region cloud/plume
    cloud_cell_frac: float = 0.30        # a grid cell this clouded (either image) is unobservable
    # rev10: the two acquisitions are rarely pixel-aligned; a few pixels of shift made
    # every road and roof edge a "change" (vegetation loss, new water, heat-map outlines).
    # AFTER is shifted onto BEFORE by phase correlation, only if that measurably
    # improves the match and the shift is small.
    align_images: bool = True
    align_max_shift: float = 16.0        # px at work size
    # rev10: water. RemoteCLIP groups are scored by their MEAN phrase probability
    # (renormalised over groups) instead of the SUM, so a group with five phrases
    # ("flood") no longer gets five times the mass of a one-phrase group.
    clip_group_score: str = "mean"       # mean | sum (pre-rev10)
    water_override_from: Tuple[str, ...] = ("vegetation", "soil")   # "ash" removed
    water_smooth_test: bool = True       # new water must not be rougher than before
    water_min_body_cells: int = 3        # a RemoteCLIP water body: >= 3 connected cells
    water_derived_min: float = 0.05      # "water widened" needs >= 5% of the tile (was 1.5%)
    # rev10: localised physical changes (a channel's mean can stay low while one part of
    # the scene changes strongly -- the dark patches in TONGA 000024)
    phys_region_thr: float = 0.5         # pixel counts as strongly changed
    phys_region_min_frac: float = 0.02   # region at least 2% of the observable tile
    caption_max_added_changes: int = 2   # measured changes the checker may add to a caption

    # ---- rev11: what KIND of darkening -------------------------------------
    # rev10 left dark deposits with no route to any disaster rule: the DKG abstained and
    # the zero-shot prior (right on 9/39 volcano pairs) called them wildfire, cyclone or
    # earthquake. Darkening is now split by what it does to DIFFERENT SURFACES, which is
    # surface physics, not an event:
    #   deposit -- darker AND changed colour on open ground (roads, bare/paved ground) as
    #              well as vegetation: a layer has settled on everything -> ashfall
    #   burn    -- darker AND changed colour on vegetation only, open ground untouched -> burned
    #   shadow  -- darker but SAME colour (only illumination changed) -> ignored
    # Building footprints (ChangeOS) are left out of "open ground" so roofs exposed by a
    # collapse are not read as a deposit; confirmed water is left out so wet ground is not.
    deposit_dark: float = 0.5            # darkening channel (luminance fell by >= 0.10)
    deposit_chroma: float = 0.025        # chromaticity shift: the material changed ...
    deposit_sat_drop: float = 0.08       # ... or the surface lost colour
    deposit_min_hit: float = 0.10        # share of a zone's surface that must be covered
    deposit_ground_min: float = 0.10     # zone needs this much open ground to judge selectivity
    deposit_to_ashfall: float = 0.8      # dark deposit -> ashfall mass per unit of covered area
    # rev11: a RemoteCLIP water body counts only if it became SMOOTHER (water smooths a
    # surface) or is very flat; ash-darkened fields keep their rows, trees and buildings and
    # were being counted as water on 10/39 Tonga pairs.
    water_body_smooth: float = 0.10      # mean "smoothing" channel over the body's cells
    water_body_flat: float = 0.015       # ... or POST local texture this low

    # rev8 (change 3): where each checklist item lies, named on a 3x3 split of the tile.
    # True = compass words (north-west ... south-east), as RSCC references use; correct for
    # orthorectified north-up tiles (xBD / RSCC). False = image words (top-left ...).
    caption_compass: bool = True
    # rev8 (change 4): "Confirmed absent" is issued only when the RemoteCLIP grid is
    # negative even at this fraction of group_area_thr AND no physical channel the surface
    # would move has fired. A weaker absence is left out of the prompt entirely.
    caption_absent_relax: float = 0.5

    # C2 purity. The legacy measurement names ("volcanic ash coverage", "burn scars",
    # "flood extent") and the DKG concept names ("ashfall", "inundation", "burned")
    # each name a disaster class in all but the word, and Qwen reasons from them
    # directly. True rewrites every one of them as the physical measurement it is, so
    # C2 supplies no hypothesis at all. False reproduces the rev4 wording.
    neutral_measurement_names: bool = True

    # ---- Qwen2.5-VL --------------------------------------------------------
    qwen_id: str = "Qwen/Qwen2.5-VL-7B-Instruct"
    load_in_4bit: bool = True
    # Both prompts end with a mandatory 'Most likely disaster: <type>' line, which the
    # caption-condition experiment parses. 320 truncated the grounded caption BEFORE
    # that line on a normal 4-6 sentence answer, so the stated class was unrecoverable.
    max_new_tokens: int = 512
    blind_max_new_tokens: int = 224

    # ---- DKG / RVG ---------------------------------------------------------
    # ISSUE 8: physical_concepts() returns strictly positive mass for ashfall, burned
    # and inundation on ANY pair, so no rule's `required` ever failed and no `veto` ever
    # cleared. Worse, inundation = mean(dark x (1-texture)) is structurally 3-15x the
    # others because (1-texture) is near 1 everywhere, so "flood" won the required term
    # by default -- it was the DKG verdict on both evaluated pairs, including a volcano
    # pair where 99% of the mass was this prior. A concept must now clear a floor before
    # it counts as present, and the physical priors are divided by a per-concept
    # baseline so they are comparable to each other instead of to nothing.
    concept_floor: float = 0.02               # mass below this does not satisfy `required`
    # "noisy_or" keeps one source's mass for one concept in [0,1]; "sum" reproduces the
    # old accumulation, which reached 1.064 for structural_damage on the earthquake pair
    # by counting the same buildings three times.
    dkg_accumulate: str = "noisy_or"          # noisy_or | sum
    # Left EMPTY on purpose: a baseline is a measurement, not a guess. Fill it with
    # `calibrate_physical_baselines()` (§19) over no-change or mixed pairs; until then
    # the DKG logs that its physical priors are uncalibrated and that flood is favoured
    # by construction. Each entry rescales a prior to "excess over typical" in [0,1].
    phys_concept_baseline: Dict[str, float] = field(default_factory=dict)
    damage_change_thr: float = 0.40
    scene_change_thr: float = 0.35
    concept_saturation: float = 0.25
    rvg_hedge_conf: float = 0.55
    rvg_hedge_check: float = 0.35
    # ISSUE 12: every check divided by a constant chosen by eye, and five of the nine
    # pinned at 1.00 on the evaluated pair, so the reported confidence was set by two
    # checks. The denominators are config now so `calibrate_rvg_denominators()` (§19)
    # can set them from the observed distribution over real pairs.
    rvg_scene_cov_ref: float = 0.25           # scene_object_consistency denominator
    rvg_obj_dmg_ref: float = 0.02
    rvg_iou_ref: float = 0.30                 # anychange_support denominator
    rvg_margin_ref: float = 0.25              # hypothesis_margin denominator
    rvg_transition_mass_ref: float = 0.05     # transition_support needs real mass, not
                                              # merely a non-zero trace (ISSUE 12)
    # ISSUE 13: caption-to-caption similarity used RemoteCLIP's TEXT tower, an
    # image-text retrieval model, which returned 0.96 for two captions that reached
    # different conclusions. "sbert" uses a sentence-similarity model and falls back to
    # "clip" only if it cannot be loaded.
    caption_sim_model: str = "sbert"          # sbert | clip
    sbert_id: str = "sentence-transformers/all-MiniLM-L6-v2"
    rvg_blind_sim_lo: float = 0.55            # cosine below this = no semantic agreement
    rvg_blind_sim_hi: float = 0.90            # cosine above this = full agreement

    out_dir: str = f"{WORK}/outputs"

    # ---- DisasterM3 as the data source -------------------------------------
    # Nothing bulk is downloaded. The Bench split is LISTED over the HF API and only the
    # files actually used are fetched one at a time (~2 MB per image). The 30 GB
    # DisasterM3_Instruct/train_images.zip is never touched by any cell here.
    dm3_repo: str = "Kingdrone-Junjue/DisasterM3"
    dm3_root: str = f"{WORK}/disasterm3"      # single-file downloads land here
    dm3_local_dir: Optional[str] = None       # a folder that already holds DisasterM3_Bench/
    dm3_seed: int = 7
    dm3_eval_per_class: int = 30              # pairs per class for the CV sample

    def __post_init__(self):
        Path(self.out_dir).mkdir(parents=True, exist_ok=True)
        Path(self.dm3_root).mkdir(parents=True, exist_ok=True)
        Path(self.sam_ckpt_dir).mkdir(parents=True, exist_ok=True)
        if self.mode not in {"dual", "object_only"}:
            raise ValueError(f"mode must be dual|object_only, got {self.mode}")
        if self.anychange_sam not in SAM_CKPT_URLS:
            raise ValueError(f"anychange_sam must be one of {sorted(SAM_CKPT_URLS)}")
        if not 0.0 < self.sda_gain <= 1.0:
            raise ValueError("sda_gain must be in (0, 1]; the (1-C) factor guarantees C' <= 1 only then")
        for name, allowed in (("resize_mode", {"letterbox", "stretch"}),
                              ("phys_scale", {"absolute", "percentile"}),
                              ("group_coverage", {"mask", "soft"}),
                              ("sam2_seed", {"object+scene", "object"}),
                              ("caption_sim_model", {"sbert", "clip"}),
                              ("caption_mode", {"look", "revise", "checklist"}),
                              ("dkg_accumulate", {"noisy_or", "sum"})):
            if getattr(self, name) not in allowed:
                raise ValueError(f"{name} must be one of {sorted(allowed)}, "
                                 f"got {getattr(self, name)!r}")
        if self.eval_classes is not None:
            bad = [c for c in self.eval_classes if c not in DISASTER_CLASSES]
            if bad:
                raise ValueError(f"eval_classes contains non-classes: {bad}")
        if self.semantic_grid not in self.patch_grids:
            raise ValueError(f"semantic_grid={self.semantic_grid} must be one of patch_grids={self.patch_grids}")

    @property
    def sam_ckpt_url(self) -> str:
        return SAM_CKPT_URLS[self.anychange_sam]


CFG = Config()
torch.manual_seed(CFG.seed); np.random.seed(CFG.seed)
RNG = np.random.default_rng(CFG.seed)

if not torch.cuda.is_available():
    raise ModelLoadError(
        "CUDA",
        "No GPU visible to PyTorch.",
        "Kaggle: Settings -> Accelerator -> GPU T4 x2 or P100, then rerun.")

log.info("GPU: %s (%.1f GB)", torch.cuda.get_device_name(0),
         torch.cuda.get_device_properties(0).total_memory / 1e9)
log.info("Mode: %s | scene: AnyChange=%s (SAM-%s) | classifier=%s | DKG=%s | RVG=%s",
         CFG.mode, CFG.use_anychange, CFG.anychange_sam,
         CFG.use_classifier, CFG.use_dkg, CFG.use_rvg)


## 3 · Branch detectors — auto-downloading, independent

Unchanged from Stage 1 apart from `null_object_output()`, which supplies a well-formed empty
object map when `use_changeos=False` so the ablation path does not special-case downstream code.

The **object branch** (ChangeOS, xBD damage grades) and the **scene branch** (AnyChange + five
physical RGB channels) never read each other's output.


In [ ]:
XBD_DAMAGE_CLASSES = {0: "background", 1: "no damage", 2: "minor damage",
                      3: "major damage", 4: "destroyed"}


# ---------------------------------------------------------------------------
# shared helpers
# ---------------------------------------------------------------------------
def abs_norm(x: np.ndarray, full_scale: float) -> np.ndarray:
    """ISSUE 5: divide by a FIXED physical range instead of this image's percentiles.

    The same amount of ground change then gives the same number on every pair, a
    no-change pair reads near zero, and the resulting means are comparable across
    images -- which is what makes them usable as evidence mass and as reported
    percentages. `full_scale` is the channel difference that saturates the map."""
    x = np.asarray(x, np.float32)
    return np.clip(x / max(float(full_scale), 1e-6), 0.0, 1.0)


def robust_norm(x: np.ndarray, lo_p: float = 1.0, hi_p: float = 99.0) -> np.ndarray:
    """Percentile normalisation to [0,1]. Robust to the handful of extreme pixels
    that make plain min-max normalisation collapse a map to near-zero.

    WARNING: this is a WITHIN-IMAGE contrast stretch. Any mean taken from a map it
    produced is not comparable between pairs -- see `abs_norm` and `Config.phys_scale`."""
    x = np.asarray(x, np.float32)
    lo, hi = np.percentile(x, lo_p), np.percentile(x, hi_p)
    if hi - lo < 1e-6:
        return np.zeros_like(x)
    return np.clip((x - lo) / (hi - lo), 0.0, 1.0)


def resize_map(a: np.ndarray, hw: Tuple[int, int], nearest: bool = False) -> np.ndarray:
    H, W = hw
    mode = Image.NEAREST if nearest else Image.BILINEAR
    return np.array(Image.fromarray(np.asarray(a, np.float32)).resize((W, H), mode))


def box_blur(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x
    t = torch.from_numpy(np.asarray(x, np.float32))[None, None]
    return F.avg_pool2d(t, k, stride=1, padding=k // 2)[0, 0].numpy()


# ---------------------------------------------------------------------------
# OBJECT BRANCH
# ---------------------------------------------------------------------------
@dataclass
class ObjectChangeOutput:
    damage_severity: np.ndarray            # HxW float [0,1], continuous-ised xBD grade
    damage_grades: np.ndarray              # HxW int, xBD class
    building_mask: np.ndarray              # HxW bool
    source: str = "changeos"

    @property
    def damaged_area_frac(self) -> float:
        return float((self.damage_grades >= 2).mean())


class ChangeOSDetector:
    """
    ChangeOS (RSE 2021) — object-based semantic change detection for building damage,
    trained on xBD. Weights auto-download via changeos.from_name(...).

    CHANGED in Stage 1: it is now fed the NATIVE-resolution pair. Previously the
    pipeline downsampled to 512 and then bicubic-upsampled to 1024, so ChangeOS
    was grading a blurred image.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        try:
            import changeos
        except Exception as e:
            raise ModelLoadError("ChangeOS (package)",
                                 f"`changeos` not importable: {type(e).__name__}: {e}",
                                 "pip install changeos  -- then RESTART THE KERNEL.")
        try:
            self.model = changeos.from_name(cfg.changeos_variant)
            if isinstance(self.model, nn.Module):
                self.model = self.model.to(cfg.device).eval()
        except Exception as e:
            raise ModelLoadError(
                "ChangeOS (weights)",
                f"changeos.from_name('{cfg.changeos_variant}') failed: {type(e).__name__}: {e}",
                "Enable Internet (weights auto-download from the ChangeOS GitHub release). "
                "Try changeos_r18 for a smaller variant.")
        log.info("ChangeOS ready: %s (xBD building damage)", cfg.changeos_variant)

    def detect(self, pre: np.ndarray, post: np.ndarray,
               out_hw: Tuple[int, int]) -> ObjectChangeOutput:
        s = self.cfg.changeos_size
        pre_r = np.array(Image.fromarray(pre).resize((s, s), Image.BICUBIC))
        post_r = np.array(Image.fromarray(post).resize((s, s), Image.BICUBIC))

        loc, dam = self.model(pre_r, post_r)          # official inference API
        loc, dam = np.asarray(loc), np.asarray(dam)
        if loc.ndim == 3: loc = loc.squeeze()
        if dam.ndim == 3: dam = dam.squeeze()

        grades = resize_map(dam, out_hw, nearest=True).astype(np.int32)
        bmask = resize_map(loc, out_hw, nearest=True) > 0.5

        sev = np.zeros(out_hw, np.float32)
        for g, v in {1: 0.10, 2: 0.45, 3: 0.75, 4: 1.0}.items():
            sev[grades == g] = v
        return ObjectChangeOutput(np.clip(sev, 0, 1), grades, bmask)


# ---------------------------------------------------------------------------
# SCENE BRANCH — part A: AnyChange (zero-shot, class-agnostic)
# ---------------------------------------------------------------------------
@dataclass
class ChangeRegion:
    """One AnyChange change instance, fully decoded from the returned MaskData.

    `cosine` is the raw `change_confidence` value AnyChange computes, which is
    -cos_sim(pre_embedding, post_embedding) over the mask, so it lives in [-1, 1] and
    is NOT a probability. `angle` is that value expressed as the bitemporal angular
    change in degrees, and `prob` is `angle` calibrated against a fixed reference.
    """
    mask: np.ndarray                       # HxW bool, decoded from the RLE
    cosine: float                          # raw MaskData['change_confidence'], [-1, 1]
    angle: float                           # angular change in degrees, [0, 180]
    prob: float                            # calibrated confidence, [0, 1]
    box: Tuple[int, int, int, int]         # XYXY, from MaskData['boxes']
    area: int                              # pixels, from MaskData['areas']
    iou_pred: float                        # SAM mask quality, MaskData['iou_preds']

    def __iter__(self):
        """`for mask, conf in regions` keeps working for older call sites."""
        return iter((self.mask, self.prob))


@dataclass
class AnyChangeOutput:
    change_prob: np.ndarray                # HxW float [0,1], FIXED-reference scaled
    raw_confidence: np.ndarray             # HxW float, angular change in DEGREES
    regions: List[ChangeRegion]            # one entry per surviving change instance
    n_raw_masks: int
    source: str = "anychange"

    @property
    def coverage(self) -> float:
        return float((self.change_prob > 0).mean())


class AnyChangeUnpackError(RuntimeError):
    """Raised when the torchange output schema is not one this wrapper understands."""


class AnyChangeDetector:
    """
    AnyChange / Segment Any Change (NeurIPS 2024) — training-free zero-shot change
    detection by bitemporal latent matching in SAM's latent space. Class-agnostic,
    so it sees ash, vegetation, water and terrain change that ChangeOS ignores.

    THE ACTUAL torchange 0.0.4 CONTRACT (read from the installed source, not assumed)
    ---------------------------------------------------------------------------------
    `AnyChange.forward(img1, img2)` returns a 3-tuple:

        (changemasks, t1_mask_data, t2_mask_data)

    all three of type `segment_anything.utils.amg.MaskData`. MaskData is a dict-like
    holder of *parallel arrays*, not a sequence: it implements `__getitem__(str)` and
    `items()`, but no `__iter__` and no `keys()`. Iterating it therefore falls back to
    the old sequence protocol, calls `__getitem__(0)`, and raises `KeyError: 0` — which
    is exactly the error this wrapper used to produce.

    Keys actually present on `changemasks` (SimpleMaskGenerator + bitemporal_match):

        rles              list[dict]      RLE masks; decode with amg.rle_to_mask
        boxes             Tensor  [N,4]   XYXY
        iou_preds         Tensor  [N]     SAM mask quality
        points            Tensor  [N,2]   prompt point
        crop_boxes        Tensor  [N,4]
        areas             Tensor  [N]     pixels, from area_from_rle
        change_confidence Tensor  [N]     -cos_sim(pre, post) over the mask, in [-1, 1]

    There is NO `segmentations` key: SimpleMaskGenerator only sets it in `generate()`,
    which the AnyChange forward path never calls. If `min_mask_region_area > 0` the
    generator calls `MaskData.to_numpy()`, so every array can be numpy instead of a
    tensor; both are handled below.

    CONFIDENCE SEMANTICS — the second bug, which a fixed `_unpack` alone would hide.
    `change_confidence = -cos_sim`, and AnyChange keeps a mask when
    `change_confidence > cos(change_confidence_threshold)`. With the author default of
    145 that is `> cos(145 deg) = -0.819`, i.e. a true angular change above 35 degrees.
    Real kept values are therefore mostly NEGATIVE. The previous scaling
    (`max(conf, 0) / 290`) mapped almost every genuine change region to exactly 0.0,
    so the scene branch would have contributed nothing to fusion even once unpacking
    worked. Confidence is now converted to an angle and calibrated against a fixed
    angular span, which keeps the map absolute and comparable across images.
    """

    MASK_KEYS = ("rles", "segmentations", "masks")
    SCORE_KEYS = ("change_confidence", "scores")

    def __init__(self, cfg: Config):
        self.cfg = cfg
        ckpt = Path(cfg.sam_ckpt_dir) / Path(cfg.sam_ckpt_url).name
        if not ckpt.is_file():
            log.info("Downloading SAM-%s checkpoint -> %s", cfg.anychange_sam, ckpt)
            try:
                urllib.request.urlretrieve(cfg.sam_ckpt_url, ckpt)
            except Exception as e:
                raise ModelLoadError("AnyChange (SAM checkpoint)",
                                     f"Download failed: {type(e).__name__}: {e}",
                                     f"Enable Internet, or fetch {cfg.sam_ckpt_url} into "
                                     f"{cfg.sam_ckpt_dir}.")
        try:
            from torchange.models.segment_any_change import AnyChange
        except Exception as e:
            raise ModelLoadError("AnyChange (package)",
                                 f"torchange not importable: {type(e).__name__}: {e}",
                                 "pip install torchange 'git+https://github.com/"
                                 "facebookresearch/segment-anything.git' -- then RESTART.")
        try:
            self.model = AnyChange(cfg.anychange_sam, sam_checkpoint=str(ckpt))
            self.model.make_mask_generator(
                points_per_side=cfg.anychange_points_per_side,
                stability_score_thresh=cfg.anychange_stability)
            self.model.set_hyperparameters(
                change_confidence_threshold=cfg.anychange_conf,
                use_normalized_feature=True, bitemporal_match=True)
        except Exception as e:
            raise ModelLoadError("AnyChange (init)",
                                 f"AnyChange construction failed: {type(e).__name__}: {e}",
                                 "The torchange API is beta; check the current signature in "
                                 "torchange/models/segment_any_change.")
        log.info("AnyChange ready: SAM-%s, %d pts/side, conf_thr=%.0f deg "
                 "(keeps angular change > %.0f deg)",
                 cfg.anychange_sam, cfg.anychange_points_per_side, cfg.anychange_conf,
                 180.0 - cfg.anychange_conf)

    # -- confidence calibration ---------------------------------------------
    @staticmethod
    def _cosine_to_angle(c: float) -> float:
        """change_confidence = -cos_sim  ->  angular change in degrees."""
        return float(np.degrees(np.arccos(np.clip(-float(c), -1.0, 1.0))))

    def _angle_to_prob(self, angle: float) -> float:
        floor = 180.0 - float(self.cfg.anychange_conf)      # the library's keep floor
        span = max(float(self.cfg.anychange_angle_span), 1e-6)
        return float(np.clip((angle - floor) / span, 0.0, 1.0))

    # -- MaskData decoding ---------------------------------------------------
    @staticmethod
    def _to_numpy(v):
        return v.detach().cpu().numpy() if torch.is_tensor(v) else v

    @classmethod
    def _as_stats(cls, changemasks) -> Dict[str, Any]:
        """Normalise any torchange return shape into a plain {key: array} mapping.

        MaskData exposes `items()`, so `dict(md.items())` is the supported way to read
        it without touching the private `_stats`.
        """
        if changemasks is None:
            return {}
        if hasattr(changemasks, "items") and not isinstance(changemasks, (list, tuple)):
            return dict(changemasks.items())         # MaskData, or the __call__ dict
        if isinstance(changemasks, (list, tuple)):
            if len(changemasks) == 0:
                return {}
            if all(isinstance(m, dict) for m in changemasks):
                out: Dict[str, Any] = {
                    "segmentations": [m.get("segmentation") for m in changemasks]}
                if all("change_confidence" in m for m in changemasks):
                    out["change_confidence"] = [m["change_confidence"] for m in changemasks]
                elif all("predicted_iou" in m for m in changemasks):
                    out["iou_preds"] = [m["predicted_iou"] for m in changemasks]
                return out
            return {"segmentations": list(changemasks)}
        raise AnyChangeUnpackError(
            f"AnyChange returned {type(changemasks)}, which is neither MaskData, dict "
            f"nor list. Print `dict(changemasks.items())` and extend _as_stats.")

    def _decode_masks(self, stats: Dict[str, Any]) -> List[np.ndarray]:
        from torchange.models.segment_any_change.segment_anything.utils.amg import rle_to_mask
        key = next((k for k in self.MASK_KEYS if k in stats), None)
        if key is None:
            raise AnyChangeUnpackError(
                f"No mask field in AnyChange output. Keys present: {sorted(stats)}. "
                f"Expected one of {self.MASK_KEYS}.")
        masks = []
        for m in stats[key]:
            if isinstance(m, dict) and "counts" in m and "size" in m:
                masks.append(rle_to_mask(m).astype(bool))     # RLE -> HxW bool
            else:
                masks.append(self._to_numpy(m).astype(bool))
        return masks

    def _unpack(self, changemasks) -> List[ChangeRegion]:
        """MaskData -> [ChangeRegion]. Masks, scores, boxes, areas and IoU all come
        out of the parallel arrays; nothing is iterated over the container itself."""
        stats = self._as_stats(changemasks)
        if not stats:
            return []
        masks = self._decode_masks(stats)
        n = len(masks)
        if n == 0:
            return []

        score_key = next((k for k in self.SCORE_KEYS if k in stats), None)
        if score_key is None:
            raise AnyChangeUnpackError(
                f"No change-confidence field in AnyChange output. Keys present: "
                f"{sorted(stats)}. Expected one of {self.SCORE_KEYS}.")
        cos = np.asarray(self._to_numpy(stats[score_key]), np.float64).reshape(-1)
        if cos.size != n:
            raise AnyChangeUnpackError(
                f"AnyChange returned {n} masks but {cos.size} confidences.")

        boxes = self._to_numpy(stats["boxes"]) if "boxes" in stats else None
        areas = self._to_numpy(stats["areas"]) if "areas" in stats else None
        ious = self._to_numpy(stats["iou_preds"]) if "iou_preds" in stats else None

        out: List[ChangeRegion] = []
        for i, m in enumerate(masks):
            if boxes is not None:
                x0, y0, x1, y1 = [int(v) for v in np.asarray(boxes[i]).reshape(-1)[:4]]
            else:
                ys, xs = np.where(m)
                x0, y0, x1, y1 = ((int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
                                  if xs.size else (0, 0, 0, 0))
            ang = self._cosine_to_angle(cos[i])
            out.append(ChangeRegion(
                mask=m, cosine=float(cos[i]), angle=ang, prob=self._angle_to_prob(ang),
                box=(x0, y0, x1, y1),
                area=int(areas[i]) if areas is not None else int(m.sum()),
                iou_pred=float(ious[i]) if ious is not None else float("nan")))
        return out

    # -- inference -----------------------------------------------------------
    def detect(self, pre: np.ndarray, post: np.ndarray) -> AnyChangeOutput:
        H, W = pre.shape[:2]
        # torchange computes SAM embeddings only while its cache is empty and clears it
        # only inside __call__; this wrapper calls forward(). Without clearing, every
        # pair after the first reused pair 1's embeddings: on the 43-pair run all 53
        # AnyChange calls returned the identical "4 regions | coverage 0.158".
        _clear = getattr(self.model, "clear_cached_embedding", None)
        if callable(_clear):
            _clear()
        try:
            out = self.model.forward(pre, post)
        except Exception as e:
            raise RuntimeError(f"AnyChange inference failed: {type(e).__name__}: {e}")
        finally:
            if callable(_clear):
                _clear()
        # forward() -> (changemasks, t1_mask_data, t2_mask_data)
        changemasks = out[0] if isinstance(out, tuple) else out

        items = self._unpack(changemasks)

        prob = np.zeros((H, W), np.float32)
        angle = np.zeros((H, W), np.float32)
        regions: List[ChangeRegion] = []
        for r in items:
            m = r.mask
            if m.shape != (H, W):
                m = resize_map(m.astype(np.float32), (H, W), nearest=True) > 0.5
                r = ChangeRegion(m, r.cosine, r.angle, r.prob, r.box, int(m.sum()), r.iou_pred)
            if not m.any():
                continue
            # overlapping instances take the maximum, never last-writer-wins
            prob[m] = np.maximum(prob[m], r.prob)
            angle[m] = np.maximum(angle[m], r.angle)
            regions.append(r)
        regions.sort(key=lambda r: -r.prob)

        if items and not regions:
            raise RuntimeError(
                f"AnyChange returned {len(items)} masks but every one was empty after "
                f"resampling to {(H, W)}. Check work_size against the mask frame.")
        if regions and float(prob.max()) == 0.0:
            lo = min(r.angle for r in regions); hi = max(r.angle for r in regions)
            raise RuntimeError(
                f"All {len(regions)} AnyChange regions calibrated to prob=0. Returned "
                f"angles span {lo:.1f}-{hi:.1f} deg but the keep floor is "
                f"{180.0 - self.cfg.anychange_conf:.1f} deg. anychange_conf and the "
                f"model hyperparameters are out of sync.")

        log.info("AnyChange: %d raw masks -> %d regions | angle %.1f-%.1f deg | "
                 "prob mean %.3f max %.3f | coverage %.3f",
                 len(items), len(regions),
                 min((r.angle for r in regions), default=0.0),
                 max((r.angle for r in regions), default=0.0),
                 float(prob[prob > 0].mean()) if (prob > 0).any() else 0.0,
                 float(prob.max()), float((prob > 0).mean()))
        return AnyChangeOutput(np.clip(prob, 0, 1), angle, regions, len(items))


# ---------------------------------------------------------------------------
# SCENE BRANCH — part B: physical RGB change channels (no weights, no training)
# ---------------------------------------------------------------------------
class ScenePhysicalChannels:
    """
    Five interpretable, RGB-native bitemporal change channels. These are the signals
    a remote-sensing analyst reads off a true-colour pair, and none of them require a
    building to be present — which is exactly the blind spot of the object branch.

      vegetation_loss : drop in VARI = (G-R)/(G+R-B), the RGB stand-in for NDVI
      brightening     : increase in luminance  -> ash fall, sediment, bare rock
      darkening       : decrease in luminance  -> burn scar, standing water, shadow
      desaturation    : loss of colour         -> dust / ash blanketing
      texture_change  : |d local std|          -> terrain deformation, rubble, debris

    They are priors, not detections: DKG weights them below the learned/CLIP evidence
    and labels their graph nodes `physical_prior` so the reasoning stays auditable.
    """
    NAMES = ["vegetation_loss", "brightening", "darkening", "desaturation", "texture_change"]

    def __init__(self, cfg: Config):
        self.cfg = cfg

    @staticmethod
    def _vari(rgb: np.ndarray) -> np.ndarray:
        r, g, b = [rgb[..., i].astype(np.float32) / 255.0 for i in range(3)]
        # rev10: VARI's denominator reaches 0 or goes negative on bluish and dark pixels
        # (water, shadow, ash), where the raw index explodes; it is bounded to [-1, 1].
        return np.clip((g - r) / (g + r - b + 1e-4), -1.0, 1.0)

    @staticmethod
    def _lum(rgb: np.ndarray) -> np.ndarray:
        r, g, b = [rgb[..., i].astype(np.float32) / 255.0 for i in range(3)]
        return 0.299 * r + 0.587 * g + 0.114 * b

    @staticmethod
    def _sat(rgb: np.ndarray) -> np.ndarray:
        x = rgb.astype(np.float32) / 255.0
        mx, mn = x.max(-1), x.min(-1)
        return (mx - mn) / (mx + 1e-4)

    def _texture(self, rgb: np.ndarray) -> np.ndarray:
        g = self._lum(rgb)
        k = self.cfg.phys_texture_win
        mean = box_blur(g, k)
        return np.sqrt(np.clip(box_blur(g * g, k) - mean * mean, 0, None))

    def __call__(self, pre: np.ndarray, post: np.ndarray,
                 invalid: Optional[np.ndarray] = None) -> Dict[str, np.ndarray]:
        ch = self._channels(pre, post)
        # rev10: pixels that are not ground in either image (cloud, haze) are excluded.
        # They are filled with the mean of the observable pixels, so every downstream
        # `.mean()` -- scene evidence, DKG priors, calibration -- is the mean over the
        # OBSERVABLE ground, and no map shows change inside a cloud.
        if invalid is not None and np.asarray(invalid).any():
            bad = np.asarray(invalid, bool)
            if bad.shape != ch["composite"].shape:
                bad = resize_map(bad.astype(np.float32), ch["composite"].shape, nearest=True) > 0.5
            ok = ~bad
            for key in list(ch):
                m = ch[key]
                if not isinstance(m, np.ndarray) or m.shape != bad.shape:
                    continue
                fill = float(m[ok].mean()) if ok.any() else 0.0
                m = m.copy(); m[bad] = fill; ch[key] = m
            ch["valid"] = ok.astype(np.float32)
        else:
            ch["valid"] = np.ones(ch["composite"].shape, np.float32)
        return ch

    def _channels(self, pre: np.ndarray, post: np.ndarray) -> Dict[str, np.ndarray]:
        k = self.cfg.phys_smooth
        v0, v1 = box_blur(self._vari(pre), k), box_blur(self._vari(post), k)
        l0, l1 = box_blur(self._lum(pre), k), box_blur(self._lum(post), k)
        s0, s1 = box_blur(self._sat(pre), k), box_blur(self._sat(post), k)
        t0, t1 = self._texture(pre), self._texture(post)

        raw = {
            "vegetation_loss": np.clip(v0 - v1, 0, None),
            "brightening":     np.clip(l1 - l0, 0, None),
            "darkening":       np.clip(l0 - l1, 0, None),
            "desaturation":    np.clip(s0 - s1, 0, None),
            "texture_change":  np.abs(t1 - t0),
        }
        if self.cfg.phys_scale == "absolute":
            ch = {n: abs_norm(m, self.cfg.phys_abs_range.get(n, 0.2))
                  for n, m in raw.items()}
        else:
            ch = {n: robust_norm(m) for n, m in raw.items()}
        # Composite scene-change energy. "max" = any ONE channel firing counts as
        # change, which saturates: off-nadir geometry alone moves texture and brightness
        # across the whole tile. "second" (default) takes the second-highest channel, so
        # a pixel counts as changed only where two independent signatures agree.
        stack = np.stack([ch[n] for n in self.NAMES])
        if str(getattr(self.cfg, "phys_composite", "second")).lower() == "max":
            comp = stack.max(0)
        else:
            comp = np.sort(stack, axis=0)[-2]
        ch["composite"] = np.clip(comp, 0, 1)
        # Helper maps for the ash/rubble/burn priors. Deliberately OUTSIDE self.NAMES:
        # `composite` must keep meaning exactly what it meant before.
        ch["post_neutrality"] = np.clip(
            1.0 - s1 / max(float(getattr(self.cfg, "phys_neutral_sat", 0.30)), 1e-6), 0, 1)
        ch["veg_mask_pre"] = (v0 > float(getattr(self.cfg, "phys_veg_thr", 0.05))).astype(np.float32)
        # Pixel facts for the caption evidence (outside NAMES; composite unchanged).
        _a = pre.astype(np.float32); _b = post.astype(np.float32)
        if _a.max() > 1.5: _a = _a / 255.0
        if _b.max() > 1.5: _b = _b / 255.0
        ch["newly_water"] = _px_newly_water(_a, _b)
        ch["edge_pre"], ch["edge_post"] = _px_edges(_a), _px_edges(_b)
        # rev11: what KIND of change a darkening is, and whether a surface smoothed
        # (outside NAMES, so `composite` keeps its meaning)
        def _chrom(x):
            c = x / (x.sum(-1, keepdims=True) + 1e-4)
            return np.stack([box_blur(c[..., i], k) for i in range(3)], -1)
        ch["chroma_shift"] = np.linalg.norm(_chrom(_b) - _chrom(_a), axis=-1).astype(np.float32)
        ch["sat_drop"] = np.clip(s0 - s1, 0, None).astype(np.float32)
        ch["smoothing"] = abs_norm(np.clip(t0 - t1, 0, None), 0.10)
        ch["texture_post"] = np.asarray(t1, np.float32)
        return ch


@dataclass
class SceneBranchOutput:
    change_prob: np.ndarray                    # AnyChange, [0,1]
    physical: Dict[str, np.ndarray]            # named channels, each [0,1]
    scene_map: np.ndarray                      # combined scene change evidence [0,1]
    regions: List[ChangeRegion]
    anychange_ran: bool
    raw_angle: Optional[np.ndarray] = None   # HxW AnyChange angle map, degrees

    def coverage(self, thr: float) -> float:
        return float((self.scene_map > thr).mean())

# ---------------------------------------------------------------------------
# STAGE 2: null outputs so an ablation switch is a config change, not a code path.
# ---------------------------------------------------------------------------
def null_object_output(hw: Tuple[int, int]) -> ObjectChangeOutput:
    """What the object branch returns when `use_changeos=False`."""
    z = np.zeros(hw, np.float32)
    return ObjectChangeOutput(z, np.zeros(hw, np.int32), np.zeros(hw, bool), source="disabled")


# ---------------------------------------------------------------------------
# Pixel facts for the caption evidence (perception diagnostic, 43 pairs).
# Model-free, and measured to separate the classes better than the vocabulary did:
#   newly-green water fraction  flood AUROC 0.90  (Pakistan 0.67, Nepal 0.20,
#                               Tonga 0.05, Guatemala 0.04, Turkey 0.04-0.06)
#   edge ratio POST/PRE         flood AUROC 0.91  (floods erase field lines: 0.78)
# ---------------------------------------------------------------------------
def _px_hsv(x: np.ndarray):
    mx, mn = x.max(2), x.min(2)
    d = mx - mn + 1e-6
    r, g, b = x[:, :, 0], x[:, :, 1], x[:, :, 2]
    h = np.where(mx == r, ((g - b) / d) % 6,
        np.where(mx == g, (b - r) / d + 2, (r - g) / d + 4)) * 60
    return h, d / (mx + 1e-6)


def _px_newly_water(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """1 where a pixel is green-cyan NOW, was NOT green-cyan BEFORE, and its hue rotated
    toward green. All three are needed: without the 'not before' test, surviving forest
    passes and the flood-vs-volcano margin collapses from 10.5x to 1.3x."""
    h0, s0 = _px_hsv(a); h1, s1 = _px_hsv(b)
    gc0 = (h0 > 60) & (h0 < 210) & (s0 > 0.10)
    gc1 = (h1 > 60) & (h1 < 210) & (s1 > 0.10)
    toward = ((h1 - h0 + 180) % 360 - 180) > 8
    out = gc1 & ~gc0 & toward
    # rev10: the hue range above also covers green VEGETATION, so regrowth, a greener
    # AFTER image or forest revealed from under a BEFORE cloud all read as "new water".
    # Standing water smooths a surface; vegetation does not. A pixel only counts if its
    # neighbourhood did not become rougher.
    _cfg = globals().get("CFG")
    if _cfg is None or bool(getattr(_cfg, "water_smooth_test", True)):
        def _std(x):
            g = x.mean(2)
            m = box_blur(g, 7); return np.sqrt(np.clip(box_blur(g * g, 7) - m * m, 0, None))
        out &= _std(b) <= _std(a) + 0.01
    return out.astype(np.float32)


def _px_edges(x: np.ndarray) -> np.ndarray:
    from scipy.ndimage import sobel
    g = x.mean(2)
    return (np.hypot(sobel(g, 0), sobel(g, 1)) > 0.08).astype(np.float32)



# ---------------------------------------------------------------------------
# rev10: what is NOT ground (clouds, haze), and the PRE/POST alignment
# ---------------------------------------------------------------------------
class CloudMasker:
    """Cloud and haze masks for BEFORE and AFTER, from physics plus one confirmation.

    A pixel is a candidate when its neighbourhood is bright, colourless and smooth. A
    connected candidate region becomes cloud only if it is (1) at least
    `cloud_min_frac` of the tile, (2) soft-edged -- its rim gradient is low, which is
    what separates a cloud from a large white roof, (3) transient -- the other image
    does not show the same bright smooth surface there, and (4) when a RemoteCLIP
    encoder is given, its crop is classified as cloud or plume. Nothing here refers to
    a disaster class, so the mask is the same for every class."""

    def __init__(self, cfg: Config, clip: Optional["RemoteCLIPEncoder"] = None):
        self.cfg, self.clip = cfg, clip

    def _feat(self, rgb: np.ndarray):
        from scipy import ndimage as ndi
        x = rgb.astype(np.float32) / 255.0
        lum = 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]
        mx, mn = x.max(-1), x.min(-1)
        sat = (mx - mn) / (mx + 1e-4)
        k = max(3, int(round(rgb.shape[0] / 64)) | 1)
        box = lambda a: ndi.uniform_filter(a, k, mode="reflect")
        std = np.sqrt(np.clip(box(lum * lum) - box(lum) ** 2, 0, None))
        cand = (box(lum) >= self.cfg.cloud_lum) & (box(sat) <= self.cfg.cloud_sat) & (std <= self.cfg.cloud_std)
        cand = ndi.binary_closing(ndi.binary_opening(cand, iterations=2), iterations=3)
        return cand, lum

    def _confirmed(self, img_full: np.ndarray, comp_small: np.ndarray) -> bool:
        if self.clip is None or not bool(getattr(self.cfg, "cloud_confirm_clip", True)):
            return True
        ys, xs = np.where(comp_small)
        H, W = img_full.shape[:2]; h, w = comp_small.shape
        y0, y1 = int(ys.min() * H / h), int((ys.max() + 1) * H / h)
        x0, x1 = int(xs.min() * W / w), int((xs.max() + 1) * W / w)
        crop = img_full[y0:max(y1, y0 + 8), x0:max(x1, x0 + 8)]
        try:
            o = self.clip.classify([crop], "scene")
            return o.groups[0] in ("cloud", "plume")
        except Exception as e:
            log.warning("cloud confirmation skipped (%s)", e)
            return True

    def __call__(self, pre: np.ndarray, post: np.ndarray):
        """(cloud_pre, cloud_post, info) as bool maps at the input resolution."""
        from scipy import ndimage as ndi
        H, W = pre.shape[:2]
        if not bool(getattr(self.cfg, "cloud_mask", True)):
            z = np.zeros((H, W), bool)
            return z, z.copy(), {"pre": 0.0, "post": 0.0, "regions": []}
        n = int(self.cfg.cloud_res)
        small = [np.asarray(Image.fromarray(np.asarray(im, np.uint8)).resize((n, n), Image.BILINEAR))
                 for im in (pre, post)]
        (cp, lp), (cq, lq) = self._feat(small[0]), self._feat(small[1])
        masks, regions = [], []
        for side, c, other, lum, full in (("pre", cp, cq, lp, pre), ("post", cq, cp, lq, post)):
            lab, nl = ndi.label(c)
            keep = np.zeros(c.shape, bool)
            g = np.hypot(ndi.sobel(lum, 0), ndi.sobel(lum, 1))
            for j in range(1, nl + 1):
                comp = lab == j
                frac = float(comp.mean())
                if frac < self.cfg.cloud_min_frac:
                    continue
                ring = ndi.binary_dilation(comp, iterations=3) & ~ndi.binary_erosion(comp, iterations=3)
                edge = float(g[ring].mean()) if ring.any() else 1.0
                persist = float(other[comp].mean())
                ok = (edge <= self.cfg.cloud_edge_max and persist <= self.cfg.cloud_persist_max)
                if ok:
                    ok = self._confirmed(full, comp)
                regions.append({"image": side, "area": round(frac, 4), "edge": round(edge, 3),
                                "persists": round(persist, 2), "cloud": bool(ok)})
                if ok:
                    keep |= comp
            keep = ndi.binary_dilation(keep, iterations=4)
            masks.append(resize_map(keep.astype(np.float32), (H, W), nearest=True) > 0.5)
        info = {"pre": round(float(masks[0].mean()), 4), "post": round(float(masks[1].mean()), 4),
                "regions": regions}
        if info["pre"] or info["post"]:
            log.info("clouds: BEFORE %.1f%% | AFTER %.1f%% of the tile excluded from measurement",
                     100 * info["pre"], 100 * info["post"])
        return masks[0], masks[1], info


def align_pair(pre_w: np.ndarray, post_w: np.ndarray, pre_n: np.ndarray, post_n: np.ndarray,
               cfg: Config):
    """Shift AFTER onto BEFORE when that measurably improves the match (rev10).

    Phase correlation on gradient magnitude (robust to the brightness change a disaster
    causes), at work size. The shift is applied only if it is small and raises the
    normalised correlation of the two gradient images by at least 2% -- otherwise the
    pair is left exactly as loaded. Returns (post_w, post_n, info)."""
    info = {"applied": False, "dy": 0.0, "dx": 0.0, "gain": 0.0}
    if not bool(getattr(cfg, "align_images", True)):
        return post_w, post_n, info
    try:
        from skimage.registration import phase_cross_correlation
        from scipy import ndimage as ndi
        def grad(img):
            g = np.asarray(img, np.float32).mean(2) / 255.0
            return np.hypot(ndi.sobel(g, 0), ndi.sobel(g, 1))
        a, b = grad(pre_w), grad(post_w)
        shift, _, _ = phase_cross_correlation(a, b, upsample_factor=4)
        dy, dx = float(shift[0]), float(shift[1])
        if max(abs(dy), abs(dx)) < 0.5 or max(abs(dy), abs(dx)) > float(cfg.align_max_shift):
            info.update(dy=round(dy, 2), dx=round(dx, 2))
            return post_w, post_n, info
        def ncc(u, v):
            u = u - u.mean(); v = v - v.mean()
            return float((u * v).sum() / (np.sqrt((u * u).sum() * (v * v).sum()) + 1e-9))
        b2 = ndi.shift(b, (dy, dx), order=1, mode="nearest")
        before, after = ncc(a, b), ncc(a, b2)
        gain = (after - before) / max(abs(before), 1e-6)
        info.update(dy=round(dy, 2), dx=round(dx, 2), gain=round(gain, 3))
        if gain < 0.02:
            return post_w, post_n, info
        s = post_n.shape[0] / max(post_w.shape[0], 1)
        post_w2 = ndi.shift(post_w.astype(np.float32), (dy, dx, 0), order=1, mode="nearest")
        post_n2 = ndi.shift(post_n.astype(np.float32), (dy * s, dx * s, 0), order=1, mode="nearest")
        info["applied"] = True
        log.info("aligned AFTER onto BEFORE: shift (%.1f, %.1f) px, gradient match +%.0f%%",
                 dy, dx, 100 * gain)
        return (np.clip(post_w2, 0, 255).astype(np.uint8), np.clip(post_n2, 0, 255).astype(np.uint8), info)
    except Exception as e:
        log.warning("alignment skipped (%s: %s)", type(e).__name__, e)
        return post_w, post_n, info


## 4 · SAM2, RemoteCLIP v2, and semantic transitions (Part 1)

`RemoteCLIPEncoder` now returns a `ClipOutput` carrying labels, confidences, the **full probability
matrix**, the **embeddings**, and group-marginalised probabilities. The vocabularies gained `lava`
and `grassland` (scene) and `buried_road` and `broken_bridge` (object), so `grassland → lava` and
`road → buried_road` are expressible.

`SemanticTransitionExtractor` runs the patch grid at **two scales** and keeps a transition only when
all three gates pass:

1. the detector says the patch changed (`transition_min_change`),
2. both labels are confident enough (`transition_min_conf`),
3. the PRE and POST **embeddings actually moved** (`cos < transition_max_embed_sim`).

Gate 3 is what removes the label flicker that a pure argmax comparison produces on near-identical
patches. Each surviving transition reports `p_transition = P(src|PRE)·P(dst|POST)` marginalised over
label groups, so `vegetation → ash` is scored as a group event rather than as one arbitrary
prompt beating another.

**Derived morphological transitions.** No single patch can express "the river widened", so
`_morphological()` compares total water area between the dates and emits `water_expanded` or
`water_receded` when the delta exceeds `water_change_thr`. The same mechanism covers vegetation loss
at scene scale.


In [ ]:
class SAM2Refiner:
    """Refines the object-branch damage map into clean instances.

    `_spread_points` is an O(n^2) farthest-point sampler over every changed pixel, so
    the candidate pool is subsampled to `sam2_point_pool` before selection.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        try:
            from sam2.sam2_image_predictor import SAM2ImagePredictor
        except Exception as e:
            raise ModelLoadError("SAM2 (package)", f"`sam2` not importable: {type(e).__name__}: {e}",
                                 "pip install 'git+https://github.com/facebookresearch/sam2.git' "
                                 "then RESTART THE KERNEL.")
        try:
            self.predictor = SAM2ImagePredictor.from_pretrained(cfg.sam2_id)
        except Exception as e:
            raise ModelLoadError("SAM2 (weights)",
                                 f"from_pretrained('{cfg.sam2_id}') failed: {type(e).__name__}: {e}",
                                 "Enable Internet. Valid ids: facebook/sam2.1-hiera-small, "
                                 "facebook/sam2.1-hiera-base-plus, facebook/sam2.1-hiera-large.")
        log.info("SAM2 ready: %s", cfg.sam2_id)

    def _spread_points(self, mask: np.ndarray, k: int):
        ys, xs = np.where(mask)
        if len(xs) == 0:
            return np.zeros((0, 2), int)
        pts = np.stack([xs, ys], 1)
        if len(pts) > self.cfg.sam2_point_pool:
            # ISSUE 6 (reproducibility): this used the module-level RNG, whose state
            # advances with every call. Two ablation rows with the SAME configuration
            # therefore sampled different prompt points and segmented a different number
            # of instances -- the rev5 earthquake sweep alternated between 11 and 12
            # instances across rows for no causal reason, which makes a 0.01 posterior
            # delta unattributable. A generator seeded from the config and the mask is
            # deterministic and independent of call order.
            gen = np.random.default_rng(
                (int(self.cfg.seed) * 1000003 + int(mask.sum())) % (2 ** 32))
            pts = pts[gen.choice(len(pts), self.cfg.sam2_point_pool, replace=False)]
        chosen = [pts[len(pts) // 2]]
        for _ in range(min(k, len(pts)) - 1):
            d = np.min(np.linalg.norm(pts[:, None] - np.array(chosen)[None], axis=2), axis=1)
            chosen.append(pts[int(d.argmax())])
        return np.array(chosen)

    def refine(self, image: np.ndarray, change_map: np.ndarray):
        binary = change_map > self.cfg.change_threshold
        if binary.sum() < self.cfg.min_instance_area:
            log.warning("Changed area below min_instance_area; no instances segmented.")
            return []
        pts = self._spread_points(binary, self.cfg.sam2_max_prompts)
        self.predictor.set_image(image)
        out = []
        for p in pts:
            masks, scores, _ = self.predictor.predict(
                point_coords=p[None].astype(np.float32),
                point_labels=np.array([1]), multimask_output=False)
            m = masks[0].astype(bool)
            if m.sum() >= self.cfg.min_instance_area:
                out.append((m, float(scores[0])))
        return self._dedup(out)

    @staticmethod
    def _dedup(items, iou_thr=0.75):
        keep = []
        for m, s in sorted(items, key=lambda t: -t[0].sum()):
            if all(np.logical_and(m, k).sum() / (np.logical_or(m, k).sum() + 1e-6) < iou_thr
                   for k, _ in keep):
                keep.append((m, s))
        return keep


# ---------------------------------------------------------------------------
# Vocabularies. Each entry maps a natural-language class to a coarse GROUP; the
# transition table reasons over groups, not over surface wording.
# STAGE 2: `lava`, `grassland` and a wider water vocabulary were added so that the
# transitions the brief asks for (grassland -> lava, river -> widened) are expressible.
# ---------------------------------------------------------------------------
SCENE_VOCAB = {
    "dense green vegetation or forest":        "vegetation",
    # narrowed: "farmland or cropland" matched dry AND flooded fields alike, so green
    # floodwater was absorbed into the vegetation group
    "dry farmland or bare cropland with visible furrows": "vegetation",
    "grassland or open pasture":               "vegetation",
    "bare soil or exposed ground":             "soil",
    "grey volcanic ash covering the ground":   "ash",
    "a dark fresh lava flow":                  "lava",
    "a burned or scorched black area":         "burn",
    "a clear river, lake or sea":              "water",
    "muddy sediment-laden brown water":        "sediment_water",
    "flooded land covered by water":           "flood",
    # Water phrasings taken from the perception diagnostic: on an open vocabulary these
    # rose on flood pairs (green opaque water 0.02 -> 0.23 of patches; water gain AUROC
    # 0.92), while our vocabulary reported flood_extent 0.000 on every pair.
    "green opaque water covering the land":    "flood",
    "brown muddy water covering fields":       "flood",
    "flooded fields with standing water":      "flood",
    "opaque green standing water over farmland": "flood",
    "a town or built-up urban area":           "built",
    "collapsed building rubble and debris":    "rubble",
    "a road or highway":                       "road",
    "a smoke or ash plume":                    "plume",
    "white cloud cover":                       "cloud",
    "a landslide scar or debris flow":         "landslide",
    "grey mud, ash or volcanic deposit over the ground": "ash",
}

OBJECT_VOCAB = {
    "an intact residential building":          "built",
    "an intact industrial building":           "built",
    "collapsed building rubble and debris":    "rubble",
    "a road or highway":                       "road",
    "a road buried under debris or ash":       "buried_road",
    "a bridge":                                "bridge",
    "a collapsed or broken bridge":            "broken_bridge",
    "a river or waterway":                     "water",
    "flooded land covered by water":           "flood",
    "dense green vegetation or forest":        "vegetation",
    "bare soil or exposed ground":             "soil",
    "grey volcanic ash covering the ground":   "ash",
    "a burned or scorched black area":         "burn",
    "a smoke or ash plume":                    "plume",
    "white cloud cover":                       "cloud",
    "a landslide scar or debris flow":         "landslide",
    "grey mud, ash or volcanic deposit over the ground": "ash",
}

SCENE_GROUPS: Tuple[str, ...] = tuple(sorted(set(SCENE_VOCAB.values())))
OBJECT_GROUPS: Tuple[str, ...] = tuple(sorted(set(OBJECT_VOCAB.values())))

# Groups whose POST coverage is read directly as scene evidence (Part 2).
WATER_GROUPS = ("water", "flood", "sediment_water")
VEG_GROUPS = ("vegetation",)

CLIP_TEMPLATES = [
    "a satellite image of {}", "an aerial photo of {}",
    "a high-resolution remote sensing image of {}", "{} seen from above",
]


@dataclass
class ClipOutput:
    """Everything one RemoteCLIP forward pass produces, kept together so that the
    embeddings (needed by the tokeniser, the attention and the classifier) never have
    to be recomputed from a second pass over the same crops."""
    labels: List[str]                  # argmax vocabulary string, per crop
    conf: np.ndarray                   # [N] argmax probability
    probs: np.ndarray                  # [N, V] full distribution over the vocabulary
    embeds: np.ndarray                 # [N, D] L2-normalised image embeddings
    groups: List[str]                  # argmax GROUP, per crop
    group_probs: np.ndarray            # [N, G] vocabulary probabilities summed per group
    group_names: Tuple[str, ...]

    def __len__(self):
        return len(self.labels)


class RemoteCLIPEncoder:
    """
    RemoteCLIP — CLIP fine-tuned on remote-sensing image-text pairs.

    Stage 1 made this a *transition* encoder (PRE vs POST rather than POST-only).
    STAGE 2 additionally exposes the embedding space itself:

      * `classify(crops, kind)`  -> ClipOutput, now including L2-normalised embeddings
                                    and group-marginalised probabilities.
      * `encode_images(crops)`   -> [N, D] embeddings, for evidence tokens.
      * `encode_texts(texts)`    -> [N, D] prompt-ensembled text embeddings, for evidence
                                    tokens, disaster prototypes and caption comparison.
      * `encode_global(img)`     -> [D] whole-image embedding.

    Those embeddings are what makes the cross-branch attention training-free and what
    initialises the classifier head, so nothing downstream needs random weights.

    The weight load is verified: `load_state_dict(..., strict=False)` silently accepts a
    total key mismatch, which would leave a randomly-initialised ViT-L-14 producing
    confident nonsense with no error anywhere in the log.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        try:
            import open_clip
            from huggingface_hub import hf_hub_download
        except Exception as e:
            raise ModelLoadError("RemoteCLIP (packages)", f"{type(e).__name__}: {e}",
                                 "pip install open_clip_torch huggingface_hub")
        try:
            self.model, _, self.preprocess = open_clip.create_model_and_transforms(
                cfg.remoteclip_arch)
            self.tokenizer = open_clip.get_tokenizer(cfg.remoteclip_arch)
            ckpt = hf_hub_download(cfg.remoteclip_repo, f"RemoteCLIP-{cfg.remoteclip_arch}.pt",
                                   cache_dir=f"{WORK}/checkpoints")
            sd = torch.load(ckpt, map_location="cpu")
            sd = sd.get("state_dict", sd)
            report = self.model.load_state_dict(sd, strict=False)
        except Exception as e:
            raise ModelLoadError("RemoteCLIP (weights)",
                                 f"Could not build/load RemoteCLIP-{cfg.remoteclip_arch}: "
                                 f"{type(e).__name__}: {e}",
                                 f"Enable Internet. Confirm the file exists at "
                                 f"https://huggingface.co/{cfg.remoteclip_repo}/tree/main .")

        n_miss, n_unexp = len(report.missing_keys), len(report.unexpected_keys)
        if n_miss > cfg.remoteclip_max_missing:
            raise ModelLoadError(
                "RemoteCLIP (key mismatch)",
                f"{n_miss} missing keys after load_state_dict, e.g. {report.missing_keys[:5]}. "
                "The checkpoint did not populate the architecture -- the encoder is still "
                "randomly initialised and every label it produces would be meaningless.",
                "Check that remoteclip_arch matches the checkpoint filename "
                "(RN50 / ViT-B-32 / ViT-L-14).")
        log.info("RemoteCLIP weights verified: %d missing / %d unexpected keys", n_miss, n_unexp)

        self.model = self.model.to(cfg.device).eval()
        self.scene_vocab = list(SCENE_VOCAB.keys())
        self.object_vocab = list(OBJECT_VOCAB.keys())
        self.vocab = {"scene": self.scene_vocab, "object": self.object_vocab}
        self.vocab_map = {"scene": SCENE_VOCAB, "object": OBJECT_VOCAB}
        self.groups = {"scene": SCENE_GROUPS, "object": OBJECT_GROUPS}
        self.text = {k: self._text_features([f for f in v]) for k, v in self.vocab.items()}
        # [V, G] one-hot vocabulary -> group matrix, for marginalising probabilities
        self.group_mat = {}
        for kind, vocab in self.vocab.items():
            M = np.zeros((len(vocab), len(self.groups[kind])), np.float32)
            for i, v in enumerate(vocab):
                M[i, self.groups[kind].index(self.vocab_map[kind][v])] = 1.0
            self.group_mat[kind] = M
        self.embed_dim = int(self.text["scene"].shape[1])
        self._text_cache: Dict[str, np.ndarray] = {}
        log.info("RemoteCLIP ready (%d scene classes / %d groups, %d object classes, D=%d)",
                 len(self.scene_vocab), len(SCENE_GROUPS), len(self.object_vocab), self.embed_dim)

    # -- encoders ------------------------------------------------------------
    @torch.no_grad()
    def _text_features(self, vocab: List[str]) -> torch.Tensor:
        """Prompt-ensembled text embeddings: average over templates, then renormalise.
        Reduces the template sensitivity that makes single-prompt zero-shot brittle."""
        feats = []
        for v in vocab:
            toks = self.tokenizer([t.format(v) for t in CLIP_TEMPLATES]).to(self.cfg.device)
            f = self.model.encode_text(toks)
            f = f / f.norm(dim=-1, keepdim=True)
            f = f.mean(0)
            feats.append(f / f.norm())
        return torch.stack(feats)

    @torch.no_grad()
    def encode_texts(self, texts: Sequence[str], ensemble: bool = True) -> np.ndarray:
        """[N, D] L2-normalised text embeddings, cached by string."""
        texts = list(texts)
        todo = [t for t in texts if (t, ensemble) not in self._text_cache]
        if todo:
            if ensemble:
                feats = self._text_features(todo).cpu().numpy()
            else:
                toks = self.tokenizer(todo).to(self.cfg.device)
                f = self.model.encode_text(toks)
                feats = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()
            for t, f in zip(todo, feats):
                self._text_cache[(t, ensemble)] = np.asarray(f, np.float32)
        if not texts:
            return np.zeros((0, self.embed_dim), np.float32)
        return np.stack([self._text_cache[(t, ensemble)] for t in texts])

    @torch.no_grad()
    def encode_images(self, crops: List[np.ndarray]) -> np.ndarray:
        """[N, D] L2-normalised image embeddings."""
        if not crops:
            return np.zeros((0, self.embed_dim), np.float32)
        out = []
        for i in range(0, len(crops), self.cfg.clip_batch):
            batch = crops[i:i + self.cfg.clip_batch]
            t = torch.stack([self.preprocess(Image.fromarray(np.asarray(c, np.uint8)))
                             for c in batch]).to(self.cfg.device)
            f = self.model.encode_image(t)
            f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.float().cpu().numpy())
        return np.concatenate(out, 0).astype(np.float32)

    def encode_global(self, img: np.ndarray) -> np.ndarray:
        return self.encode_images([img])[0]

    def classify(self, crops: List[np.ndarray], kind: str) -> ClipOutput:
        """One pass: embeddings, vocabulary probabilities and group probabilities."""
        vocab, groups = self.vocab[kind], self.groups[kind]
        if not crops:
            return ClipOutput([], np.zeros(0, np.float32), np.zeros((0, len(vocab)), np.float32),
                              np.zeros((0, self.embed_dim), np.float32), [],
                              np.zeros((0, len(groups)), np.float32), groups)
        E = self.encode_images(crops)
        T = self.text[kind].float().cpu().numpy()
        logits = 100.0 * E @ T.T
        logits = logits - logits.max(1, keepdims=True)
        P = np.exp(logits); P = P / P.sum(1, keepdims=True)
        idx = P.argmax(1)
        labels = [vocab[i] for i in idx]
        GP = P @ self.group_mat[kind]
        if str(getattr(self.cfg, "clip_group_score", "mean")) == "mean":
            # rev10: a group's score is the MEAN of its phrases' probabilities, renormalised
            # over groups. With the sum, "flood" (5 phrases) collected up to 5x the mass of
            # a one-phrase group on any ambiguous patch and cleared group_area_thr far more
            # easily. Each group is now one competitor, whatever its number of phrases, and
            # a cell's group and label are read from these group scores.
            M = self.group_mat[kind]
            GP = GP / np.maximum(M.sum(0, keepdims=True), 1.0)
            GP = GP / np.maximum(GP.sum(1, keepdims=True), 1e-12)
            gi = GP.argmax(1)
            idx = np.array([int(np.argmax(np.where(M[:, g] > 0, P[r], -1.0))) for r, g in enumerate(gi)])
            labels = [vocab[i] for i in idx]
            conf = GP[np.arange(len(gi)), gi]
            return ClipOutput(labels, conf.astype(np.float32), P.astype(np.float32), E,
                              [groups[g] for g in gi], GP.astype(np.float32), groups)
        return ClipOutput(labels, P[np.arange(len(idx)), idx].astype(np.float32), P.astype(np.float32),
                          E, [self.vocab_map[kind][l] for l in labels], GP.astype(np.float32), groups)

    # Stage 1 call signature, kept so nothing older breaks.
    def classify_batch(self, crops: List[np.ndarray], kind: str):
        o = self.classify(crops, kind)
        return o.labels, o.conf, o.probs

    # -- caption comparison (used by RVG v3) ---------------------------------
    @torch.no_grad()
    def encode_sentences(self, text: str) -> np.ndarray:
        """A caption is longer than CLIP's 77-token context, so it is embedded
        sentence-wise and mean-pooled rather than truncated to its first clause."""
        parts = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text or "") if len(s.strip()) > 3]
        if not parts:
            return np.zeros(self.embed_dim, np.float32)
        E = self.encode_texts(parts, ensemble=False)
        v = E.mean(0)
        n = np.linalg.norm(v)
        return (v / n).astype(np.float32) if n > 1e-8 else v.astype(np.float32)

    def caption_similarity(self, a: str, b: str) -> float:
        ea, eb = self.encode_sentences(a), self.encode_sentences(b)
        if np.linalg.norm(ea) < 1e-8 or np.linalg.norm(eb) < 1e-8:
            return 0.0
        return float(np.dot(ea, eb))


In [ ]:
@dataclass
class SemanticTransition:
    """One PRE -> POST semantic change (Part 1).

    Everything the brief asks a transition to carry is a field here: the pre and post
    semantic labels, a confidence, the patch coordinates, the RemoteCLIP embedding
    similarity between the matched PRE and POST content, and the group-marginalised
    transition probability P(src | pre) * P(dst | post).
    """
    src_label: str
    dst_label: str
    src_group: str
    dst_group: str
    src_conf: float                  # P(src_label | PRE patch)
    dst_conf: float                  # P(dst_label | POST patch)
    confidence: float                # joint CLIP confidence x change gate x embedding motion
    transition_prob: float           # P(src_group | PRE) * P(dst_group | POST)
    embed_similarity: float          # cosine(PRE patch embedding, POST patch embedding)
    change_magnitude: float          # local scene-change energy inside the patch
    weight: float                    # area share of the scene (or 1/N for objects)
    where: Tuple[int, int, int, int] # patch coordinates, XYXY
    level: str                       # "scene" | "object"
    kind: str = "patch"              # "patch" | "object" | "derived"
    grid: int = 0                    # which patch grid produced it (0 for object/derived)

    @property
    def changed(self) -> bool:
        return self.src_group != self.dst_group

    @property
    def name(self) -> str:
        return f"{self.src_group}->{self.dst_group}"

    def phrase(self) -> str:
        return f"{self.src_label} changing into {self.dst_label}"

    def __repr__(self):
        return (f"{self.src_group}->{self.dst_group} (conf={self.confidence:.2f}, "
                f"p={self.transition_prob:.2f}, sim={self.embed_similarity:.2f}, "
                f"w={self.weight:.3f}, {self.level}/{self.kind})")


# Stage 1 name, kept so older call sites and saved records still resolve.
Transition = SemanticTransition


@dataclass
class PatchGrid:
    """The raw per-cell record for one patch grid, kept because the tokeniser, the
    attention and every transition visualisation all read from it."""
    g: int
    boxes: List[Tuple[int, int, int, int]]
    pre: ClipOutput
    post: ClipOutput
    mags: np.ndarray                 # [N] mean scene-change magnitude per cell
    sims: np.ndarray                 # [N] cosine(PRE, POST) embedding similarity
    obs: Optional[np.ndarray] = None # [N] rev10: cell observable (not clouded in either image)


@dataclass
class SceneSemantics:
    """Everything the semantic scene branch produces for one image pair."""
    transitions: List[SemanticTransition]
    grids: Dict[int, PatchGrid]
    group_cov_pre: Dict[str, float]      # soft coverage per SCENE group, PRE
    group_cov_post: Dict[str, float]     # ... POST
    water_pre: float
    water_post: float
    veg_pre: float
    veg_post: float
    global_pre: np.ndarray               # [D] whole-image PRE embedding
    global_post: np.ndarray              # [D] whole-image POST embedding
    enabled: bool = True
    obs_frac: float = 1.0                # rev10: share of the main grid that is observable

    @property
    def main_grid(self) -> Optional[PatchGrid]:
        return self.grids.get(max(self.grids)) if self.grids else None

    @property
    def water_delta(self) -> float:
        return self.water_post - self.water_pre

    @property
    def veg_delta(self) -> float:
        return self.veg_pre - self.veg_post

    def changed(self) -> List[SemanticTransition]:
        return [t for t in self.transitions if t.changed]


def empty_scene_semantics(dim: int) -> SceneSemantics:
    z = np.zeros(dim, np.float32)
    return SceneSemantics([], {}, {g: 0.0 for g in SCENE_GROUPS}, {g: 0.0 for g in SCENE_GROUPS},
                          0.0, 0.0, 0.0, 0.0, z, z, enabled=False)


class SemanticTransitionExtractor:
    """
    Scene-level semantic change understanding (Part 1).

    Stage 1 tiled PRE and POST on ONE 8x8 grid and kept a cell whose argmax class
    changed. Stage 2:

      1. runs a MULTI-SCALE grid (8x8 and 16x16 by default), so a transition that only
         exists at one scale -- a buried road at 32 px, an ash field at 128 px -- is not
         averaged away by the other;
      2. keeps the full probability vectors, so a transition carries an actual
         probability P(src|PRE) * P(dst|POST) rather than only the argmax confidences;
      3. gates on RemoteCLIP EMBEDDING MOTION as well as on detector change magnitude.
         Two patches whose embeddings are nearly identical cannot be a semantic
         transition no matter what the two argmaxes say, which is what kills the label
         flicker that used to invent transitions on unchanged ground;
      4. derives morphological transitions that no single-patch class change can express
         -- `river -> widened` is a change in the AREA of the water class, not in the
         class of any one patch.
    """

    def __init__(self, cfg: Config, clip: RemoteCLIPEncoder):
        self.cfg, self.clip = cfg, clip

    # -- helpers -------------------------------------------------------------
    @staticmethod
    def _cells(H: int, W: int, g: int):
        ys = np.linspace(0, H, g + 1).astype(int)
        xs = np.linspace(0, W, g + 1).astype(int)
        return [(int(xs[j]), int(ys[i]), int(xs[j + 1]), int(ys[i + 1]))
                for i in range(g) for j in range(g)]

    def _grid(self, pre, post, change_map, g, invalid=None) -> PatchGrid:
        H, W = pre.shape[:2]
        boxes = self._cells(H, W, g)
        # rev10: a cell more than `cloud_cell_frac` clouded in EITHER image cannot be
        # compared; it is excluded from coverage, transitions and the water override.
        if invalid is not None:
            inv = np.asarray(invalid, bool)
            frac = float(getattr(self.cfg, "cloud_cell_frac", 0.30))
            obs = np.array([float(inv[y0:y1, x0:x1].mean()) < frac for (x0, y0, x1, y1) in boxes], bool)
        else:
            obs = np.ones(len(boxes), bool)
        cpre = [pre[y0:y1, x0:x1] for (x0, y0, x1, y1) in boxes]
        cpost = [post[y0:y1, x0:x1] for (x0, y0, x1, y1) in boxes]
        mags = np.array([float(change_map[y0:y1, x0:x1].mean()) for (x0, y0, x1, y1) in boxes],
                        np.float32)
        op, oq = self.clip.classify(cpre, "scene"), self.clip.classify(cpost, "scene")
        oq = _water_override(cpre, cpost, oq, self.cfg, obs=obs)
        sims = np.sum(op.embeds * oq.embeds, axis=1).astype(np.float32)
        return PatchGrid(g, boxes, op, oq, mags, sims, obs)

    def _coverage(self, grid: PatchGrid) -> Tuple[Dict[str, float], Dict[str, float]]:
        """Group coverage over equal-area cells.

        ISSUE 7: the mean group PROBABILITY is a soft share with a non-zero floor, not
        an area. It is what made "volcanic ash coverage 2%" appear on a flood image and
        be handed to Qwen as a measurement. With `group_coverage="mask"` a cell counts
        towards a group only when that group clears `group_area_thr`, so the number is
        the fraction of the image actually classified as that group, and it is exactly
        zero when nothing is. The soft version is kept for comparison."""
        # rev10: fractions are of the OBSERVABLE cells (clouded cells are not ground)
        o = grid.obs if grid.obs is not None else np.ones(len(grid.boxes), bool)
        if not o.any():
            z = {g: 0.0 for g in SCENE_GROUPS}
            return z, dict(z)
        if self.cfg.group_coverage == "mask":
            thr = self.cfg.group_area_thr
            pre = {g: float((grid.pre.group_probs[o, i] >= thr).mean())
                   for i, g in enumerate(SCENE_GROUPS)}
            post = {g: float((grid.post.group_probs[o, i] >= thr).mean())
                    for i, g in enumerate(SCENE_GROUPS)}
            return pre, post
        pre = {g: float(grid.pre.group_probs[o, i].mean()) for i, g in enumerate(SCENE_GROUPS)}
        post = {g: float(grid.post.group_probs[o, i].mean()) for i, g in enumerate(SCENE_GROUPS)}
        return pre, post

    # -- scene ---------------------------------------------------------------
    def scene(self, pre: np.ndarray, post: np.ndarray, change_map: np.ndarray,
              invalid: Optional[np.ndarray] = None) -> SceneSemantics:
        cfg = self.cfg
        grids = {g: self._grid(pre, post, change_map, g, invalid) for g in cfg.patch_grids}
        main = grids[cfg.semantic_grid]
        cov_pre, cov_post = self._coverage(main)

        water_pre = float(sum(cov_pre[g] for g in WATER_GROUPS))
        water_post = float(sum(cov_post[g] for g in WATER_GROUPS))
        veg_pre = float(sum(cov_pre[g] for g in VEG_GROUPS))
        veg_post = float(sum(cov_post[g] for g in VEG_GROUPS))

        # ISSUE 17: with cell_w = 1/(g*g)/n_grids each grid contributed half the image
        # area, so the SAME ground was counted once at 8x8 and once at 16x16 and
        # `transition_mass` was an average of two scales dressed as an area fraction.
        # The weight is now the cell's true area share of ONE image, and the per-grid
        # share is recorded on the transition so a consumer can de-duplicate by scale.
        out: List[SemanticTransition] = []
        n_grids = max(len(grids), 1)
        for g, pg in grids.items():
            cell_w = 1.0 / (g * g) / n_grids
            gp_pre, gp_post = pg.pre.group_probs, pg.post.group_probs
            for k, box in enumerate(pg.boxes):
                if pg.obs is not None and not pg.obs[k]:
                    continue                       # rev10: clouded in one of the images
                sg, dg = pg.pre.groups[k], pg.post.groups[k]
                if sg == dg:
                    continue
                sim = float(pg.sims[k])
                if sim > cfg.transition_max_embed_sim:
                    continue                       # the two patches are the same content
                mag = float(pg.mags[k])
                if mag < cfg.transition_min_change:
                    continue
                p_tr = float(gp_pre[k, SCENE_GROUPS.index(sg)] * gp_post[k, SCENE_GROUPS.index(dg)])
                if p_tr < cfg.transition_min_prob:
                    continue
                gate = min(1.0, mag / max(cfg.scene_change_thr, 1e-6))
                motion = float(np.clip((1.0 - sim) / 0.15, 0.0, 1.0))
                conf = float(np.sqrt(pg.pre.conf[k] * pg.post.conf[k]) * gate * (0.5 + 0.5 * motion))
                if conf < cfg.transition_min_conf:
                    continue
                out.append(SemanticTransition(
                    src_label=pg.pre.labels[k], dst_label=pg.post.labels[k],
                    src_group=sg, dst_group=dg,
                    src_conf=float(pg.pre.conf[k]), dst_conf=float(pg.post.conf[k]),
                    confidence=conf, transition_prob=p_tr, embed_similarity=sim,
                    change_magnitude=mag, weight=cell_w, where=box, level="scene",
                    kind="patch", grid=g))

        out.extend(self._morphological(main, water_pre, water_post))
        out.sort(key=lambda t: -(t.confidence * t.weight))
        log.info("Semantic transitions: %d kept over grids %s | water %.3f->%.3f | veg %.3f->%.3f",
                 len(out), tuple(grids), water_pre, water_post, veg_pre, veg_post)
        return SceneSemantics(out, grids, cov_pre, cov_post, water_pre, water_post,
                              veg_pre, veg_post,
                              self.clip.encode_global(pre), self.clip.encode_global(post),
                              obs_frac=float(main.obs.mean()) if main.obs is not None else 1.0)

    def _morphological(self, grid: PatchGrid, water_pre: float,
                       water_post: float) -> List[SemanticTransition]:
        """`river -> widened` is a change in the AREA of a class, not in the class of any
        one patch, so it is derived from the coverage delta and attributed to the cells
        that actually gained water."""
        d = water_post - water_pre
        # rev10: 1.5% of the tile was enough, and a BEFORE cloud or a greener AFTER image
        # produced that easily (TONGA 000076, 000222, 000891, 000918)
        if abs(d) < float(getattr(self.cfg, "water_derived_min", self.cfg.water_change_thr)):
            return []
        wi = [SCENE_GROUPS.index(g) for g in WATER_GROUPS]
        gain = grid.post.group_probs[:, wi].sum(1) - grid.pre.group_probs[:, wi].sum(1)
        if grid.obs is not None:
            gain = np.where(grid.obs, gain, -1.0)
        cells = np.argsort(-gain)[:max(1, int(0.25 * len(gain)))] if d > 0 else []
        cells = [c for c in cells if gain[c] > 0.05]
        if d > 0 and not cells:
            return []
        widened = d > 0
        x0 = min(grid.boxes[c][0] for c in cells) if cells else 0
        y0 = min(grid.boxes[c][1] for c in cells) if cells else 0
        x1 = max(grid.boxes[c][2] for c in cells) if cells else grid.boxes[-1][2]
        y1 = max(grid.boxes[c][3] for c in cells) if cells else grid.boxes[-1][3]
        conf = float(np.clip(abs(d) / max(3 * self.cfg.water_change_thr, 1e-6), 0.0, 1.0))
        return [SemanticTransition(
            src_label="a clear river, lake or sea",
            dst_label=("a widened river or expanded water body" if widened
                       else "a receded or dried water body"),
            src_group="water", dst_group=("water_expanded" if widened else "water_receded"),
            src_conf=float(water_pre), dst_conf=float(water_post),
            confidence=conf, transition_prob=float(min(1.0, abs(d) * 4)),
            embed_similarity=0.0, change_magnitude=float(abs(d)), weight=float(abs(d)),
            where=(int(x0), int(y0), int(x1), int(y1)), level="scene", kind="derived")]

    # -- object --------------------------------------------------------------
    def object(self, pre: np.ndarray, post: np.ndarray,
               boxes: List[Tuple[int, int, int, int]]
               ) -> Tuple[Dict[int, SemanticTransition], np.ndarray]:
        """PRE-crop vs POST-crop transitions for the object branch. Returns a mapping
        BOX INDEX -> transition (a box whose crop is degenerate is simply absent) AND
        the POST-crop embeddings, which become the object evidence tokens without a
        second RemoteCLIP pass."""
        if not boxes:
            return {}, np.zeros((0, self.clip.embed_dim), np.float32)
        cpre = [pre[y0:y1 + 1, x0:x1 + 1] for (x0, y0, x1, y1) in boxes]
        cpost = [post[y0:y1 + 1, x0:x1 + 1] for (x0, y0, x1, y1) in boxes]
        keep = [i for i in range(len(boxes)) if cpre[i].size and cpost[i].size]
        op = self.clip.classify([cpre[i] for i in keep], "object")
        oq = self.clip.classify([cpost[i] for i in keep], "object")
        sims = (np.sum(op.embeds * oq.embeds, 1) if len(keep)
                else np.zeros(0, np.float32))

        # ISSUE 16: this returns one transition per KEPT box, but the caller indexes the
        # result by the full mask index. A single degenerate crop (a 1-pixel bbox at an
        # image edge) therefore paired every later instance with another instance's
        # transition, silently. The transitions are returned keyed BY BOX INDEX.
        out: Dict[int, SemanticTransition] = {}
        total = float(len(boxes))
        gi = {g: i for i, g in enumerate(OBJECT_GROUPS)}
        for n, i in enumerate(keep):
            sg, dg = op.groups[n], oq.groups[n]
            out[i] = (SemanticTransition(
                src_label=op.labels[n], dst_label=oq.labels[n], src_group=sg, dst_group=dg,
                src_conf=float(op.conf[n]), dst_conf=float(oq.conf[n]),
                confidence=float(np.sqrt(op.conf[n] * oq.conf[n])),
                transition_prob=float(op.group_probs[n, gi[sg]] * oq.group_probs[n, gi[dg]]),
                embed_similarity=float(sims[n]), change_magnitude=float(1.0 - sims[n]),
                weight=1.0 / total, where=boxes[i], level="object", kind="object"))
        embeds = np.zeros((len(boxes), self.clip.embed_dim), np.float32)
        for n, i in enumerate(keep):
            embeds[i] = oq.embeds[n]
        return out, embeds          # out: {box index -> transition}


# ---------------------------------------------------------------------------
# Water override for the patch classifier.
#
# Perception diagnostic, 43 pairs: our vocabulary put Pakistan floodwater in
# "dense green vegetation" (vegetation 0.26 -> 0.74, flood 0.00 -> 0.02), and the
# `flood_extent` field was 0.000 on EVERY pair — AUROC 0.50, i.e. no information.
# On an open vocabulary RemoteCLIP itself saw the water (green opaque water 0.02 ->
# 0.23 of patches; open-vocab water gain AUROC 0.92). So the water is perceivable; the
# vocabulary was not asking for it, and argmax can still prefer a green-field phrase.
#
# After classification, a POST cell labelled vegetation, ash or soil is re-assigned to
# the flood group when its pixels say it BECAME water (green-cyan now, not before, hue
# rotated toward green). Measured on real 64-px patches: 55% of cells flip on Pakistan
# floods, 0% on Tonga, 3.7% on Guatemala, 0.9% on Turkey. It only ever moves cells INTO
# the flood group, so it cannot create ash, burn or rubble.
# ---------------------------------------------------------------------------
WATER_OVERRIDE_FROM = {"vegetation", "ash", "soil"}
WATER_OVERRIDE_LABEL = "opaque green standing water over farmland"


def _cell_became_water(pre_cell: np.ndarray, post_cell: np.ndarray) -> float:
    a = np.asarray(pre_cell, np.float32)
    b = np.asarray(post_cell, np.float32)
    if a.size == 0 or b.size == 0:
        return 0.0
    if a.max() > 1.5: a = a / 255.0
    if b.max() > 1.5: b = b / 255.0
    return float(_px_newly_water(a, b).mean())


def _water_override(cpre, cpost, oq, cfg, obs=None):
    if "flood" not in SCENE_GROUPS:
        return oq
    fi = list(SCENE_GROUPS).index("flood")
    thr = float(getattr(cfg, "water_override_thr", 0.35))
    # rev10: "ash" is no longer a source (an ash cell must not become water), and a cell
    # clouded in either image is skipped (forest under a BEFORE cloud is not new water)
    src = set(getattr(cfg, "water_override_from", WATER_OVERRIDE_FROM))
    n = 0
    for k in range(len(cpost)):
        if obs is not None and not obs[k]:
            continue
        if oq.groups[k] not in src:
            continue
        if _cell_became_water(cpre[k], cpost[k]) < thr:
            continue
        oq.groups[k] = "flood"
        oq.labels[k] = WATER_OVERRIDE_LABEL
        row = np.zeros_like(oq.group_probs[k]); row[fi] = 1.0
        oq.group_probs[k] = row                      # counts as flood under mask coverage
        n += 1
    if n:
        log.info("water override: %d/%d cells reassigned to flood", n, len(cpost))
    return oq


## 5 · Evidence aggregation — what the reasoners actually consume

`SceneEvidence` (20 fields) and `ObjectEvidence` (14 fields) are the interface between perception
and reasoning. Everything downstream — the tokeniser, the DKG, the classifier's tabular features,
the RVG and the Qwen prompt — reads these two objects, never the raw maps.

Scene: ash / lava / burn-scar / flood / rubble / landslide-scar coverage, water area before and
after, vegetation loss, terrain deformation, brightness, desaturation, texture, smoke, cloud
obscuration, scene change coverage, transition mass.
Object: destroyed / damaged buildings, buried and damaged roads, collapsed bridges, damaged
infrastructure, the xBD grade histogram by area, instance count, damaged area fraction, and the
RemoteCLIP label counts.

`SceneEvidence.vector()` and the object counts are also the tabular half of the classifier input,
which is why they are defined once, here, in a fixed field order (`SCENE_FIELDS`, `OBJECT_FIELDS`).


In [ ]:
# ---------------------------------------------------------------------------
# Fixed field orders. Every downstream consumer -- DKG, the classifier feature
# vector, the evidence package, the ablation tables -- indexes these tuples, so the
# feature layout is stable across runs and across saved checkpoints.
# ---------------------------------------------------------------------------
SCENE_FIELDS: Tuple[str, ...] = (
    "ash_coverage", "lava_coverage", "burn_scars", "flood_extent", "river_change",
    "water_expansion", "smoke_regions", "cloud_obscuration", "rubble_coverage",
    "landslide_scars", "bare_soil_gain", "vegetation_loss", "terrain_deformation",
    "brightness_change", "darkening", "desaturation", "texture_change",
    "scene_change_coverage", "anychange_coverage", "transition_mass")

OBJECT_FIELDS: Tuple[str, ...] = (
    "n_instances_norm", "damaged_buildings_frac", "destroyed_buildings_frac",
    "collapsed_bridges_frac", "damaged_roads_frac", "buried_roads_frac",
    "damaged_infrastructure_frac", "damaged_area_frac", "mean_severity",
    "severity_in_buildings", "grade_minor", "grade_major", "grade_destroyed",
    "object_transition_mass")

PHYS_FIELDS: Tuple[str, ...] = tuple(ScenePhysicalChannels.NAMES)

# Scene evidence -> DKG concepts (Part 2). Weight is how strongly the measurement
# implies the concept; the mass itself is the measured area fraction.
SCENE_EVIDENCE_CONCEPTS: Dict[str, List[Tuple[str, float]]] = {
    "ash_coverage":       [("ashfall", 1.0)],
    "lava_coverage":      [("lava_flow", 1.0), ("ashfall", 0.3)],
    "burn_scars":         [("burned", 1.0)],
    "flood_extent":       [("inundation", 1.0)],
    "water_expansion":    [("water_expansion", 1.0), ("inundation", 0.5)],
    "smoke_regions":      [("plume", 1.0)],
    "cloud_obscuration":  [("obscuration", 1.0)],
    "rubble_coverage":    [("structural_damage", 0.7), ("debris_scatter", 0.6)],
    "landslide_scars":    [("mass_movement", 1.0), ("terrain_change", 0.6)],
    "bare_soil_gain":     [("surface_change", 0.6), ("vegetation_loss", 0.4)],
    "vegetation_loss":    [("vegetation_loss", 1.0)],
    "terrain_deformation": [("terrain_change", 1.0)],
    "brightness_change":  [("surface_change", 0.5)],
    # rev11: darkening split by surface (see darkening_types); added to the values after
    # construction, so SCENE_FIELDS -- the classifier head's inputs -- is unchanged
    "dark_deposit_cover": [("ashfall", 0.8), ("surface_change", 0.3)],
    "burn_cover":         [("burned", 1.0), ("vegetation_loss", 0.3)],
}

# Object evidence -> DKG concepts (Part 2).
OBJECT_EVIDENCE_CONCEPTS: Dict[str, List[Tuple[str, float]]] = {
    "damaged_buildings_frac":      [("structural_damage", 1.0)],
    "destroyed_buildings_frac":    [("structural_damage", 1.0), ("debris_scatter", 0.7)],
    "collapsed_bridges_frac":      [("infrastructure_damage", 1.0)],
    "damaged_roads_frac":          [("infrastructure_damage", 0.8)],
    "buried_roads_frac":           [("infrastructure_damage", 0.8), ("ashfall", 0.4)],
    "damaged_area_frac":           [("structural_damage", 0.6)],
}

BUILDING_GROUPS = ("built", "rubble")
ROAD_GROUPS = ("road", "buried_road")
BRIDGE_GROUPS = ("bridge", "broken_bridge")
BURIAL_GROUPS = ("ash", "landslide", "rubble", "flood", "lava")


def _confirmed_water(sem: SceneSemantics, physical: Dict[str, np.ndarray], cfg, return_cells: bool = False):
    """rev10: share of the OBSERVABLE main-grid cells holding confirmed water in POST.
    A RemoteCLIP water cell (flood or sediment-laden water, by the coverage rule) is
    confirmed if (a) the pixel test finds new water over >= 15% of that cell, or (b) it
    belongs to a 4-connected body of at least `water_min_body_cells` water cells."""
    g = sem.main_grid if getattr(sem, "enabled", True) else None
    if g is None:
        return 0.0
    from scipy import ndimage as ndi
    idx = [SCENE_GROUPS.index(x) for x in ("flood", "sediment_water") if x in SCENE_GROUPS]
    thr = float(getattr(cfg, "group_area_thr", 0.5)) if cfg is not None else 0.5
    obs = g.obs if g.obs is not None else np.ones(len(g.boxes), bool)
    wet = (g.post.group_probs[:, idx].max(1) >= thr) & obs
    if not wet.any() or not obs.any():
        return 0.0
    nw = physical.get("newly_water") if physical else None
    px = np.zeros(len(g.boxes), bool)
    if nw is not None:
        px = np.array([float(np.asarray(nw)[y0:y1, x0:x1].mean()) >= 0.15 for (x0, y0, x1, y1) in g.boxes])
    lab, n = ndi.label(wet.reshape(g.g, g.g))
    sizes = np.bincount(lab.ravel())
    body = (sizes[lab.ravel()] >= int(getattr(cfg, "water_min_body_cells", 3) if cfg is not None else 3)) & (lab.ravel() > 0)
    # rev11: a body of RemoteCLIP "water" cells counts only if those cells became smoother
    # or are very flat -- water smooths a surface; ash-darkened ground does not
    sm, tp = (physical.get("smoothing"), physical.get("texture_post")) if physical else (None, None)
    if sm is not None and tp is not None:
        smooth_ok = np.array([float(np.asarray(sm)[y0:y1, x0:x1].mean()) >= float(getattr(cfg, "water_body_smooth", 0.10))
                              or float(np.asarray(tp)[y0:y1, x0:x1].mean()) <= float(getattr(cfg, "water_body_flat", 0.015))
                              for (x0, y0, x1, y1) in g.boxes])
        body &= smooth_ok
    conf = wet & (px | body)
    if return_cells:
        return conf
    return float(conf.sum() / obs.sum())


def darkening_types(physical: Dict[str, np.ndarray], cfg, buildings: Optional[np.ndarray] = None,
                    water: Optional[np.ndarray] = None) -> dict:
    """rev11: split darkening by what it does to DIFFERENT SURFACES, per 3x3 zone.

    A pixel is `covered` when it darkened strongly AND its material changed (its
    chromaticity shifted or it lost colour) -- a shadow darkens without changing colour.
    In each zone the covered share of OPEN ground (not vegetation, not a building, not
    water) is compared with the covered share of vegetation:
      deposit   open ground covered at least half as much as vegetation (and >= min_hit)
      burn      vegetation covered (>= min_hit), open ground nearly untouched (< 1/4 of it)
      ambiguous neither the zone nor the rest of the tile has enough open ground to tell
    A zone with too little open ground of its own is judged against the open ground of
    the whole tile.
    Returns covers as fractions of the observable tile, plus where they are."""
    z0 = {"deposit": 0.0, "burn": 0.0, "ambiguous": 0.0, "shadow": 0.0,
          "where_deposit": "", "where_burn": "", "zones": [],
          "zones_deposit": [[0.0] * 3 for _ in range(3)], "zones_burn": [[0.0] * 3 for _ in range(3)]}
    need = ("darkening", "chroma_shift", "sat_drop", "veg_mask_pre")
    if not physical or any(k not in physical for k in need):
        return z0
    valid = np.asarray(physical.get("valid", np.ones_like(physical["darkening"])), np.float32) > 0.5
    excl = ~valid
    if water is not None:
        excl |= np.asarray(water, bool)
    dark = np.asarray(physical["darkening"]) >= float(getattr(cfg, "deposit_dark", 0.5))
    material = ((np.asarray(physical["chroma_shift"]) >= float(getattr(cfg, "deposit_chroma", 0.025)))
                | (np.asarray(physical["sat_drop"]) >= float(getattr(cfg, "deposit_sat_drop", 0.08))))
    veg = np.asarray(physical["veg_mask_pre"]) > 0.5
    hit = dark & material & ~excl
    shadow = dark & ~material & ~excl
    ground = ~veg & ~excl
    if buildings is not None:
        b = np.asarray(buildings, bool)
        if b.shape != ground.shape:
            b = resize_map(b.astype(np.float32), ground.shape, nearest=True) > 0.5
        ground &= ~b
    vegz = veg & ~excl
    n_ok = max(int(valid.sum()), 1)
    H, W = hit.shape
    ys, xs = np.linspace(0, H, 4).astype(int), np.linspace(0, W, 4).astype(int)
    mh = float(getattr(cfg, "deposit_min_hit", 0.10)); gmin = float(getattr(cfg, "deposit_ground_min", 0.10))
    dep, brn = np.zeros_like(hit), np.zeros_like(hit)
    # open ground of the WHOLE tile: the reference for zones that have none of their own
    # (a settled layer is regional -- roads elsewhere are covered too; a fire leaves them)
    hg_tile = float(hit[ground].mean()) if ground.sum() >= gmin * hit.size else None
    out = dict(z0); zones = []
    for i in range(3):
        for j in range(3):
            sl = (slice(ys[i], ys[i + 1]), slice(xs[j], xs[j + 1]))
            zg, zv, zh = ground[sl], vegz[sl], hit[sl]
            npx = zg.size
            hg = float(zh[zg].mean()) if zg.sum() > 0 else 0.0
            hv = float(zh[zv].mean()) if zv.sum() > 0 else 0.0
            if zg.sum() >= gmin * npx:
                if hg >= max(mh, 0.5 * hv):
                    kind = "deposit"; dep[sl] |= zh
                elif hv >= mh and hg < 0.25 * hv:
                    kind = "burn"; brn[sl] |= zh & zv
                else:
                    kind = "none"
            elif hv >= mh and hg_tile is not None:
                if hg_tile >= max(mh, 0.5 * hv):
                    kind = "deposit"; dep[sl] |= zh
                elif hg_tile < 0.25 * hv:
                    kind = "burn"; brn[sl] |= zh & zv
                else:
                    kind = "ambiguous"; out["ambiguous"] += float(zh.sum()) / n_ok
            else:
                kind = "ambiguous" if hv >= mh else "none"
                if kind == "ambiguous":
                    out["ambiguous"] += float(zh.sum()) / n_ok
            zones.append({"zone": (i, j), "kind": kind, "open_ground_hit": round(hg, 3), "vegetation_hit": round(hv, 3)})
    out["deposit"] = round(float(dep.sum()) / n_ok, 4)
    out["burn"] = round(float(brn.sum()) / n_ok, 4)
    out["ambiguous"] = round(out["ambiguous"], 4)
    out["shadow"] = round(float(shadow.sum()) / n_ok, 4)
    compass = bool(getattr(cfg, "caption_compass", True))
    out["where_deposit"] = _where(_zone_mass_map(dep.astype(np.float32)), compass) if dep.any() else ""
    out["where_burn"] = _where(_zone_mass_map(brn.astype(np.float32)), compass) if brn.any() else ""
    # rev12: where each cover lies, as 3x3 shares, so the caption step can look there
    out["zones_deposit"] = _zones_list(_zone_mass_map(dep.astype(np.float32)))
    out["zones_burn"] = _zones_list(_zone_mass_map(brn.astype(np.float32)))
    out["zones"] = zones
    return out


@dataclass
class SceneEvidence:
    """Scene-level evidence, exactly the list the brief enumerates: ash coverage,
    vegetation loss, smoke regions, terrain deformation, river change, flood extent,
    burn scars, texture change, brightness change and the semantic transitions.

    Semantic fields come from RemoteCLIP group coverage (PRE and POST), physical fields
    from the unlearned RGB channels. They are kept in ONE object so DKG, the classifier
    and the caption prompt all read the same numbers.
    """
    values: Dict[str, float]
    transitions: List[SemanticTransition]
    group_cov_pre: Dict[str, float]
    group_cov_post: Dict[str, float]
    semantic: bool = True                    # False when use_transitions is off

    def __getitem__(self, k: str) -> float:
        return float(self.values.get(k, 0.0))

    def vector(self) -> np.ndarray:
        return np.array([self[f] for f in SCENE_FIELDS], np.float32)

    def top_groups(self, n: int = 5) -> List[Tuple[str, float]]:
        d = {g: self.group_cov_post.get(g, 0.0) - self.group_cov_pre.get(g, 0.0)
             for g in self.group_cov_post}
        return sorted(d.items(), key=lambda kv: -kv[1])[:n]

    @classmethod
    def build(cls, sem: SceneSemantics, physical: Dict[str, np.ndarray],
              scalars: Dict[str, float], buildings: Optional[np.ndarray] = None) -> "SceneEvidence":
        cov_pre = sem.group_cov_pre if sem.enabled else {g: 0.0 for g in SCENE_GROUPS}
        cov_post = sem.group_cov_post if sem.enabled else {g: 0.0 for g in SCENE_GROUPS}
        phys_mean = {k: float(v.mean()) for k, v in physical.items()}
        changed = [t for t in sem.transitions if t.changed]
        # Pixel water counts only above 0.15. Per pair on the 43-pair set, 0.15 catches
        # 10/14 floods with 1/29 false positives; 0.12 caught the same floods with 2/29.
        _pxw = float(phys_mean.get("newly_water", 0.0))
        _pxw = _pxw if _pxw > 0.15 else 0.0
        _semw = cov_post.get("flood", 0.0) + cov_post.get("sediment_water", 0.0)
        # rev10: water counts where it is CONFIRMED -- by the pixel test in that cell, or by
        # forming a connected body of RemoteCLIP water cells. rev9 required the green-cyan
        # pixel test everywhere, so brown floodwater could never count (Pakistan's water
        # is green; most floods are not), while the override made the two "tests" the same
        # test on overridden cells. Scattered single cells (vocabulary noise) do not count.
        _cfg = globals().get("CFG")
        _conf = _confirmed_water(sem, physical, _cfg)
        if _cfg is None or bool(getattr(_cfg, "water_requires_agreement", True)):
            _flood = _conf
        else:
            _flood = max(_semw, _pxw)

        v = {
            # sediment-laden water is water: it was added to ash for Guatemala's lahar,
            # but measured 0.000 on every Guatemala pair and 0.011 on Nepal's floods.
            "ash_coverage":        cov_post.get("ash", 0.0),
            "lava_coverage":       cov_post.get("lava", 0.0),
            "burn_scars":          cov_post.get("burn", 0.0),
            # was 0.000 on all 43 pairs (AUROC 0.50): now semantic flood + turbid water,
            # or the pixel test where the vocabulary still misses it
            "flood_extent":        _flood,
            "river_change":        float(sem.water_delta),
            # rev10: an expansion counts only as far as the new water is confirmed
            "water_expansion":     (min(max(0.0, float(sem.water_delta)), _conf)
                                    if _cfg is None or bool(getattr(_cfg, "water_requires_agreement", True))
                                    else max(0.0, float(sem.water_delta))),
            "smoke_regions":       cov_post.get("plume", 0.0),
            "cloud_obscuration":   cov_post.get("cloud", 0.0),
            "rubble_coverage":     cov_post.get("rubble", 0.0),
            "landslide_scars":     cov_post.get("landslide", 0.0),
            "bare_soil_gain":      max(0.0, cov_post.get("soil", 0.0) - cov_pre.get("soil", 0.0)),
            # the semantic drop in vegetation and the unlearned VARI channel measure the
            # same thing two independent ways; the stronger of the two is reported and
            # their disagreement is what RVG audits.
            "vegetation_loss":     max(float(sem.veg_delta), phys_mean.get("vegetation_loss", 0.0)),
            "terrain_deformation": phys_mean.get("texture_change", 0.0),
            "brightness_change":   phys_mean.get("brightening", 0.0),
            "darkening":           phys_mean.get("darkening", 0.0),
            "desaturation":        phys_mean.get("desaturation", 0.0),
            "texture_change":      phys_mean.get("texture_change", 0.0),
            "scene_change_coverage": float(scalars.get("scene_coverage", 0.0)),
            "anychange_coverage":  float(scalars.get("anychange_coverage", 0.0)),
            "transition_mass":     float(sum(t.confidence * t.weight for t in changed)),
        }
        _se = cls({k: float(v[k]) for k in SCENE_FIELDS}, changed, cov_pre, cov_post,
                   semantic=sem.enabled)
        _ep, _eq = phys_mean.get("edge_pre", 0.0), phys_mean.get("edge_post", 0.0)
        _se.extra = {"newly_water": _pxw, "flood_semantic": float(_semw), "water_confirmed": float(_conf),
                     "observable": float(getattr(sem, "obs_frac", 1.0)),
                     "edge_ratio": (_eq / _ep) if _ep > 1e-6 else 1.0,
                     "brightening": phys_mean.get("brightening", 0.0),
                     "darkening": phys_mean.get("darkening", 0.0),
                     "desaturation": phys_mean.get("desaturation", 0.0),
                     "texture_change": phys_mean.get("texture_change", 0.0),
                     "water_pre": float(getattr(sem, "water_pre", 0.0)),
                     "water_post": float(getattr(sem, "water_post", 0.0)),
                     "veg_pre": float(getattr(sem, "veg_pre", 0.0)),
                     "veg_post": float(getattr(sem, "veg_post", 0.0))}
        # rev11: the kind of darkening, with confirmed water and building footprints left out
        _water = None
        try:
            _cells = _confirmed_water(sem, physical, _cfg, return_cells=True)
            if not isinstance(_cells, float) and sem.main_grid is not None:
                _water = np.zeros(physical["darkening"].shape, bool)
                for _k, (x0, y0, x1, y1) in enumerate(sem.main_grid.boxes):
                    if _cells[_k]: _water[y0:y1, x0:x1] = True
            if "newly_water" in physical:
                _nw = np.asarray(physical["newly_water"]) >= 0.5
                _water = _nw if _water is None else (_water | _nw)
        except Exception:
            pass
        _dt = darkening_types(physical, _cfg, buildings=buildings, water=_water)
        _se.values["dark_deposit_cover"] = float(_dt["deposit"])
        _se.values["burn_cover"] = float(_dt["burn"])
        _se.extra["darkening_types"] = _dt
        return _se


@dataclass
class ObjectEvidence:
    """Object-level evidence: damaged / destroyed buildings, collapsed bridges, damaged
    and buried roads, the ChangeOS xBD grade histogram and the RemoteCLIP object labels."""
    values: Dict[str, float]
    counts: Dict[str, int]
    grade_hist: Dict[int, float]             # xBD grade -> pixel fraction
    labels: Dict[str, int]                   # RemoteCLIP POST label -> instance count
    transitions: List[SemanticTransition]
    n_instances: int

    def __getitem__(self, k: str) -> float:
        return float(self.values.get(k, 0.0))

    def vector(self) -> np.ndarray:
        return np.array([self[f] for f in OBJECT_FIELDS], np.float32)

    @classmethod
    def build(cls, obj: ObjectChangeOutput, instances: List["Instance"],
              cfg: Config) -> "ObjectEvidence":
        n = len(instances)
        thr = cfg.damage_change_thr
        c = Counter()
        labels = Counter()
        for i in instances:
            t = i.transition
            dst = t.dst_group if t else ""
            src = t.src_group if t else ""
            if t is not None:
                labels[t.dst_label] += 1
            damaged = (i.damage_grade or 0) >= 2 or i.change_score >= thr
            destroyed = (i.damage_grade or 0) >= 4 or dst == "rubble"
            if dst in BUILDING_GROUPS or src in BUILDING_GROUPS:
                if destroyed:
                    c["destroyed_buildings"] += 1
                elif damaged:
                    c["damaged_buildings"] += 1
            if src in BRIDGE_GROUPS or dst in BRIDGE_GROUPS:
                if dst == "broken_bridge" or damaged:
                    c["collapsed_bridges"] += 1
            if src in ROAD_GROUPS or dst in ROAD_GROUPS:
                if dst == "buried_road" or (src == "road" and dst in BURIAL_GROUPS):
                    c["buried_roads"] += 1
                elif damaged:
                    c["damaged_roads"] += 1
        c["damaged_infrastructure"] = (c["collapsed_bridges"] + c["damaged_roads"]
                                       + c["buried_roads"])

        grades = obj.damage_grades
        hist = {g: float((grades == g).mean()) for g in (1, 2, 3, 4)}
        sev = obj.damage_severity
        denom = float(max(n, 1))
        v = {
            "n_instances_norm":           min(1.0, n / 12.0),
            "damaged_buildings_frac":     c["damaged_buildings"] / denom,
            "destroyed_buildings_frac":   c["destroyed_buildings"] / denom,
            "collapsed_bridges_frac":     c["collapsed_bridges"] / denom,
            "damaged_roads_frac":         c["damaged_roads"] / denom,
            "buried_roads_frac":          c["buried_roads"] / denom,
            "damaged_infrastructure_frac": c["damaged_infrastructure"] / denom,
            "damaged_area_frac":          float(obj.damaged_area_frac),
            "mean_severity":              float(sev.mean()),
            "severity_in_buildings":      float(sev[obj.building_mask].mean())
                                          if obj.building_mask.any() else 0.0,
            "grade_minor":                hist[2],
            "grade_major":                hist[3],
            "grade_destroyed":            hist[4],
            "object_transition_mass":     float(sum(t.confidence * t.weight for i_ in instances
                                                    for t in ([i_.transition] if i_.transition
                                                              and i_.transition.changed else []))),
        }
        # Buildings counted from the ChangeOS footprint mask, each given the xBD level
        # (0 none .. 3 destroyed) of the most severe grade covering >= 30% of it. RSCC
        # references quote exactly this ("all 18 buildings remained intact, Level 0").
        c["buildings_detected"] = 0
        _pts: List[Tuple[float, float, int]] = []   # rev8: (cx, cy, xBD level) per building
        try:
            _bm = np.asarray(obj.building_mask, bool)
            _gr = np.asarray(obj.damage_grades)
            if _gr.shape != _bm.shape:
                _gr = resize_map(_gr.astype(np.float32), _bm.shape, nearest=True)
            if _bm.any():
                from scipy.ndimage import label as _cc_label
                _lab, _nbl = _cc_label(_bm)
                _sizes = np.bincount(_lab.ravel())
                _g = np.clip(np.asarray(_gr), 0, 4).astype(np.int64)
                _tab = np.bincount(_lab.ravel() * 5 + _g.ravel(),
                                   minlength=(_nbl + 1) * 5).reshape(_nbl + 1, 5)
                _min = max(12, int(5e-5 * _bm.size))
                _yy, _xx = np.indices(_bm.shape)
                _sy = np.bincount(_lab.ravel(), weights=_yy.ravel(), minlength=_nbl + 1)
                _sx = np.bincount(_lab.ravel(), weights=_xx.ravel(), minlength=_nbl + 1)
                for _j in range(1, _nbl + 1):
                    if _sizes[_j] < _min:
                        continue
                    _tot = max(int(_sizes[_j]), 1); _lvl = 0
                    for _gg in (4, 3, 2):
                        if _tab[_j, _gg:].sum() / _tot >= 0.30:
                            _lvl = _gg - 1; break
                    c[f"bld_level{_lvl}"] += 1
                    c["buildings_detected"] += 1
                    _pts.append((float(_sx[_j] / _tot / _bm.shape[1]),
                                 float(_sy[_j] / _tot / _bm.shape[0]), int(_lvl)))
        except Exception as _e:
            log.warning("building count failed: %s", _e)
        ev = cls({k: float(v[k]) for k in OBJECT_FIELDS}, dict(c), hist, dict(labels),
                 [i.transition for i in instances if i.transition is not None], n)
        # rev8: building centroids as tile fractions, so the caption checklist can say
        # WHERE the damaged buildings are. Not a feature: the classifier and the DKG read
        # `values` and `counts` only.
        ev.extra = {"buildings": _pts}
        return ev


## 6 · Evidence tokenisation and cross-branch attention (Part 6)

`EvidenceTokeniser` turns the two evidence objects into a common token space. Every token carries a
short natural-language phrase, a strength in `[0,1]`, a source tag, and a **RemoteCLIP text or image
embedding** — so scene tokens and object tokens live in the same vector space and are directly
comparable.

| Token kind | Built from | Embedding |
|---|---|---|
| scene | one per changed POST patch of the main grid | RemoteCLIP **image** embedding of that patch |
| object | one per SAM2 instance | RemoteCLIP **image** embedding of its POST crop |
| transition | one per distinct transition *type*, pooled by area share | RemoteCLIP **text** embedding of its phrase |
| physical | the five physical channels | RemoteCLIP **text** embedding of its phrase |

*(Rev1-rev5 of this table had the image and text columns swapped — the code was always right; the documentation was not. Which modality sits on each side is the whole justification for training-free cosine attention, so it matters.)*

`CrossBranchAttention` then computes a softmax over **cosine similarities** between scene tokens and
object tokens, weights each context vector by token strength, and fuses the result residually into a
single embedding `z`. There are no randomly initialised projection layers: the attention is
training-free, so it is meaningful on the very first image. When one branch has no tokens (an
ablation, or a scene with no detected objects) the attention degrades gracefully to the other
branch's mean rather than producing NaNs.

The two alignment scores it reports — `align_scene_to_object` and `align_object_to_scene` — are used
later by the RVG as a direct measure of whether the branches are describing the same event.


In [ ]:
PHYS_PHRASES = {
    # Describe what each channel MEASURES, never a cause. The perception diagnostic
    # showed the old causes were wrong on this data: darkening is highest on floods
    # (0.50) and ashfall (0.44) and never on a burn, yet read "burned or covered by
    # water"; brightening is high only on Turkish rubble (0.47) and read "ash or sediment".
    "vegetation_loss": "green vegetation has decreased",
    "brightening":     "the ground surface has become paler and brighter",
    "darkening":       "the ground surface has become darker",
    "desaturation":    "colours have faded toward grey",
    "texture_change":  "the surface has become rougher and more broken up",
}


@dataclass
class EvidenceToken:
    """One unit of evidence, projected into RemoteCLIP space (Part 6).

    Region tokens carry an IMAGE embedding of the region they describe; symbolic tokens
    (transitions, physical channels) carry a prompt-ensembled TEXT embedding. CLIP aligns
    both into one space, which is exactly why cosine attention across them is meaningful
    without training a single parameter.
    """
    kind: str                                  # scene | object | transition | physical
    name: str
    text: str                                  # natural-language form, for prompts and viz
    value: float                               # evidence strength in [0,1]
    embedding: np.ndarray                      # [D], L2-normalised
    where: Optional[Tuple[int, int, int, int]] = None
    meta: Dict[str, Any] = field(default_factory=dict)


class EvidenceTokeniser:
    """Turns every evidence source into tokens on one common embedding space."""

    def __init__(self, cfg: Config, clip: RemoteCLIPEncoder):
        self.cfg, self.clip = cfg, clip

    def _norm_values(self, vals: List[float]) -> List[float]:
        if not vals:
            return []
        m = max(vals)
        return [float(v / m) if m > 1e-8 else 0.0 for v in vals]

    def scene_tokens(self, sem: SceneSemantics) -> List[EvidenceToken]:
        """One token per changed POST patch, embedded with its own image embedding."""
        pg = sem.main_grid
        if pg is None:
            return []
        score = pg.mags * (1.0 - np.clip(pg.sims, 0.0, 1.0))
        order = np.argsort(-score)[:self.cfg.max_scene_tokens]
        order = [int(k) for k in order if score[k] > 1e-4]
        vals = self._norm_values([float(score[k]) for k in order])
        toks = []
        for v, k in zip(vals, order):
            toks.append(EvidenceToken(
                "scene", f"patch{k}", pg.post.labels[k], v, pg.post.embeds[k],
                where=pg.boxes[k],
                meta={"pre_label": pg.pre.labels[k], "post_label": pg.post.labels[k],
                      "sim": float(pg.sims[k]), "change": float(pg.mags[k]),
                      "pre_group": pg.pre.groups[k], "post_group": pg.post.groups[k]}))
        return toks

    def object_tokens(self, instances: List["Instance"],
                      embeds: np.ndarray) -> List[EvidenceToken]:
        """One token per SAM2 instance, embedded with its POST crop embedding."""
        idx = list(np.argsort([-i.refined_score for i in instances])[:self.cfg.max_object_tokens])
        vals = self._norm_values([float(instances[i].refined_score) for i in idx])
        toks = []
        for v, i in zip(vals, idx):
            inst = instances[i]
            e = embeds[i] if i < len(embeds) else np.zeros(self.clip.embed_dim, np.float32)
            if np.linalg.norm(e) < 1e-6:                 # instance had no valid crop
                continue
            grade = XBD_DAMAGE_CLASSES.get(inst.damage_grade or 0, "unknown")
            toks.append(EvidenceToken(
                "object", f"inst{i}", inst.label or "changed object", v, e, where=inst.bbox,
                meta={"grade": grade, "area": int(inst.area),
                      "change": float(inst.change_score),
                      "refined": float(inst.refined_score),
                      "transition": inst.transition.name if inst.transition else None}))
        return toks

    def transition_tokens(self, transitions: List[SemanticTransition]) -> List[EvidenceToken]:
        """One token per distinct transition TYPE, embedded from its natural-language
        phrase. Cells sharing a transition are pooled so the token weight is the total
        area share of that transition, not the count of cells."""
        agg: Dict[str, Dict[str, Any]] = {}
        for t in transitions:
            if not t.changed:
                continue
            a = agg.setdefault(t.name, {"w": 0.0, "conf": 0.0, "n": 0, "phrase": t.phrase(),
                                        "src": t.src_group, "dst": t.dst_group,
                                        "prob": 0.0, "where": t.where})
            a["w"] += t.weight
            a["conf"] = max(a["conf"], t.confidence)
            a["prob"] = max(a["prob"], t.transition_prob)
            a["n"] += 1
        items = sorted(agg.items(), key=lambda kv: -(kv[1]["w"] * kv[1]["conf"])
                       )[:self.cfg.max_transition_tokens]
        if not items:
            return []
        E = self.clip.encode_texts([a["phrase"] for _, a in items])
        vals = self._norm_values([a["w"] * a["conf"] for _, a in items])
        return [EvidenceToken("transition", name, a["phrase"], v, E[i], where=a["where"],
                              meta={"area_share": round(a["w"], 4), "cells": a["n"],
                                    "conf": round(a["conf"], 3), "prob": round(a["prob"], 3),
                                    "src": a["src"], "dst": a["dst"]})
                for i, ((name, a), v) in enumerate(zip(items, vals))]

    def physical_tokens(self, physical: Dict[str, np.ndarray]) -> List[EvidenceToken]:
        names = [n for n in ScenePhysicalChannels.NAMES if n in physical]
        if not names:
            return []
        E = self.clip.encode_texts([PHYS_PHRASES[n] for n in names])
        vals = [float(physical[n].mean()) for n in names]
        return [EvidenceToken("physical", n, PHYS_PHRASES[n], float(np.clip(v, 0, 1)), E[i],
                              meta={"channel_mean": round(v, 4)})
                for i, (n, v) in enumerate(zip(names, vals))]

    def __call__(self, sem: SceneSemantics, instances: List["Instance"],
                 obj_embeds: np.ndarray, physical: Dict[str, np.ndarray]
                 ) -> Dict[str, List[EvidenceToken]]:
        return {
            "scene": self.scene_tokens(sem),
            "object": self.object_tokens(instances, obj_embeds),
            "transition": self.transition_tokens(sem.transitions),
            "physical": self.physical_tokens(physical),
        }


@dataclass
class FusedTokens:
    tokens: Dict[str, List[EvidenceToken]]
    A_scene_to_object: np.ndarray        # [Ns, No] attention weights
    A_object_to_scene: np.ndarray        # [No, Ns]
    scene_repr: np.ndarray               # [D] scene summary after cross-attention
    object_repr: np.ndarray              # [D] object summary after cross-attention
    fused_embedding: np.ndarray          # [4D] concat summary, for inspection / probing
    z: np.ndarray                        # [D] the embedding the classifier consumes
    global_post: np.ndarray              # [D] whole-image POST embedding
    align_scene_to_object: float         # mean attended cosine, scene querying objects
    align_object_to_scene: float
    attention_map: np.ndarray            # HxW spatial attention received by scene regions
    enabled: bool = True

    def counts(self) -> Dict[str, int]:
        return {k: len(v) for k, v in self.tokens.items()}


class CrossBranchAttention:
    """
    Training-free cross-branch attention (Part 6).

        S = Q E_k^T                      cosine similarity, both sides L2-normalised
        A = softmax(S / tau)             tau = attention_temperature
        ctx_i = sum_j A_ij * v_j * E_j   value-weighted context, then renormalised

    There is no randomly initialised projection anywhere in this operator: the queries,
    keys and values are all RemoteCLIP embeddings, so an untrained pipeline still
    produces a meaningful alignment instead of noise. Scene evidence attends to object
    evidence and object evidence attends to scene evidence; the two attended contexts
    are added back as residuals, so the fused representation degrades gracefully to the
    single-branch summary when one branch is empty or disabled.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg

    @staticmethod
    def _stack(tokens: List[EvidenceToken], D: int) -> Tuple[np.ndarray, np.ndarray]:
        if not tokens:
            return np.zeros((0, D), np.float32), np.zeros(0, np.float32)
        E = np.stack([t.embedding for t in tokens]).astype(np.float32)
        n = np.linalg.norm(E, axis=1, keepdims=True)
        E = E / np.clip(n, 1e-8, None)
        return E, np.array([t.value for t in tokens], np.float32)

    @staticmethod
    def _softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
        x = x - x.max(axis=axis, keepdims=True)
        e = np.exp(x)
        return e / np.clip(e.sum(axis=axis, keepdims=True), 1e-12, None)

    @staticmethod
    def _unit(v: np.ndarray) -> np.ndarray:
        n = float(np.linalg.norm(v))
        return (v / n).astype(np.float32) if n > 1e-8 else np.asarray(v, np.float32)

    def _attend(self, Q: np.ndarray, K: np.ndarray, vk: np.ndarray):
        if Q.shape[0] == 0 or K.shape[0] == 0:
            return (np.zeros((Q.shape[0], K.shape[0]), np.float32),
                    np.zeros((Q.shape[0], Q.shape[1]), np.float32), 0.0)
        S = Q @ K.T
        A = self._softmax(S / max(self.cfg.attention_temperature, 1e-6), axis=1)
        ctx = A @ (vk[:, None] * K)
        align = float((A * S).sum(1).mean())
        return A.astype(np.float32), ctx.astype(np.float32), align

    def __call__(self, tokens: Dict[str, List[EvidenceToken]], global_post: np.ndarray,
                 hw: Tuple[int, int], enabled: bool = True) -> FusedTokens:
        cfg, D = self.cfg, int(global_post.shape[0])
        Es, vs = self._stack(tokens.get("scene", []), D)
        Eo, vo = self._stack(tokens.get("object", []), D)
        Et, vt = self._stack(tokens.get("transition", []), D)

        def wmean(E, v):
            if E.shape[0] == 0:
                return np.zeros(D, np.float32)
            w = v / max(float(v.sum()), 1e-8)
            return self._unit((w[:, None] * E).sum(0))

        scene_mean, object_mean, trans_mean = wmean(Es, vs), wmean(Eo, vo), wmean(Et, vt)

        if enabled:
            A_so, ctx_s, al_so = self._attend(Es, Eo, vo)
            A_os, ctx_o, al_os = self._attend(Eo, Es, vs)
        else:
            A_so = np.zeros((Es.shape[0], Eo.shape[0]), np.float32)
            A_os = np.zeros((Eo.shape[0], Es.shape[0]), np.float32)
            ctx_s = np.zeros((Es.shape[0], D), np.float32)
            ctx_o = np.zeros((Eo.shape[0], D), np.float32)
            al_so = al_os = 0.0

        r = cfg.attention_residual if enabled else 0.0
        scene_repr = self._unit(scene_mean + r * wmean(ctx_s, vs) if Es.shape[0] else scene_mean)
        object_repr = self._unit(object_mean + r * wmean(ctx_o, vo) if Eo.shape[0] else object_mean)

        # the embedding the learned head consumes: whole-image context, attended scene
        # evidence and attended object evidence, in one L2-normalised CLIP-space vector.
        z = self._unit(cfg.emb_w_global * self._unit(global_post)
                       + cfg.emb_w_scene * scene_repr
                       + cfg.emb_w_object * object_repr)
        fused = np.concatenate([self._unit(global_post), scene_repr, object_repr, trans_mean])

        # spatial map: how much attention each scene region received from the object branch
        amap = np.zeros(hw, np.float32)
        if A_os.size and tokens.get("scene"):
            recv = A_os.sum(0) / max(A_os.shape[0], 1)
            for t, r_ in zip(tokens["scene"], recv):
                if t.where is not None:
                    x0, y0, x1, y1 = t.where
                    amap[y0:y1, x0:x1] = np.maximum(amap[y0:y1, x0:x1], float(r_))
        elif tokens.get("scene"):
            for t in tokens["scene"]:
                if t.where is not None:
                    x0, y0, x1, y1 = t.where
                    amap[y0:y1, x0:x1] = np.maximum(amap[y0:y1, x0:x1], float(t.value))
        if amap.max() > 0:
            amap = amap / amap.max()

        return FusedTokens(tokens, A_so, A_os, scene_repr, object_repr, fused, z,
                           self._unit(global_post), al_so, al_os, amap, enabled=enabled)


## 7 · Multi-scale fusion + SDA v2

Unchanged from Stage 1 and deliberately so: fusion is the controlled variable in the Stage 1 vs
Stage 2 comparison. Convex weights `0.45 / 0.35 / 0.20` over object, AnyChange and physical
evidence, with a `max(fused, 0.9·object)` floor that protects graded xBD damage from being diluted
by a large low-confidence scene response, then the smooth Gaussian ambiguity gate of SDA v2.


In [ ]:
@dataclass
class FusedEvidence:
    fused: np.ndarray                    # HxW [0,1] multi-scale change field
    object_map: np.ndarray               # HxW [0,1] ChangeOS severity
    scene_map: np.ndarray                # HxW [0,1] AnyChange + physical
    anychange_map: np.ndarray            # HxW [0,1]
    physical: Dict[str, np.ndarray]
    scalars: Dict[str, float]


class MultiScaleFusion:
    """
    The single meeting point of the two branches.

    Fusion is a fixed convex combination, NOT a max. Max-fusion (the old FusedDetector)
    let whichever branch happened to saturate first overwrite the other, which destroyed
    the graded xBD semantics wherever AnyChange fired. A convex combination keeps both
    contributions recoverable and keeps the fused field on a stable, comparable scale.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        w = np.array([cfg.fuse_w_object, cfg.fuse_w_anychange, cfg.fuse_w_physical], np.float32)
        self.w = w / w.sum()

    def __call__(self, obj: ObjectChangeOutput, scene: SceneBranchOutput) -> FusedEvidence:
        o = obj.damage_severity.astype(np.float32)
        a = scene.change_prob.astype(np.float32)
        p = scene.physical["composite"].astype(np.float32)

        fused = self.w[0] * o + self.w[1] * a + self.w[2] * p
        # a confident single branch should still reach a decisive value, so take the
        # convex mixture but never below the strongest *object* evidence: graded
        # building damage is a hard fact and must not be diluted by empty surroundings.
        fused = np.clip(np.maximum(fused, o * 0.9), 0.0, 1.0)

        thr_s, thr_o = self.cfg.scene_change_thr, self.cfg.change_threshold
        scalars = {
            "object_damaged_area_frac": obj.damaged_area_frac,
            "object_severity_mean": float(o.mean()),
            "object_severity_in_buildings": float(o[obj.building_mask].mean())
                                            if obj.building_mask.any() else 0.0,
            "anychange_coverage": float((a > 0.05).mean()),
            "anychange_mean": float(a.mean()),
            "scene_coverage": float((scene.scene_map > thr_s).mean()),
            "scene_mean": float(scene.scene_map.mean()),
            "fused_coverage": float((fused > thr_o).mean()),
            "fused_mean": float(fused.mean()),
        }
        for name in ScenePhysicalChannels.NAMES:
            scalars[f"phys_{name}"] = float(scene.physical[name].mean())
        return FusedEvidence(fused, o, scene.scene_map, a, scene.physical, scalars)


class SubtleDamageAttention:
    """
    SDA v2 — continuous, smooth-gated subtle-change amplification.

        w(C)  = exp(-(C-mu)^2 / 2 sigma^2)        smooth ambiguity gate (no dead zone)
        A     = w(C) * (K * C) * E(pre, post)     coherence AND structure energy
        C'    = C + gamma * (1 - C) * A           saturating, monotone, C' <= 1

    See the markdown above for the four properties this buys. Parameter-free apart from
    (mu, sigma, gamma, K): it is a deterministic operator over real detector output, not
    a randomly initialised network dressed up as inference.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def _box(self, x, k):
        return F.avg_pool2d(x, k, stride=1, padding=k // 2)

    def _structure_energy(self, pre, post, hw):
        """Multi-scale bitemporal difference energy from the real image pair."""
        a = torch.from_numpy(pre.astype(np.float32) / 255).permute(2, 0, 1)[None]
        b = torch.from_numpy(post.astype(np.float32) / 255).permute(2, 0, 1)[None]
        maps = []
        for s in (1, 2, 4, 8):
            aa = F.avg_pool2d(a, s) if s > 1 else a
            bb = F.avg_pool2d(b, s) if s > 1 else b
            d = (aa - bb).pow(2).sum(1, keepdim=True).sqrt()
            d = F.interpolate(d, size=hw, mode="bilinear", align_corners=False)
            d = (d - d.amin()) / (d.amax() - d.amin() + 1e-6)
            maps.append(d)
        return torch.stack(maps).mean(0)

    def __call__(self, change: np.ndarray, pre: np.ndarray, post: np.ndarray):
        hw = change.shape
        C = torch.from_numpy(change.astype(np.float32))[None, None]
        E = self._structure_energy(pre, post, hw)
        K = self._box(C, self.cfg.sda_coherence_k)
        W = torch.exp(-((C - self.cfg.sda_mu) ** 2) / (2 * self.cfg.sda_sigma ** 2))
        A = W * K * E
        refined = torch.clamp(C + self.cfg.sda_gain * (1.0 - C) * A, 0.0, 1.0)
        return (refined[0, 0].numpy(), A[0, 0].numpy(), E[0, 0].numpy())

    @staticmethod
    def report(before: np.ndarray, after: np.ndarray, thr: float) -> Dict[str, float]:
        b, a = (before > thr).sum(), (after > thr).sum()
        return {"area_before": int(b), "area_after": int(a),
                "area_gain_pct": round(float(100.0 * (a / max(b, 1) - 1)), 2),
                "mean_before": round(float(before.mean()), 4),
                "mean_after": round(float(after.mean()), 4),
                "max_boost": round(float((after - before).max()), 4)}


class SubtleDamageAttentionLearned(nn.Module):
    """Trainable SDA for when xBD/RSCC supervision is available (kept for Stage 2)."""
    def __init__(self, in_ch: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(in_ch + 1, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1), nn.Sigmoid())

    def forward(self, feat, change_prob):
        a = self.gate(torch.cat([feat, change_prob], 1))
        return torch.clamp(change_prob + a * (1 - change_prob), 0, 1), a


## 8 · Dynamic Knowledge Graph v3 (Part 2)

The graph now reasons over **four evidence streams** instead of three:

| Source | Weight | Feeds |
|---|---|---|
| `physical` | 0.6 | unlearned RGB priors (ash brightening, vegetation loss, texture) |
| `transition` | 1.0 | `SemanticTransition` objects from both branches |
| `scene` | 0.9 | aggregated `SceneEvidence` (ash / flood / burn / landslide / water / smoke) |
| `object` | 1.0 | aggregated `ObjectEvidence` (destroyed buildings, buried roads, bridges, xBD grades) |

Every concept node records **which sources contributed and how much**, so `infer()` can report, per
hypothesis, both the supporting evidence and the **conflicting** evidence — the concepts that argue
against it — with provenance. `required` entries may be tuples read as OR, so `volcanic_eruption`
fires on ashfall **or** lava flow rather than demanding both.

Per-instance `structural_damage` mass was removed to stop the same building being counted twice now
that `ObjectEvidence` aggregates it; instances remain in the graph as nodes purely for
explainability. `infer()` returns a full ranking with a softmax over `dkg_temperature`, so the DKG
can act as one *probabilistic opinion* in the pool below rather than as a hard verdict.


In [ ]:
import networkx as nx

EVIDENCE_SOURCES = ("physical", "transition", "scene", "object")

# How far each evidence stream is trusted when concept mass is accumulated. The physical
# channels are unlearned priors, so they are deliberately trusted least; they exist to
# corroborate, not to decide.
SOURCE_WEIGHT = {"physical": 0.6, "transition": 1.0, "scene": 0.9, "object": 1.0}

# (src_group, dst_group) -> [(concept, weight), ...]   "*" is a wildcard
TRANSITION_CONCEPTS: Dict[Tuple[str, str], List[Tuple[str, float]]] = {
    ("vegetation", "ash"):            [("ashfall", 1.0), ("vegetation_loss", 1.0)],
    ("soil", "ash"):                  [("ashfall", 1.0), ("surface_change", 0.5)],
    ("built", "ash"):                 [("ashfall", 1.0), ("structural_damage", 0.3)],
    ("water", "ash"):                 [("ashfall", 0.8), ("sediment_load", 0.6)],
    ("*", "ash"):                     [("ashfall", 0.9)],

    ("vegetation", "lava"):           [("lava_flow", 1.0), ("vegetation_loss", 0.9)],
    ("*", "lava"):                    [("lava_flow", 1.0), ("terrain_change", 0.4)],

    ("vegetation", "burn"):           [("burned", 1.0), ("vegetation_loss", 1.0)],
    ("*", "burn"):                    [("burned", 0.9)],

    ("vegetation", "soil"):           [("vegetation_loss", 1.0), ("surface_change", 0.5),
                                       ("ashfall", 0.25)],  # lahar/ash deposit often reads as bare soil
    ("vegetation", "landslide"):      [("mass_movement", 1.0), ("vegetation_loss", 0.8),
                                       ("terrain_change", 0.8)],
    ("*", "landslide"):               [("mass_movement", 0.9), ("terrain_change", 0.6)],

    ("*", "flood"):                   [("inundation", 1.0)],
    ("vegetation", "water"):          [("inundation", 0.9), ("water_expansion", 0.8)],
    ("soil", "water"):                [("inundation", 0.8), ("water_expansion", 0.8)],
    ("built", "water"):               [("inundation", 1.0), ("structural_damage", 0.4)],
    ("water", "water_expanded"):      [("water_expansion", 1.0), ("inundation", 0.6)],
    ("water", "water_receded"):       [("surface_change", 0.5)],
    ("water", "sediment_water"):      [("sediment_load", 1.0), ("water_expansion", 0.4)],
    ("*", "sediment_water"):          [("sediment_load", 0.8), ("inundation", 0.5)],

    ("built", "rubble"):              [("structural_damage", 1.0), ("debris_scatter", 0.7)],
    ("*", "rubble"):                  [("structural_damage", 0.8), ("debris_scatter", 0.7)],
    ("built", "soil"):                [("structural_damage", 0.7), ("debris_scatter", 0.5)],
    ("road", "buried_road"):          [("infrastructure_damage", 1.0)],
    ("bridge", "broken_bridge"):      [("infrastructure_damage", 1.0), ("structural_damage", 0.5)],
    ("road", "*"):                    [("infrastructure_damage", 0.8)],
    ("bridge", "*"):                  [("infrastructure_damage", 1.0)],

    ("*", "plume"):                   [("plume", 1.0)],
    ("*", "cloud"):                   [("obscuration", 1.0)],
    ("water", "soil"):                [("surface_change", 0.6), ("terrain_change", 0.4)],
}


def transition_concepts(src: str, dst: str) -> List[Tuple[str, float]]:
    for key in ((src, dst), (src, "*"), ("*", dst)):
        if key in TRANSITION_CONCEPTS:
            return TRANSITION_CONCEPTS[key]
    return []


class DynamicKnowledgeGraph:
    """
    DKG v3 — a scene graph over FOUR independent evidence streams (Part 2).

    v2 reasoned over physical priors, semantic transitions and per-instance damage.
    v3 additionally ingests the aggregated SCENE evidence (ash coverage, burn scars,
    flood extent, river change, smoke, landslide scars, terrain deformation) and the
    aggregated OBJECT evidence (damaged / destroyed buildings, collapsed bridges,
    damaged and buried roads, xBD grade histogram), so a hypothesis can be raised by
    scene-scale evidence alone, by object-scale evidence alone, or by both agreeing.

    Per-instance structural damage is no longer added as concept mass -- the aggregated
    ObjectEvidence covers it -- so the two object routes cannot double count. The
    instances remain in the graph as nodes, which is what keeps the explanation legible.

    `required` entries may be a tuple, read as OR: a volcanic eruption is evidenced by
    ashfall OR a lava flow, and demanding both would make the rule unsatisfiable on a
    lava-dominated scene.
    """

    RULES = [
        dict(name="volcanic_eruption", w=0.92,
             required=[("ashfall", "lava_flow")],
             support=["plume", "vegetation_loss", "terrain_change", "surface_change",
                      "sediment_load", "infrastructure_damage"],
             veto=["burned", "inundation"]),
        dict(name="flood", w=0.88,
             required=[("inundation", "water_expansion")],
             support=["water_expansion", "sediment_load", "structural_damage",
                      "infrastructure_damage"],
             veto=["burned", "ashfall", "lava_flow"]),
        dict(name="wildfire", w=0.90,
             required=["burned"],
             support=["plume", "vegetation_loss"],
             veto=["ashfall", "inundation", "lava_flow"]),
        dict(name="earthquake", w=0.68,
             required=["structural_damage"],
             support=["infrastructure_damage", "debris_scatter", "surface_change",
                      "terrain_change", "debris_dust"],
             veto=["ashfall", "burned", "inundation", "lava_flow"]),
        dict(name="cyclone", w=0.72,
             required=["structural_damage", "vegetation_loss"],
             support=["inundation", "infrastructure_damage", "debris_scatter",
                      "debris_dust"],
             veto=["ashfall", "burned", "mass_movement", "lava_flow"]),
        dict(name="landslide", w=0.85,
             required=["mass_movement"],
             support=["vegetation_loss", "terrain_change", "infrastructure_damage",
                      "debris_scatter"],
             veto=["burned"]),
        dict(name="tsunami", w=0.80,
             required=["inundation", "debris_scatter"],
             support=["structural_damage", "water_expansion", "sediment_load"],
             veto=["burned", "ashfall", "lava_flow"]),
    ]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        if not cfg.phys_concept_baseline:
            log.warning("DKG physical priors are UNCALIBRATED (phys_concept_baseline is "
                        "empty). inundation = mean(dark x (1-texture)) is structurally "
                        "3-15x ashfall and burned because (1-texture) is near 1 "
                        "everywhere, so `flood` wins the required term by default. Run "
                        "calibrate_physical_baselines() before reporting any DKG number.")
        names = [r["name"] for r in self.RULES]
        missing = set(DISASTER_CLASSES) - set(names)
        if missing:
            raise ValueError(f"DKG rules do not cover {missing}")
        self.order = [names.index(d) for d in DISASTER_CLASSES]

    # -- evidence extraction -------------------------------------------------
    def raw_physical_concepts(self, phys: Dict[str, np.ndarray]) -> Dict[str, float]:
        """Interpretable priors read off the RGB change channels. Each is a
        conjunction, so a single channel alone cannot manufacture a concept.

        These are NOT comparable to each other as they stand (ISSUE 8): `inundation`
        multiplies one mid-sized map by `(1 - texture)`, whose mean is about 0.85, while
        `ashfall` and `burned` multiply two small maps. Use `physical_concepts`, which
        divides each by its measured baseline."""
        veg, bright = phys["vegetation_loss"], phys["brightening"]
        dark, desat, tex = phys["darkening"], phys["desaturation"], phys["texture_change"]
        neutral = phys.get("post_neutrality")
        vegm = phys.get("veg_mask_pre")

        # ASHFALL -- ash settles as a DARK, colourless blanket. The old prior asked for
        # `bright AND colourless`, which is pale concrete dust, i.e. earthquake rubble;
        # measured on this dataset it scored EQ 0.0260 vs Tonga 0.0041. Asking for
        # `dark AND colourless POST surface` reverses that: Tonga 0.0520, EQ 0.0270.
        if neutral is not None:
            gate = (np.maximum(neutral, veg)
                    if bool(getattr(self.cfg, "phys_ash_vegloss_path", True)) else neutral)
            ashfall = float((dark * gate).mean())
        else:
            ashfall = float((bright * desat).mean())       # fallback = old behaviour

        # DEBRIS_DUST -- the signal the old ashfall prior was actually measuring. It is
        # evidence FOR structural collapse, so it is kept and routed to the earthquake
        # side instead of being thrown away.
        # rev10: bright AND colourless over formerly VEGETATED ground is a pale cover on
        # vegetation (ash, dust, sediment, drying) -- not rubble dust; it now goes to the
        # neutral surface_change instead of supporting earthquake. Over built or bare
        # ground it stays debris_dust.
        if vegm is not None:
            debris_dust = float((bright * desat * (1.0 - vegm)).mean())
            pale_veg = float((bright * desat * vegm).mean())
        else:
            debris_dust, pale_veg = float((bright * desat).mean()), 0.0

        # BURNED -- fire darkens VEGETATION specifically; ashfall darkens everything, and
        # on these tiles coats roofs and roads MORE than vegetation (Tonga: non-veg
        # -0.119 vs veg -0.038). Gating on that selectivity removes the volcano->wildfire
        # confusion. `phys_burn_selectivity` = 0 restores the ungated prior.
        burned = float((dark * veg).mean())
        w_sel = float(getattr(self.cfg, "phys_burn_selectivity", 1.0))
        if vegm is not None and w_sel > 0:
            net = dark - bright                      # >0 where the surface darkened
            nv = 1.0 - vegm
            sel = (float((net * vegm).sum() / max(float(vegm.sum()), 1.0))
                   - float((net * nv).sum() / max(float(nv.sum()), 1.0)))
            gate = float(np.clip(sel / max(float(getattr(self.cfg, "phys_burn_sel_ref", 0.05)), 1e-6), 0, 1))
            burned *= (1.0 - w_sel) + w_sel * gate

        return {
            "ashfall":        ashfall,
            "debris_dust":    debris_dust,
            "burned":         burned,
            "inundation":     float((dark * (1.0 - tex)).mean()),    # dark AND smooth
            "vegetation_loss": float(veg.mean()),
            "terrain_change": float(tex.mean()),
            "surface_change": float((bright * (1.0 - desat)).mean()) + pale_veg,
        }

    def physical_concepts(self, phys: Dict[str, np.ndarray]) -> Dict[str, float]:
        """The raw priors rescaled to EXCESS OVER TYPICAL, so they can be compared.

        With a baseline b measured over a pair sample, a prior p becomes
        clip((p - b) / b, 0, 1): zero at or below what an ordinary pair produces, one at
        twice it. Without a baseline the raw value is passed through unchanged and
        __init__ has already warned that the comparison is not meaningful."""
        raw = self.raw_physical_concepts(phys)
        base = self.cfg.phys_concept_baseline
        if not base:
            return raw
        out = {}
        for k, v in raw.items():
            b = float(base.get(k, 0.0))
            out[k] = float(np.clip((v - b) / b, 0.0, 1.0)) if b > 1e-9 else float(v)
        return out

    def _squash(self, m: float) -> float:
        k = self.cfg.concept_saturation
        return float(m / (m + k)) if m > 0 else 0.0

    # -- graph ---------------------------------------------------------------
    def build(self, fused: FusedEvidence, transitions: List[SemanticTransition],
              instances: List["Instance"], scene_ev: Optional[SceneEvidence] = None,
              obj_ev: Optional[ObjectEvidence] = None):
        G = nx.DiGraph()
        G.add_node("SCENE", kind="scene",
                   scene_coverage=round(fused.scalars["scene_coverage"], 3),
                   object_damage=round(fused.scalars["object_damaged_area_frac"], 4),
                   fused_mean=round(fused.scalars["fused_mean"], 3))

        concepts: Dict[str, Dict[str, float]] = {}
        provenance: Dict[str, List[Tuple[str, str, float]]] = defaultdict(list)

        def add(concept: str, source: str, mass: float, from_node: Optional[str] = None,
                why: str = ""):
            if mass <= 0:
                return
            m = mass * SOURCE_WEIGHT[source]
            entry = concepts.setdefault(concept, {s: 0.0 for s in EVIDENCE_SOURCES})
            # ISSUE 8 (companion): a plain sum is double counting. `structural_damage`
            # receives damaged_buildings_frac, destroyed_buildings_frac AND
            # damaged_area_frac -- three views of the SAME buildings -- so on the
            # earthquake pair its total reached 1.064, and the docstring's claim that a
            # mass is "on the same [0,1] scale as the coverage fractions" was false.
            # Noisy-OR combines repeated evidence for one concept from one source as
            # 1 - prod(1 - m_i): it is monotone, it saturates at 1, and two views of one
            # building can no longer add up to more than one building.
            if self.cfg.dkg_accumulate == "noisy_or":
                entry[source] = 1.0 - (1.0 - entry[source]) * (1.0 - min(m, 1.0))
            else:
                entry[source] += m
            provenance[concept].append((source, why or (from_node or source), round(m, 4)))
            cnode = f"concept:{concept}"
            if cnode not in G:
                G.add_node(cnode, kind="concept")
            if from_node:
                G.add_edge(from_node, cnode, rel="supports", weight=round(m, 3))

        # (1) physical priors -------------------------------------------------
        for name, mass in self.physical_concepts(fused.physical).items():
            nid = f"phys:{name}"
            G.add_node(nid, kind="physical_prior", mass=round(mass, 4))
            G.add_edge("SCENE", nid, rel="measures")
            add(name, "physical", mass, nid, why=f"physical channel {name}={mass:.3f}")

        # (2) aggregated SCENE evidence (new in v3) ---------------------------
        if scene_ev is not None:
            for fieldname, targets in SCENE_EVIDENCE_CONCEPTS.items():
                mass = max(0.0, scene_ev[fieldname])
                if mass <= 1e-4:
                    continue
                nid = f"scene:{fieldname}"
                G.add_node(nid, kind="scene_evidence", mass=round(mass, 4))
                G.add_edge("SCENE", nid, rel="measures")
                for concept, cw in targets:
                    add(concept, "scene", cw * mass, nid,
                        why=f"scene {fieldname}={mass:.3f}")

        # (3) semantic transitions -------------------------------------------
        for i, t in enumerate(transitions):
            if not t.changed:
                continue
            nid = f"trans{i}:{t.src_group}->{t.dst_group}"
            G.add_node(nid, kind="transition", level=t.level, src=t.src_group, dst=t.dst_group,
                       conf=round(t.confidence, 3), weight=round(t.weight, 4),
                       prob=round(t.transition_prob, 3))
            G.add_edge("SCENE", nid, rel="observes")
            for concept, cw in transition_concepts(t.src_group, t.dst_group):
                # t.weight is the area share of the cell, so summing over cells gives
                # "fraction of the scene showing this evidence" -- a mass on the same
                # [0,1] scale as the coverage fractions and the object fractions.
                add(concept, "transition", cw * t.confidence * t.weight, nid,
                    why=f"{t.name} over {100*t.weight:.1f}% of the scene")

        # (4) aggregated OBJECT evidence (new in v3) --------------------------
        if obj_ev is not None:
            for fieldname, targets in OBJECT_EVIDENCE_CONCEPTS.items():
                mass = max(0.0, obj_ev[fieldname])
                if mass <= 1e-4:
                    continue
                nid = f"objev:{fieldname}"
                G.add_node(nid, kind="object_evidence", mass=round(mass, 4),
                           count=obj_ev.counts.get(fieldname.replace("_frac", ""), 0))
                G.add_edge("SCENE", nid, rel="measures")
                for concept, cw in targets:
                    add(concept, "object", cw * mass, nid,
                        why=f"object {fieldname}={mass:.3f}")

        # (5) object instances: nodes for explainability, transition mass only --
        n_obj = max(len(instances), 1)
        for i, inst in enumerate(instances):
            nid = f"obj{i}"
            attrs = dict(kind="object", label=inst.label, change=round(inst.change_score, 3),
                         conf=round(inst.label_conf, 3), area=int(inst.area))
            if inst.transition is not None:
                attrs["transition"] = inst.transition.name
            if inst.damage_grade is not None:
                attrs["damage"] = XBD_DAMAGE_CLASSES.get(inst.damage_grade, "unknown")
            G.add_node(nid, **attrs)
            G.add_edge("SCENE", nid, rel="contains")
            if inst.change_score < self.cfg.damage_change_thr:
                continue
            if inst.transition is not None and inst.transition.changed:
                for concept, cw in transition_concepts(inst.transition.src_group,
                                                       inst.transition.dst_group):
                    add(concept, "object",
                        cw * inst.transition.confidence * inst.change_score / n_obj, nid,
                        why=f"instance {i} {inst.transition.name}")

        self._provenance = dict(provenance)
        return G, concepts

    # -- inference -----------------------------------------------------------
    @staticmethod
    def _required_mass(total: Dict[str, float], req) -> Tuple[float, str]:
        """`req` is a concept name or a tuple read as OR; returns (mass, winning name)."""
        alts = (req,) if isinstance(req, str) else tuple(req)
        best = max(alts, key=lambda c: total.get(c, 0.0))
        return float(total.get(best, 0.0)), best

    def infer(self, G, concepts: Dict[str, Dict[str, float]]):
        total = {c: sum(v.values()) for c, v in concepts.items()}
        prov = getattr(self, "_provenance", {})
        ranking = []
        # ISSUE 8: `m <= 0.0` was never true, because every physical prior is strictly
        # positive on any image pair -- so no rule was ever ruled out and no veto ever
        # cleared. A concept must now carry at least `concept_floor` mass to count as
        # present, for `required` and for `veto` alike.
        floor = self.cfg.concept_floor
        for rule in self.RULES:
            req_masses, req_names, req_raw = [], [], []
            for r in rule["required"]:
                m, name = self._required_mass(total, r)
                req_raw.append(m)
                req_masses.append(self._squash(m)); req_names.append(name)
            missing = [n for n, m in zip(req_names, req_raw) if m < floor]
            sup = [self._squash(total.get(c, 0.0)) for c in rule["support"]]
            veto_hits = [(c, round(total.get(c, 0.0), 3)) for c in rule["veto"]
                         if total.get(c, 0.0) >= floor]
            veto_mass = sum(self._squash(total.get(c, 0.0)) for c in rule["veto"]
                            if total.get(c, 0.0) >= floor)
            supporting = [{"concept": n, "mass": round(total.get(n, 0.0), 3),
                           "by_source": {s: round(m, 3)
                                         for s, m in concepts.get(n, {}).items() if m > 0},
                           "role": "required"} for n in req_names if total.get(n, 0.0) > 0]
            supporting += [{"concept": c, "mass": round(total.get(c, 0.0), 3),
                            "by_source": {s: round(m, 3)
                                          for s, m in concepts.get(c, {}).items() if m > 0},
                            "role": "support"}
                           for c in rule["support"] if total.get(c, 0.0) > 0]
            entry = {"disaster": rule["name"],
                     "supporting": supporting,
                     "conflicting": [{"concept": c, "mass": m} for c, m in veto_hits]}
            if missing:
                entry.update(score=0.0, required=0.0, support=0.0, veto=1.0,
                             reason=f"required concept(s) below the {floor:.2f} floor: "
                                    + ",".join(missing))
            else:
                req_term = float(min(req_masses))          # weakest link limits the claim
                sup_term = float(np.mean(sup)) if sup else 0.0
                veto_term = 1.0 / (1.0 + veto_mass)
                entry.update(score=round(float(rule["w"] * req_term
                                               * (0.75 + 0.25 * sup_term) * veto_term), 3),
                             required=round(req_term, 3), support=round(sup_term, 3),
                             veto=round(veto_term, 3), reason="")
            ranking.append(entry)

        by_name = {r["disaster"]: r for r in ranking}
        scores = np.array([by_name[d]["score"] for d in DISASTER_CLASSES], np.float32)
        if scores.max() <= 0:
            probs = np.full(len(DISASTER_CLASSES), 1.0 / len(DISASTER_CLASSES), np.float32)
        else:
            z = (scores - scores.max()) / max(self.cfg.dkg_temperature, 1e-6)
            e = np.exp(z); probs = (e / e.sum()).astype(np.float32)
        for i, d in enumerate(DISASTER_CLASSES):
            by_name[d]["prob"] = round(float(probs[i]), 4)

        ranking.sort(key=lambda r: -r["score"])
        best = ranking[0]
        margin = best["score"] - (ranking[1]["score"] if len(ranking) > 1 else 0.0)
        disaster = best["disaster"] if best["score"] > 0 else "undetermined"
        matched = []
        if disaster != "undetermined":
            matched = [s["concept"] for s in best["supporting"]]
            G.add_node(f"disaster:{disaster}", kind="disaster", score=best["score"])
            for c in matched:
                if f"concept:{c}" in G:
                    G.add_edge(f"concept:{c}", f"disaster:{disaster}", rel="implies")

        return {
            "disaster": disaster,
            "score": best["score"] if disaster != "undetermined" else 0.0,
            "margin": round(float(margin), 3),
            "matched_concepts": sorted(set(matched)),
            "ranking": ranking,
            "top_k": [{"disaster": r["disaster"], "score": r["score"], "prob": r["prob"],
                       "supporting": r["supporting"][:5], "conflicting": r["conflicting"]}
                      for r in ranking[:3]],
            "probs": {d: round(float(probs[i]), 4) for i, d in enumerate(DISASTER_CLASSES)},
            "prob_vector": probs,
            "concept_mass": {c: round(v, 3) for c, v in sorted(total.items(), key=lambda kv: -kv[1])},
            "concept_by_source": {c: {s: round(m, 3) for s, m in v.items() if m > 0}
                                  for c, v in concepts.items()},
            "concept_provenance": {c: v[:6] for c, v in prov.items()},
            "obscuration_mass": round(float(total.get("obscuration", 0.0)), 3),
        }

    @staticmethod
    def null_output() -> dict:
        """What `use_dkg=False` returns: an uninformative, correctly shaped opinion."""
        p = np.full(len(DISASTER_CLASSES), 1.0 / len(DISASTER_CLASSES), np.float32)
        return {"disaster": "undetermined", "score": 0.0, "margin": 0.0, "matched_concepts": [],
                "ranking": [], "top_k": [],
                "probs": {d: float(p[i]) for i, d in enumerate(DISASTER_CLASSES)},
                "prob_vector": p, "concept_mass": {}, "concept_by_source": {},
                "concept_provenance": {}, "obscuration_mass": 0.0, "disabled": True}


def concept_vector(dkg_out: dict) -> np.ndarray:
    """The concept masses as a fixed-length vector, so the classifier reads exactly the
    evidence the graph reasoned over."""
    m = dkg_out.get("concept_mass", {})
    return np.array([float(m.get(c, 0.0)) for c in CONCEPTS], np.float32)


## 9 · Learned disaster classifier + calibrated fusion (Part 3)

**The DKG is not replaced.** It is pooled with exactly one CLIP-derived source.

`ClassifierFeatures` = the attention-fused embedding `z` (RemoteCLIP dimension, frozen) concatenated
with a fixed-layout tabular vector `t` built from `SceneEvidence`, `ObjectEvidence`, the physical
channels, the DKG concept mass and the attention fields (see `build_features`). Nothing in the backbone is fine-tuned; only the
head trains.

```
logits = logit_scale · (z · Pᵀ)  +  MLP([t ; z])
```

`P` are frozen RemoteCLIP text prototypes of the seven disaster descriptions and the MLP's final
layer is **zero-initialised**, so at construction `logits` equal the zero-shot logits exactly. Two
consequences: the classifier is useful before any training, and training is a well-posed residual
problem rather than learning a disaster classifier from a few hundred images.

**Log-linear opinion pooling of TWO sources.** The symbolic DKG is pooled with one CLIP-derived
opinion, in log space:

```
p(d) ∝ Π_s  p_s(d) ^ w_s          s ∈ {DKG, one CLIP source}
```

A geometric pool assumes its inputs are conditionally independent. The CLIP prior (whole-image POST
embedding vs the seven prototypes) and the head are **not** independent: the untrained head *is* the
prior (zero-initialised residual), and the trained head's input `z` shares a 40% whole-image
component with it. Pooling both would count one computation twice and overstate confidence, so the
pool carries only one of them at a time:

| head state | DKG | CLIP prior | head |
|---|---|---|---|
| untrained (default) | 0.65 | 0.35 | computed, not pooled |
| trained (§18) | 0.40 | computed, not pooled | 0.60 |

All three are still computed every run, and `DisasterPosterior.dependence` reports their pairwise
rank correlation, JSD and top-1 agreement, so the duplication is measured rather than asserted.
`cross_validate_leave_event_out` (§19) aggregates it over a dataset. The weights are fixed priors, not
tuned values; a sensitivity sweep belongs with the dataset run.

`DisasterPosterior` carries the pooled probabilities, top-k, margin, entropy and each source's own
verdict — the margin and entropy are what the RVG reads as a calibration signal.


In [ ]:
# Natural-language prototypes for the seven target types. These are what the head is
# INITIALISED from, so an untrained head is a calibrated zero-shot classifier rather
# than noise -- the same reason the attention uses CLIP cosines instead of random
# projections.
# STATE prompts — what the POST tile LOOKS LIKE from directly overhead, after the event.
# Rewritten from ground-level photographer wording to nadir satellite appearance,
# because what actually gets scored is a nadir POST tile.
DISASTER_PROMPTS: Dict[str, List[str]] = {
    "volcanic_eruption": ["an aerial view of grey volcanic ash and mud covering the ground",
                          "a satellite image of grey ash deposit burying vegetation and roads",
                          "overhead view of a grey-brown volcanic mudflow across a valley"],
    "flood":             ["an aerial view of brown muddy water covering land and fields",
                          "a satellite image of standing flood water over roads and buildings",
                          "overhead view of a swollen river spreading across the floodplain"],
    "wildfire":          ["an aerial view of black charred ground after a fire",
                          "a satellite image of a dark burn scar across vegetation",
                          "overhead view of blackened burnt forest"],
    "earthquake":        ["an aerial view of collapsed buildings and rubble in a town",
                          "a satellite image of a damaged built-up area with debris and intact vegetation",
                          "overhead view of a dense urban block reduced to rubble"],
    "cyclone":           ["an aerial view of wind-damaged roofs and scattered debris",
                          "a satellite image of a coastal town with flattened trees and storm damage",
                          "overhead view of damaged buildings and flooded farmland after a storm"],
    "landslide":         ["an aerial view of a bare brown soil scar on a vegetated slope",
                          "a satellite image of a debris flow running down a hillside",
                          "overhead view of mud and soil burying a road on a slope"],
    "tsunami":           ["an aerial view of a flooded coastline with scattered debris",
                          "a satellite image of a coastal strip covered by water and wreckage",
                          "overhead view of coastal buildings washed away leaving standing water"],
}

# TRANSITION prompts — what CHANGED between PRE and POST. Scored against the PRE->POST
# embedding direction, which makes this reasoner bi-temporal instead of reading the POST
# tile alone. A grey valley is ambiguous; "vegetation replaced by grey ash" is not.
DISASTER_TRANSITION_PROMPTS: Dict[str, List[str]] = {
    "volcanic_eruption": ["green vegetation replaced by grey volcanic ash and mud",
                          "clear ground buried under a grey ash and mudflow deposit",
                          "a river and valley filled in by grey volcanic debris"],
    "flood":             ["dry land replaced by standing brown water",
                          "fields and roads submerged under spreading flood water",
                          "a river widening to cover the surrounding land"],
    "wildfire":          ["green forest replaced by black burnt ground",
                          "healthy vegetation turning into a dark burn scar",
                          "vegetated land becoming charred and blackened"],
    "earthquake":        ["intact buildings replaced by rubble and debris",
                          "a built-up block collapsing into debris while vegetation stays green",
                          "standing structures reduced to broken debris"],
    "cyclone":           ["intact roofs and trees replaced by scattered storm debris",
                          "buildings damaged and vegetation flattened by wind",
                          "a coastal settlement becoming damaged and partly flooded"],
    "landslide":         ["a vegetated hillside replaced by a bare soil scar",
                          "slope vegetation stripped away by a descending debris flow",
                          "a road buried by soil sliding down a slope"],
    "tsunami":           ["a dry coastline replaced by water and floating debris",
                          "coastal buildings swept away leaving standing water",
                          "shoreline land inundated and littered with wreckage"],
}

ATTENTION_FIELDS: Tuple[str, ...] = (
    "align_scene_to_object", "align_object_to_scene", "n_scene_tokens", "n_object_tokens")


def tabular_layout(use_concepts: Optional[bool] = None
                   ) -> Tuple[Dict[str, Tuple[int, int]], int]:
    """ISSUE 10: the DKG concept-mass block used to be 15 of these features, so a
    TRAINED head read the graph it is pooled against and the two sources stopped being
    separate. `Config.clf_use_concepts` controls it; default False."""
    if use_concepts is None:
        use_concepts = bool(getattr(CFG, "clf_use_concepts", False))
    blocks = [("scene", len(SCENE_FIELDS)), ("object", len(OBJECT_FIELDS)),
              ("physical", len(PHYS_FIELDS))]
    if use_concepts:
        blocks.append(("concepts", len(CONCEPTS)))
    blocks.append(("attention", len(ATTENTION_FIELDS)))
    layout, i = {}, 0
    for name, n in blocks:
        layout[name] = (i, i + n); i += n
    return layout, i


TAB_LAYOUT, TAB_DIM = tabular_layout()


@dataclass
class ClassifierFeatures:
    """The frozen-feature bundle the learned head consumes.

    `z` is a RemoteCLIP-space embedding (attention-fused scene + object + global);
    `t` is the interpretable tabular vector (scene evidence, object evidence, physical
    channels, DKG concept mass, attention alignment). Keeping the two separate is what
    lets the head be text-initialised on `z` while learning a residual on `t`.
    """
    z: np.ndarray
    t: np.ndarray
    global_post: np.ndarray
    scene_repr: np.ndarray
    object_repr: np.ndarray
    layout: Dict[str, Tuple[int, int]] = field(default_factory=lambda: dict(TAB_LAYOUT))

    def block(self, name: str) -> np.ndarray:
        a, b = self.layout[name]
        return self.t[a:b]

    def masked(self, *blocks: str, drop_embedding: Optional[str] = None) -> "ClassifierFeatures":
        """A copy with whole evidence blocks zeroed — the per-image contribution probe."""
        t = self.t.copy()
        for b in blocks:
            if b not in self.layout:             # "concepts" is absent by default
                continue
            a, c = self.layout[b]
            t[a:c] = 0.0
        z = self.z
        if drop_embedding in ("scene", "object"):
            keep = self.object_repr if drop_embedding == "scene" else self.scene_repr
            w = CFG.emb_w_object if drop_embedding == "scene" else CFG.emb_w_scene
            v = CFG.emb_w_global * self.global_post + w * keep
            n = float(np.linalg.norm(v))
            z = (v / n).astype(np.float32) if n > 1e-8 else v.astype(np.float32)
        return ClassifierFeatures(z, t, self.global_post, self.scene_repr,
                                  self.object_repr, dict(self.layout))


def build_features(scene_ev: SceneEvidence, obj_ev: ObjectEvidence,
                   physical: Dict[str, np.ndarray], dkg_out: dict,
                   fused_tokens: FusedTokens, cfg: Config) -> ClassifierFeatures:
    counts = fused_tokens.counts()
    attn = np.array([
        float(np.clip((fused_tokens.align_scene_to_object + 1) / 2, 0, 1)),
        float(np.clip((fused_tokens.align_object_to_scene + 1) / 2, 0, 1)),
        min(1.0, counts.get("scene", 0) / max(cfg.max_scene_tokens, 1)),
        min(1.0, counts.get("object", 0) / max(cfg.max_object_tokens, 1)),
    ], np.float32)
    parts = [scene_ev.vector(), obj_ev.vector(),
             np.array([float(physical[n].mean()) for n in PHYS_FIELDS], np.float32)]
    if "concepts" in TAB_LAYOUT:                 # ISSUE 10: off by default
        parts.append(concept_vector(dkg_out))
    parts.append(attn)
    t = np.concatenate(parts).astype(np.float32)
    if t.shape[0] != TAB_DIM:
        raise ValueError(f"tabular feature length {t.shape[0]} != TAB_DIM {TAB_DIM}")
    return ClassifierFeatures(fused_tokens.z.astype(np.float32), t,
                              fused_tokens.global_post.astype(np.float32),
                              fused_tokens.scene_repr.astype(np.float32),
                              fused_tokens.object_repr.astype(np.float32))


class DisasterClassifierHead(nn.Module):
    """
    A lightweight, TEXT-INITIALISED classifier head (Part 3).

        logits = s * (z . P^T)  +  MLP([t ; z])

    `P` are the frozen RemoteCLIP text prototypes of the seven disaster descriptions and
    the MLP's last layer is zero-initialised, so at construction

        logits == zero-shot logits

    exactly. Training on labelled pairs therefore learns a RESIDUAL over a sensible
    prior instead of climbing out of a random initialisation, which is what makes a head
    trainable on the few hundred labelled pairs a capstone can realistically assemble.
    Every foundation model stays frozen; only this head has gradients.
    """
    def __init__(self, prototypes: np.ndarray, tab_dim: int, emb_dim: int,
                 hidden: int = 128, dropout: float = 0.1,
                 classes: Tuple[str, ...] = DISASTER_CLASSES):
        super().__init__()
        self.classes = tuple(classes)
        self.register_buffer("proto", torch.as_tensor(prototypes, dtype=torch.float32))
        self.logit_scale = nn.Parameter(torch.tensor(math.log(20.0)))
        self.temperature = nn.Parameter(torch.tensor(1.0), requires_grad=False)
        self.norm = nn.LayerNorm(tab_dim + emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim + emb_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, len(self.classes)))
        nn.init.zeros_(self.mlp[-1].weight); nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, z: torch.Tensor, t: torch.Tensor):
        zs = self.logit_scale.exp().clamp(max=100.0) * (z @ self.proto.T)
        residual = self.mlp(self.norm(torch.cat([t, z], -1)))
        return zs + residual, zs, residual


class LearnedDisasterClassifier:
    """Trains and serves the head. Foundation models are frozen; `fit` touches only
    `DisasterClassifierHead`. With no labelled data the head is still usable — it is
    then exactly the zero-shot prototype classifier, and `is_trained` says so, which is
    what the evidence-pooling weights key off."""

    def __init__(self, cfg: Config, clip: RemoteCLIPEncoder):
        self.cfg, self.clip = cfg, clip
        self.classes = DISASTER_CLASSES
        proto = np.stack([self._prototype(d) for d in self.classes])
        self.prototypes = proto
        self.transition_prototypes = np.stack(
            [self._transition_prototype(d) for d in self.classes])
        self.head = DisasterClassifierHead(proto, TAB_DIM, clip.embed_dim,
                                           cfg.clf_hidden, cfg.clf_dropout,
                                           self.classes).to(cfg.device).eval()
        self.is_trained = False
        self.history: Dict[str, Any] = {}
        if Path(cfg.clf_ckpt).is_file():
            try:
                self.load(cfg.clf_ckpt)
                log.info("Disaster head: loaded trained checkpoint %s", cfg.clf_ckpt)
            except Exception as e:                       # a stale checkpoint must not
                log.warning("Ignoring unloadable head checkpoint (%s: %s)",  # kill the run
                            type(e).__name__, e)
        log.info("Disaster head ready: %d classes, tab_dim=%d, emb_dim=%d, trained=%s",
                 len(self.classes), TAB_DIM, clip.embed_dim, self.is_trained)

    def _prototype(self, disaster: str) -> np.ndarray:
        E = self.clip.encode_texts(DISASTER_PROMPTS[disaster])
        v = E.mean(0)
        return (v / max(float(np.linalg.norm(v)), 1e-8)).astype(np.float32)

    def _transition_prototype(self, disaster: str) -> np.ndarray:
        """Prototype for the PRE->POST direction, not the POST state."""
        E = self.clip.encode_texts(DISASTER_TRANSITION_PROMPTS[disaster])
        v = E.mean(0)
        return (v / max(float(np.linalg.norm(v)), 1e-8)).astype(np.float32)

    # -- inference -----------------------------------------------------------
    @torch.no_grad()
    def predict(self, feats: ClassifierFeatures, k: int = 3) -> dict:
        self.head.eval()
        z = torch.as_tensor(feats.z, dtype=torch.float32, device=self.cfg.device)[None]
        t = torch.as_tensor(feats.t, dtype=torch.float32, device=self.cfg.device)[None]
        logits, zs, res = self.head(z, t)
        logits = logits / self.head.temperature.clamp(min=1e-3)
        p = torch.softmax(logits, -1)[0].float().cpu().numpy()
        order = np.argsort(-p)[:k]
        return {
            "probs": p.astype(np.float32),
            "probs_by_class": {c: round(float(p[i]), 4) for i, c in enumerate(self.classes)},
            "top_k": [{"disaster": self.classes[i], "prob": round(float(p[i]), 4)} for i in order],
            "label": self.classes[int(p.argmax())],
            "logits": logits[0].float().cpu().numpy(),
            "prototype_logits": zs[0].float().cpu().numpy(),
            "residual_norm": float(res.norm()),
            "embedding": feats.z.astype(np.float32),
            "is_trained": self.is_trained,
        }

    def zero_shot(self, global_post: np.ndarray,
                  global_pre: Optional[np.ndarray] = None, k: int = 3) -> dict:
        """The CLIP prior. Three modes, set by `Config.zeroshot_mode`:

          "post"  — POST tile vs STATE prototypes. The original behaviour, and the reason
                    this reasoner failed on the volcano pair: a single POST tile cannot
                    say that anything CHANGED, so a grey lahar valley scored closer to
                    the earthquake prompts than to the volcano ones.
          "diff"  — the PRE->POST embedding direction vs TRANSITION prototypes. Genuinely
                    bi-temporal: it scores what changed, not what is there.
          "both"  — average of the two distributions (default). Arithmetic, not geometric,
                    so one term scoring a class near zero cannot veto it outright.

        Falls back to "post" when no PRE embedding is supplied, or when the two embeddings
        are too close for the direction to carry signal (`zeroshot_min_diff_norm`) — which
        is exactly what a genuine no-change pair looks like.

        This is NOT independent of the head. The untrained head is exactly the "post"
        computation (zero-initialised residual), so the two are never pooled together:
        `Config.pool_weights_*` gives each head state one CLIP-derived source.
        """
        mode = str(getattr(self.cfg, "zeroshot_mode", "both")).lower()
        scale = float(self.cfg.zeroshot_logit_scale)

        def _soft(sim: np.ndarray) -> np.ndarray:
            s = scale * np.asarray(sim, np.float32)
            s = s - s.max()
            p = np.exp(s)
            return (p / p.sum()).astype(np.float32)

        p_state = _soft(self.prototypes @ np.asarray(global_post, np.float32))

        p_diff, diff_norm = None, 0.0
        if mode in ("diff", "both") and global_pre is not None:
            d = np.asarray(global_post, np.float32) - np.asarray(global_pre, np.float32)
            diff_norm = float(np.linalg.norm(d))
            if diff_norm >= float(getattr(self.cfg, "zeroshot_min_diff_norm", 0.05)):
                p_diff = _soft(self.transition_prototypes @ (d / diff_norm))

        if p_diff is None:
            p, used = p_state, "post"
        elif mode == "diff":
            p, used = p_diff, "diff"
        else:
            w = float(getattr(self.cfg, "zeroshot_diff_weight", 0.5))
            p = ((1.0 - w) * p_state + w * p_diff).astype(np.float32)
            p = (p / p.sum()).astype(np.float32)
            used = "both"

        order = np.argsort(-p)[:k]
        out = {"probs": p,
               "probs_by_class": {c: round(float(p[i]), 4) for i, c in enumerate(self.classes)},
               "top_k": [{"disaster": self.classes[i], "prob": round(float(p[i]), 4)}
                         for i in order],
               "label": self.classes[int(p.argmax())],
               "mode": used, "diff_norm": round(diff_norm, 4),
               "state_label": self.classes[int(p_state.argmax())],
               "state_probs_by_class": {c: round(float(p_state[i]), 4)
                                        for i, c in enumerate(self.classes)}}
        if p_diff is not None:
            out["diff_label"] = self.classes[int(p_diff.argmax())]
            out["diff_probs_by_class"] = {c: round(float(p_diff[i]), 4)
                                          for i, c in enumerate(self.classes)}
        return out

    def reset(self):
        """Re-initialise the head from the text prototypes (untrained, zero-shot state).
        Used between cross-validation folds so no fold sees another fold's weights."""
        self.head = DisasterClassifierHead(self.prototypes, TAB_DIM, self.clip.embed_dim,
                                           self.cfg.clf_hidden, self.cfg.clf_dropout,
                                           self.classes).to(self.cfg.device).eval()
        self.is_trained = False
        self.history = {}

    # -- training ------------------------------------------------------------
    def fit(self, features: List[ClassifierFeatures], labels: List[str],
            epochs: Optional[int] = None, val_frac: Optional[float] = None,
            verbose: bool = True, save_ckpt: bool = True) -> dict:
        cfg = self.cfg
        unknown = sorted(set(labels) - set(self.classes))
        if unknown:
            raise ValueError(f"labels contain classes the head cannot predict: {unknown}")
        if len(features) != len(labels):
            raise ValueError("features and labels must be the same length")
        if len(features) < 8:
            raise ValueError(f"only {len(features)} labelled pairs; the head needs at least 8 "
                             "(and realistically a few hundred) to beat its zero-shot init")

        Z = torch.tensor(np.stack([f.z for f in features]), dtype=torch.float32)
        T = torch.tensor(np.stack([f.t for f in features]), dtype=torch.float32)
        y = torch.tensor([self.classes.index(l) for l in labels], dtype=torch.long)

        g = torch.Generator().manual_seed(cfg.seed)
        perm = torch.randperm(len(y), generator=g)
        n_val = max(2, int((val_frac if val_frac is not None else cfg.clf_val_frac) * len(y)))
        vi, ti = perm[:n_val], perm[n_val:]
        dev = cfg.device
        Ztr, Ttr, ytr = Z[ti].to(dev), T[ti].to(dev), y[ti].to(dev)
        Zva, Tva, yva = Z[vi].to(dev), T[vi].to(dev), y[vi].to(dev)

        counts = torch.bincount(ytr, minlength=len(self.classes)).float()
        w = torch.where(counts > 0, counts.sum() / (counts * (counts > 0).sum()),
                        torch.zeros_like(counts)).to(dev)
        lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
        opt = torch.optim.AdamW([p for p in self.head.parameters() if p.requires_grad],
                                lr=cfg.clf_lr, weight_decay=cfg.clf_weight_decay)
        epochs = epochs or cfg.clf_epochs
        best, best_state, hist = -1.0, None, []
        for ep in range(epochs):
            self.head.train()
            idx = torch.randperm(len(ytr), device=dev)
            tot = 0.0
            for s in range(0, len(idx), cfg.clf_batch):
                b = idx[s:s + cfg.clf_batch]
                opt.zero_grad()
                logits, _, _ = self.head(Ztr[b], Ttr[b])
                loss = lossf(logits, ytr[b])
                loss.backward(); opt.step()
                tot += float(loss) * len(b)
            self.head.eval()
            with torch.no_grad():
                va_logits, _, _ = self.head(Zva, Tva)
                va_acc = float((va_logits.argmax(-1) == yva).float().mean())
                tr_logits, _, _ = self.head(Ztr, Ttr)
                tr_acc = float((tr_logits.argmax(-1) == ytr).float().mean())
            hist.append({"epoch": ep, "loss": tot / max(len(idx), 1),
                         "train_acc": tr_acc, "val_acc": va_acc})
            if va_acc > best:
                best = va_acc
                best_state = {k: v.detach().clone() for k, v in self.head.state_dict().items()}
            if verbose and (ep % max(1, epochs // 10) == 0 or ep == epochs - 1):
                log.info("head epoch %3d | loss %.4f | train %.3f | val %.3f",
                         ep, hist[-1]["loss"], tr_acc, va_acc)
        if best_state is not None:
            self.head.load_state_dict(best_state)
        self._calibrate(Zva, Tva, yva)
        self.is_trained = True
        self.history = {"best_val_acc": best, "n_train": int(len(ti)), "n_val": int(n_val),
                        "epochs": epochs, "curve": hist,
                        "temperature": float(self.head.temperature)}
        if save_ckpt:
            self.save(cfg.clf_ckpt)
        log.info("Head trained: best val acc %.3f on %d/%d split, T=%.3f%s",
                 best, len(ti), n_val, float(self.head.temperature),
                 f" -> {cfg.clf_ckpt}" if save_ckpt else " (checkpoint not written)")
        return self.history

    @torch.no_grad()
    def _calibrate(self, Z, T, y):
        """Temperature scaling on the held-out split, so the probabilities the pool
        receives are calibrated rather than merely ranked."""
        logits, _, _ = self.head(Z, T)
        best_t, best_nll = 1.0, float("inf")
        for t in np.linspace(0.5, 5.0, 46):
            nll = float(F.cross_entropy(logits / t, y))
            if nll < best_nll:
                best_nll, best_t = nll, float(t)
        self.head.temperature.fill_(best_t)

    # -- persistence ---------------------------------------------------------
    def save(self, path: str):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": self.head.state_dict(), "classes": list(self.classes),
                    "tab_dim": TAB_DIM, "emb_dim": self.clip.embed_dim,
                    "is_trained": self.is_trained, "history": self.history}, path)

    def load(self, path: str):
        ck = torch.load(path, map_location="cpu")
        if tuple(ck["classes"]) != self.classes or ck["tab_dim"] != TAB_DIM \
                or ck["emb_dim"] != self.clip.embed_dim:
            raise ValueError("checkpoint layout does not match the current feature layout")
        self.head.load_state_dict(ck["state_dict"])
        self.head.to(self.cfg.device).eval()
        self.is_trained = bool(ck.get("is_trained", False))
        self.history = ck.get("history", {})


# ---------------------------------------------------------------------------
# Calibrated evidence fusion (Part 3): logarithmic opinion pooling.
# ---------------------------------------------------------------------------
@dataclass
class DisasterPosterior:
    probs: np.ndarray
    classes: Tuple[str, ...]
    sources: Dict[str, np.ndarray]
    weights: Dict[str, float]
    # pairwise agreement of every source that was COMPUTED (pooled or not); see
    # `source_dependence`. Reported, never used to weight the pool.
    dependence: Dict[str, Dict[str, Any]] = field(default_factory=dict)

    @property
    def label(self) -> str:
        return self.classes[int(self.probs.argmax())]

    @property
    def prob(self) -> float:
        return float(self.probs.max())

    @property
    def margin(self) -> float:
        p = np.sort(self.probs)[::-1]
        return float(p[0] - p[1]) if len(p) > 1 else float(p[0])

    @property
    def entropy(self) -> float:
        p = np.clip(self.probs, 1e-9, 1.0)
        return float(-(p * np.log(p)).sum() / np.log(len(p)))

    def top_k(self, k: int = 3) -> List[dict]:
        order = np.argsort(-self.probs)[:k]
        return [{"disaster": self.classes[i], "prob": round(float(self.probs[i]), 4)}
                for i in order]

    def source_table(self) -> Dict[str, Dict[str, float]]:
        return {s: {self.classes[i]: round(float(p[i]), 4) for i in range(len(self.classes))}
                for s, p in self.sources.items()}

    def as_dict(self) -> dict:
        return {"label": self.label, "prob": round(self.prob, 4),
                "margin": round(self.margin, 4), "entropy": round(self.entropy, 4),
                "top_k": self.top_k(), "weights": self.weights,
                "dependence": self.dependence,
                "sources": {s: self.classes[int(p.argmax())] for s, p in self.sources.items()},
                "probs": {c: round(float(self.probs[i]), 4) for i, c in enumerate(self.classes)}}


def log_linear_pool(sources: Dict[str, np.ndarray], weights: Dict[str, float],
                    eps: float = 1e-6) -> DisasterPosterior:
    """Logarithmic opinion pooling — the normalised weighted geometric mean

        p(d) ∝ prod_s p_s(d)^{w_s}

    chosen over a linear (arithmetic) pool because it is externally Bayesian and
    conjunctive: a class that ANY well-weighted source considers near-impossible is
    suppressed, which is the behaviour we want from a veto-carrying knowledge graph.
    An uninformative source contributes a uniform distribution and therefore nothing,
    so a disabled module degrades the pool gracefully instead of biasing it.
    """
    use = {s: np.asarray(p, np.float64) for s, p in sources.items()
           if p is not None and float(weights.get(s, 0.0)) > 0}
    if not use:
        n = len(DISASTER_CLASSES)
        return DisasterPosterior(np.full(n, 1.0 / n, np.float32), DISASTER_CLASSES, {}, {})
    tot = sum(float(weights[s]) for s in use)
    L = np.zeros(len(DISASTER_CLASSES), np.float64)
    for s, p in use.items():
        p = p / max(p.sum(), 1e-12)
        L += (float(weights[s]) / tot) * np.log(p + eps)
    L -= L.max()
    p = np.exp(L); p = p / p.sum()
    return DisasterPosterior(p.astype(np.float32), DISASTER_CLASSES,
                             {s: np.asarray(v, np.float32) for s, v in use.items()},
                             {s: round(float(weights[s]) / tot, 3) for s in use})



def pool_posterior(cfg: Config, sources: Dict[str, Optional[np.ndarray]],
                   head_trained: bool) -> DisasterPosterior:
    """The ONE place the pooling weights are chosen. Both the pipeline and the
    cross-validation harness call this, so a reported number cannot come from a
    different pooling rule than the one the pipeline runs."""
    w = dict(cfg.pool_weights_trained if head_trained else cfg.pool_weights_untrained)
    post = log_linear_pool(sources, w)
    post.dependence = source_dependence(sources, pooled=set(post.weights))
    return post


def source_dependence(sources: Dict[str, Optional[np.ndarray]],
                      pooled: Sequence[str] = ()) -> Dict[str, Dict[str, Any]]:
    """Pairwise dependence between every computed source, for the report.

    Log-linear pooling treats its inputs as conditionally independent. Where two sources
    are one computation (the CLIP prior and the head) that assumption overstates
    confidence, so the pair is reported -- rank correlation of the class probabilities,
    Jensen-Shannon divergence, top-1 agreement, and whether both were pooled -- and the
    pool itself is built from sources that are not known duplicates."""
    def _rank(x):
        return np.argsort(np.argsort(x)).astype(np.float64)

    names = [n for n, p in sources.items() if p is not None]
    out: Dict[str, Dict[str, Any]] = {}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            pa = np.clip(np.asarray(sources[a], np.float64), 1e-9, 1.0); pa = pa / pa.sum()
            pb = np.clip(np.asarray(sources[b], np.float64), 1e-9, 1.0); pb = pb / pb.sum()
            ra, rb = _rank(pa), _rank(pb)
            rc = float(np.corrcoef(ra, rb)[0, 1]) if ra.std() > 0 and rb.std() > 0 else 0.0
            m = 0.5 * (pa + pb)
            jsd = 0.5 * float((pa * np.log(pa / m)).sum()) + 0.5 * float((pb * np.log(pb / m)).sum())
            out[f"{a}~{b}"] = {"rank_corr": round(rc, 3), "jsd": round(jsd, 4),
                               "same_top1": bool(pa.argmax() == pb.argmax()),
                               "both_pooled": bool(a in pooled and b in pooled)}
    return out


## 10 · Reliability Verification Gate v3 (Part 4)

Nine weighted checks, each a continuous score in `[0,1]`:

`scene_object_consistency`, `anychange_support`, `transition_support`, `object_support`,
`hypothesis_margin`, `classifier_graph_agreement`, `not_obscured`, `blind_caption_agreement`,
`caption_support`.

**The blind-caption test.** Qwen is run twice. Pass 1 sees *only the two images* and is asked what
happened and which disaster it is. Pass 2 sees the images **and** the evidence package. If the blind
verdict disagrees with the pipeline verdict, or the two captions are semantically far apart
(RemoteCLIP sentence similarity) while the evidence-fed caption mentions far more specifics, the
gate treats that as a warning: confidence drops, `hedge` is raised, and the caption is asked to state
its own uncertainty. This is the closest thing the notebook has to an independent check on whether
the evidence pipeline is leading the VLM somewhere the images do not support.

The gate reports `weakest_source` (which evidence stream dragged the confidence down, via the
`SOURCE_OF` map) and a written `uncertainty_reason`. Set `use_blind_pass=False` to skip the extra
generation and save roughly one Qwen forward pass.

**Two checks changed with the two-source pool.** `caption_support` now compares the class the
caption *states* on its closing line against the posterior, instead of counting disaster keywords
in its body — keyword counting scored a caption 1.00 for the phrase “ruling out wildfires” while
the posterior was wildfire, and so *raised* confidence on a caption that contradicted the
pipeline. `classifier_graph_agreement` is now scored only when the learned head is actually one of
the pooled sources; while the head is untrained it duplicates the CLIP prior and is excluded from
the pool, so auditing the posterior against it penalised the system for disagreeing with an
opinion it never used. When the check is dropped its weight is removed and the remaining checks
are renormalised.


In [ ]:
DISASTER_KEYWORDS: Dict[str, List[str]] = {
    "volcanic_eruption": ["volcan", "ash", "erupt", "tephra", "pyroclastic", "lava", "lahar"],
    "flood":             ["flood", "inundat", "submerg", "overflow", "standing water"],
    "wildfire":          ["wildfire", "fire", "burn", "scorch", "charred"],
    "earthquake":        ["earthquake", "seismic", "collaps", "rubble"],
    "cyclone":           ["cyclone", "hurricane", "typhoon", "wind", "storm"],
    "landslide":         ["landslide", "mudslide", "debris flow", "slope failure", "slid"],
    "tsunami":           ["tsunami", "tidal wave", "storm surge", "swept"],
}

# Terms the grounded caption is expected to reuse if it actually read the evidence.
EVIDENCE_TERMS = ["ash", "vegetation", "building", "road", "bridge", "river", "water",
                  "flood", "burn", "debris", "rubble", "smoke", "plume", "terrain",
                  "coverage", "%", "destroyed", "damaged"]


# ISSUE 13: caption-to-caption similarity used RemoteCLIP's text tower, an image-text
# retrieval model with a 77-token context. It returned 0.96 for a blind and a grounded
# caption that reached DIFFERENT conclusions, which saturated the blind-pass check.
# A sentence-similarity model is the right tool; CLIP stays as the fallback.
_SBERT = {"model": None, "tried": False}


def caption_similarity(a: str, b: str, clip: "RemoteCLIPEncoder", cfg: Config) -> float:
    """Cosine similarity between two captions, in [0, 1]."""
    if cfg.caption_sim_model == "sbert":
        if not _SBERT["tried"]:
            _SBERT["tried"] = True
            try:
                from sentence_transformers import SentenceTransformer
                _SBERT["model"] = SentenceTransformer(cfg.sbert_id, device=cfg.device)
                log.info("Caption similarity: %s", cfg.sbert_id)
            except Exception as e:
                log.warning("sentence-transformers unavailable (%s: %s); caption "
                            "similarity falls back to the CLIP text tower, which is "
                            "not a sentence-similarity model -- say so when reporting.",
                            type(e).__name__, e)
        m = _SBERT["model"]
        if m is not None:
            if not (a or "").strip() or not (b or "").strip():
                return 0.0
            e = m.encode([a, b], normalize_embeddings=True, show_progress_bar=False)
            return float(np.dot(e[0], e[1]))
    return clip.caption_similarity(a, b)


def disaster_from_text(text: str) -> Tuple[str, float, Dict[str, int]]:
    """Read a disaster claim out of free-form caption text by keyword voting.
    Returns (label, normalised margin, per-class hit counts)."""
    t = (text or "").lower()
    # word-start matching ("char" used to fire on "characterized", "ash" on "washed");
    # short keywords must match a whole word
    _whole = {"ash", "fire", "wind"}
    hits = {d: sum(len(re.findall(r"\b" + re.escape(k) + (r"\b" if k in _whole else ""), t))
                   for k in ks) for d, ks in DISASTER_KEYWORDS.items()}
    tot = sum(hits.values())
    if tot == 0:
        return "unclear", 0.0, hits
    ranked = sorted(hits.items(), key=lambda kv: -kv[1])
    top, second = ranked[0][1], (ranked[1][1] if len(ranked) > 1 else 0)
    return ranked[0][0], float((top - second) / max(top, 1)), hits


def evidence_grounding(text: str, terms: Sequence[str] = EVIDENCE_TERMS) -> float:
    t = (text or "").lower()
    return float(sum(1 for k in terms if k in t) / max(len(terms), 1))


_STATED_ALIASES = {
    "volcanic eruption": "volcanic_eruption", "volcanic": "volcanic_eruption",
    "volcano": "volcanic_eruption", "eruption": "volcanic_eruption",
    "flood": "flood", "flooding": "flood", "floods": "flood",
    "wildfire": "wildfire", "wildfires": "wildfire", "fire": "wildfire",
    "earthquake": "earthquake", "earthquakes": "earthquake",
    "cyclone": "cyclone", "hurricane": "cyclone", "typhoon": "cyclone",
    "tropical cyclone": "cyclone", "landslide": "landslide", "landslides": "landslide",
    "tsunami": "tsunami", "unclear": "unclear"}


def stated_disaster(text: str) -> str:
    """The class a caption STATES on its closing 'Most likely disaster: <type>' line
    (both the blind prompt and the grounded prompt request it). Falls back to keyword
    voting only when the line is absent. Used by the caption-condition experiment so a
    caption's label is what it says, not what a keyword counter infers from its body."""
    text = (text or "").replace("*", "")      # "**Most likely disaster:** x" was missed
    m = re.findall(r"most likely disaster\s*[:\-]\s*([A-Za-z _\-]+)", text or "", flags=re.I)
    if m:
        tok = re.sub(r"[\s_\-]+", " ", m[-1].strip().lower())
        if tok in _STATED_ALIASES:
            return _STATED_ALIASES[tok]
        for k in sorted(_STATED_ALIASES, key=len, reverse=True):
            if k in tok:
                return _STATED_ALIASES[k]
    return disaster_from_text(text)[0]


@dataclass
class BlindComparison:
    """The blind-caption verification (Part 4): Qwen with the images alone versus Qwen
    with the images plus the structured evidence."""
    blind_caption: str
    grounded_caption: str
    blind_disaster: str
    grounded_disaster: str
    pipeline_disaster: str
    semantic_similarity: float
    blind_grounding: float
    grounded_grounding: float
    agreement: float
    label_match: bool
    explanation: str

    def as_dict(self) -> dict:
        d = asdict(self)
        for k in ("semantic_similarity", "blind_grounding", "grounded_grounding", "agreement"):
            d[k] = round(float(d[k]), 3)
        return d


def compare_blind_and_grounded(clip: RemoteCLIPEncoder, cfg: Config, blind: str,
                               grounded: Optional[str], pipeline_disaster: str) -> BlindComparison:
    """`grounded` may be empty on the provisional pass, before the grounded caption
    exists. The label check is meaningful either way; the semantic-similarity term is
    simply dropped rather than scored against an empty string."""
    # Both prompts require a closing 'Most likely disaster: <type>' line; keyword
    # voting is only the fallback. Counting keywords mislabels a caption that RULES
    # a class OUT ("no burn scars, so not a wildfire" scored as a wildfire claim).
    b_lab = stated_disaster(blind)
    g_lab = stated_disaster(grounded or "")
    match = (b_lab == pipeline_disaster)
    partial = 0.5 if b_lab == "unclear" else 0.0
    label_term = 1.0 if match else partial
    if grounded:
        sim = caption_similarity(blind, grounded, clip, cfg)
        sim_n = float(np.clip((sim - cfg.rvg_blind_sim_lo)
                              / max(cfg.rvg_blind_sim_hi - cfg.rvg_blind_sim_lo, 1e-6), 0.0, 1.0))
        agreement = float(np.clip(0.6 * label_term + 0.4 * sim_n, 0.0, 1.0))
    else:
        sim, agreement = 0.0, float(label_term)
    sim_txt = f" (caption similarity {sim:.2f})" if grounded else ""
    if match:
        why = (f"the blind caption reached '{b_lab}' from the images alone, independently of "
               f"the evidence package{sim_txt}")
    elif b_lab == "unclear":
        why = (f"the blind caption named no disaster type, so the '{pipeline_disaster}' claim "
               f"rests on the structured evidence rather than on what is plainly visible"
               f"{sim_txt}")
    else:
        why = (f"the blind caption suggested '{b_lab}' while the pipeline concluded "
               f"'{pipeline_disaster}'; the two readings disagree{sim_txt}, so confidence is "
               f"reduced and the caption should hedge")
    return BlindComparison(blind, grounded, b_lab, g_lab, pipeline_disaster, float(sim),
                           evidence_grounding(blind), evidence_grounding(grounded),
                           agreement, bool(match), why)


class ReliabilityVerificationGate:
    """
    RVG v3 — reliability verification across every producer in the pipeline (Part 4).

    v2 checked the two branches against each other. v3 additionally verifies the LEARNED
    classifier against the graph, and the generated caption against a BLIND caption that
    Qwen wrote from the images alone with no access to the evidence package. That blind
    pass is the only check in the system whose input is not derived from the pipeline's
    own features, which makes it the one thing that can catch the pipeline agreeing with
    itself.

    Every check scores in [0,1]; the weighted mean is squeezed into [0.05, 0.95] because
    a system that reports 1.00 from a handful of heuristic checks is overclaiming.
    """
    WEIGHTS = {
        "scene_object_consistency": 0.14,
        "anychange_support":        0.10,
        "transition_support":       0.16,
        "object_support":           0.10,
        "hypothesis_margin":        0.10,
        "classifier_graph_agreement": 0.14,
        "not_obscured":             0.06,
        "blind_caption_agreement":  0.10,
        "caption_support":          0.10,
    }
    # which producer each check is auditing -- reported as `weakest_source`
    SOURCE_OF = {
        "scene_object_consistency": "cross-branch",
        "anychange_support":        "scene branch",
        "transition_support":       "semantic transitions",
        "object_support":           "object branch",
        "hypothesis_margin":        "DKG",
        "classifier_graph_agreement": "learned classifier",
        "not_obscured":             "scene branch",
        "blind_caption_agreement":  "caption (blind pass)",
        "caption_support":          "caption",
    }
    SCENE_DRIVEN = {"volcanic_eruption", "wildfire", "flood", "landslide"}
    OBJECT_DRIVEN = {"earthquake", "cyclone", "tsunami"}

    def __init__(self, cfg: Config):
        self.cfg = cfg

    @staticmethod
    def _soft_iou(a: np.ndarray, b: np.ndarray) -> float:
        inter = float(np.minimum(a, b).sum())
        union = float(np.maximum(a, b).sum())
        return inter / union if union > 1e-6 else 0.0

    def verify(self, posterior: DisasterPosterior, dkg_out: dict, clf_out: Optional[dict],
               fused: FusedEvidence, transitions: List[SemanticTransition],
               instances: List["Instance"], scene_ev: SceneEvidence,
               blind: Optional[BlindComparison] = None,
               caption: Optional[str] = None, clf_pooled: bool = True,
               caption_primed: bool = False) -> dict:
        s = fused.scalars
        d = posterior.label if posterior.entropy < 0.985 else "undetermined"
        checks: Dict[str, float] = {}
        notes: Dict[str, str] = {}
        drop: set = set()          # checks that are not applicable to this configuration

        # 1. do the two branches tell a consistent story?
        scene_cov, obj_dmg = s["scene_coverage"], s["object_damaged_area_frac"]
        if d == "undetermined":
            # agreement means BOTH branches are quiet. A quiet object branch alone is
            # exactly the false confirmation v1 produced on an ash-covered scene.
            checks["scene_object_consistency"] = float(np.clip(1.0 - 3.0 * scene_cov, 0, 1))
            notes["scene_object_consistency"] = f"scene_coverage={scene_cov:.3f} with no hypothesis"
        elif d in self.SCENE_DRIVEN:
            checks["scene_object_consistency"] = float(
                np.clip(scene_cov / max(self.cfg.rvg_scene_cov_ref, 1e-6), 0, 1))
            notes["scene_object_consistency"] = f"scene-driven hypothesis, coverage={scene_cov:.3f}"
        else:
            checks["scene_object_consistency"] = float(
                np.clip(obj_dmg / max(self.cfg.rvg_obj_dmg_ref, 1e-6), 0, 1))
            notes["scene_object_consistency"] = f"object-driven hypothesis, damaged={obj_dmg:.4f}"

        # 2. do the two INDEPENDENT scene detectors overlap? (learned SAM latent
        #    matching vs unlearned physical channels)
        _ac = np.asarray(fused.anychange_map, np.float32)
        if float(_ac.max()) <= 1e-6:
            # Nothing to overlap with: scoring 0 would punish the hypothesis for a
            # detector that stayed silent. Not applicable, so not scored.
            drop.add("anychange_support")
            notes["anychange_support"] = "not scored: AnyChange produced no regions"
        else:
            iou = self._soft_iou(_ac, fused.physical["composite"])
            checks["anychange_support"] = float(
                np.clip(iou / max(self.cfg.rvg_iou_ref, 1e-6), 0, 1))
            notes["anychange_support"] = f"AnyChange/physical soft-IoU={iou:.3f}"

        # 3. does the semantic transition evidence back the winning hypothesis?
        n_ch = sum(1 for t in transitions if t.changed)
        if d == "undetermined":
            checks["transition_support"] = float(np.clip(1.0 - n_ch / 8.0, 0, 1))
            notes["transition_support"] = f"{n_ch} semantic transitions with no hypothesis"
        else:
            rule = next((r for r in DynamicKnowledgeGraph.RULES if r["name"] == d), None)
            mass = dkg_out.get("concept_mass", {})
            by_src = dkg_out.get("concept_by_source", {})
            # ISSUE 12: this used to pass on a non-zero TRACE of semantic mass -- 0.002
            # counted the same as 0.3 -- which is why it read 1.00 on every pair. The
            # semantic streams must now carry at least `rvg_transition_mass_ref`.
            ref = max(self.cfg.rvg_transition_mass_ref, 1e-6)
            got, want, lvl = [], [], []
            for req in (rule["required"] if rule else []):
                _, name = DynamicKnowledgeGraph._required_mass(mass, req)
                want.append(name)
                srcs = by_src.get(name, {})
                sem_mass = float(srcs.get("transition", 0.0)) + float(srcs.get("scene", 0.0))
                lvl.append(round(sem_mass, 3))
                got.append(float(np.clip(sem_mass / ref, 0.0, 1.0)))
            checks["transition_support"] = float(np.mean(got)) if got else 0.0
            notes["transition_support"] = (f"semantic (transition/scene) mass {lvl} for "
                                           f"{want} against a {ref:.2f} reference; "
                                           f"{n_ch} transitions total")

        # 4. object branch
        graded = [i.damage_grade for i in instances if i.damage_grade not in (None, 0)]
        if d in self.OBJECT_DRIVEN:
            checks["object_support"] = 1.0 if any(g >= 2 for g in graded) else 0.0
            notes["object_support"] = f"{sum(1 for g in graded if g >= 2)} instances graded >= minor"
        else:
            # Was 1.0 whenever SAM2 returned ANY instance, which it always does -- so a
            # volcano or flood tile with zero buildings scored a free 1.00 and diluted
            # the checks that carry signal. A check with nothing to say is not scored.
            drop.add("object_support")
            notes["object_support"] = (f"not scored: '{d}' is not an object-driven "
                                       f"hypothesis ({len(instances)} instances present, "
                                       f"none required)")

        # 5. how decisively did the fused posterior beat the runner-up?
        checks["hypothesis_margin"] = float(
            np.clip(posterior.margin / max(self.cfg.rvg_margin_ref, 1e-6), 0, 1))
        notes["hypothesis_margin"] = (f"posterior margin={posterior.margin:.3f}, "
                                      f"entropy={posterior.entropy:.3f}")

        # 6. do the learned head and the knowledge graph agree? (Part 3/4)
        #    Only audited when the head is one of the POOLED sources. While the head is
        #    untrained it duplicates the CLIP prior and is excluded from the pool, so
        #    scoring the posterior against it penalises the system for disagreeing with
        #    an opinion it never used -- which is what dragged rev4's confidence down.
        if not clf_pooled:
            drop.add("classifier_graph_agreement")
            notes["classifier_graph_agreement"] = (
                "learned head is not a pooled source here (untrained head duplicates the "
                "CLIP prior); check not scored")
        elif clf_out is None or dkg_out.get("disaster") in (None, "undetermined"):
            checks["classifier_graph_agreement"] = 0.5
            notes["classifier_graph_agreement"] = "one of the two reasoners produced no verdict"
        else:
            g = dkg_out["disaster"]
            p_clf = float(clf_out["probs"][DISASTER_CLASSES.index(g)])
            same = clf_out["label"] == g
            checks["classifier_graph_agreement"] = float(np.clip(0.5 * same + 0.5 * p_clf * 2, 0, 1))
            notes["classifier_graph_agreement"] = (
                f"DKG={g}, classifier={clf_out['label']} "
                f"(p({g})={p_clf:.2f}, trained={clf_out['is_trained']})")

        # 7. cloud / plume obscuration
        obs = max(dkg_out.get("obscuration_mass", 0.0), scene_ev["cloud_obscuration"])
        checks["not_obscured"] = float(np.clip(1.0 - obs, 0, 1))
        notes["not_obscured"] = f"cloud/plume obscuration={obs:.3f}"

        # 8/9. captions -------------------------------------------------------
        w = dict(self.WEIGHTS)
        for _k in drop:
            w.pop(_k, None)
        if blind is None:
            w.pop("blind_caption_agreement")
            notes["blind_caption_agreement"] = "blind pass not run"
        else:
            checks["blind_caption_agreement"] = blind.agreement
            notes["blind_caption_agreement"] = blind.explanation
        if caption is None:
            w.pop("caption_support")
            notes["caption_support"] = "not yet generated"
        elif caption_primed:
            # rev8: the system caption (C1) is written WITH the posterior label in its
            # prompt, so the class it states is an echo of the posterior, not support for
            # it. Scoring it would raise confidence by construction. The blind-caption
            # check is the independent caption-based test, and it stays.
            w.pop("caption_support")
            notes["caption_support"] = ("caption was written with the posterior label "
                                        "(C1); its stated class is an echo, not scored")
        else:
            # Substring counting scored a caption 1.00 for "ruling out wildfires" when
            # the posterior was wildfire -- it counted the word, not the claim, and it
            # RAISED confidence on a caption that contradicted the pipeline. Under C2
            # the caption can legitimately disagree, so the check reads the class the
            # caption actually states and scores agreement with the posterior.
            said = stated_disaster(caption)
            ground = evidence_grounding(caption)
            if d == "undetermined":
                agree = 1.0
            elif said == "unclear":
                agree = 0.5                      # committed to nothing: neither support nor conflict
            else:
                agree = 1.0 if said == d else 0.0
            checks["caption_support"] = float(np.clip(0.6 * agree
                                                      + 0.4 * min(ground / 0.4, 1.0), 0, 1))
            notes["caption_support"] = (f"caption states '{said}' vs posterior '{d}', "
                                        f"evidence grounding {ground:.2f}")

        tot = sum(w.values())
        agreement = sum(w[k] * checks[k] for k in w) / tot
        conf = 0.05 + 0.90 * agreement
        weakest = min(((k, v) for k, v in checks.items() if k in w), key=lambda kv: kv[1])
        hedge = bool(conf < self.cfg.rvg_hedge_conf or weakest[1] < self.cfg.rvg_hedge_check
                     or (blind is not None and not blind.label_match
                         and blind.blind_disaster != "unclear"))
        reason = (f"weakest evidence source: {self.SOURCE_OF[weakest[0]]} "
                  f"({weakest[0]}={weakest[1]:.2f}; {notes[weakest[0]]})")
        if blind is not None and not blind.label_match:
            reason += " | " + blind.explanation
        return {
            "disaster": d,
            "confidence": round(float(conf), 3),
            "raw_agreement": round(float(agreement), 3),
            "hedge": hedge,
            "checks": {k: round(v, 3) for k, v in checks.items()},
            "weights": {k: round(v / tot, 3) for k, v in w.items()},
            "weakest_check": weakest[0],
            "weakest_source": self.SOURCE_OF[weakest[0]],
            "uncertainty_reason": reason,
            "caption_included": caption is not None,
            "blind": blind.as_dict() if blind is not None else None,
            "notes": notes,
            "rationale": "; ".join(f"{k}={checks[k]:.2f} ({notes[k]})" for k in checks),
        }

    @staticmethod
    def null_output(posterior: DisasterPosterior) -> dict:
        """`use_rvg=False`: the posterior's own maximum, with no verification claimed."""
        return {"disaster": posterior.label, "confidence": round(posterior.prob, 3),
                "raw_agreement": 0.0, "hedge": False, "checks": {}, "weights": {},
                "weakest_check": "none", "weakest_source": "none",
                "uncertainty_reason": "RVG disabled (ablation)", "caption_included": False,
                "blind": None, "notes": {}, "rationale": "RVG disabled", "disabled": True}


## 11 · Evidence, close-ups and the Qwen prompts (Part 5) — rev12

**The system caption is EVIDENCE → LOOK → WRITE → VERIFY** (`CFG.caption_mode = "look"`, `look_then_write`).

1. **Evidence** (`caption_evidence`). Every valid change the pipeline measured, with no cap, each with its 3×3 location: typed darkening (deposit, burn), located physical changes (darker, paler, faded, vegetation loss, texture), RemoteCLIP changes, surfaces (water confirmed by both tests, ash, burn, smoke, lava, landslide), damaged buildings (ChangeOS), AnyChange regions and SAM2 object changes. A RemoteCLIP "cloud" label on ground the cloud test calls clear is kept as a *pale veil* (ash, dust or haze) when it is the AFTER class, and excluded when it is the BEFORE class. Excluded items are kept with the reason.
2. **Look** (`group_evidence`, `look_at_groups`). Items are grouped by place; Qwen gets a BEFORE/AFTER close-up of each group (`look_prompt`, no evidence words) and says what changed. Items its answer does not describe get one guided look with a plain cue (`ITEM_CUE`). Status per item: *visible* or *not visible*.
3. **Write** (`compose_prompt`). One caption from scratch from the full images, the verdict and Qwen's own notes per region; sentences scale with the number of regions (≤ `caption_max_sentences`).
4. **Verify** (`verify_caption`). Coverage of every visible item (its kind of change in a clause at its place, or its note's content words / sentence similarity), event named, AFTER clouds mentioned, no confident absence asserted, no damaged buildings called intact, no measured place called unchanged. A failed check means a rewrite **from scratch** with the omitted notes flagged; the earlier caption is never shown.

`look_from_record` rebuilds the regions from a saved record, so §21 writes C2 (no verdict) and the grounding tests from the same looks.

---

**rev9–rev11 caption (comparison C5): DRAFT → CHECK → REVISE** (`CFG.caption_mode = "revise"`).

1. **Draft.** Qwen describes the BEFORE and AFTER images with no measurements at all — this is the blind pass, so it costs nothing extra. The words and the structure are the model's own.
2. **Check.** Code (`check_draft`), not the LLM, compares each sentence with the measurements (`measured_facts`). In priority order: the disaster type against the pipeline's verdict; claims contradicted by a **confident** absence (two measurements agree); large water described as dry ground; buildings called intact where ChangeOS detects damage; and large measured changes the draft misses (with where they are). Negations are respected ("no standing water", "without any sign of fire"). A collapse claim ChangeOS cannot see is logged as *unverifiable* and left alone (`caption_check_buildings = "add_only"`).
3. **Revise.** Qwen receives its own draft, the images and only the corrections (`revision_prompt`), and keeps every other sentence. If nothing needs correcting the draft is the caption.

Every edit is stored with its claim, measurement and value, and re-checked against the revised caption (`resolved`); anything still contradicting a measurement is reported as `residual`. **The measurements decide what the caption says; they never supply its words.**

The disaster label is one of the checks (`label_in_prompt = True`): when the draft and the pipeline disagreed in rev8, the pipeline was right 27 times and the draft once. `label_in_prompt = False` gives the no-label ablation (C2). The rev8 evidence checklist (`build_evidence_package`, `caption()`) is kept as the comparison condition C4 because it is copied rather than used.

RVG: the system caption states the posterior by construction, so `caption_support` is not scored for it; the blind-caption check remains the independent caption test.

**rev10 additions to the check.**
- **Clouds.** If clouds cover part of an image, the caption must say they hide that area, and must not describe a change under a BEFORE cloud.
- **Measured changes.** The checker adds up to two of the strongest measured changes the draft misses: located physical regions (darker, paler, faded colours, vegetation loss) and RemoteCLIP changes by their true area. They are passed in neutral words; Qwen looks and says what they are.
- **Contradicted "unchanged" claims.** A draft sentence that calls the vegetation unchanged where loss was measured is rewritten.
- **Location-aware mentions.** A sentence counts as mentioning a change only if it does not name a different place. *(rev12: these tests were mostly off until now — `_ZONE_WORDS` held literal backspace characters, so single directions never matched.)*


In [ ]:
def _pct(x: float) -> str:
    return f"{100.0 * float(x):.0f}%"


# ---------------------------------------------------------------------------
# C2 PURITY: every name that encodes a disaster hypothesis, rewritten as the physical
# measurement it actually is. "volcanic ash coverage 1%" is a softmax share of the
# prompt "grey volcanic ash covering the ground" -- naming it that way hands Qwen the
# volcanic hypothesis without naming the class, and the rev4 caption reasoned straight
# off these words ("no burn scars ... ruling out wildfires"). Under C2 the captioner
# must get measurements and nothing else.
# ---------------------------------------------------------------------------
NEUTRAL_SCENE_TERMS: Dict[str, str] = {
    "ash_coverage":      "grey low-saturation particulate ground cover",
    "lava_coverage":     "fresh dark unvegetated rock cover",
    "burn_scars":        "blackened ground cover",
    "flood_extent":      "standing-water ground cover",
    "smoke_regions":     "airborne plume cover",
    "cloud_obscuration": "bright opaque veil cover",
    "landslide_scars":   "exposed downslope scar cover",
    "rubble_coverage":   "broken-surface debris cover",
}

NEUTRAL_CONCEPT_TERMS: Dict[str, str] = {
    "ashfall": "grey particulate deposit", "lava_flow": "fresh dark rock",
    "burned": "blackened surface", "inundation": "dark smooth standing water",
    "water_expansion": "water-area increase", "sediment_load": "turbid water",
    "vegetation_loss": "vegetation decrease", "terrain_change": "surface-texture change",
    "surface_change": "surface-brightness change",
    "mass_movement": "downslope material displacement",
    "structural_damage": "built-structure disruption", "debris_scatter": "scattered debris",
    "infrastructure_damage": "linear-infrastructure disruption",
    "plume": "airborne plume", "obscuration": "opaque veil",
}

NEUTRAL_GROUP_TERMS: Dict[str, str] = {
    "ash": "grey particulate cover", "lava": "fresh dark rock", "burn": "blackened ground",
    "flood": "standing water", "landslide": "exposed slope scar", "plume": "airborne plume",
    "cloud": "opaque veil", "rubble": "broken debris", "sediment_water": "turbid water",
    "water_expanded": "enlarged water body", "water_receded": "reduced water body",
    "buried_road": "covered road", "broken_bridge": "discontinuous bridge",
}


def _neutral(term: str, table: Dict[str, str], on: bool) -> str:
    """A measurement name with no disaster class implied, when `on`."""
    return table.get(term, term.replace("_", " ")) if on else term.replace("_", " ")


@dataclass
class EvidencePackage:
    """The evidence Qwen captions from (Part 5) -- a CHECKLIST, not a script.

    rev8. rev7 handed Qwen finished English sentences, and the C2 captions restated them
    in order: 17% of a C2 caption's word pairs came straight from the evidence, against
    1% for the baseline, and flood captions from different pairs were more alike
    (ROUGE-L 0.43) than the references were (0.19). Every line is now a terse measured
    fact, `thing: change | extent | where`, so Qwen has to look at the images and put
    the change into its own words, and every claim carries the part of the scene it was
    measured in.

    Two conditions, set by `label_in_prompt`:
      * True  -- C1, THE SYSTEM CAPTION. The pipeline's own fused verdict (never the
                 ground truth) heads the package, the checklist is worded in that event
                 type's vocabulary, and Qwen is told to open the caption with the event.
      * False -- C2, the no-label ablation. The same checklist and the same locations in
                 neutral words, with no class name or synonym anywhere. C1 - C2 is the
                 effect of the label.
    `transitions`, `graph` and `reliability` have been empty since rev7 and are kept so
    saved records and older call sites still resolve.
    """
    scene: List[str]                 # checklist: measured changes
    objects: List[str]               # buildings and roads
    transitions: List[str]
    classifier: List[str]            # C1 only: the verdict and its vocabulary
    graph: List[str]
    reliability: List[str]
    disaster: str
    confidence: float
    hedge: bool
    label_in_prompt: bool = False
    absent: List[str] = field(default_factory=list)     # CONFIDENT absences only (rev8)
    regions: List[str] = field(default_factory=list)    # where the change is strongest

    def _section(self, title: str, lines: List[str]) -> str:
        return f"{title}:\n" + "\n".join(f"- {l}" for l in lines)

    def to_prompt(self) -> str:
        """Only non-empty sections are printed: an empty section rendered as
        '- none detected' reads as a finding."""
        secs = []
        if self.label_in_prompt and self.classifier:
            secs.append("\n".join(self.classifier))
        if self.scene:
            secs.append(self._section("Checklist of measured changes, BEFORE -> AFTER",
                                      self.scene))
        if self.objects:
            secs.append(self._section("Structures", self.objects))
        if self.regions:
            secs.append(self._section("Where the change is strongest (look at these areas)",
                                      self.regions))
        if self.absent:
            secs.append("Confirmed absent: " + ", ".join(self.absent))
        return "\n\n".join(secs)

    def to_markdown(self) -> str:
        return self.to_prompt().replace("\n- ", "\n  - ")

    def as_dict(self) -> dict:
        return {"scene": self.scene, "transitions": self.transitions, "objects": self.objects,
                "classifier": self.classifier, "graph": self.graph,
                "reliability": self.reliability, "disaster": self.disaster,
                "confidence": self.confidence, "hedge": self.hedge,
                "label_in_prompt": self.label_in_prompt, "absent": self.absent,
                "regions": self.regions}


def _extent(f: float) -> str:
    """A fraction in words. RSCC references almost never use numbers (0.2-0.6% contain a
    '%'), and every figure placed in the evidence was quoted back into the caption."""
    if f >= 0.50: return "most of the scene"
    if f >= 0.20: return "large parts of the scene"
    if f >= 0.05: return "scattered parts of the scene"
    return "small isolated patches"


CAPTION_FLOOR = 0.02        # at or below this a surface is not reported as present

# ---------------------------------------------------------------------------
# Checklist wording. key -> (thing, change). NEUTRAL wording carries no class name or
# synonym, so C2 stays label-free; `_cc_leaks` in §21 checks this on every pair.
# ---------------------------------------------------------------------------
CHECK_NEUTRAL: Dict[str, Tuple[str, str]] = {
    "water_new":      ("ground", "dry -> standing water"),
    "water_more":     ("water", "spread beyond the water already present"),
    "water_wider":    ("open water", "widened"),
    "water_less":     ("open water", "shrunk"),
    "ash":            ("ground", "-> grey ash-like deposit"),
    "lava":           ("ground", "-> fresh dark rock"),
    "burn":           ("ground", "-> blackened, charred-looking"),
    "smoke":          ("view", "smoke or haze plume present"),
    "landslide":      ("slopes", "-> bare scars of displaced soil"),
    "rubble":         ("built-up surface", "-> broken rubble and debris"),
    "edges_lost":     ("field boundaries, tracks, fine lines", "much less visible"),
    "edges_new":      ("outlines", "new edges and breaks"),
    "brightening":    ("ground tone", "paler, dustier"),
    "darkening":      ("ground tone", "noticeably darker"),
    "desaturation":   ("colours", "faded toward grey"),
    "texture_change": ("surfaces", "rougher, more broken up"),
    "veg_loss":       ("green vegetation", "decreased"),
    "veg_gain":       ("green cover", "increased"),
    "no_change":      ("surface cover", "no strong measured change"),
}

# Plain names for RemoteCLIP scene groups, used when a changed region is described as
# "<pre> -> <post>". Neutral for the same reason as CHECK_NEUTRAL.
GROUP_WORDS: Dict[str, str] = {
    "vegetation": "vegetation", "soil": "bare ground", "ash": "grey ash-like deposit",
    "lava": "fresh dark rock", "burn": "blackened ground", "water": "open water",
    "sediment_water": "muddy water", "flood": "standing water", "built": "buildings",
    "rubble": "broken rubble", "road": "road", "plume": "smoke or haze", "cloud": "cloud",
    "landslide": "bare displaced soil"}

_BLD_LEVEL_WORDS = {3: "destroyed", 2: "with major damage", 1: "with minor damage",
                    0: "with no visible damage"}

# ---------------------------------------------------------------------------
# rev8 (change 2): once the pipeline has decided the event type, the C1 checklist is
# worded in that type's vocabulary -- the words RSCC references use ("inundated",
# "ash-covered", "collapsed"). An override only RENAMES a fact that was measured; it
# never adds one. `words` is offered to Qwen with the instruction to use them only where
# they fit what it sees. Every class in DISASTER_CLASSES has an entry (asserted below),
# so a posterior on any class gets its own wording, not just the three evaluated ones.
# ---------------------------------------------------------------------------
DISASTER_VOCAB: Dict[str, dict] = {
    "flood": {
        "name": "flooding",
        "words": ("flooding", "inundated", "submerged", "floodwater", "waterlogged"),
        "check": {"water_new":   ("ground", "dry -> floodwater, inundated"),
                  "water_more":  ("floodwater", "spread beyond the original water"),
                  "water_wider": ("flooded area", "widened")},
        "groups": {"flood": "floodwater", "sediment_water": "muddy floodwater"},
        "buildings": {}},
    "volcanic_eruption": {
        "name": "volcanic eruption",
        "words": ("volcanic eruption", "ashfall", "ash-covered", "volcanic deposits", "buried"),
        "check": {"ash":   ("ground", "-> volcanic ash deposit (ashfall)"),
                  "lava":  ("ground", "-> fresh lava rock"),
                  "smoke": ("view", "ash plume or haze present")},
        "groups": {"ash": "volcanic ash", "lava": "lava", "plume": "ash plume"},
        "buildings": {}},
    "earthquake": {
        "name": "earthquake",
        "words": ("earthquake", "collapsed buildings", "rubble", "debris", "cracked"),
        "check": {"rubble": ("buildings", "-> collapsed rubble and debris")},
        "groups": {"rubble": "collapsed-building rubble"},
        "buildings": {3: "collapsed", 2: "heavily damaged"}},
    "wildfire": {
        "name": "wildfire",
        "words": ("wildfire", "burn scars", "charred", "scorched", "smoke"),
        "check": {"burn":  ("ground", "-> burn scar, charred"),
                  "smoke": ("view", "wildfire smoke present")},
        "groups": {"burn": "burn scar", "plume": "smoke"},
        "buildings": {3: "burned out"}},
    "cyclone": {
        "name": "cyclone",
        "words": ("cyclone", "storm damage", "wind-damaged roofs", "uprooted trees", "debris"),
        "check": {"rubble":    ("built-up surface", "-> storm debris"),
                  "water_new": ("ground", "dry -> storm floodwater")},
        "groups": {"rubble": "storm debris", "flood": "storm floodwater"},
        "buildings": {}},
    "landslide": {
        "name": "landslide",
        "words": ("landslide", "slope failure", "debris flow", "displaced soil", "buried"),
        "check": {"landslide": ("slopes", "-> landslide scars of displaced soil and debris")},
        "groups": {"landslide": "landslide scar"},
        "buildings": {}},
    "tsunami": {
        "name": "tsunami",
        "words": ("tsunami", "inundation", "swept away", "debris", "coastal flooding"),
        "check": {"water_new": ("ground", "dry -> inundated"),
                  "rubble":    ("built-up surface", "-> tsunami debris")},
        "groups": {"flood": "inundated ground", "rubble": "tsunami debris"},
        "buildings": {3: "swept away"}},
}
_missing_vocab = set(DISASTER_CLASSES) - set(DISASTER_VOCAB)
assert not _missing_vocab, f"DISASTER_VOCAB has no entry for {sorted(_missing_vocab)}"


def _check_words(key: str, label: Optional[str]) -> Tuple[str, str]:
    over = DISASTER_VOCAB.get(label, {}).get("check", {}) if label else {}
    return over.get(key, CHECK_NEUTRAL[key])


def _group_word(g: str, label: Optional[str]) -> str:
    over = DISASTER_VOCAB.get(label, {}).get("groups", {}) if label else {}
    return over.get(g, GROUP_WORDS.get(g, g.replace("_", " ")))


# ---------------------------------------------------------------------------
# rev8 (change 3): WHERE each claim is. A 3x3 split of the tile, named the way RSCC
# references locate changes ("the southeastern quadrant", "the western flank"). xBD and
# RSCC tiles are orthorectified north-up; CFG.caption_compass=False switches to
# image-relative words (top-left, ...) for imagery that is not.
# ---------------------------------------------------------------------------
_ZONE_STYLE = {
    True:  {"cells": (("north-west", "north", "north-east"),
                      ("west", "centre", "east"),
                      ("south-west", "south", "south-east")),
            "rows": ("northern", "central", "southern"),
            "cols": ("western", "central", "eastern")},
    False: {"cells": (("top-left", "top", "top-right"),
                      ("left", "centre", "right"),
                      ("bottom-left", "bottom", "bottom-right")),
            "rows": ("top", "middle", "bottom"),
            "cols": ("left", "middle", "right")},
}


def _zone_mass_map(m: Optional[np.ndarray]) -> np.ndarray:
    """Sum of a non-negative HxW map inside each of the 3x3 zones."""
    z = np.zeros((3, 3))
    if m is None:
        return z
    m = np.nan_to_num(np.asarray(m, np.float64), nan=0.0)
    if m.ndim != 2 or m.size == 0:
        return z
    H, W = m.shape
    ys = np.linspace(0, H, 4).astype(int)
    xs = np.linspace(0, W, 4).astype(int)
    for i in range(3):
        for j in range(3):
            z[i, j] = float(np.clip(m[ys[i]:ys[i + 1], xs[j]:xs[j + 1]], 0, None).sum())
    return z


def _zone_mass_boxes(boxes: Sequence[Tuple[int, int, int, int]], W: int, H: int) -> np.ndarray:
    """Area of each box credited to the zone holding its centre. W, H are the TILE size
    (normalising by the selected boxes' own extent would spread any cluster over the
    whole tile)."""
    z = np.zeros((3, 3))
    if not boxes or W <= 0 or H <= 0:
        return z
    for (x0, y0, x1, y1) in boxes:
        cx, cy = 0.5 * (x0 + x1) / W, 0.5 * (y0 + y1) / H
        z[min(int(cy * 3), 2), min(int(cx * 3), 2)] += max(x1 - x0, 1) * max(y1 - y0, 1)
    return z


def _zone_mass_points(pts: Sequence[Tuple[float, float, float]]) -> np.ndarray:
    """(cx, cy, weight) with cx, cy as fractions of the tile."""
    z = np.zeros((3, 3))
    for cx, cy, w in pts:
        z[min(max(int(cy * 3), 0), 2), min(max(int(cx * 3), 0), 2)] += w
    return z


def _zones_arr(z) -> np.ndarray:
    """A 3x3 zone mass as an array (zeros when missing or malformed)."""
    a = np.asarray(z if z is not None else np.zeros((3, 3)), np.float64)
    return np.nan_to_num(a.reshape(3, 3)) if a.size == 9 else np.zeros((3, 3))


def _zones_list(z) -> List[List[float]]:
    """rev12: a 3x3 zone mass normalised to shares, as plain lists (JSON-safe), so every
    measured change carries WHERE it is, not only a phrase naming it."""
    a = _zones_arr(z)
    t = float(a.sum())
    return (a / t if t > 0 else a).round(4).tolist()


def _where(z: np.ndarray, compass: bool = True) -> str:
    """Name where a mass sits, as RSCC references do.

    '' when there is none; 'throughout the scene' when it is spread out; a side or band
    ('the western side', 'the northern part') when one column or row holds most of it; two
    adjacent bands ('the western and central parts'); otherwise the one to three zones
    holding most of it."""
    st = _ZONE_STYLE[bool(compass)]
    z = np.asarray(z, np.float64)
    tot = float(z.sum())
    if not np.isfinite(tot) or tot <= 0:
        return ""
    s = z / tot
    if int((s >= 0.06).sum()) >= 7:
        return "throughout the scene"
    col, row = s.sum(0), s.sum(1)
    for marg, axis, names, word in ((col, 1, st["cols"], "side"), (row, 0, st["rows"], "part")):
        k = int(np.argmax(marg))
        band = s[:, k] if axis == 1 else s[k, :]
        if marg[k] >= 0.60 and int((band >= 0.10).sum()) >= 2 and float(band.max()) < 0.50:
            return f"the {names[k]} {word}" if names[k] not in ("central", "middle") \
                else ("the central strip" if axis == 1 else "the middle band")
    for marg, names in ((col, st["cols"]), (row, st["rows"])):
        for a_, b_ in ((0, 1), (1, 2)):
            if marg[a_] + marg[b_] >= 0.85 and int((s >= 0.06).sum()) >= 4:
                return f"the {names[a_]} and {names[b_]} parts"
    order = sorted(((float(s[i, j]), i, j) for i in range(3) for j in range(3)), reverse=True)
    picked, cum = [], 0.0
    for v, i, j in order:
        if v <= 0:
            break
        picked.append((i, j)); cum += v
        if cum >= 0.60 or len(picked) == 3:
            break
    if cum < 0.45:
        return "throughout the scene"
    names = [st["cells"][i][j] for i, j in picked]
    return "the " + (names[0] if len(names) == 1
                     else ", ".join(names[:-1]) + " and " + names[-1])


def _grid(sem) -> Optional["PatchGrid"]:
    if sem is None or not getattr(sem, "enabled", True):
        return None
    return sem.main_grid


def _grid_dims(sem) -> Tuple[int, int]:
    """(W, H) of the image the semantic grid tiles."""
    g = _grid(sem)
    if g is None or not g.boxes:
        return 0, 0
    return max(b[2] for b in g.boxes), max(b[3] for b in g.boxes)


def _box_where(sem, boxes, compass: bool) -> str:
    W, H = _grid_dims(sem)
    return _where(_zone_mass_boxes(boxes, W, H), compass)


def _group_boxes(sem, groups: Sequence[str], cfg: Config,
                 thr: Optional[float] = None) -> List[Tuple[int, int, int, int]]:
    """Grid cells classified POST as one of `groups` (the same mask rule as the coverage
    fields), preferring the cells that were something else BEFORE -- i.e. where the
    surface is new. Falls back to every POST cell when none of them is new."""
    g = _grid(sem)
    idx = [SCENE_GROUPS.index(x) for x in groups if x in SCENE_GROUPS]
    if g is None or not idx:
        return []
    thr = cfg.group_area_thr if thr is None else thr
    post_hit = g.post.group_probs[:, idx].max(1) >= thr
    new = post_hit & np.array([pg not in groups for pg in g.pre.groups], bool)
    keep = new if new.any() else post_hit
    return [g.boxes[k] for k in np.where(keep)[0]]


def _veg_loss_boxes(sem) -> List[Tuple[int, int, int, int]]:
    g = _grid(sem)
    if g is None:
        return []
    return [g.boxes[k] for k in range(len(g.boxes))
            if g.pre.groups[k] in VEG_GROUPS and g.post.groups[k] not in VEG_GROUPS]


def _dominant_transition(sem, mask: np.ndarray, label: Optional[str]) -> str:
    """The most common PRE -> POST group change among the grid cells a region covers."""
    g = _grid(sem)
    if g is None or mask is None:
        return ""
    mask = np.asarray(mask, bool)
    W, H = _grid_dims(sem)
    if mask.shape != (H, W):
        return ""
    cnt = Counter()
    for k, (x0, y0, x1, y1) in enumerate(g.boxes):
        cell = mask[y0:y1, x0:x1]
        if cell.size and cell.mean() >= 0.5 and g.pre.groups[k] != g.post.groups[k]:
            cnt[(g.pre.groups[k], g.post.groups[k])] += 1
    if not cnt:
        return ""
    (a, b), n = cnt.most_common(1)[0]
    return f"{_group_word(a, label)} -> {_group_word(b, label)}" if n >= 2 else ""


# ---------------------------------------------------------------------------
# Physical channels. Thresholds set from the per-event means measured on the 43 pairs:
#   brightening  Turkey 0.47 vs 0.13-0.19 elsewhere      (earthquake AUROC 0.95)
#   darkening    Pakistan 0.50, Tonga 0.44 vs 0.14-0.32
#   desaturation Turkey 0.25-0.30, Tonga 0.22 vs <=0.17  (earthquake AUROC 0.83)
#   texture      Turkey 0.50-0.57 vs <=0.33              (earthquake AUROC 1.00)
# The checklist states what changed, never a cause (rev7: "burned" pushed volcanoes
# toward wildfire).
# ---------------------------------------------------------------------------
CAPTION_PHYS: Tuple[Tuple[str, float], ...] = (
    ("brightening", 0.30), ("darkening", 0.35),
    ("desaturation", 0.25), ("texture_change", 0.40))
_PHYS_THR = dict(CAPTION_PHYS)
EDGE_LOST, EDGE_NEW = 0.85, 1.15
WATER_GAIN = 0.05

#   (scene field, checklist key, RemoteCLIP groups that locate it)
CAPTION_SURFACES: Tuple[Tuple[str, str, Tuple[str, ...]], ...] = (
    ("ash_coverage",    "ash",       ("ash",)),
    ("lava_coverage",   "lava",      ("lava",)),
    ("burn_scars",      "burn",      ("burn",)),
    ("smoke_regions",   "smoke",     ("plume",)),
    ("landslide_scars", "landslide", ("landslide",)),
    ("rubble_coverage", "rubble",    ("rubble",)),
)


# ---------------------------------------------------------------------------
# rev8 (change 4): CONFIDENT absence. "Confirmed absent" is a hard veto, so it is only
# issued when two independent measurements agree:
#   1. the RemoteCLIP grid is negative even at a RELAXED threshold
#      (CFG.caption_absent_relax x group_area_thr), not just at the mask threshold; and
#   2. none of the physical channels this surface would move has fired -- using the
#      same thresholds that would have put that channel into the checklist.
# rev7 vetoed on the semantic reading alone. The plume group never fired on any of the
# 15 pairs, so smoke was forbidden on TONGA-VOLCANO_000509, where the RSCC reference and
# the baseline both describe thick smoke -- and its desaturation channel HAD fired.
# An absence that is not confident is simply left out: no veto, and the word is not put
# in front of the model (rev7's list was echoed back as "no evidence of standing water,
# fresh dark rock, or smoke or haze plume").
#   (name, RemoteCLIP groups, scene field, physical cues that would show it)
# rubble and landslide scars are never vetoed: rubble read 0.009-0.025 on Turkey pairs
# whose buildings ChangeOS graded as damaged.
# ---------------------------------------------------------------------------
ABSENCE_CHECKS: Tuple[Tuple[str, Tuple[str, ...], str, Tuple[str, ...]], ...] = (
    ("standing water",        ("flood", "sediment_water"), "flood_extent",  ("newly_water", "water_gain")),
    ("grey ash-like deposit", ("ash",),                    "ash_coverage",  ("desaturation", "darkening")),
    ("fresh dark rock",       ("lava",),                   "lava_coverage", ("darkening",)),
    ("blackened ground",      ("burn",),                   "burn_scars",    ("darkening",)),
    ("smoke or haze plume",   ("plume", "cloud"),          "smoke_regions", ("desaturation", "brightening", "edges_lost")),
)


def _phys_fired(cue: str, x: dict) -> bool:
    if cue == "newly_water":
        return float(x.get("newly_water", 0.0)) > 0.0          # already gated at 0.15
    if cue == "water_gain":
        return float(x.get("water_post", 0.0)) - float(x.get("water_pre", 0.0)) >= WATER_GAIN
    if cue == "edges_lost":
        return float(x.get("edge_ratio", 1.0)) < EDGE_LOST
    return float(x.get(cue, 0.0)) >= _PHYS_THR[cue]


def confident_absences(scene_ev: SceneEvidence, sem, cfg: Config) -> Tuple[List[str], Dict[str, str]]:
    """(names that are confidently absent, why each surface was or was not vetoed)."""
    x = getattr(scene_ev, "extra", {}) or {}
    g = _grid(sem)
    relaxed = cfg.caption_absent_relax * cfg.group_area_thr
    out, why = [], {}
    for name, groups, fld, cues in ABSENCE_CHECKS:
        if float(scene_ev[fld]) > CAPTION_FLOOR:
            why[name] = "present"; continue
        if g is None:
            why[name] = "no semantic grid: absence not confident"; continue
        idx = [SCENE_GROUPS.index(s) for s in groups if s in SCENE_GROUPS]
        rel = float((g.post.group_probs[:, idx].max(1) >= relaxed).mean()) if idx else 1.0
        if rel > CAPTION_FLOOR:
            why[name] = f"weak semantic trace ({rel:.3f} of cells at p>={relaxed:.2f})"; continue
        fired = [c for c in cues if _phys_fired(c, x)]
        if fired:
            why[name] = "physical cue fired: " + ", ".join(fired); continue
        why[name] = "confident"
        out.append(name)
    return out, why


def _building_lines(obj_ev: ObjectEvidence, compass: bool = True,
                    label: Optional[str] = None) -> List[str]:
    """Building count and xBD levels from the ChangeOS footprint mask -- the four-level
    scale 35-40% of RSCC references quote -- plus where the damaged ones are."""
    c = obj_ev.counts
    nb = int(c.get("buildings_detected", 0))
    if nb <= 0:
        return ["buildings: none detected"]
    words = dict(_BLD_LEVEL_WORDS)
    words.update(DISASTER_VOCAB.get(label, {}).get("buildings", {}) if label else {})
    L = [int(c.get(f"bld_level{i}", 0)) for i in range(4)]
    lines = []
    if L[0] == nb:
        lines.append(f"buildings: {nb} detected, all intact (xBD damage level 0)")
    else:
        parts = [f"{L[k]} {words[k]} (xBD level {k})" for k in (3, 2, 1, 0) if L[k]]
        line = f"buildings: {nb} detected; " + ", ".join(parts)
        pts = [(cx, cy, 1.0) for cx, cy, lvl in
               (getattr(obj_ev, "extra", {}) or {}).get("buildings", []) if lvl >= 1]
        where = _where(_zone_mass_points(pts), compass) if pts else ""
        if where:
            line += f" | where damaged: {where}"
        lines.append(line)
    rd = int(c.get("buried_roads", 0)) + int(c.get("damaged_roads", 0))
    if rd:
        lines.append(f"roads: {rd} segment{'s' if rd > 1 else ''} buried or damaged")
    return lines


def _strongest_change_lines(change_map: Optional[np.ndarray], regions, sem, cfg: Config,
                            label: Optional[str], k: int = 3) -> List[str]:
    """Where the heat map concentrates, plus the strongest AnyChange regions, each with
    its size, location and the dominant PRE -> POST change inside it."""
    compass = cfg.caption_compass
    out = []
    if change_map is not None:
        cm = np.asarray(change_map, np.float32)
        hit = cm >= cfg.change_threshold
        frac = float(hit.mean())
        if frac >= 0.005:
            out.append(f"overall change | extent: {_extent(frac)} | where: "
                       f"{_where(_zone_mass_map(np.where(hit, cm, 0.0)), compass)}")
    cand = []
    for r in (regions or []):
        m = np.asarray(getattr(r, "mask", None), bool)
        if m.ndim != 2 or not m.any():
            continue
        frac = float(m.mean())
        if float(getattr(r, "prob", 0.0)) >= 0.5 and frac >= 0.01:
            cand.append((float(r.prob) * frac, frac, m))
    for _, frac, m in sorted(cand, key=lambda t: -t[0])[:k]:
        size = "small" if frac < 0.03 else ("medium" if frac < 0.10 else "large")
        s = f"{size} changed area | where: {_where(_zone_mass_map(m), compass)}"
        tr = _dominant_transition(sem, m, label)
        if tr:
            s += f" | measured: {tr}"
        out.append(s)
    return out


def build_evidence_package(scene_ev: SceneEvidence, obj_ev: ObjectEvidence,
                           sem: SceneSemantics, posterior: DisasterPosterior,
                           clf_out: Optional[dict], dkg_out: dict, rvg_out: dict,
                           max_transitions: int = 6,
                           label_in_prompt: bool = False,
                           neutral: bool = True,
                           change_map: Optional[np.ndarray] = None,
                           physical: Optional[Dict[str, np.ndarray]] = None,
                           regions: Optional[list] = None,
                           cfg: Optional[Config] = None) -> EvidencePackage:
    """The checklist Qwen captions from (rev8).

      * every measured change is one terse line, `thing: change | extent | where`, so the
        caption has to be written from the images, not paraphrased from the prompt;
      * each line carries WHERE it was measured: RemoteCLIP grid cells for surfaces,
        physical-channel maps for tone and texture, ChangeOS footprints for damaged
        buildings, and a separate block for the strongest heat-map / AnyChange regions;
      * C1 (`label_in_prompt=True`, the system caption) is headed by the pipeline's fused
        verdict and worded in that event type's vocabulary; C2 is identical but neutral;
      * "Confirmed absent" lists only confident absences (see ABSENCE_CHECKS).
    Pass `change_map`, `physical` and `regions` (predict() and `package_from_result` do)
    to get locations; without them the checklist is built without the location fields.
    `max_transitions` and `neutral` are kept for signature compatibility.
    """
    cfg = cfg or CFG
    compass = cfg.caption_compass
    x = getattr(scene_ev, "extra", {}) or {}
    label = (posterior.label if label_in_prompt and posterior.label in DISASTER_VOCAB
             else None)
    lines: List[str] = []

    def add(key: str, extent: str = "", where: str = ""):
        what, change = _check_words(key, label)
        s = f"{what}: {change}"
        if extent:
            s += f" | extent: {extent}"
        if where and not (where == "throughout the scene" and extent == "most of the scene"):
            s += f" | where: {where}"
        lines.append(s)

    # water ------------------------------------------------------------------
    fe = float(scene_ev["flood_extent"])
    w0, w1 = float(x.get("water_pre", 0.0)), float(x.get("water_post", 0.0))
    if fe > CAPTION_FLOOR:
        W, H = _grid_dims(sem)
        z = _zone_mass_boxes(_group_boxes(sem, ("flood", "sediment_water"), cfg), W, H)
        if z.sum() == 0 and physical is not None and "newly_water" in physical:
            z = _zone_mass_map(physical["newly_water"])      # the pixel test found it
        add("water_new" if w0 < 0.05 else "water_more", _extent(fe), _where(z, compass))
    if w1 - w0 >= WATER_GAIN:
        add("water_wider")
    elif w0 - w1 >= WATER_GAIN:
        add("water_less")

    # other surfaces -----------------------------------------------------------
    for fld, key, groups in CAPTION_SURFACES:
        v = float(scene_ev[fld])
        if v > CAPTION_FLOOR:
            add(key, _extent(v), _box_where(sem, _group_boxes(sem, groups, cfg), compass))

    # boundaries ----------------------------------------------------------------
    er = float(x.get("edge_ratio", 1.0))
    e0 = physical.get("edge_pre") if physical else None
    e1 = physical.get("edge_post") if physical else None
    if er < EDGE_LOST:
        add("edges_lost", where=_where(_zone_mass_map(np.clip(e0 - e1, 0, None)), compass)
            if e0 is not None and e1 is not None else "")
    elif er > EDGE_NEW:
        add("edges_new", where=_where(_zone_mass_map(np.clip(e1 - e0, 0, None)), compass)
            if e0 is not None and e1 is not None else "")

    # physical channels -----------------------------------------------------------
    for key, thr in CAPTION_PHYS:
        if float(x.get(key, 0.0)) >= thr:
            m = physical.get(key) if physical else None
            add(key, where=_where(_zone_mass_map(np.where(np.asarray(m) >= thr, m, 0.0)), compass)
                if m is not None else "")

    # vegetation: skipped when standing water dominates, because green water raises the
    # vegetation reading ("vegetation dramatically increased" on floods) --------------
    if fe <= 0.15:
        if float(scene_ev["vegetation_loss"]) >= 0.25:
            add("veg_loss", where=_box_where(sem, _veg_loss_boxes(sem), compass))
        elif float(x.get("veg_post", 0.0)) - float(x.get("veg_pre", 0.0)) >= 0.15:
            add("veg_gain")

    if not lines:
        add("no_change")

    absent, _why = confident_absences(scene_ev, sem, cfg)
    objects = _building_lines(obj_ev, compass, label)
    where_lines = _strongest_change_lines(change_map, regions, sem, cfg, label)

    classifier: List[str] = []
    if label_in_prompt:
        if label:
            v = DISASTER_VOCAB[label]
            classifier = [f"Event identified by the analysis pipeline: {v['name']}.",
                          "Wording for this event type (use it only where it fits what you "
                          "see): " + ", ".join(v["words"]) + "."]
        else:
            classifier = ["The analysis pipeline did not settle on an event type; judge it "
                          "from the images and the checklist."]

    pkg = EvidencePackage(lines, objects, [], classifier, [], [],
                          posterior.label, float(rvg_out["confidence"]),
                          bool(rvg_out["hedge"]), label_in_prompt=label_in_prompt,
                          absent=absent, regions=where_lines)
    pkg.absence_reasons = _why          # not part of the prompt; kept for the §21 audit
    return pkg


def package_from_result(r, label_in_prompt: bool, cfg: Optional[Config] = None) -> EvidencePackage:
    """The evidence package of a finished PredictionResult under either condition, built
    exactly as predict() builds it (same maps, same locations). §19 and §21 use this, so
    no harness constructs a package that predict() would not."""
    return build_evidence_package(r.scene_evidence, r.object_evidence, r.semantics,
                                  r.posterior, r.classifier, r.dkg, r.rvg_provisional,
                                  label_in_prompt=label_in_prompt,
                                  change_map=r.refined_map, physical=r.scene.physical,
                                  regions=r.scene.regions, cfg=cfg)


# ---------------------------------------------------------------------------
# Caption prompts. The baseline (images only) and both pipeline conditions share the
# SAME style block, so a difference in the scores comes from the evidence and the label,
# not from being asked for a different kind of text.
#
# Measured on 11,738 RSCC references for our three classes: 2-3 sentences, ~50 words,
# 0.2-0.6% contain a percentage, 86-99% name the disaster, opened by the change as the
# subject ("Flooding transforms...", "Post-earthquake assessment reveals...").
# rev8: every condition is now asked to open by naming the disaster (rev7 asked no one,
# and the verdict line is stripped before scoring, so the pipeline's main advantage
# never reached the scored text), and to say where changes are. Both instructions are
# in the SHARED block, so the baseline gets them too. The example uses a placeholder,
# never a class name: a class word in shared scaffolding would prime every caption
# toward that class and would also hide that word from the filename-leak audit.
# No few-shot examples: in an earlier run their nouns were copied into 11 of 15
# captions as if they were observations.
# ---------------------------------------------------------------------------
_LOC_HINT = ("using words such as north, south-east or centre (north is at the top of "
             "the image)" if CFG.caption_compass else
             "using words such as top-left, bottom or centre")

CAPTION_STYLE = (
    "Write ONE change caption of 2 to 3 sentences, about 50 words, describing how the "
    "place changed from BEFORE to AFTER.\n"
    "- Begin the first sentence by naming the kind of disaster that caused the change, as "
    "its subject (for example \"<Disaster> has ...\"); then say what the scene looked like "
    "before and what it looks like after, linking the changes in one flowing description.\n"
    f"- Where a change is concentrated in one part of the scene, say where, {_LOC_HINT}.\n"
    "- Mention buildings, roads, fields, vegetation, water or bare ground only where they "
    "are actually present.\n"
    "- Describe extent with words such as widespread, partial, scattered or isolated. Do "
    "not use numbers or percentages.\n"
    "- Do not mention image quality, confidence, measurements, the heat map, or how the "
    "analysis was done.\n"
    "- No headings, bullet points or markdown. Never invent place names, dates or "
    "casualties.\n")

_VERDICT_TYPES = ", ".join(c.replace("_", " ") for c in DISASTER_CLASSES)
CAPTION_VERDICT = ("\nAfter the caption, on a new final line, write exactly:\n"
                   "Most likely disaster: <type>\n"
                   f"where <type> is one of: {_VERDICT_TYPES}, or unclear.")

# Baseline (condition C3) and the RVG blind check. Images only.
BLIND_PROMPT = (
    "You are given two satellite images of the same place: (1) BEFORE an event and (2) "
    "AFTER it.\n\n"
    + CAPTION_STYLE
    + "- Describe only what you can see in the two images.\n"
    + CAPTION_VERDICT)


class QwenCaptioner:
    """Qwen2.5-VL-7B-Instruct in 4-bit.

    Calls:
      * `blind_caption` -- images only, no evidence. Condition C3 (the baseline) and the
        RVG blind check; it must never see the evidence.
      * `look`          -- rev12: a close-up of one part of the scene (BEFORE and AFTER
        crops) and the question "what changed here?". No evidence words; on a second,
        guided look only a plain cue ("parts of it look darker").
      * `compose`       -- rev12 SYSTEM CAPTION: the full images, the verdict and Qwen's
        OWN notes on every region; the caption is written once, from scratch (see
        `look_then_write`). A rewrite, when verification finds an omission, is again
        from scratch -- the earlier caption is never shown.
      * `revise`        -- rev9-rev11 system caption, kept as comparison C5: its own
        draft, the images and only the corrections the measurement check produced.
      * `caption`       -- images, heat map and the evidence CHECKLIST (rev8; kept as the
        comparison condition C4, because it copies the checklist).
    All share the RSCC caption style: open by naming the disaster, say where changes are,
    no figures, no talk of measurements or confidence.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        try:
            from transformers import (AutoProcessor, Qwen2_5_VLForConditionalGeneration,
                                      BitsAndBytesConfig)
        except Exception as e:
            raise ModelLoadError("Qwen2.5-VL (transformers)", f"{type(e).__name__}: {e}",
                                 "pip install -U 'transformers>=4.51.0' accelerate qwen-vl-utils "
                                 "then RESTART THE KERNEL.")
        quant = None
        if cfg.load_in_4bit:
            try:
                import bitsandbytes  # noqa: F401
                quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=cfg.dtype,
                                           bnb_4bit_use_double_quant=True)
            except Exception as e:
                raise ModelLoadError("Qwen2.5-VL (bitsandbytes)", f"{type(e).__name__}: {e}",
                                     "pip install -U bitsandbytes, or set CFG.load_in_4bit=False.")
        try:
            self.processor = AutoProcessor.from_pretrained(cfg.qwen_id)
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                cfg.qwen_id, torch_dtype=cfg.dtype, device_map="auto", quantization_config=quant)
            self.model.eval()
        except Exception as e:
            raise ModelLoadError("Qwen2.5-VL (weights)",
                                 f"from_pretrained('{cfg.qwen_id}') failed: {type(e).__name__}: {e}",
                                 "Enable Internet (~16 GB download). Keep load_in_4bit=True if OOM.")
        log.info("Qwen2.5-VL ready: %s (4bit=%s)", cfg.qwen_id, cfg.load_in_4bit)

    @torch.no_grad()
    def _generate(self, images: List[Image.Image], prompt: str, max_new_tokens: int) -> str:
        msgs = [{"role": "user", "content": [{"type": "image"} for _ in images]
                 + [{"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], images=images, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        return self.processor.batch_decode(gen, skip_special_tokens=True)[0].strip()

    def blind_caption(self, pre: Image.Image, post: Image.Image) -> str:
        return self._generate([pre, post], BLIND_PROMPT, self.cfg.blind_max_new_tokens)

    @staticmethod
    def build_prompt(evidence: str, hedge: bool = False,
                     label_in_prompt: bool = False) -> str:
        """Grounded caption prompt (C1 and C2), in the RSCC reference style.

        rev8 (change 3): the evidence is presented as a CHECKLIST to verify against the
        images, not as text to restate -- Qwen describes what it sees in its own words and
        uses the checklist to decide what to include, where it is, and what to rule out.
        The label itself lives in the evidence (C1 only), so this scaffolding contains no
        class name for either condition. `hedge` is accepted for compatibility and unused:
        RSCC references never discuss confidence."""
        named = ("- The first sentence must name the event identified at the top of the "
                 "findings, using the wording given for it where it fits.\n"
                 if label_in_prompt else "")
        return (
            "You are given three images of the same place: (1) BEFORE an event, (2) AFTER "
            "the event, and (3) a change heat map in which brighter areas changed more.\n\n"
            "An image-analysis pipeline measured this pair; its findings are below. Treat "
            "them as a checklist, not as text to copy: look at the BEFORE and AFTER images, "
            "describe what you see in your own words, and use the checklist to decide what "
            "to include, where it is, and what to rule out.\n\n"
            f"{evidence}\n\n"
            + CAPTION_STYLE
            + named
            + "- Include the checklist changes you can see in the images. You may add a "
              "change the checklist does not list only if it is clearly visible.\n"
            + "- Never mention anything listed under 'Confirmed absent', not even to say it "
              "is absent.\n"
            + "- If buildings are reported intact you may say they remain intact; if none "
              "were detected, do not describe buildings.\n"
            + CAPTION_VERDICT)

    def caption(self, pre: Image.Image, post: Image.Image, damage: Image.Image,
                evidence: str, hedge: bool = False, label_in_prompt: bool = False) -> str:
        prompt = self.build_prompt(evidence, hedge, label_in_prompt)
        return self._generate([pre, post, damage], prompt, self.cfg.max_new_tokens)

    def look(self, pre_crop: Image.Image, post_crop: Image.Image, whole: bool = False,
             cues: Sequence[str] = ()) -> str:
        """rev12: what changed in ONE part of the scene. `cues` (guided look only) are
        plain descriptions of what was measured there; the open look has none."""
        return self._generate([pre_crop, post_crop], look_prompt(whole, cues),
                              int(getattr(self.cfg, "look_max_new_tokens", 96)))

    def compose(self, pre: Image.Image, post: Image.Image, prompt: str) -> str:
        """rev12: write the caption from the full images and the notes in `prompt`."""
        return self._generate([pre, post], prompt, self.cfg.max_new_tokens)

    def revise(self, pre: Image.Image, post: Image.Image, draft_body: str,
               sent_edits: List[dict]) -> str:
        """rev9: rewrite ONLY the flagged claims of Qwen's own draft (images attached, so
        new words come from what it sees)."""
        return self._generate([pre, post], revision_prompt(draft_body, sent_edits),
                              self.cfg.max_new_tokens)


# ═══════════════════════════════════════════════════════════════════════════════
# rev9 · DRAFT -> CHECK -> REVISE  (the system caption)
#
# rev8 handed Qwen a checklist and Qwen wrote the checklist out: 34% of its sentences
# restated a checklist line, the copy rate stayed at 0.147 (rev7 0.15), and flood
# captions of different pairs were more alike (ROUGE-L 0.39) than the references (0.17).
# A 7B model treats any list of facts as the outline of its answer.
#
# rev9 separates WHO WRITES from WHAT IS TRUE:
#   1. DRAFT  -- Qwen describes the two images with no measurements at all (this is the
#                blind pass, so it costs nothing extra). Its words, its structure.
#   2. CHECK  -- code, not the LLM, compares every sentence of the draft with the
#                measurements: disaster type, water, ash, fire, smoke, lava, buildings.
#                A claim is supported, contradicted (only by a CONFIDENT measurement), or
#                not checkable; a large measured change the draft misses is flagged.
#   3. REVISE -- Qwen gets its own draft, the images and ONLY the corrections, and keeps
#                everything else. If nothing needs correcting, the draft IS the caption.
# Every edit is logged with the claim, the measurement and its value, and after the
# revision it is checked again (`resolved`). The measurements decide what the caption
# says; they never supply its words.
# ═══════════════════════════════════════════════════════════════════════════════

# what a sentence ASSERTS, per surface (matched after a negation check)
CLAIM_LEX: Dict[str, str] = {
    "water":     r"\b(flood\w*|inundat\w*|submerg\w*|standing water|floodwater\w*|under ?water|"
                 r"water-?logged|swamp\w*|waters? (has|have) (risen|spread|covered))",
    "ash":       r"\b(ash|ashfall|ash-\w+|volcanic (deposit|material|debris|ash)|tephra)\b",
    "burn":      r"\b(burn\w*|charred|scorch\w*|blacken\w*|wildfire\w*|fires?|blaze\w*)\b",
    "smoke":     r"\b(smoke|smoky|plume\w*|haze|hazy)\b",
    "lava":      r"\blava\b",
    "landslide": r"\b(landslide\w*|mudslide\w*|slope failure\w*|displaced soil|debris flow\w*)\b",
    "bare":      r"\b(bare (ground|earth|soil|land)|exposed (earth|soil|ground)|barren|"
                 r"dry (ground|land|soil|earth)|arid|cleared (land|area|ground))\b",
    "collapse":  r"\b(collaps\w*|destroy\w*|rubble|flatten\w*|razed|demolish\w*|levell?ed)\b",
    "intact":    r"\b(intact|undamaged|unaffected|unharmed|unscathed|no (visible |apparent )?damage)\b",
    "building":  r"\b(buildings?|houses?|homes?|roofs?|structures?|dwellings?|villages?)\b",
    "damage":    r"\b(damag\w*|collaps\w*|destroy\w*|rubble|roofless|roof loss\w*|crack\w*|ruin\w*)\b",
}
_NEG_BEFORE = (r"\b(no|not|without|absence of|absent|none|nor|lack\w*|free of|unaffected by|"
               r"rather than|instead of|neither|never)\b[^.;]{0,40}$")

# the confident-absence names (ABSENCE_CHECKS) each surface corresponds to
SURFACE_ABSENT_NAME = {"water": "standing water", "ash": "grey ash-like deposit",
                       "burn": "blackened ground", "smoke": "smoke or haze plume",
                       "lava": "fresh dark rock"}
# scene field and RemoteCLIP groups of each surface, and the size from which a surface
# the draft does not mention is worth adding (a LARGE measured change, not every trace)
SURFACE_SOURCE = {"water":     ("flood_extent",    ("flood", "sediment_water"), 0.20),
                  "ash":       ("ash_coverage",    ("ash",),                    0.10),
                  "burn":      ("burn_scars",      ("burn",),                   0.10),
                  "smoke":     ("smoke_regions",   ("plume",),                  0.10),
                  "lava":      ("lava_coverage",   ("lava",),                   0.05),
                  "landslide": ("landslide_scars", ("landslide",),              0.10)}
SURFACE_WORDS = {"water": "standing water", "ash": "a grey ash-like deposit",
                 "burn": "blackened, charred ground", "smoke": "a smoke or haze plume",
                 "lava": "fresh dark rock", "landslide": "bare scars of displaced soil"}
_SURFACE_GROUP = {"water": "flood", "ash": "ash", "burn": "burn", "smoke": "plume",
                  "lava": "lava", "landslide": "landslide"}
EVENT_PHRASE = {"flood": "flooding", "volcanic_eruption": "a volcanic eruption",
                "earthquake": "an earthquake", "wildfire": "a wildfire",
                "cyclone": "a cyclone", "landslide": "a landslide", "tsunami": "a tsunami"}


def _sentences(text: str) -> List[str]:
    return [s.strip() for s in _re_split_sent(text) if s.strip()]


def _re_split_sent(text: str) -> List[str]:
    import re as _r
    return _r.split(r"(?<=[.!?])\s+", (text or "").strip())


def caption_body(raw: str) -> str:
    """The caption without its closing 'Most likely disaster:' line."""
    import re as _r
    return _r.split(r"\n?\s*\**\s*most likely disaster", raw or "", flags=_r.I)[0].strip()


def asserts(sentence: str, surface: str) -> bool:
    """True if `sentence` asserts the surface: a match NOT preceded, within the same
    clause, by a negation ('no standing water', 'without any sign of fire')."""
    import re as _r
    s = sentence or ""
    for m in _r.finditer(CLAIM_LEX[surface], s, flags=_r.I):
        if not _r.search(_NEG_BEFORE, s[:m.start()], flags=_r.I):
            return True
    return False


def _surface_words(surface: str, label: Optional[str]) -> str:
    over = DISASTER_VOCAB.get(label, {}).get("groups", {}) if label else {}
    g = _SURFACE_GROUP[surface]
    return over.get(g, SURFACE_WORDS[surface]) if g in over else SURFACE_WORDS[surface]


def measured_facts(scene_ev: SceneEvidence, obj_ev: ObjectEvidence, sem: SceneSemantics,
                   posterior: DisasterPosterior, cfg: Optional[Config] = None,
                   physical: Optional[Dict[str, np.ndarray]] = None,
                   quality: Optional[dict] = None, anychange_regions=None,
                   instances=None) -> dict:
    """Everything the checker compares a caption against, as plain numbers and words.
    Kept as a dict so it can be logged, corrupted for the grounding tests, and
    re-checked against the caption.

    rev12: every located item also carries `zones` (its 3x3 zone mass, as shares), so the
    look step can crop a close-up of it; AnyChange regions and SAM2 object changes are
    included, and `changes_all` keeps the RemoteCLIP changes labelled "cloud"."""
    cfg = cfg or CFG
    compass = cfg.caption_compass
    x = getattr(scene_ev, "extra", {}) or {}
    W_, H_ = _grid_dims(sem)
    surfaces = {}
    for s, (fld, groups, _thr) in SURFACE_SOURCE.items():
        v = float(scene_ev[fld])
        where, zm = "", np.zeros((3, 3))
        if v > CAPTION_FLOOR:
            zm = _zone_mass_boxes(_group_boxes(sem, groups, cfg), W_, H_)
            where = _where(zm, compass)
            if not where and s == "water" and physical is not None and "newly_water" in physical:
                zm = _zone_mass_map(physical["newly_water"])
                where = _where(zm, compass)
        surfaces[s] = {"value": round(v, 4), "where": where, "zones": _zones_list(zm)}
    surfaces["water"]["semantic"] = round(float(x.get("flood_semantic", 0.0)), 4)
    surfaces["water"]["pixel"] = round(float(x.get("newly_water", 0.0)), 4)
    absent, why = confident_absences(scene_ev, sem, cfg)
    c = obj_ev.counts
    L = [int(c.get(f"bld_level{i}", 0)) for i in range(4)]
    pts = (getattr(obj_ev, "extra", {}) or {}).get("buildings", [])
    dmg = [(cx, cy, 1.0) for cx, cy, lvl in pts if lvl >= 1]
    q = quality or {}
    cl = q.get("clouds", {}) or {}
    zc = {k: (_zone_mass_map(np.asarray(q[f"cloud_{k}"], np.float32))
              if q.get(f"cloud_{k}") is not None and float(cl.get(k, 0.0)) > 0 else np.zeros((3, 3)))
          for k in ("pre", "post")}
    clouds = {"pre": float(cl.get("pre", 0.0)), "post": float(cl.get("post", 0.0)),
              "where_pre": _where(zc["pre"], compass), "where_post": _where(zc["post"], compass),
              "zones_pre": _zones_list(zc["pre"]), "zones_post": _zones_list(zc["post"])}
    # rev12: AnyChange regions (change of unknown kind) and SAM2 object changes, located
    any_out, obj_out = [], []
    for q_ in (anychange_regions or []):
        m_ = np.asarray(getattr(q_, "mask", np.zeros((1, 1))), bool)
        fr_ = float(m_.mean()) if m_.size else 0.0
        if float(getattr(q_, "prob", 0.0)) >= 0.5 and fr_ >= CAPTION_FLOOR:
            zm_ = _zone_mass_map(m_.astype(np.float32))
            any_out.append({"area": round(fr_, 4), "prob": round(float(q_.prob), 3),
                            "where": _where(zm_, compass), "zones": _zones_list(zm_)})
    for i_ in (instances or []):
        tr_ = getattr(i_, "transition", None)
        m_ = np.asarray(getattr(i_, "mask", np.zeros((1, 1))), bool)
        fr_ = float(m_.mean()) if m_.size else 0.0
        if tr_ is not None and tr_.changed and float(i_.change_score) >= float(cfg.damage_change_thr) \
                and fr_ >= CAPTION_FLOOR:
            zm_ = _zone_mass_map(m_.astype(np.float32))
            obj_out.append({"src": tr_.src_group, "dst": tr_.dst_group, "area": round(fr_, 4),
                            "where": _where(zm_, compass), "zones": _zones_list(zm_)})
    return {"label": posterior.label, "prob": round(float(posterior.prob), 4),
            "surfaces": surfaces, "absent": list(absent), "absence_reasons": why,
            "buildings": int(c.get("buildings_detected", 0)), "levels": L,
            "damaged": int(L[1] + L[2] + L[3]),
            "damaged_where": _where(_zone_mass_points(dmg), compass) if dmg else "",
            "damaged_zones": _zones_list(_zone_mass_points(dmg)) if dmg else _zones_list(None),
            "clouds": clouds, "observable": round(float(getattr(sem, "obs_frac", 1.0)), 3),
            "align": dict(q.get("align", {}) or {}),
            "regions": physical_regions(physical, cfg),
            "darkening": {k: v for k, v in (getattr(scene_ev, "extra", {}) or {}).get("darkening_types", {}).items()
                          if k != "zones"},
            "changes": semantic_changes(sem, cfg),
            "changes_all": semantic_changes_all(sem, cfg, min_area=float(getattr(cfg, "caption_min_area", 0.02))),
            "anychange": any_out, "objects": obj_out}


def _edit(kind, check, instruction, sentence=None, measurement=None, value=None,
          where="", surfaces=()):
    return {"kind": kind, "check": check, "sentence": sentence, "measurement": measurement,
            "value": value, "where": where, "surfaces": list(surfaces),
            "instruction": instruction, "sent": False, "resolved": None}


# ---------------------------------------------------------------------------
# rev10: the MEASURED changes a caption should contain, and the image quality it must
# respect. Physical channels are read as LOCATED REGIONS of strong change -- a channel's
# whole-image mean can stay low while one part of the scene changes strongly -- and
# RemoteCLIP transitions by their true area. All wording is neutral: a darker patch is
# "darker ground", and Qwen, looking at the images, says what it is.
# ---------------------------------------------------------------------------
REGION_CHANNELS = ("darkening", "brightening", "vegetation_loss", "desaturation")
TYPED_DARK = ("deposit", "burn")
CHANGE_LEX = {
    "darkening":       r"\b(dark\w*|black\w*|shadow\w*|charred|scorch\w*|burn\w*|ash\w*|soot\w*)\b",
    "brightening":     r"\b(pale\w*|bright\w*|whit\w*|light-colou?red|dust\w*|ash\w*|gr[ae]y\w*)\b",
    "desaturation":    r"\b(gr[ae]y\w*|dull\w*|faded|colou?rless|wash\w* out|ash\w*)\b",
    "cloud":           r"\b(cloud\w*|haze|hazy|mist\w*|fog\w*|overcast)\b",
}
_VEG_WORD = r"\b(vegetation|trees?|forest\w*|crops?|fields?|foliage|plants?|grass\w*|greenery|canopy|orchard\w*|plantation\w*)\b"
_LOSS_WORD = (r"\b(loss|lost|los(e|ing)|stripped|clear(ed|ing)|thinn\w*|bare|barren|dead|died|dying|remov\w*|"
              r"destroy\w*|damag\w*|burn\w*|buried|cover\w*|flatten\w*|wilt\w*|brown\w*|replac\w*)\b")
_GROUP_LEX = {
    "water": r"\b(water\w*|flood\w*|inundat\w*|submerg\w*|river\w*|lake\w*|pond\w*|lagoon\w*)\b",
    "flood": r"\b(water\w*|flood\w*|inundat\w*|submerg\w*)\b",
    "sediment_water": r"\b(water\w*|flood\w*|mud\w*|silt\w*|sediment\w*)\b",
    "ash": r"\b(ash\w*|gr[ae]y\w*|deposit\w*|tephra)\b",
    "lava": r"\b(lava|molten|dark rock)\b",
    "burn": r"\b(burn\w*|charred|scorch\w*|blacken\w*|fire\w*)\b",
    "soil": r"\b(bare|soil\w*|exposed|barren|dirt|cleared|earthen)\b",
    "vegetation": _VEG_WORD,
    "built": r"\b(buildings?|houses?|homes?|roofs?|settlement\w*|urban|village\w*|town\w*)\b",
    "rubble": r"\b(rubble|debris|collaps\w*|ruins?)\b",
    "road": r"\b(roads?|streets?|highways?|tracks?)\b",
    "plume": r"\b(smoke|plume\w*|haze|hazy)\b",
    "landslide": r"\b(landslide\w*|slid\w*|scar\w*|displaced|slope\w*)\b",
}


# rev12: these patterns held 24 literal backspace characters (0x08) where \b was meant,
# so the single directions (north, western, centre, top, left, ...) never matched: only
# composite ones ("lower left", "north-west") did, and every location test treated
# "the western and central parts" as naming no place at all.
_ZONE_WORDS = [  # (pattern, 3x3 cells) -- composite directions first
    (r"north-?west|upper[- ]left|top[- ]left", {(0, 0)}), (r"north-?east|upper[- ]right|top[- ]right", {(0, 2)}),
    (r"south-?west|lower[- ]left|bottom[- ]left", {(2, 0)}), (r"south-?east|lower[- ]right|bottom[- ]right", {(2, 2)}),
    (r"\bnorth(ern)?\b|\btop\b|\bupper\b", {(0, 0), (0, 1), (0, 2)}),
    (r"\bsouth(ern)?\b|\bbottom\b|\blower\b", {(2, 0), (2, 1), (2, 2)}),
    (r"\bwest(ern)?\b|\bleft\b", {(0, 0), (1, 0), (2, 0)}), (r"\beast(ern)?\b|\bright\b", {(0, 2), (1, 2), (2, 2)}),
    (r"\bcent(re|er|ral)\b|\bmiddle\b", {(1, 1), (0, 1), (2, 1), (1, 0), (1, 2)}),
    (r"throughout|whole|entire|across the (scene|image|area)|everywhere|widespread", {(i, j) for i in range(3) for j in range(3)}),
]


def _zones_of(text: str) -> set:
    """The 3x3 cells a phrase refers to; empty when it names no place."""
    import re as _r
    t, out = (text or "").lower(), set()
    for pat, cells in _ZONE_WORDS:
        if _r.search(pat, t):
            out |= cells
            t = _r.sub(pat, " ", t)
    return out


_UNCHANGED = (r"\b(undisturbed|unchanged|intact|untouched|unaffected|unharmed|unaltered|undamaged|"
              r"as before|largely the same|remains? (green|the same|unchanged|intact|lush)|"
              r"no (visible |apparent |significant )?change)")


def _says_unchanged(sentence: str) -> bool:
    import re as _r
    return bool(_r.search(_UNCHANGED, sentence or "", _r.I))


def _same_place(sentence: str, where: str) -> bool:
    """A sentence covers a change at `where` unless it names a DIFFERENT place."""
    zs, zw = _zones_of(sentence), _zones_of(where)
    return not zs or not zw or bool(zs & zw)


def _in_place(where: str) -> str:
    if not where: return ""
    return "across the scene" if where == "throughout the scene" else f"in {where}"


def physical_regions(physical: Optional[Dict[str, np.ndarray]], cfg: Optional[Config] = None) -> List[dict]:
    """Located strong changes per physical channel, over the OBSERVABLE ground."""
    cfg = cfg or CFG
    if not physical:
        return []
    from scipy import ndimage as ndi
    valid = np.asarray(physical.get("valid", np.ones_like(physical["composite"])), bool)
    n_ok = max(int(valid.sum()), 1)
    out = []
    for ch in REGION_CHANNELS + ("texture_change",):
        m = physical.get(ch)
        if m is None:
            continue
        strong = (np.asarray(m) >= cfg.phys_region_thr) & valid
        strong = ndi.binary_opening(strong, iterations=2)
        lab, n = ndi.label(strong)
        if n == 0:
            continue
        sizes = np.bincount(lab.ravel())[1:]
        big = np.isin(lab, 1 + np.where(sizes >= cfg.phys_region_min_frac * n_ok)[0])
        frac = float(big.sum() / n_ok)
        if frac <= 0:
            continue
        zm = _zone_mass_map(big.astype(np.float32))
        out.append({"kind": ch, "area": round(frac, 4),
                    "where": _where(zm, cfg.caption_compass), "zones": _zones_list(zm)})
    return sorted(out, key=lambda r: -r["area"])


def semantic_changes(sem, cfg: Optional[Config] = None, min_area: float = 0.05) -> List[dict]:
    """RemoteCLIP PRE->POST changes on the main grid, by their TRUE area (a patch
    transition's weight is its area share divided by the number of grids)."""
    cfg = cfg or CFG
    g = sem.main_grid if getattr(sem, "enabled", True) else None
    if g is None:
        return []
    n_grids = max(len(getattr(sem, "grids", {}) or {}), 1)
    W, H = _grid_dims(sem)
    agg: Dict[Tuple[str, str], List] = {}
    for t in sem.transitions:
        if not t.changed or t.kind != "patch" or t.grid != g.g:
            continue
        if "cloud" in (t.src_group, t.dst_group):
            continue       # RemoteCLIP's "cloud" on observable ground is not a cloud (often ash or haze)
        agg.setdefault((t.src_group, t.dst_group), []).append(t)
    out = []
    for (a, b), ts in agg.items():
        area = float(sum(t.weight for t in ts) * n_grids)
        if area >= min_area:
            zm = _zone_mass_boxes([t.where for t in ts], W, H)
            out.append({"kind": "transition", "src": a, "dst": b, "area": round(area, 4),
                        "where": _where(zm, cfg.caption_compass), "zones": _zones_list(zm)})
    return sorted(out, key=lambda r: -r["area"])


def semantic_changes_all(sem, cfg: Optional[Config] = None, min_area: float = 0.02) -> List[dict]:
    """rev12: every RemoteCLIP PRE->POST change on the main grid by true area, INCLUDING the
    ones labelled "cloud". Clouded cells are already excluded from the transitions (rev10),
    so a "cloud" label left here sits on ground the cloud test calls clear: as an AFTER
    class it is a pale veil (ash, dust or haze); as a BEFORE class it means the BEFORE
    class itself is unreliable. `caption_evidence` decides which of these is caption
    material; this function only measures."""
    cfg = cfg or CFG
    g = sem.main_grid if getattr(sem, "enabled", True) else None
    if g is None:
        return []
    n_grids = max(len(getattr(sem, "grids", {}) or {}), 1)
    W, H = _grid_dims(sem)
    agg: Dict[Tuple[str, str], List] = {}
    for t in sem.transitions:
        if t.changed and t.kind == "patch" and t.grid == g.g:
            agg.setdefault((t.src_group, t.dst_group), []).append(t)
    out = []
    for (a, b), ts in agg.items():
        area = float(sum(t.weight for t in ts) * n_grids)
        if area >= min_area:
            zm = _zone_mass_boxes([t.where for t in ts], W, H)
            out.append({"kind": "transition", "src": a, "dst": b, "area": round(area, 4),
                        "where": _where(zm, cfg.caption_compass), "zones": _zones_list(zm)})
    return sorted(out, key=lambda r: -r["area"])


def _re_cloud(sentence: str) -> bool:
    import re as _r
    return bool(_r.search(CHANGE_LEX["cloud"], sentence or "", _r.I))


def _mentions_group(group: str, sentence: str, where: str = "") -> bool:
    import re as _r
    pat = _GROUP_LEX.get(group)
    return (bool(pat and _r.search(pat, sentence or "", _r.I)) and _same_place(sentence, where)
            and not _says_unchanged(sentence))


def _mentions_channel(kind: str, sents: List[str], where: str = "") -> bool:
    """rev10: a sentence covers a measured change only if it is about the same kind of
    change AND does not name a different place (the cloud sentence in TONGA 000024 --
    "forest in the south-west cleared" -- does not cover vegetation loss in the north-east)."""
    import re as _r
    if kind == "vegetation_loss":
        return any(_r.search(_VEG_WORD, x, _r.I) and _r.search(_LOSS_WORD, x, _r.I) and _same_place(x, where)
                   and not _says_unchanged(x) for x in sents)
    if kind in TYPED_DARK:                       # any mention of a dark cover / blackening
        kind = "darkening"
    pat = CHANGE_LEX.get(kind)
    return bool(pat) and any(_r.search(pat, x, _r.I) and _same_place(x, where) and not _says_unchanged(x)
                             for x in sents)


def _surface_like(group: str, label: Optional[str]) -> str:
    w = _group_word(group, label)
    return w if w.split()[0] in ("the", "a", "an") else f"{w}"


def check_draft(draft_raw: str, facts: dict, label_check: bool = True,
                cfg: Optional[Config] = None) -> List[dict]:
    """Compare a caption with the measurements. Returns the edit list, in priority order:
    disaster type, claims contradicted by a confident measurement, a building claim that
    ChangeOS positively contradicts, then large measured changes the caption misses.
    The first `caption_max_edits` are marked `sent`; the rest, and every claim the
    measurements cannot settle, are logged but not sent."""
    cfg = cfg or CFG
    body = caption_body(draft_raw)
    sents = _sentences(body)
    label = facts.get("label") if label_check else None
    label = label if label in DISASTER_VOCAB else None
    S = facts["surfaces"]
    edits: List[dict] = []

    # 1 · disaster type (C1 only: the pipeline's verdict is the system's output)
    if label:
        said = stated_disaster(draft_raw)
        if said != label:
            was = EVENT_PHRASE.get(said)
            edits.append(_edit(
                "disaster_type", f"draft: {said} | pipeline: {label}",
                f"The event was {EVENT_PHRASE[label]}"
                + (f", not {was}" if was else "")
                + ". Name it as the cause in the first sentence, and reword anything that "
                  f"attributes the change to {was or 'another event'}.",
                sentence=sents[0] if sents else None, measurement="posterior",
                value=facts.get("prob")))

    # 1b · rev10: clouds. A cloud is not ground; Qwen's draft read the AFTER cloud in
    #      TONGA 000024 as "forest cleared to bare ground". The caption must say the area
    #      is hidden, and must not describe a change under a BEFORE cloud.
    cl = facts.get("clouds", {}) or {}
    said_cloud = any(_re_cloud(x) for x in sents)
    if cl.get("post", 0.0) >= 0.05 and not said_cloud:
        edits.append(_edit(
            "cloud", "clouds in the AFTER image",
            f"Clouds cover part of the AFTER image ({cl.get('where_post') or 'part of the scene'}). "
            "Say that clouds hide that area, and do not describe the ground under them as changed.",
            measurement="cloud mask", value=cl["post"], where=cl.get("where_post", "")))
    if cl.get("pre", 0.0) >= 0.05:
        edits.append(_edit(
            "cloud", "clouds in the BEFORE image",
            f"Clouds covered part of the BEFORE image ({cl.get('where_pre') or 'part of the scene'}), "
            "so that area cannot be compared: do not describe a change there.",
            measurement="cloud mask", value=cl["pre"], where=cl.get("where_pre", "")))

    # 2 · claims contradicted by a CONFIDENT absence (two measurements agree)
    for s in sents:
        hit = [k for k, name in SURFACE_ABSENT_NAME.items()
               if name in facts["absent"] and asserts(s, k)]
        if hit:
            names = " or ".join(SURFACE_ABSENT_NAME[k] for k in hit)
            edits.append(_edit(
                "contradiction", "absent: " + ", ".join(SURFACE_ABSENT_NAME[k] for k in hit),
                f'Rewrite this sentence so it describes what is actually there: "{s}" There is '
                f'no {names} in the scene.',
                sentence=s, measurement="confident absence", surfaces=hit))

    # 3 · positive contradiction: large water (both signals agree) described as dry ground
    w = S["water"]
    if w["value"] >= 0.5 and w.get("pixel", 0.0) > 0:
        for s in sents:
            if asserts(s, "bare") and not asserts(s, "water"):
                edits.append(_edit(
                    "contradiction", "water described as dry ground",
                    f'Rewrite this sentence: "{s}" What looks like dry or bare ground is '
                    f"{_surface_words('water', label)} covering {_extent(w['value'])}"
                    + (f", in {w['where']}" if w['where'].startswith('the ') else
                       (f", {w['where']}" if w['where'] else "")) + ".",
                    sentence=s, measurement="flood_extent", value=w["value"],
                    where=w["where"], surfaces=["water"]))
                break

    # 4 · buildings. ChangeOS detections are used where they are POSITIVE (damage seen);
    #     "no damage" is only enforced when caption_check_buildings == "full", because its
    #     reliability on ash-covered or Turkish imagery is not established (see the
    #     visual-check sheet in §21).
    words = dict(_BLD_LEVEL_WORDS)
    words.update(DISASTER_VOCAB.get(label, {}).get("buildings", {}) if label else {})
    L, nd = facts["levels"], facts["damaged"]
    bsents = [s for s in sents if asserts(s, "building")]
    if nd >= 1 and cfg.caption_check_buildings in ("add_only", "full"):
        breakdown = ", ".join(f"{L[k]} {words[k]}" for k in (3, 2, 1) if L[k])
        loc = f" ({facts['damaged_where']})" if facts.get("damaged_where") else ""
        intact = [s for s in bsents if asserts(s, "intact")]
        if intact:
            edits.append(_edit(
                "contradiction", "buildings said intact; ChangeOS sees damage",
                f'Rewrite this sentence: "{intact[0]}" Some buildings are damaged: '
                f"{breakdown}{loc}.", sentence=intact[0], measurement="ChangeOS xBD levels",
                value=L, where=facts.get("damaged_where", "")))
        elif not any(asserts(s, "damage") for s in bsents):
            edits.append(_edit(
                "missing", "damaged buildings not mentioned",
                f"Add, in your own words from what you see: some buildings are damaged "
                f"({breakdown}){loc}.", measurement="ChangeOS xBD levels", value=L,
                where=facts.get("damaged_where", "")))
    if nd == 0 and facts["buildings"] >= 5:
        for s in bsents:
            if asserts(s, "collapse"):
                if cfg.caption_check_buildings == "full":
                    edits.append(_edit(
                        "contradiction", "collapse claimed; ChangeOS sees none",
                        f'Rewrite this sentence: "{s}" The buildings show no visible damage.',
                        sentence=s, measurement="ChangeOS xBD levels", value=L))
                else:
                    e = _edit("unverifiable", "collapse claimed; ChangeOS sees none",
                              "", sentence=s, measurement="ChangeOS xBD levels", value=L)
                    e["why"] = ("ChangeOS reliability on this imagery is not established; "
                                "claim left as written")
                    edits.append(e)
                break

    # 5 · large measured changes the draft does not mention (a surface already
    #     corrected by an edit above is not added a second time)
    covered = {k for e in edits if e["kind"] != "unverifiable" for k in e["surfaces"]}
    for surf, (_fld, _g, thr) in SURFACE_SOURCE.items():
        v = S[surf]["value"]
        if surf in covered:
            continue
        if surf == "water" and S["water"].get("pixel", 0.0) <= 0:
            continue                                # water must be seen by both tests
        if v >= thr and not any(asserts(s, surf) for s in sents):
            where = S[surf]["where"]
            edits.append(_edit(
                "missing", f"{surf} not mentioned",
                f"Add, in your own words from what you see: {_surface_words(surf, label)} "
                f"now covers {_extent(v)}" + (f", in {where}" if where.startswith("the ")
                                              else (f", {where}" if where else "")) + ".",
                measurement=_fld, value=v, where=where, surfaces=[surf]))

    # 6 · rev10: the strongest MEASURED changes the draft does not mention (located
    #     physical regions and RemoteCLIP changes), in neutral words -- Qwen looks at that
    #     part of the images and says what it is. At most `caption_max_added_changes`.
    covered = {k for e in edits if e["kind"] != "unverifiable" for k in e["surfaces"]}
    cands = []
    dk = facts.get("darkening", {}) or {}
    for kind_, key_ in (("deposit", "where_deposit"), ("burn", "where_burn")):
        if float(dk.get(kind_, 0.0)) >= 0.02:
            cands.append({"kind": kind_, "area": float(dk[kind_]), "where": dk.get(key_, ""), "label": kind_})
    for r_ in facts.get("regions", []):
        if r_["kind"] in REGION_CHANNELS:
            if r_["kind"] == "darkening" and any(c_["kind"] in ("deposit", "burn") for c_ in cands):
                continue                     # the typed darkening says more
            cands.append(dict(r_, label=r_["kind"]))
    for c_ in facts.get("changes", []):
        cands.append(dict(c_, label=f"{c_['src']}->{c_['dst']}"))
    cands.sort(key=lambda r_: -r_["area"])
    added = 0
    for c_ in cands:
        if added >= int(getattr(cfg, "caption_max_added_changes", 2)):
            break
        if c_["kind"] == "transition":
            if c_["dst"] in covered or c_["dst"] == "cloud" or any(_mentions_group(c_["dst"], x, c_["where"]) for x in sents):
                continue
            what = (f"{_surface_like(c_['src'], label)} {_in_place(c_['where'])} has become "
                    f"{_surface_like(c_['dst'], label)}").replace("  ", " ")
            surf = [c_["dst"]]
        else:
            if _mentions_channel(c_["kind"], sents, c_["where"]):
                continue
            import re as _r
            stale = [x for x in sents if c_["kind"] == "vegetation_loss" and _says_unchanged(x)
                     and _r.search(_VEG_WORD, x, _r.I) and _same_place(x, c_["where"])]
            if stale:
                # the draft says the vegetation is unchanged where loss was measured
                edits.append(_edit("contradiction", "vegetation said unchanged; measured loss",
                                   f'Rewrite this sentence: "{stale[0]}" Green vegetation '
                                   f"{_in_place(c_['where'])} has thinned or disappeared.",
                                   sentence=stale[0], measurement="vegetation_loss", value=c_["area"],
                                   where=c_.get("where", ""), surfaces=["vegetation_loss"]))
                added += 1
                continue
            what = {"deposit": f"a dark layer now covers roads and open ground {_in_place(c_['where'])}",
                    "burn": f"vegetation {_in_place(c_['where'])} has been blackened",
                    "darkening": f"the ground {_in_place(c_['where'])} has become noticeably darker",
                    "brightening": f"the ground {_in_place(c_['where'])} has become paler and lighter",
                    "vegetation_loss": f"green vegetation {_in_place(c_['where'])} has thinned or disappeared",
                    "desaturation": f"colours {_in_place(c_['where'])} have faded toward grey"}[c_["kind"]]
            surf = [c_["kind"]]
        edits.append(_edit("added", f"measured change not mentioned: {c_['label']}",
                           f"Add, in your own words from what you see: {what}. Look at that part of "
                           "the AFTER image and describe what it is.",
                           measurement=c_["kind"], value=c_["area"], where=c_.get("where", ""),
                           surfaces=surf))
        added += 1

    budget = int(cfg.caption_max_edits)
    for e in edits:
        if e["kind"] != "unverifiable" and budget > 0:
            e["sent"] = True; budget -= 1
    return edits


def revision_prompt(draft_body: str, sent: List[dict]) -> str:
    fixes = "\n".join(f"{i}. {e['instruction']}" for i, e in enumerate(sent, 1))
    return ("You are given two satellite images of the same place: (1) BEFORE an event and "
            "(2) AFTER it. You wrote this change caption for them:\n\n"
            f"\"{draft_body}\"\n\n"
            "An image-analysis pipeline measured the same two images and found these "
            "problems with your caption. Fix ONLY these:\n"
            f"{fixes}\n\n"
            "Rules:\n"
            "- Keep every other sentence as you wrote it.\n"
            "- Write any new or rewritten words yourself, from what you see in the images. "
            "Do not mention measurements, the pipeline, the heat map or these instructions.\n"
            "- Keep 2 to 4 sentences, about the same length as before. No numbers or "
            "percentages, no headings or bullet points.\n"
            + CAPTION_VERDICT)


def resolve_edits(final_raw: str, edits: List[dict], facts: dict) -> List[dict]:
    """After revision, check each SENT edit against the new caption."""
    body = caption_body(final_raw)
    sents = _sentences(body)
    for e in edits:
        if not e["sent"]:
            continue
        if e["kind"] == "disaster_type":
            e["resolved"] = stated_disaster(final_raw) == facts["label"]
        elif e["check"].startswith("absent"):
            e["resolved"] = not any(asserts(s, k) for s in sents for k in e["surfaces"])
        elif e["check"] == "water described as dry ground":
            e["resolved"] = any(asserts(s, "water") for s in sents)
        elif e["check"].startswith("buildings said intact") or e["check"].startswith("damaged"):
            e["resolved"] = any(asserts(s, "damage") for s in sents if asserts(s, "building"))
        elif e["check"] == "vegetation said unchanged; measured loss":
            e["resolved"] = _mentions_channel("vegetation_loss", sents, e.get("where", ""))
        elif e["kind"] == "cloud":
            e["resolved"] = any(_re_cloud(x) for x in sents) if "AFTER" in e["check"] else True
        elif e["kind"] == "added":
            k = e["surfaces"][0] if e["surfaces"] else ""
            e["resolved"] = (_mentions_channel(k, sents, e.get("where", "")) if k in REGION_CHANNELS + TYPED_DARK
                             else any(_mentions_group(k, x, e.get("where", "")) for x in sents))
        elif e["kind"] == "missing":
            e["resolved"] = any(asserts(s, k) for s in sents for k in e["surfaces"])
        elif e["check"].startswith("collapse"):
            e["resolved"] = not any(asserts(s, "collapse") for s in sents
                                    if asserts(s, "building"))
    return edits


def revise_caption(captioner: "QwenCaptioner", pre: Image.Image, post: Image.Image,
                   draft_raw: str, facts: dict, label_check: bool = True,
                   cfg: Optional[Config] = None) -> Tuple[str, List[dict], List[dict]]:
    """(final caption, edit log, residual problems after revision).

    No edit to send -> the draft is returned untouched and no generation is spent."""
    cfg = cfg or CFG
    edits = check_draft(draft_raw, facts, label_check, cfg)
    sent = [e for e in edits if e["sent"]]
    final = draft_raw
    if sent:
        try:
            out = captioner.revise(pre, post, caption_body(draft_raw), sent)
            final = out if caption_body(out) else draft_raw
        except Exception as ex:                       # keep the draft, say so in the log
            log.warning("revision failed (%s: %s); keeping the draft", type(ex).__name__, ex)
            for e in sent:
                e["resolved"] = False
        if final is not draft_raw:
            resolve_edits(final, edits, facts)
    residual = [e for e in check_draft(final, facts, label_check, cfg)
                if e["kind"] in ("disaster_type", "contradiction")]
    return final, edits, residual


def corrupt_facts(facts: dict) -> Tuple[dict, dict]:
    """Grounding test (§21 I): flip the verdict and the water reading. If the caption
    follows the flipped measurement where it contradicts the draft, the measurement is
    what decides the caption -- the evidence is USED, not decoration."""
    import copy as _copy
    f = _copy.deepcopy(facts)
    cyc = {"flood": "volcanic_eruption", "volcanic_eruption": "earthquake",
           "earthquake": "flood"}
    old = f.get("label")
    f["label"] = cyc.get(old, "flood")
    had = f["surfaces"]["water"]["value"] >= 0.20 and f["surfaces"]["water"].get("pixel", 0) > 0
    if had:
        f["surfaces"]["water"].update(value=0.0, pixel=0.0, semantic=0.0, where="")
        if "standing water" not in f["absent"]:
            f["absent"].append("standing water")
    else:
        f["surfaces"]["water"].update(value=0.8, pixel=0.8, semantic=0.8,
                                      where="throughout the scene")
        f["absent"] = [a for a in f["absent"] if a != "standing water"]
    return f, {"label": [old, f["label"]], "water_present": [had, not had]}


# ═══════════════════════════════════════════════════════════════════════════════
# rev12 · EVIDENCE -> LOOK -> WRITE -> VERIFY  (the system caption, caption_mode="look")
#
# rev9-rev11 kept Qwen's blind draft and patched it ("Keep every other sentence as you
# wrote it", at most 4 corrections, at most 2 of them adding a measured change). On the
# 82 sample2 pairs that gave: the draft copied verbatim, the additions appended at the
# end in the pipeline's own words, and only 9/82 captions holding every detected change
# (volcano 58%, earthquake 61% of changes in the caption).
#
# rev12 separates WHERE TO LOOK and WHAT IS VALID (the measurements) from WHAT TO SAY
# (Qwen, looking at the images):
#   1. EVIDENCE -- `caption_evidence` lists every valid change the pipeline measured:
#                  located physical changes, the typed darkening, RemoteCLIP changes,
#                  confirmed surfaces, damaged buildings, AnyChange regions, SAM2 object
#                  changes. Nothing is capped; an item is excluded only for a stated
#                  reason (below the size floor, a shadow, a BEFORE class the cloud test
#                  rejects, water the pixel test does not confirm, contradicted by a
#                  confident absence). A change in two separate places becomes two items.
#   2. LOOK     -- items are grouped by where they are. For each group Qwen sees a
#                  close-up crop of BEFORE and AFTER and says what changed there; the
#                  prompt holds no evidence words. An item the answer does not describe
#                  gets ONE guided look with a plain cue ("parts of it look darker").
#                  An item Qwen still cannot see is "not_visible": logged, not captioned
#                  (CFG.caption_require_visual_confirmation=False includes it anyway).
#   3. WRITE    -- Qwen writes the caption once, from scratch: the full images, the
#                  verdict and its OWN notes on every region. The blind draft is not
#                  passed, so there is nothing to keep and nothing to append to.
#   4. VERIFY   -- code checks that every visible item is in the caption (its kind or
#                  its note, at its place), that the event is named, that clouds are
#                  mentioned, and that no confident measurement is contradicted. If not,
#                  the caption is written AGAIN FROM SCRATCH with the omitted notes
#                  flagged -- the earlier caption is never shown, so it cannot be patched.
#                  The best attempt is kept, and every item's status is logged.
# ═══════════════════════════════════════════════════════════════════════════════
import re as _re12

_CELL_SHARE = 0.12   # a 3x3 cell belongs to an item when it holds >= this share of it
_FULL_CELLS = 7      # an item spread over >= 7 of the 9 cells is "throughout the scene"

# plain cues for the GUIDED look only (never in the open look, never in the caption)
ITEM_CUE: Dict[str, str] = {
    "deposit": "the roads and open ground look darker and differently coloured, as if a layer now covers them",
    "burn": "the vegetation looks darker, as if blackened",
    "darkening": "parts of it look darker",
    "brightening": "parts of it look paler or lighter",
    "vegetation_loss": "there is less green vegetation",
    "desaturation": "the colours look greyer and more faded",
    "texture_change": "the surface looks rougher or more broken up",
    "pale_cover": "a pale, hazy or dusty-looking cover lies over part of it",
    "buildings": "some buildings look damaged",
    "anychange": "something in it looks different",
}
SURFACE_CUE: Dict[str, str] = {
    "water": "there seems to be water where the ground was dry",
    "ash": "a grey, ash-like layer seems to cover the ground",
    "burn": "the ground looks blackened or charred",
    "smoke": "there seems to be smoke or haze",
    "lava": "there seems to be fresh dark rock",
    "landslide": "there seem to be bare scars of displaced soil",
}
# words that count as describing each kind of change (looks AND final caption)
_COVER_LEX: Dict[str, str] = {
    "darkening": r"\b(dark\w*|black\w*|shadow\w*|charred|scorch\w*|burn\w*|ash\w*|soot\w*|dim\w*|deposit\w*)\b",
    "deposit": (r"\b(dark\w*|black\w*|ash\w*|deposit\w*|layers?|coat\w*|cover\w*|blanket\w*|dust\w*|"
                r"gr[ae]y\w*|mud\w*|sediment\w*|soot\w*|tephra|buried|debris)\b"),
    "burn": r"\b(burn\w*|charred|scorch\w*|blacken\w*|fires?|black\w*|dark\w*)\b",
    "brightening": (r"\b(pale\w*|bright\w*|whit\w*|lighter|light-colou?red|dust\w*|bare|exposed|"
                    r"sand\w*|bleach\w*|wash\w*)\b"),
    "desaturation": r"\b(gr[ae]y\w*|fad\w*|dull\w*|colou?rless|wash\w*|ash\w*|brown\w*|muted|drab|pale\w*)\b",
    "texture_change": (r"\b(rough\w*|debris|rubble|broken|crack\w*|fragment\w*|scatter\w*|disturb\w*|"
                       r"scar\w*|churn\w*|jumbl\w*|uneven|textur\w*|collaps\w*)\b"),
    "pale_cover": (r"\b(haz\w*|dust\w*|pale\w*|veil\w*|ash\w*|smok\w*|mist\w*|whit\w*|gr[ae]y\w*|"
                   r"cloud\w*|fog\w*|film|layers?|plume\w*)\b"),
    "buildings": r"\b(damag\w*|collaps\w*|destroy\w*|rubble|roofless|ruin\w*|flatten\w*|debris)\b",
    "anychange": (r"\b(chang\w*|differ\w*|new\w*|now|replac\w*|appear\w*|disappear\w*|remov\w*|built|"
                  r"construct\w*|clear\w*|damag\w*|cover\w*|became|become|turn\w*)\b"),
}
_BUILDING_WORD = r"\b(buildings?|houses?|homes?|roofs?|structures?|settlement\w*|urban|village\w*|town\w*|blocks?)\b"
_ITEM_WEIGHT = {"surface": 3, "buildings": 3, "deposit": 2, "burn": 2, "transition": 1, "pale_cover": 1,
                "brightening": 1, "darkening": 1, "vegetation_loss": 1, "desaturation": 1,
                "texture_change": 1, "anychange": 0}
_DST_ABSENT = {"ash": "grey ash-like deposit", "burn": "blackened ground", "plume": "smoke or haze plume",
               "lava": "fresh dark rock", "flood": "standing water", "sediment_water": "standing water",
               "water": "standing water"}
_PLACE_WORDS = _re12.compile(
    r"^(north\w*|south\w*|east\w*|west\w*|cent\w*|middle|upper|lower|left|right|top|bottom|corner|side|sides|"
    r"quadrant|edge|edges)$")
_NOTE_STOP = set((
    "before after image images area areas scene part parts this that these those there their with from have "
    "been were which where while some more less most also into onto over under near small large much many "
    "other than then they them what when your look looks looked appear appears appeared seem seems seemed "
    "visible clearly likely possibly slightly mostly around along across within still remain remains remained "
    "show shows shown now being does done made make whole same").split())
_NO_CHANGE = _re12.compile(
    r"\b(no (visible |apparent |noticeable |significant |obvious |discernible |major |clear )?"
    r"(change|difference)s?|(look|looks|appear|appears|remain|remains|seem|seems|is|are) "
    r"(largely |mostly |essentially |virtually |relatively )?(the same|unchanged|identical|similar)|"
    r"nothing (has )?changed|unchanged)\b", _re12.I)
_CHANGE_CUE = _re12.compile(
    r"\b(now|became|become|has been|have been|replaced|covered|turned|appears? to have|darker|lighter|"
    r"paler|flooded|destroyed|damaged|collapsed|cleared|new|no longer|instead|disappeared|removed)\b", _re12.I)


def _says_no_change(text: str) -> bool:
    """A look answer that reports no change (and describes none)."""
    t = (text or "").strip()
    return bool(_NO_CHANGE.search(t)) and not _CHANGE_CUE.search(t)


def _clauses(text: str) -> List[str]:
    parts = []
    for s_ in _sentences(text or "") or [text or ""]:
        parts += [c for c in _re12.split(r"[,;:]|\b(?:while|whereas|but|although|though)\b", s_, flags=_re12.I)
                  if c and c.strip()]
    return parts


def _lex_hit(pattern: Optional[str], text: str) -> bool:
    return bool(pattern) and bool(_re12.search(pattern, text or "", _re12.I))


def _kind_match(item: dict, text: str) -> bool:
    """Does `text` describe this kind of change? Clause by clause, ignoring clauses that
    call something unchanged ("the buildings remain intact, while the fields flooded")."""
    k = item["kind"]
    for c in _clauses(text):
        if _says_unchanged(c):
            continue
        if k == "vegetation_loss":
            if (_lex_hit(_VEG_WORD, c) and _lex_hit(_LOSS_WORD, c)) or _lex_hit(r"\b(deforest\w*|denud\w*)\b", c):
                return True
        elif k == "transition":
            dst, src = item.get("dst", ""), item.get("src", "")
            if _lex_hit(_GROUP_LEX.get(dst), c):
                return True
            if src in VEG_GROUPS and dst not in VEG_GROUPS and _lex_hit(_VEG_WORD, c) and _lex_hit(_LOSS_WORD, c):
                return True
        elif k == "surface":
            if _lex_hit(_GROUP_LEX.get(_SURFACE_GROUP.get(item.get("surface", ""), "")), c):
                return True
        elif k == "buildings":
            if _lex_hit(_COVER_LEX["buildings"], c) and (_lex_hit(_BUILDING_WORD, c) or _lex_hit(r"\b(rubble|collaps\w*|ruin\w*)\b", c)):
                return True
        elif _lex_hit(_COVER_LEX.get(k), c):
            return True
    return False


def _place_ok(zones: set, cells) -> bool:
    """Is a place named in the text the item's place? No place named -> not contradicted.
    Otherwise the named cells and the item's cells must overlap by at least half of the
    smaller set ("the south" does not cover a change on "the western side", although the
    two share the south-west corner)."""
    cells = {tuple(c) for c in (cells or [])}
    if not zones or not cells:
        return True
    return len(zones & cells) / max(min(len(zones), len(cells)), 1) >= 0.5


def _mentions_item(item: dict, sentence: str) -> bool:
    """A caption sentence covers an item when one of its clauses describes that kind of
    change at the item's place (the clause's own place words, else the sentence's)."""
    zs_sent = _zones_of(sentence)
    for c in _clauses(sentence):
        if _says_unchanged(c) or not _kind_match(item, c):
            continue
        if _place_ok(_zones_of(c) or zs_sent, item.get("cells")):
            return True
    return False


def _content_words(text: str) -> set:
    return {w for w in _re12.findall(r"[a-z]+", (text or "").lower())
            if len(w) >= 4 and w not in _NOTE_STOP and not _PLACE_WORDS.match(w)}


def _note_overlap(note: str, sentence: str) -> Tuple[int, float]:
    N, S_ = _content_words(note), _content_words(sentence)
    shared = len(N & S_)
    return shared, shared / max(len(N), 1)


def _sbert_model(cfg: Config):
    """The sentence-similarity model §10 already uses (loaded once). None when the
    config does not use it or it cannot load -- coverage then rests on the lexical tests."""
    if getattr(cfg, "caption_sim_model", "sbert") != "sbert":
        return None
    if not _SBERT["tried"]:
        _SBERT["tried"] = True
        try:
            from sentence_transformers import SentenceTransformer
            _SBERT["model"] = SentenceTransformer(cfg.sbert_id, device=cfg.device)
            log.info("Caption similarity: %s", cfg.sbert_id)
        except Exception as e:
            log.warning("sentence-transformers unavailable (%s: %s); caption coverage uses the "
                        "lexical tests only", type(e).__name__, e)
    return _SBERT["model"]


# ── 1 · EVIDENCE ─────────────────────────────────────────────────────────────
def _split_by_place(base: dict, zones, area: float, cfg: Config,
                    floor_exempt: bool = False) -> Tuple[List[dict], List[dict]]:
    """One item per separate place: the cells holding >= _CELL_SHARE of the item, split
    into 8-connected clusters on the 3x3 grid. A fragment below the size floor is dropped
    (and logged) unless the whole item would vanish."""
    z = _zones_arr(zones)
    t = float(z.sum())
    floor = 0.0 if floor_exempt else float(cfg.caption_min_area)
    if t <= 0:
        return [dict(base, area=round(area, 4), zones=_zones_list(None), cells=[],
                     where=base.get("where", ""))], []
    s = z / t
    if _where(z, cfg.caption_compass) == "throughout the scene":
        # spread out (the same rule `_where` names): one item over every cell it touches
        return [dict(base, area=round(area, 4), zones=_zones_list(z), where="throughout the scene",
                     cells=sorted((i, j) for i in range(3) for j in range(3) if s[i, j] > 0))], []
    cells = {(i, j) for i in range(3) for j in range(3) if s[i, j] >= _CELL_SHARE}
    if not cells:
        i, j = np.unravel_index(int(np.argmax(s)), s.shape)
        cells = {(int(i), int(j))}
    clusters, seen = [], set()
    for c in sorted(cells):
        if c in seen:
            continue
        comp, stack = set(), [c]
        while stack:
            a = stack.pop()
            if a in seen:
                continue
            seen.add(a); comp.add(a)
            stack += [(a[0] + di, a[1] + dj) for di in (-1, 0, 1) for dj in (-1, 0, 1)
                      if (a[0] + di, a[1] + dj) in cells and (a[0] + di, a[1] + dj) not in seen]
        clusters.append(comp)
    if len(clusters) == 1 or len(cells) >= _FULL_CELLS:
        cl = set().union(*clusters)
        return [dict(base, area=round(area, 4), zones=_zones_list(z), cells=sorted(cl),
                     where=_where(z, cfg.caption_compass) or base.get("where", ""))], []
    items, dropped = [], []
    for cl in clusters:
        zm = np.zeros((3, 3))
        for (i, j) in cl:
            zm[i, j] = z[i, j]
        a = area * float(zm.sum()) / t
        it = dict(base, area=round(a, 4), zones=_zones_list(zm), cells=sorted(cl),
                  where=_where(zm, cfg.caption_compass))
        (items if a >= floor else dropped).append(it)
    if not items:
        # every fragment is small but together they pass the floor: keep each one, located,
        # rather than one item spanning places that do not touch
        return [dict(d, fragment=True) for d in dropped], []
    return items, [dict(d, why=f"a separate fragment below the size floor ({floor:.0%} of the tile)")
                   for d in dropped]


def caption_evidence(facts: dict, cfg: Optional[Config] = None) -> Tuple[List[dict], List[dict]]:
    """Every VALID change the pipeline measured, as caption items, and every excluded one
    with the reason. Valid = measured on observable ground (clouds are masked upstream),
    at least `caption_min_area` of the tile (damaged buildings: any), and not contradicted
    by another measurement. Nothing is capped."""
    cfg = cfg or CFG
    floor = float(cfg.caption_min_area)
    items: List[dict] = []
    excluded: List[dict] = []
    absent = set(facts.get("absent", []) or [])
    S = facts.get("surfaces", {}) or {}
    w = S.get("water", {}) or {}
    water_ok = float(w.get("value", 0.0)) > CAPTION_FLOOR and float(w.get("pixel", 0.0)) > 0

    def add(base: dict, zones, area: float, floor_exempt: bool = False):
        area = float(area)
        if area < floor and not floor_exempt:
            excluded.append(dict(base, area=round(area, 4), where=base.get("where", ""),
                                 why=f"below the size floor ({floor:.0%} of the tile)"))
            return
        its, drop = _split_by_place(base, zones, area, cfg, floor_exempt)
        items.extend(its); excluded.extend(drop)

    # typed darkening (rev11): deposit and burn; a shadow is not a change
    dk = facts.get("darkening", {}) or {}
    for kind_, lab_ in (("deposit", "dark deposit on open ground and vegetation"),
                        ("burn", "blackened vegetation (burn-like)")):
        v_ = float(dk.get(kind_, 0.0) or 0.0)
        if v_ > 0:
            add({"kind": kind_, "label": lab_, "source": "darkening by surface",
                 "where": dk.get(f"where_{kind_}", "")}, dk.get(f"zones_{kind_}"), v_)
    if float(dk.get("shadow", 0.0) or 0.0) >= floor:
        excluded.append({"kind": "shadow", "label": "darker but the same colours", "source": "darkening by surface",
                         "area": round(float(dk["shadow"]), 4), "where": "",
                         "why": "a shadow: darker without a change of colour, which the pipeline ignores"})
    typed_cells = {tuple(c) for it in items if it["kind"] in ("deposit", "burn") for c in it["cells"]}

    # located physical changes
    for r_ in facts.get("regions", []) or []:
        k = r_.get("kind")
        if k not in REGION_CHANNELS + ("texture_change",):
            continue
        n0 = len(items)
        add({"kind": k, "label": f"{k.replace('_', ' ')} (strong, located)", "source": "physical RGB channel",
             "where": r_.get("where", "")}, r_.get("zones"), r_.get("area", 0.0))
        if k == "darkening" and typed_cells:
            keep = []
            for it in items[n0:]:
                if {tuple(c) for c in it["cells"]} <= typed_cells:
                    excluded.append(dict(it, why="the same darkening, already typed as deposit or burn"))
                else:
                    keep.append(it)
            items[n0:] = keep

    # RemoteCLIP grid changes (and SAM2 object changes): validity of the classes
    def add_transition(c_: dict, source: str):
        src, dst = c_.get("src", ""), c_.get("dst", "")
        base = {"src": src, "dst": dst, "source": source, "where": c_.get("where", "")}
        if src == "cloud":
            excluded.append(dict(base, kind="transition", area=c_.get("area", 0.0),
                                 label=f"cloud -> {GROUP_WORDS.get(dst, dst)}",
                                 why="the BEFORE class is 'cloud' on ground the cloud test calls clear, "
                                     "so the BEFORE class is unreliable"))
            return
        if dst in ("flood", "sediment_water", "water") and not water_ok:
            excluded.append(dict(base, kind="transition", area=c_.get("area", 0.0),
                                 label=f"{GROUP_WORDS.get(src, src)} -> {GROUP_WORDS.get(dst, dst)}",
                                 why="water that the pixel test does not confirm (rev9 agreement rule)"))
            return
        if _DST_ABSENT.get(dst) in absent:
            excluded.append(dict(base, kind="transition", area=c_.get("area", 0.0),
                                 label=f"{GROUP_WORDS.get(src, src)} -> {GROUP_WORDS.get(dst, dst)}",
                                 why=f"contradicted by a confident absence: {_DST_ABSENT[dst]}"))
            return
        if dst == "cloud":
            add(dict(base, kind="pale_cover",
                     label=f"{GROUP_WORDS.get(src, src)} -> pale veil (RemoteCLIP 'cloud'; the cloud test "
                           "found no cloud: ash, dust or haze)"), c_.get("zones"), c_.get("area", 0.0))
            return
        add(dict(base, kind="transition", label=f"{GROUP_WORDS.get(src, src)} -> {GROUP_WORDS.get(dst, dst)}"),
            c_.get("zones"), c_.get("area", 0.0))

    for c_ in (facts.get("changes_all") if facts.get("changes_all") is not None else facts.get("changes", [])) or []:
        add_transition(c_, "RemoteCLIP grid")
    for c_ in facts.get("objects", []) or []:
        add_transition(c_, "SAM2 instance + RemoteCLIP")

    # surfaces the scene branch measured
    for s_, (fld_, _g, _thr) in SURFACE_SOURCE.items():
        f_ = S.get(s_, {}) or {}
        v_ = float(f_.get("value", 0.0))
        if v_ <= CAPTION_FLOOR:
            continue
        base = {"kind": "surface", "surface": s_, "label": SURFACE_WORDS[s_], "source": f"scene evidence ({fld_})",
                "where": f_.get("where", "")}
        if s_ == "water" and float(f_.get("pixel", 0.0)) <= 0:
            excluded.append(dict(base, area=round(v_, 4), why="semantic water only; the pixel test disagrees"))
            continue
        if SURFACE_ABSENT_NAME.get(s_) in absent:
            excluded.append(dict(base, area=round(v_, 4), why="contradicted by a confident absence"))
            continue
        add(base, f_.get("zones"), v_)

    # buildings: ChangeOS damage it DETECTS (any amount); its "no damage" is not trusted
    L = list(facts.get("levels", [0, 0, 0, 0]) or [0, 0, 0, 0]) + [0, 0, 0, 0]
    nd = int(facts.get("damaged", 0) or 0)
    if nd >= 1:
        add({"kind": "buildings", "label": f"{nd} damaged buildings (levels 1-3: {L[1]}/{L[2]}/{L[3]})",
             "levels": L[:4], "source": "ChangeOS", "where": facts.get("damaged_where", "")},
            facts.get("damaged_zones"), max(float(nd) / max(int(facts.get("buildings", 1) or 1), 1), 0.0),
            floor_exempt=True)
    elif int(facts.get("buildings", 0) or 0) >= 5 and cfg.caption_check_buildings != "full":
        excluded.append({"kind": "buildings", "label": f"{facts['buildings']} buildings graded undamaged",
                         "source": "ChangeOS", "area": 0.0, "where": "",
                         "why": "ChangeOS 'no damage' is not trusted on this imagery (caption_check_buildings)"})

    # AnyChange: a change of unknown kind; the look says what it is
    for q_ in facts.get("anychange", []) or []:
        add({"kind": "anychange", "label": "AnyChange region (change of unknown kind)", "source": "AnyChange (SAM)",
             "where": q_.get("where", ""), "prob": q_.get("prob")}, q_.get("zones"), q_.get("area", 0.0))

    for k, it in enumerate(items):
        it["id"] = f"e{k}"
        it["cue"] = (f"what looked like {GROUP_WORDS.get(it['src'], it['src'])} now looks like "
                     f"{GROUP_WORDS.get(it['dst'], it['dst'])}" if it["kind"] == "transition"
                     else SURFACE_CUE.get(it.get("surface", ""), "") if it["kind"] == "surface"
                     else ITEM_CUE.get(it["kind"], ITEM_CUE["anychange"]))
        it["status"] = "pending"
    return items, excluded


# ── 2 · LOOK ─────────────────────────────────────────────────────────────────
def group_evidence(items: List[dict], cfg: Optional[Config] = None) -> List[dict]:
    """Group items by place: one close-up per group. Items overlapping an existing group
    (overlap coefficient >= 0.5, union <= 6 cells) join it; past `caption_max_regions`
    groups an item joins the group it overlaps most (or the nearest), so nothing is dropped
    -- that group's crop just grows. Items spread throughout the scene share one group
    that looks at the whole images."""
    cfg = cfg or CFG
    whole = [it for it in items if len(it["cells"]) >= _FULL_CELLS or not it["cells"]]
    local = sorted([it for it in items if it not in whole],
                   key=lambda it: (-_ITEM_WEIGHT.get(it["kind"], 0), -float(it["area"])))
    cap = max(int(cfg.caption_max_regions) - (1 if whole else 0), 1)
    groups: List[dict] = []

    def _dist(a: set, b: set) -> int:
        return min(max(abs(p[0] - q[0]), abs(p[1] - q[1])) for p in a for q in b)

    for it in local:
        c = {tuple(x) for x in it["cells"]}
        best, bo = None, 0.0
        for g in groups:
            ov = len(c & g["cells"]) / max(min(len(c), len(g["cells"])), 1)
            if ov > bo:
                best, bo = g, ov
        if best is not None and bo >= 0.5 and len(best["cells"] | c) <= 6:
            best["items"].append(it); best["cells"] |= c
        elif len(groups) < cap:
            groups.append({"cells": set(c), "items": [it]})
        else:
            tgt = best if best is not None and bo > 0 else min(groups, key=lambda g: _dist(g["cells"], c))
            tgt["items"].append(it); tgt["cells"] |= c
    if whole:
        groups.append({"cells": {(i, j) for i in range(3) for j in range(3)}, "items": whole})
    for k, g in enumerate(groups):
        g["id"] = k
        g["whole"] = len(g["cells"]) >= _FULL_CELLS
        zm = np.zeros((3, 3))
        for it in g["items"]:
            zm += _zones_arr(it["zones"]) * max(float(it["area"]), 1e-6)
            it["group"] = k
        mask = np.zeros((3, 3))
        for (i, j) in g["cells"]:
            mask[i, j] = 1.0
        # the place named in the notes is the area the close-up shows
        g["place"] = "throughout the scene" if g["whole"] else (_where(mask, cfg.caption_compass)
                                                               or _where(zm, cfg.caption_compass))
        g["cells"] = sorted(g["cells"])
    return groups


def look_prompt(whole: bool = False, cues: Sequence[str] = ()) -> str:
    """The question for one close-up. The open look carries NO evidence; the guided look
    only plain cues about appearance, never a measurement, a class name or a cause."""
    what = ("two satellite images of the same place" if whole
            else "two close-up satellite images of the same small area")
    p = f"You are given {what}: (1) BEFORE an event and (2) AFTER it.\n"
    if cues:
        p += ("Look closely at the AFTER image: " + "; ".join(cues) + ".\n"
              "In one or two short sentences, say what this is and what was there before. ")
    else:
        p += "In one or two short sentences, say what covered this area before and what covers it now. "
    p += ("Name the surfaces you can see (for example buildings, roads, fields, trees, water, bare soil, "
          "a dark or pale layer) and how they changed. Describe only what you can see and do not guess the "
          "cause. If you see no difference, answer exactly: no visible change.")
    return p


def _crop_pair(pre: Image.Image, post: Image.Image, cells, cfg: Config):
    """BEFORE/AFTER crops of a group's cells with a margin, upscaled so the short close-up
    is not lost in Qwen's patching. (pre, post, box, whole)."""
    W, H = pre.size
    cells = [tuple(c) for c in cells]
    if not cells or len(cells) >= _FULL_CELLS:
        return pre, post, (0, 0, W, H), True
    rows, cols = [c[0] for c in cells], [c[1] for c in cells]
    m = float(cfg.caption_crop_margin) * max(W, H)
    box = (int(max(0, min(cols) * W / 3 - m)), int(max(0, min(rows) * H / 3 - m)),
           int(min(W, (max(cols) + 1) * W / 3 + m)), int(min(H, (max(rows) + 1) * H / 3 + m)))
    if (box[2] - box[0]) * (box[3] - box[1]) >= 0.85 * W * H:
        return pre, post, (0, 0, W, H), True
    a, b = pre.crop(box), post.crop(box)
    side = max(a.size)
    if side < int(cfg.caption_crop_min_side):
        f = int(cfg.caption_crop_min_side) / side
        size = (max(1, int(round(a.size[0] * f))), max(1, int(round(a.size[1] * f))))
        a, b = a.resize(size, Image.LANCZOS), b.resize(size, Image.LANCZOS)
    return a, b, box, False


def look_at_groups(captioner: "QwenCaptioner", pre: Image.Image, post: Image.Image,
                   groups: List[dict], cfg: Optional[Config] = None) -> List[Tuple[Image.Image, Image.Image]]:
    """Ask Qwen what changed in each group's close-up; set each item's status. Returns
    the crops (kept out of the JSON records)."""
    cfg = cfg or CFG
    crops = []
    for g in groups:
        a, b, box, whole = _crop_pair(pre, post, g["cells"], cfg)
        crops.append((a, b))
        note1 = captioner.look(a, b, whole=whole).strip()
        open_ok = not _says_no_change(note1)
        undescribed = [it for it in g["items"] if not (open_ok and _kind_match(it, note1))]
        guided = None
        if undescribed and cfg.caption_guided_look:
            cues = list(dict.fromkeys(it["cue"] for it in sorted(undescribed, key=lambda it: -float(it["area"]))
                                      if it.get("cue")))
            guided = captioner.look(a, b, whole=whole, cues=cues[:4]).strip()
        guided_ok = guided is not None and not _says_no_change(guided)
        for it in g["items"]:
            if open_ok and _kind_match(it, note1):
                it["status"], it["seen_by"] = "visible", "open look"
            elif guided_ok and _kind_match(it, guided):
                it["status"], it["seen_by"] = "visible", "guided look"
            elif guided_ok:
                it["status"], it["seen_by"] = "visible", "guided look (in its own words)"
            elif cfg.caption_require_visual_confirmation:
                it["status"], it["seen_by"] = "not_visible", ""
            else:
                it["status"], it["seen_by"] = "unconfirmed", ""
        g.update(box=list(box), whole=whole, note_open=note1, note_guided=guided,
                 notes=[n for n, ok in ((note1, open_ok), (guided, guided_ok)) if ok and n])
    return crops


def _group_note(g: dict) -> str:
    """What the caption must say about a group: Qwen's own notes, plus (only when visual
    confirmation is switched off) the plain cue of an item it could not see."""
    parts = list(g.get("notes") or [])
    for it in g["items"]:
        if it["status"] == "unconfirmed" and it.get("cue"):
            parts.append(it["cue"][0].upper() + it["cue"][1:] + ".")
    return " ".join(p.strip() for p in parts if p and p.strip())


# ── 3 · WRITE ────────────────────────────────────────────────────────────────
def _event_subject(label: str) -> str:
    ph = EVENT_PHRASE.get(label, label.replace("_", " "))
    return ph[0].upper() + ph[1:]


def compose_prompt(label: Optional[str], groups: List[dict], facts: dict, cfg: Optional[Config] = None,
                   flagged: Sequence[int] = (), rules: Sequence[str] = ()) -> str:
    """The caption prompt: full images + verdict + Qwen's OWN notes per region. The blind
    draft and any earlier attempt are never included, so nothing can be kept or appended to."""
    cfg = cfg or CFG
    notes = []
    for g in groups:
        text = _group_note(g)
        if not text:
            continue
        place = ("Across the whole scene" if g["whole"] else
                 f"{'On' if g['place'].endswith('side') else 'In'} {g['place']}")
        mark = "  (this was left out of your last caption: include it)" if g["id"] in set(flagged) else ""
        notes.append(f"- {place}: {text}{mark}")
    cl = facts.get("clouds", {}) or {}
    cloud_lines = []
    if float(cl.get("post", 0.0)) >= 0.05:
        cloud_lines.append(f"- Clouds cover {cl.get('where_post') or 'part of the scene'} in the AFTER image; "
                           "the ground under them cannot be seen.")
    if float(cl.get("pre", 0.0)) >= 0.05:
        cloud_lines.append(f"- Clouds covered {cl.get('where_pre') or 'part of the scene'} in the BEFORE image, "
                           "so that area cannot be compared.")
    n_sent = int(min(max(2, 1 + math.ceil(0.75 * len(notes)) + (1 if float(cl.get("post", 0.0)) >= 0.05 else 0)),
                     int(cfg.caption_max_sentences)))
    head = "You are given two satellite images of the same place: (1) BEFORE an event and (2) AFTER it.\n"
    if label:
        head += f"The event that caused the change was {EVENT_PHRASE.get(label, label.replace('_', ' '))}.\n"
    head += ("\nYou looked closely at parts of the scene and noted what changed there:\n" + "\n".join(notes) + "\n"
             if notes else "\nA close look found no clear change in any single part of the scene.\n")
    if cloud_lines:
        head += "\n" + "\n".join(cloud_lines) + "\n"
    opener = (f"- Begin the first sentence with {EVENT_PHRASE.get(label, label.replace('_', ' '))} as its subject "
              f"(\"{_event_subject(label)} has ...\"), then describe the changes.\n" if label else
              "- Begin the first sentence by naming the kind of disaster that caused the change, as its subject "
              "(\"<Disaster> has ...\").\n")
    return (head
            + f"\nWrite ONE change caption of about {n_sent} sentences describing how the place changed from "
              "BEFORE to AFTER, as one flowing description.\n"
            + opener
            + ("- Include every note above, each with where it is (north is at the top of the image). You may "
               "combine notes that describe the same change, but leave none out.\n" if notes else "")
            + "- Write in your own words while looking at the two images; do not copy the notes word for word "
              "and do not list or number them.\n"
            + "- Do not call any area unchanged where a note describes a change.\n"
            + ("- Where clouds hide the ground, say that clouds cover that area; do not describe the ground "
               "under them.\n" if cloud_lines else "")
            + "- Describe extent with words such as widespread, partial, scattered or isolated. No numbers or "
              "percentages, no headings or bullet points.\n"
            + "- Do not mention close-ups, notes, image quality, measurements or how the analysis was done. "
              "Never invent place names, dates or casualties.\n"
            + "".join(f"- {r_}\n" for r_ in rules)
            + CAPTION_VERDICT)


# ── 4 · VERIFY ───────────────────────────────────────────────────────────────
def _names_event(body: str, label: str) -> bool:
    keys = [k for k, v in _STATED_ALIASES.items() if v == label and k != "unclear"]
    return any(_re12.search(r"\b" + _re12.escape(k) + r"\b", body or "", _re12.I) for k in keys)


def verify_caption(raw: str, groups: List[dict], items: List[dict], facts: dict,
                   label: Optional[str], cfg: Optional[Config] = None) -> dict:
    """Coverage and consistency of a written caption. An item is required when Qwen saw
    it (status visible) or confirmation is off (unconfirmed). It is covered when a
    sentence names its kind of change at its place, or carries its group's note (shared
    content words, or sentence similarity when the model is available) at that place."""
    cfg = cfg or CFG
    body = caption_body(raw)
    sents = _sentences(body)
    req = [it for it in items if it["status"] in ("visible", "unconfirmed")]
    by_group = {g["id"]: g for g in groups}
    model = _sbert_model(cfg) if req and sents else None
    sims = {}
    if model is not None:
        notes = {g["id"]: _group_note(g) for g in groups if _group_note(g)}
        if notes:
            gids = list(notes)
            emb = model.encode([notes[i] for i in gids] + sents, normalize_embeddings=True, show_progress_bar=False)
            ne, se = emb[:len(gids)], emb[len(gids):]
            for a_, gid in enumerate(gids):
                sims[gid] = [float(np.dot(ne[a_], se[b_])) for b_ in range(len(sents))]
    cov = {}
    for it in req:
        g = by_group.get(it.get("group"))
        place = it.get("where", "") or (g["place"] if g else "")
        hit = ("", "")
        for b_, s_ in enumerate(sents):
            if _mentions_item(it, s_):
                hit = ("kind", s_); break
        if not hit[0] and g is not None and _group_note(g):
            note = _group_note(g)
            for b_, s_ in enumerate(sents):
                if not _place_ok(_zones_of(s_), g["cells"]) or _says_no_change(s_):
                    continue
                shared, ratio = _note_overlap(note, s_)
                if shared >= 2 and ratio >= float(cfg.caption_cover_overlap):
                    hit = ("note words", s_); break
                if g["id"] in sims and sims[g["id"]][b_] >= float(cfg.caption_cover_sim):
                    hit = ("note similarity", s_); break
        cov[it["id"]] = {"covered": bool(hit[0]), "by": hit[0], "sentence": hit[1]}
    missing_items = [it["id"] for it in req if not cov[it["id"]]["covered"]]
    missing_groups = sorted({it["group"] for it in req if not cov[it["id"]]["covered"] and "group" in it})

    problems, rules = [], []
    event_ok = True
    if label:
        event_ok = _names_event(body, label) and stated_disaster(raw) == label
        if not event_ok:
            problems.append({"kind": "event", "check": f"caption does not name {label} as the event"})
            rules.append(f"The event was {EVENT_PHRASE.get(label, label)}: name it as the cause in the first "
                         "sentence and do not attribute the change to anything else.")
    cl = facts.get("clouds", {}) or {}
    cloud_ok = not (float(cl.get("post", 0.0)) >= 0.05 and not any(_re_cloud(s_) for s_ in sents))
    if not cloud_ok:
        problems.append({"kind": "cloud", "check": "clouds in the AFTER image not mentioned"})
        rules.append(f"Say that clouds cover {cl.get('where_post') or 'part of the scene'}.")
    contradictions = []
    absent = set(facts.get("absent", []) or [])
    for s_ in sents:
        for k_, name_ in SURFACE_ABSENT_NAME.items():
            if name_ in absent and asserts(s_, k_):
                contradictions.append({"kind": "absent", "check": f"says {name_}; measured none", "sentence": s_})
                rules.append(f"Do not describe {name_}: there is none in this scene (and do not mention its absence).")
        if int(facts.get("damaged", 0) or 0) >= 1 and asserts(s_, "building") and asserts(s_, "intact"):
            contradictions.append({"kind": "buildings", "check": "buildings said intact; ChangeOS sees damage",
                                   "sentence": s_})
            rules.append("Some buildings are damaged: do not describe the buildings as intact.")
        if _says_unchanged(s_):
            zs = _zones_of(s_)
            if zs:
                for it in req:
                    if it["cells"] and _place_ok(zs, it["cells"]) and not _mentions_item(it, s_):
                        contradictions.append({"kind": "unchanged", "sentence": s_,
                                               "check": f"calls a place unchanged where {it['label']} was seen"})
                        rules.append(f"Do not call {it['where'] or 'that area'} unchanged: a change was noted there.")
                        break
    rules = list(dict.fromkeys(rules))
    ok = event_ok and cloud_ok and not contradictions and not missing_items
    return {"ok": ok, "event_ok": event_ok, "cloud_ok": cloud_ok, "items": cov,
            "required": len(req), "covered": len(req) - len(missing_items),
            "missing_items": missing_items, "missing_groups": missing_groups,
            "problems": problems, "contradictions": contradictions, "rules": rules,
            "sentences": len(sents), "words": len(_re12.findall(r"[A-Za-z']+", body))}


def _attempt_score(v: dict) -> Tuple:
    return (v["event_ok"], -len(v["contradictions"]), v["covered"], v["cloud_ok"])


def compose_and_verify(captioner: "QwenCaptioner", pre: Image.Image, post: Image.Image,
                       groups: List[dict], items: List[dict], facts: dict, label: Optional[str],
                       cfg: Optional[Config] = None) -> Tuple[str, List[dict], dict]:
    """WRITE then VERIFY; rewrite from scratch (omitted notes flagged, broken rules
    stated) up to `caption_max_rewrites` times. Returns (caption, attempts, final check)."""
    cfg = cfg or CFG
    attempts, flagged, rules = [], (), []
    for _k in range(1 + int(cfg.caption_max_rewrites)):
        prompt = compose_prompt(label, groups, facts, cfg, flagged, rules)
        raw = captioner.compose(pre, post, prompt).strip()
        v = verify_caption(raw, groups, items, facts, label, cfg)
        attempts.append({"prompt": prompt, "caption": raw, "check": v})
        if v["ok"]:
            break
        flagged = tuple(v["missing_groups"])
        rules = list(dict.fromkeys(rules + v["rules"]))
    best = max(attempts, key=lambda a_: _attempt_score(a_["check"]))
    return best["caption"], attempts, best["check"]


def look_then_write(captioner: "QwenCaptioner", pre: Image.Image, post: Image.Image, facts: dict,
                    cfg: Optional[Config] = None, label_check: bool = True) -> dict:
    """The rev12 system caption. Returns the caption and its full record: the evidence
    items (with status), the excluded items (with reasons), the groups and Qwen's notes,
    every attempt with its check, and a coverage summary. `crops` holds the close-ups
    (PIL images; `look_record` drops them for JSON)."""
    cfg = cfg or CFG
    label = facts.get("label") if label_check else None
    label = label if label in DISASTER_VOCAB else None
    items, excluded = caption_evidence(facts, cfg)
    groups = group_evidence(items, cfg)
    crops = look_at_groups(captioner, pre, post, groups, cfg)
    raw, attempts, v = compose_and_verify(captioner, pre, post, groups, items, facts, label, cfg)
    return {"caption": raw, "label": label, "items": items, "excluded": excluded, "groups": groups,
            "attempts": attempts, "coverage": coverage_summary(items, excluded, v, attempts), "crops": crops}


def coverage_summary(items: List[dict], excluded: List[dict], v: dict, attempts: List[dict]) -> dict:
    """Per-item final status and the counts the trace and §21 report."""
    for it in items:
        c = v["items"].get(it["id"])
        if c is not None:
            it["final"] = ("covered" if c["covered"] else "missing") + (" (unconfirmed)" if it["status"] == "unconfirmed" else "")
            it["covered_by"] = c["by"]
        else:
            it["final"] = it["status"]            # not_visible
    lab = lambda it: f"{it['label']} @ {it.get('where') or '-'}"
    return {"valid": len(items), "visible": sum(it["status"] == "visible" for it in items),
            "required": v["required"], "covered": v["covered"],
            "missing": [lab(it) for it in items if it["final"].startswith("missing")],
            "not_visible": [lab(it) for it in items if it["status"] == "not_visible"],
            "excluded": len(excluded), "event_named": v["event_ok"], "clouds_ok": v["cloud_ok"],
            "contradictions": [c["check"] for c in v["contradictions"]], "attempts": len(attempts),
            "complete": bool(v["ok"]), "sentences": v["sentences"], "words": v["words"]}


def look_record(lw: dict) -> dict:
    """JSON-safe copy of a look_then_write result (no crops)."""
    import copy as _copy
    out = {k: _copy.deepcopy(v) for k, v in lw.items() if k != "crops"}
    for g in out.get("groups", []):
        g["items"] = [it["id"] if isinstance(it, dict) else it for it in g["items"]]
        g["cells"] = [list(c) for c in g["cells"]]
    return out


def look_from_record(rec: dict) -> Tuple[List[dict], List[dict]]:
    """(groups, items) rebuilt from a `look_record`, so the same looks can be written up
    again under another condition (C2 without the label, the grounding tests) without
    spending a single extra look."""
    import copy as _copy
    items = _copy.deepcopy(rec.get("items", []))
    for it in items:
        it["cells"] = [tuple(c) for c in it.get("cells", [])]
        it["status"] = it.get("status", "visible")
    by_id = {it["id"]: it for it in items}
    groups = []
    for g in _copy.deepcopy(rec.get("groups", [])):
        g["items"] = [by_id[i] if isinstance(i, str) else i for i in g["items"]]
        g["cells"] = [tuple(c) for c in g.get("cells", [])]
        groups.append(g)
    return groups, items


def corrupt_label(facts: dict) -> Tuple[dict, str]:
    """Grounding test (§21 I, rev12): the verdict cycled to another class."""
    import copy as _copy
    f = _copy.deepcopy(facts)
    cyc = {"flood": "volcanic_eruption", "volcanic_eruption": "earthquake", "earthquake": "flood"}
    f["label"] = cyc.get(facts.get("label"), "flood")
    return f, f["label"]


## 12 · Pipeline — the whole Stage 2 chain

`predict(pre_path, post_path)` is still the only entry point and still needs nothing but two paths.
The order is: object branch → scene branch → fusion → SDA → evidence aggregation → tokenisation →
cross-branch attention → DKG → classifier → zero-shot → pooling → provisional RVG → blind caption →
measured facts → **system caption** (rev12: evidence → close-up looks → write → verify) → final RVG.
The result carries the caption's full record in `r.look` and the close-ups in `r.look_crops`.

`extract_features()` runs the same chain but stops after the attention step and returns
`ClassifierFeatures` — that is what §18 uses to build a training set without regenerating captions.

Every ablation switch is read here, in one place, so a switched-off module changes exactly one thing
in this sequence.


In [ ]:
@dataclass
class Instance:
    mask: np.ndarray
    bbox: Tuple[int, int, int, int]
    area: int
    change_score: float                    # 90th pct of OBJECT severity inside the mask
    refined_score: float                   # 90th pct of the SDA-refined FUSED field
    sam_score: float
    label: str = ""
    label_conf: float = 0.0
    damage_grade: Optional[int] = None
    transition: Optional[SemanticTransition] = None


@dataclass
class PredictionResult:
    pre: np.ndarray
    post: np.ndarray
    obj: ObjectChangeOutput
    scene: SceneBranchOutput
    semantics: SceneSemantics
    fused: FusedEvidence
    refined_map: np.ndarray
    sda_attention: np.ndarray
    structure_energy: np.ndarray
    sda_report: Dict[str, float]
    scene_evidence: SceneEvidence
    object_evidence: ObjectEvidence
    tokens: FusedTokens
    features: ClassifierFeatures
    scene_transitions: List[SemanticTransition]
    object_transitions: List[SemanticTransition]
    instances: List[Instance]
    graph: Any
    dkg: dict
    classifier: Optional[dict]
    zeroshot: dict
    posterior: DisasterPosterior
    rvg_provisional: dict
    rvg: dict
    blind: Optional[BlindComparison]
    blind_caption: str
    caption: str
    package: EvidencePackage
    timings: Dict[str, float]
    mode: str
    switches: Dict[str, bool]
    # rev9: draft -> check -> revise
    draft_caption: str = ""
    edits: List[dict] = field(default_factory=list)
    facts: dict = field(default_factory=dict)
    residual: List[dict] = field(default_factory=list)
    quality: dict = field(default_factory=dict)          # rev10: clouds, alignment
    # rev12: look -> write -> verify. `look` is JSON-safe (evidence items with status,
    # excluded items with reasons, regions with Qwen's notes, every attempt and its check,
    # coverage); `look_crops` holds the BEFORE/AFTER close-ups Qwen looked at.
    look: dict = field(default_factory=dict)
    look_crops: list = field(default_factory=list)

    @property
    def disaster(self) -> str:
        return self.posterior.label

    @property
    def confidence(self) -> float:
        return float(self.rvg["confidence"])

    def summary(self) -> str:
        return (f"[{self.mode}] disaster={self.disaster} "
                f"p={self.posterior.prob:.2f} confidence={self.confidence:.2f} "
                f"hedge={self.rvg['hedge']} | dkg={self.dkg.get('disaster')} "
                f"clf={self.classifier['label'] if self.classifier else 'off'} "
                f"zs={self.zeroshot['label']} | instances={len(self.instances)} "
                f"transitions={len(self.scene_evidence.transitions)} "
                f"scene_coverage={self.fused.scalars['scene_coverage']:.3f}")

    def top_k(self, k: int = 3) -> List[dict]:
        return self.posterior.top_k(k)


class DisasterCaptioningPipeline:
    """
    Stage 2 execution order, and why:

      1. SCENE branch   — AnyChange + physical channels -> scene change map -> MULTI-SCALE
         patch-grid PRE vs POST RemoteCLIP semantic transitions. Reads nothing from the
         object branch.
      2. OBJECT branch  — ChangeOS at native resolution -> SAM2 instances -> PRE-crop vs
         POST-crop RemoteCLIP transitions. Reads ONE thing from the scene branch: the
         change map that seeds SAM2 (`sam2_seed="object+scene"`), without which the
         branch is empty on any disaster that damages no buildings.
      3. MULTI-SCALE FUSION — the branches meet here for the first time.
      4. SDA v2 on the fused continuous field.
      5. EVIDENCE — SceneEvidence and ObjectEvidence: the two branches expressed as the
         named quantities the graph, the classifier and the caption all consume.
      6. TOKENISE + CROSS-BRANCH ATTENTION — evidence into one RemoteCLIP space; scene
         attends to objects and objects attend to scene; produces `z`.
      7. DKG v3 (symbolic) and the LEARNED HEAD (parametric) reason in parallel over the
         same evidence, and a zero-shot CLIP prior is read off the POST image.
      8. TWO opinions are pooled log-linearly into one posterior: the DKG plus ONE
         CLIP-derived source (the prior while the head is untrained, the head once it is
         trained). The third is computed and its correlation reported, never pooled.
      9. RVG v3 verifies, Qwen writes a BLIND caption (the C3 baseline and the RVG
         blind check), the measured facts are collected, and the SYSTEM caption is
         written by `caption_mode`: "look" (rev12, default) lists every valid measured
         change, has Qwen describe a close-up of each region, writes the caption once
         from the images + verdict + those notes, and verifies every visible change is
         in it; "revise" (rev9) and "checklist" (rev8) remain as comparisons. RVG then
         verifies again with both captions in hand.

    Every `CFG.use_*` switch is honoured here, so an ablation never needs a second code
    path. `predict(pre_path, post_path)` remains the only call required.
    """

    def __init__(self, cfg, changeos, anychange, physical, refiner, clip, trans,
                 fusion, sda, tokeniser, attention, dkg, classifier, rvg, captioner):
        self.cfg = cfg
        self.changeos, self.anychange, self.physical = changeos, anychange, physical
        self.refiner, self.clip, self.trans = refiner, clip, trans
        self.fusion, self.sda = fusion, sda
        self.tokeniser, self.attention = tokeniser, attention
        self.dkg, self.classifier, self.rvg = dkg, classifier, rvg
        self.captioner = captioner
        self.cloud = CloudMasker(cfg, clip)          # rev10

    # -- io ------------------------------------------------------------------
    @staticmethod
    def _square(img: Image.Image, n: int, mode: str) -> np.ndarray:
        """ISSUE 15: resize((n, n)) stretches a non-square tile, and every area
        fraction downstream is then measured on a distorted image. Letterboxing keeps
        the aspect ratio and pads with black; the padding is a constant region, so the
        bitemporal difference there is exactly zero and it adds no spurious change."""
        if mode == "stretch":
            return np.array(img.resize((n, n), Image.BICUBIC))
        w, h = img.size
        s = n / max(w, h)
        rw, rh = max(1, int(round(w * s))), max(1, int(round(h * s)))
        canvas = Image.new("RGB", (n, n), (0, 0, 0))
        canvas.paste(img.resize((rw, rh), Image.BICUBIC), ((n - rw) // 2, (n - rh) // 2))
        return np.array(canvas)

    def _prepare(self, pre_path, post_path) -> dict:
        """rev10: load both images, shift AFTER onto BEFORE, and find what is not ground
        (cloud, haze) in either. Every measurement uses this, including calibration."""
        (pre_w, pre_n), (post_w, post_n) = self._load(pre_path), self._load(post_path)
        post_w, post_n, align = align_pair(pre_w, post_w, pre_n, post_n, self.cfg)
        cpre, cpost, clouds = self.cloud(pre_w, post_w)
        return {"pre_w": pre_w, "pre_n": pre_n, "post_w": post_w, "post_n": post_n,
                "invalid": cpre | cpost, "cloud_pre": cpre, "cloud_post": cpost,
                "clouds": clouds, "align": align}

    def _load(self, path):
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(f"Image not found: {p}")
        img = Image.open(p).convert("RGB")
        mode = self.cfg.resize_mode
        n = min(self.cfg.native_max, max(img.size))
        native = self._square(img, n, mode)
        work = self._square(img, self.cfg.work_size, mode)
        return work, native

    @staticmethod
    def _bbox(m):
        ys, xs = np.where(m)
        return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

    @staticmethod
    def _heat(m):
        return Image.fromarray((plt.cm.inferno(m)[..., :3] * 255).astype(np.uint8))

    @staticmethod
    def _score_in(mask, field, pct=90):
        v = field[mask]
        return float(np.percentile(v, pct)) if v.size else 0.0

    def switches(self) -> Dict[str, bool]:
        c = self.cfg
        return {"changeos": c.use_changeos, "anychange": c.use_anychange,
                "transitions": c.use_transitions, "sda": c.use_sda, "dkg": c.use_dkg,
                "classifier": c.use_classifier, "rvg": c.use_rvg,
                "attention": c.use_attention, "blind_pass": c.use_blind_pass,
                "zeroshot": c.use_zeroshot, "label_in_prompt": c.label_in_prompt,
                "neutral_names": c.neutral_measurement_names}

    # -- branch 1: OBJECT ----------------------------------------------------
    def _object_branch(self, pre_w, post_w, pre_n, post_n,
                       scene_map: Optional[np.ndarray] = None):
        cfg = self.cfg
        hw = (cfg.work_size, cfg.work_size)
        if not cfg.use_changeos:
            return null_object_output(hw), [], [], np.zeros((0, self.clip.embed_dim), np.float32)

        obj = self.changeos.detect(pre_n, post_n, hw)
        # ISSUE 4: seeding SAM2 from damage_severity alone means "xBD grade >= 2", so on
        # a flood, a wildfire or an eruption the object branch returned zero instances
        # and took the object tokens and the cross-branch attention down with it. The
        # seed is now the union of graded building damage and confident scene change, so
        # SAM2 segments whatever actually changed. The xBD grade of a scene-seeded
        # instance is still read from ChangeOS, so it is simply ungraded (None) when no
        # building is there -- nothing is invented.
        seed = obj.damage_severity.astype(np.float32)
        if cfg.sam2_seed == "object+scene" and scene_map is not None:
            scene_seed = np.where(np.asarray(scene_map, np.float32) >= cfg.sam2_scene_seed_thr,
                                  np.asarray(scene_map, np.float32), 0.0)
            seed = np.maximum(seed, scene_seed)
        masks = self.refiner.refine(post_w, seed)

        boxes, keep = [], []
        for m, s in masks:
            if m.sum() == 0:
                continue
            boxes.append(self._bbox(m)); keep.append((m, s))

        if cfg.mode == "dual" and cfg.use_transitions:
            transitions, embeds = self.trans.object(pre_w, post_w, boxes)
        else:
            # legacy behaviour: POST crop only, no temporal comparison
            crops = [post_w[y0:y1 + 1, x0:x1 + 1] for (x0, y0, x1, y1) in boxes]
            o = self.clip.classify(crops, "object")
            transitions = {k: SemanticTransition(
                o.labels[k], o.labels[k], o.groups[k], o.groups[k], float(o.conf[k]),
                float(o.conf[k]), float(o.conf[k]), float(o.conf[k]), 1.0, 0.0,
                1.0 / max(len(boxes), 1), b, "object", "object") for k, b in enumerate(boxes)}
            embeds = o.embeds if len(o) else np.zeros((0, self.clip.embed_dim), np.float32)

        instances = []
        for k, (m, s) in enumerate(keep):
            t = transitions.get(k)          # ISSUE 16: keyed by box index, never by rank
            grade = None
            vals = obj.damage_grades[m]; vals = vals[vals > 0]
            if vals.size:
                grade = int(np.bincount(vals).argmax())
            instances.append(Instance(
                mask=m, bbox=boxes[k], area=int(m.sum()),
                change_score=self._score_in(m, obj.damage_severity),
                refined_score=0.0, sam_score=s,
                label=(t.dst_label if t else ""), label_conf=(t.confidence if t else 0.0),
                damage_grade=grade, transition=t))
        return obj, instances, list(transitions.values()), embeds

    # -- branch 2: SCENE -----------------------------------------------------
    def _scene_branch(self, pre_w, post_w, invalid=None):
        cfg = self.cfg
        hw = (cfg.work_size, cfg.work_size)
        phys = self.physical(pre_w, post_w, invalid)

        if cfg.mode == "object_only" or not cfg.use_anychange or self.anychange is None:
            any_prob = np.zeros(hw, np.float32)
            any_angle = np.zeros(hw, np.float32)
            regions, ran = [], False
        else:
            ac = self.anychange.detect(pre_w, post_w)
            any_prob, regions, ran = ac.change_prob, ac.regions, True
            any_angle = ac.raw_confidence          # degrees, for inspection/plots
            if invalid is not None and np.asarray(invalid).any():
                # rev10: a change region that is mostly cloud is a cloud, not a change
                inv = np.asarray(invalid, bool)
                regions = [r for r in regions if float(inv[r.mask].mean()) < 0.5]
                any_prob = np.zeros(hw, np.float32)
                for r in regions:
                    any_prob[r.mask] = np.maximum(any_prob[r.mask], r.prob)
                any_prob[inv] = 0.0

        if cfg.mode == "object_only":
            scene_map = np.zeros(hw, np.float32)
            phys = {k: np.zeros(hw, np.float32) for k in phys}
        else:
            # scene evidence = strongest of the learned zero-shot detector and the
            # unlearned physical composite; they are independent, so a union is the
            # right combination and their overlap is what RVG later audits.
            scene_map = np.clip(np.maximum(any_prob, phys["composite"]), 0, 1)
            if invalid is not None and np.asarray(invalid).any():
                scene_map = scene_map.copy(); scene_map[np.asarray(invalid, bool)] = 0.0   # rev10

        if cfg.mode == "dual" and cfg.use_transitions:
            sem = self.trans.scene(pre_w, post_w, scene_map, invalid)
        else:
            sem = empty_scene_semantics(self.clip.embed_dim)
            sem.global_pre = self.clip.encode_global(pre_w)
            sem.global_post = self.clip.encode_global(post_w)

        scene = SceneBranchOutput(any_prob, phys, scene_map, regions, ran, any_angle)
        return scene, sem

    # -- reasoning -----------------------------------------------------------
    def _reason(self, fused, sem, scene_ev, obj_ev, instances, tokens):
        cfg = self.cfg
        if cfg.use_dkg:
            G, concepts = self.dkg.build(fused, sem.transitions, instances, scene_ev, obj_ev)
            dkg_out = self.dkg.infer(G, concepts)
        else:
            G, dkg_out = nx.DiGraph(), DynamicKnowledgeGraph.null_output()

        feats = build_features(scene_ev, obj_ev, fused.physical, dkg_out, tokens, cfg)
        clf_out = self.classifier.predict(feats) if cfg.use_classifier else None
        zs = self.classifier.zero_shot(sem.global_post, sem.global_pre)

        # every source is COMPUTED (so its dependence on the others is reported); which
        # of them enter the product is decided by the weights in pool_posterior().
        sources: Dict[str, Optional[np.ndarray]] = {}
        if cfg.use_zeroshot:
            sources["zeroshot"] = zs["probs"]
        if clf_out is not None:
            sources["classifier"] = clf_out["probs"]
        if cfg.use_dkg:
            sources["dkg"] = dkg_out["prob_vector"]
        posterior = pool_posterior(cfg, sources,
                                   head_trained=bool(clf_out and clf_out["is_trained"]))
        return G, dkg_out, feats, clf_out, zs, posterior

    # -- main ----------------------------------------------------------------
    def predict(self, pre_path, post_path, generate_caption: bool = True) -> PredictionResult:
        cfg = self.cfg
        T, clock = {}, time.perf_counter
        prep = self._prepare(pre_path, post_path)                      # rev10: aligned, clouds found
        pre_w, pre_n, post_w, post_n = prep["pre_w"], prep["pre_n"], prep["post_w"], prep["post_n"]

        # The scene branch runs FIRST so its change map can seed SAM2 (ISSUE 4). It still
        # reads nothing from the object branch, so the two remain independent and the
        # only coupling is the seed -- which is what makes the object branch usable on a
        # disaster that damages no buildings.
        t0 = clock()
        scene, sem = self._scene_branch(pre_w, post_w, prep["invalid"])
        T["scene_branch"] = clock() - t0
        torch.cuda.empty_cache()
        log.info("SCENE branch: anychange_ran=%s, %d changed transitions, coverage %.3f",
                 scene.anychange_ran, len(sem.changed()),
                 scene.coverage(cfg.scene_change_thr))

        t0 = clock()
        obj, instances, obj_tr, obj_embeds = self._object_branch(
            pre_w, post_w, pre_n, post_n, scene_map=scene.scene_map)
        T["object_branch"] = clock() - t0
        log.info("OBJECT branch: %d instances (seed=%s), damaged area %.4f",
                 len(instances), cfg.sam2_seed, obj.damaged_area_frac)

        t0 = clock(); fused = self.fusion(obj, scene); T["fusion"] = clock() - t0

        t0 = clock()
        if cfg.use_sda:
            refined, attention, energy = self.sda(fused.fused, pre_w, post_w)
        else:
            refined = fused.fused.copy()
            attention = np.zeros_like(refined)
            energy = np.zeros_like(refined)
        sda_rep = self.sda.report(fused.fused, refined, cfg.change_threshold)
        T["sda"] = clock() - t0
        for i in instances:
            i.refined_score = self._score_in(i.mask, refined)

        # ---- evidence, tokens, attention ----------------------------------
        t0 = clock()
        scene_ev = SceneEvidence.build(sem, scene.physical, fused.scalars,
                                       buildings=getattr(obj, "building_mask", None))    # rev11
        if bool(getattr(cfg, "water_requires_agreement", True)) and \
                scene_ev.extra.get("water_confirmed", 0.0) < float(getattr(cfg, "water_derived_min", 0.05)):
            # rev10: "water widened" is a claim about water; without confirmed water it goes
            _n0 = len(sem.transitions)
            sem.transitions = [t for t in sem.transitions
                               if not (t.kind == "derived" and t.dst_group == "water_expanded")]
            if len(sem.transitions) < _n0:
                log.info("'water widened' dropped: no confirmed new water")
        obj_ev = ObjectEvidence.build(obj, instances, cfg)
        toks = self.tokeniser(sem, instances, obj_embeds, scene.physical)
        tokens = self.attention(toks, sem.global_post, (cfg.work_size, cfg.work_size),
                                enabled=cfg.use_attention)
        T["evidence"] = clock() - t0
        log.info("Evidence: tokens=%s | align s->o %.3f, o->s %.3f",
                 tokens.counts(), tokens.align_scene_to_object, tokens.align_object_to_scene)

        # ---- reasoning ------------------------------------------------------
        t0 = clock()
        G, dkg_out, feats, clf_out, zs, posterior = self._reason(
            fused, sem, scene_ev, obj_ev, instances, tokens)
        T["reasoning"] = clock() - t0
        log.info("DKG=%s (%.2f) | %s=%s | zero-shot=%s | POSTERIOR=%s (%.2f)",
                 dkg_out.get("disaster"), dkg_out.get("score", 0.0),
                 "classifier" if "classifier" in posterior.weights else "head(not pooled)",
                 clf_out["label"] if clf_out else "off", zs["label"],
                 posterior.label, posterior.prob)

        # ---- verification and captions --------------------------------------
        t0 = clock()
        # which sources actually entered the product -- RVG must not audit an opinion
        # that was computed but excluded (the untrained head duplicates the CLIP prior).
        clf_pooled = "classifier" in posterior.weights
        rvg_prov = (self.rvg.verify(posterior, dkg_out, clf_out, fused, sem.transitions,
                                    instances, scene_ev, clf_pooled=clf_pooled)
                    if cfg.use_rvg else ReliabilityVerificationGate.null_output(posterior))
        T["rvg"] = clock() - t0

        blind_caption, blind = "", None
        # rev9: the revise mode needs Qwen's own draft, which IS the blind pass
        if generate_caption and (cfg.use_blind_pass or cfg.caption_mode == "revise"):
            t0 = clock()
            blind_caption = self.captioner.blind_caption(Image.fromarray(pre_w),
                                                         Image.fromarray(post_w))
            T["qwen_blind"] = clock() - t0
            blind = compare_blind_and_grounded(self.clip, cfg, blind_caption, "", posterior.label)
            if cfg.use_rvg:
                rvg_prov = self.rvg.verify(posterior, dkg_out, clf_out, fused, sem.transitions,
                                           instances, scene_ev, blind=blind,
                                           clf_pooled=clf_pooled)

        # rev8: the maps that locate each checklist item travel with the evidence.
        package = build_evidence_package(scene_ev, obj_ev, sem, posterior, clf_out,
                                         dkg_out, rvg_prov,
                                         label_in_prompt=cfg.label_in_prompt,
                                         neutral=cfg.neutral_measurement_names,
                                         change_map=refined, physical=scene.physical,
                                         regions=scene.regions, cfg=cfg)
        caption, draft, edits, residual = "", "", [], []
        quality = {"clouds": prep["clouds"], "align": prep["align"],
                   "cloud_pre": prep["cloud_pre"], "cloud_post": prep["cloud_post"]}
        facts = measured_facts(scene_ev, obj_ev, sem, posterior, cfg, physical=scene.physical,
                               quality=quality, anychange_regions=scene.regions, instances=instances)
        look, look_crops = {}, []
        if generate_caption:
            t0 = clock()
            if cfg.caption_mode == "look":
                draft = blind_caption
                lw = look_then_write(self.captioner, Image.fromarray(pre_w), Image.fromarray(post_w),
                                     facts, cfg=cfg, label_check=cfg.label_in_prompt)
                caption, look_crops = lw["caption"], lw["crops"]
                look = look_record(lw)
                cov = lw["coverage"]
                # what the kept caption still contradicts (same shape as the rev9 residual)
                residual = [{"kind": "contradiction", "check": c} for c in cov["contradictions"]]
                log.info("CAPTION (look): %d valid changes, %d visible, %d in the caption%s | %d region(s), "
                         "%d attempt(s)%s", cov["valid"], cov["visible"], cov["covered"],
                         "" if cov["covered"] == cov["required"] else f" (missing: {'; '.join(cov['missing'])})",
                         len(lw["groups"]), cov["attempts"],
                         "" if not cov["contradictions"] else f" | still contradicting: {cov['contradictions']}")
            elif cfg.caption_mode == "revise":
                draft = blind_caption
                caption, edits, residual = revise_caption(
                    self.captioner, Image.fromarray(pre_w), Image.fromarray(post_w),
                    draft, facts, label_check=cfg.label_in_prompt, cfg=cfg)
            else:
                caption = self.captioner.caption(
                    Image.fromarray(pre_w), Image.fromarray(post_w), self._heat(refined),
                    package.to_prompt(), hedge=bool(rvg_prov["hedge"]),
                    label_in_prompt=cfg.label_in_prompt)
            T["qwen"] = clock() - t0
            if blind is not None:
                blind = compare_blind_and_grounded(self.clip, cfg, blind_caption, caption,
                                                   posterior.label)

        rvg_final = (self.rvg.verify(posterior, dkg_out, clf_out, fused, sem.transitions,
                                     instances, scene_ev, blind=blind,
                                     caption=caption if caption else None,
                                     clf_pooled=clf_pooled,
                                     caption_primed=bool(cfg.label_in_prompt))
                     if cfg.use_rvg else ReliabilityVerificationGate.null_output(posterior))
        T["total"] = sum(T.values())

        return PredictionResult(
            pre_w, post_w, obj, scene, sem, fused, refined, attention, energy, sda_rep,
            scene_ev, obj_ev, tokens, feats, sem.transitions, obj_tr, instances, G,
            dkg_out, clf_out, zs, posterior, rvg_prov, rvg_final, blind, blind_caption,
            caption, package, T, cfg.mode, self.switches(),
            draft_caption=draft, edits=edits, facts=facts, residual=residual, quality=quality,
            look=look, look_crops=look_crops)

    # -- feature extraction for training (no captioner, no Qwen) -------------
    def extract_features(self, pre_path, post_path) -> ClassifierFeatures:
        """Everything except the two language passes. This is what `train_head` calls
        per labelled pair, so training never pays for caption generation."""
        return self.predict(pre_path, post_path, generate_caption=False).features


## 13 · Load everything

Weights download on first run: ChangeOS (~0.2 GB), SAM for AnyChange (~0.4 GB), SAM2 (~0.2 GB),
RemoteCLIP (~0.6 GB), Qwen2.5-VL-7B 4-bit (~6 GB). Later runs use the cache.

All fifteen components load unconditionally, whatever the ablation switches say, so that switching a
module off never also changes what got loaded.


In [ ]:
t0 = time.perf_counter()

# Every model is loaded unconditionally, regardless of the CFG.use_* switches: the
# switches select what the PIPELINE runs, and an ablation that had to reload weights
# would not be a controlled comparison.
changeos_det = ChangeOSDetector(CFG)
anychange_det = AnyChangeDetector(CFG)
physical   = ScenePhysicalChannels(CFG)
refiner    = SAM2Refiner(CFG)
clip       = RemoteCLIPEncoder(CFG)
trans      = SemanticTransitionExtractor(CFG, clip)
fusion     = MultiScaleFusion(CFG)
sda        = SubtleDamageAttention(CFG)
tokeniser  = EvidenceTokeniser(CFG, clip)
attention  = CrossBranchAttention(CFG)
dkg        = DynamicKnowledgeGraph(CFG)
classifier = LearnedDisasterClassifier(CFG, clip)
rvg        = ReliabilityVerificationGate(CFG)
captioner  = QwenCaptioner(CFG)

pipeline = DisasterCaptioningPipeline(
    CFG, changeos_det, anychange_det, physical, refiner, clip, trans, fusion, sda,
    tokeniser, attention, dkg, classifier, rvg, captioner)

log.info("All models loaded in %.1fs | VRAM %.2f GB",
         time.perf_counter() - t0, torch.cuda.memory_allocated() / 1e9)
log.info("OBJECT: ChangeOS+SAM2+RemoteCLIP | SCENE: AnyChange+physical+RemoteCLIP transitions")
log.info("REASONING: DKG v3 (symbolic) + learned head (%s) + zero-shot prior -> log-linear pool",
         "trained" if classifier.is_trained else "zero-shot init")


# ---------------------------------------------------------------------------
# ISSUE 9: one taxonomy, checked. Five independent tables assert the class list;
# any of them can drift without a single error until a confusion matrix is wrong.
# ---------------------------------------------------------------------------
def assert_taxonomy_consistent(strict: bool = True) -> dict:
    """Every table that names a disaster class must name the same ones."""
    rep = {"classes": list(DISASTER_CLASSES)}
    problems = []

    missing = set(DISASTER_CLASSES) - {r["name"] for r in DynamicKnowledgeGraph.RULES}
    extra = {r["name"] for r in DynamicKnowledgeGraph.RULES} - set(DISASTER_CLASSES)
    if missing or extra:
        problems.append(f"DKG rules: missing {sorted(missing)}, extra {sorted(extra)}")

    for name, table in (("DISASTER_PROMPTS", DISASTER_PROMPTS),
                        ("DISASTER_TRANSITION_PROMPTS", DISASTER_TRANSITION_PROMPTS),
                        ("DISASTER_KEYWORDS", DISASTER_KEYWORDS)):
        m, e = set(DISASTER_CLASSES) - set(table), set(table) - set(DISASTER_CLASSES)
        if m or e:
            problems.append(f"{name}: missing {sorted(m)}, extra {sorted(e)}")

    # rev9: the DisasterM3 loader was removed in rev7, so DM3_TO_CLASS may not exist;
    # the reachability check only applies when it does.
    _dm3 = globals().get("DM3_TO_CLASS") or {}
    unreachable = sorted(set(DISASTER_CLASSES) - set(_dm3.values())) if _dm3 else []
    rep["not_reachable_from_disasterm3"] = unreachable
    if unreachable:
        problems.append(f"no DisasterM3 event maps to {unreachable} -- those classes can be "
                        f"PREDICTED but never SCORED, so they only ever add false positives")

    aliased = sorted({a for a in _STATED_ALIASES.values() if a != "unclear"}
                     - set(DISASTER_CLASSES))
    if aliased:
        problems.append(f"caption aliases resolve to non-classes: {aliased}")

    rep["problems"] = problems
    rep["ok"] = not problems
    for p in problems:
        log.warning("TAXONOMY: %s", p)
    if problems and strict:
        raise AssertionError("taxonomy inconsistent: " + " | ".join(problems))
    if not problems:
        log.info("Taxonomy consistent across DKG rules, prompts, keywords and DM3_TO_CLASS "
                 "(%d classes)", len(DISASTER_CLASSES))
    return rep


## 18 · Training the head (optional, needs labels)

The head works untrained. The labelled pairs now come from DisasterM3 itself (§13b), so no CSV is
needed:

```python
dm3_pairs = dm3_labeled_pairs(per_class=30)     # fetches ~4 MB per pair
build_dm3_manifest(dm3_pairs)                   # optional, for reproducibility
```

`load_manifest` still reads a CSV if you have your own labelled pairs:

```
pre,post,disaster,caption,event
/path/pre_0001.png,/path/post_0001.png,flood,optional reference caption,midwest_flooding
```

`disaster` is mapped through `normalize_disaster` (flooding→flood, fire→wildfire, hurricane→cyclone, …) and
must then be one of `DISASTER_CLASSES`. `event` groups pairs for leave-event-out splits; if the column is
absent it is inferred from the file name, in either DisasterM3 (`libya_flood_96_pre_96.png`) or xBD
(`midwest_flooding_00000133_pre_disaster.png`) form. To fit one head on everything:

```python
pairs = load_manifest("/kaggle/input/<your-dataset>/train.csv")
history = train_head_on_pairs(pipeline, dm3_pairs)    # features cached, head fit + temperature-calibrated
```

Feature extraction is the expensive part and is cached, so re-training the head after changing its
width or dropout costs seconds. The fit uses a random validation split of the training pairs, early stopping on validation
accuracy, and post-hoc temperature scaling; `is_trained` then swaps the pool from DKG + CLIP prior to
DKG + trained head. **Do not report accuracy from a head trained this way on the same events you score** —
use `cross_validate_leave_event_out` (§19), which retrains per fold on held-out events. xBD's `disaster_type` metadata is the obvious first source of labels.


In [ ]:
@dataclass
class LabeledPair:
    pre: str
    post: str
    disaster: str
    reference_caption: Optional[str] = None
    event: Optional[str] = None          # grouping key for leave-event-out splits


# Label mapping and `infer_event` now live in §13b (the DisasterM3 loader), so the CSV
# path and the DisasterM3 path cannot disagree about what a label or an event is.
def canonical_label(raw: str) -> str:
    """Any dataset's wording -> a class in DISASTER_CLASSES, else the slug unchanged
    (so `load_manifest` can report it as out of scope)."""
    return normalize_disaster(raw) or re.sub(r"[\s\-]+", "_", (raw or "").strip().lower())


def load_manifest(path: str) -> List[LabeledPair]:
    """CSV with columns: pre,post,disaster[,caption][,event]. Paths may be relative to the
    CSV. `disaster` accepts DisasterM3 / xBD vocabulary (flooding, fire, hurricane, ...)
    via LABEL_ALIASES. `event` groups pairs for leave-event-out splits; when the column is
    absent it is inferred from the file name (midwest_flooding_00000133_* -> midwest_flooding).

    No dataset ships with this notebook and none is downloaded: supply your own
    labelled pairs (xBD `disaster_type` metadata, RSCC, or your own annotation) and the
    head trains on them. Without a manifest the head stays in its text-initialised
    zero-shot mode and the pipeline runs exactly as it does above.
    """
    import csv
    root = Path(path).parent
    pairs, bad = [], []
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            d = canonical_label(row.get("disaster") or "")
            if d not in DISASTER_CLASSES:
                bad.append(d); continue
            pre, post = Path(row["pre"]), Path(row["post"])
            pre = pre if pre.is_absolute() else root / pre
            post = post if post.is_absolute() else root / post
            pairs.append(LabeledPair(str(pre), str(post), d,
                                     (row.get("caption") or "").strip() or None,
                                     (row.get("event") or "").strip() or infer_event(str(pre))))
    if bad:
        log.warning("Skipped %d rows with labels outside %s: %s",
                    len(bad), DISASTER_CLASSES, sorted(set(bad))[:5])
    log.info("Manifest: %d usable pairs, %d events | class counts %s",
             len(pairs), len({p.event for p in pairs}),
             dict(Counter(p.disaster for p in pairs)))
    return pairs


# ---------------------------------------------------------------------------
# DisasterM3 -> LabeledPair. This is what feeds `cross_validate_leave_event_out`;
# no CSV is required, though `build_dm3_manifest` writes one for the record.
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# YOUR OWN FOLDER OF PAIRS -> List[LabeledPair]
# No manifest file needed. Point this at the dataset you downloaded; it finds the
# pre/post pairs, reads the class from the folder or the filename, groups tiles into
# events for the CV split, and TELLS YOU WHAT IT FOUND before anything runs.
# ---------------------------------------------------------------------------
IMG_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

# The pre/post naming schemes this understands. One folder can mix all of them.
#   <event>_<idx>_pre_disaster.png   xBD / DisasterM3 Bench   guatemala_volcano_00000011_pre_disaster
#   <event>_<idx>_pre.png            tile exports             turkey_earthquake0_195_pre
#   <event>_pre_<idx>.png            DisasterM3 release       bata_explosion_pre_0
#   <event>_before.png / _t1 / _A    other conventions
# The modality token can sit at the END of the stem or BETWEEN the event and the index,
# and a pairer that only looked at the end silently dropped the third form entirely --
# no error, just missing pairs.
PAIR_SUFFIXES: Tuple[Tuple[str, str], ...] = (
    ("_pre_disaster", "_post_disaster"),
    ("_pre", "_post"),
    ("_before", "_after"),
    ("_t1", "_t2"),
    ("_A", "_B"),
)
# Non-optical modalities. The pipeline is RGB-only: ChangeOS, AnyChange and RemoteCLIP
# are all trained on optical imagery, so a SAR post paired with an optical pre would
# produce confident nonsense. Dropped, and counted in the scan report.
NON_OPTICAL_TOKENS = ("sar_post", "sar_pre", "sar")
_MID_RE = re.compile(r"^(?P<event>.+?)_(?P<mod>sar_post|sar_pre|pre|post)_"
                     r"(?P<idx>\d+(?:_\d+)*)$", re.I)


def _pair_key(stem: str) -> Optional[Tuple[str, str]]:
    """(pairing key, modality) for one file stem, or None when it names no modality.

    The key is what a pre and its post share; the modality says which half this is."""
    low = stem.lower()
    for pre_s, post_s in PAIR_SUFFIXES:                 # token at the END of the stem
        for suf, mod in ((pre_s, "pre"), (post_s, "post")):
            if low.endswith(suf.lower()):
                return stem[: -len(suf)], mod
    m = _MID_RE.match(stem)                             # token in the MIDDLE
    if m:
        mod = m["mod"].lower()
        return f'{m["event"]}#{m["idx"]}', ("sar_post" if mod in NON_OPTICAL_TOKENS else mod)
    return None


def _label_from_path(p: Path, root: Path) -> Optional[str]:
    """Class from the folder tree first (a `.../earthquake/` directory is explicit),
    then from the file name. Returns None when nothing in the path names a class."""
    for part in list(p.relative_to(root).parts)[:-1][::-1]:
        lab = normalize_disaster(part)
        if lab:
            return lab
    return normalize_disaster(p.stem) or event_to_class(infer_event(str(p)))


def scan_pair_folder(root: str, class_from: str = "auto",
                     label_overrides: Optional[Dict[str, str]] = None,
                     captions: bool = True, recursive: bool = True,
                     per_class: Optional[int] = None, seed: int = 7,
                     verbose: bool = True) -> List[LabeledPair]:
    """Every pre/post pair under `root`, as LabeledPairs.

    class_from : "auto"   folder name, then file name            (default)
                 "folder" the parent directory only -- use this when the file names
                          carry no class and the folders do
                 "name"   the file name only
    label_overrides : {event or folder name: class} for anything the rules miss.
    captions : read `<pre stem>.txt` (or the stem without its pre suffix) as the
               reference caption, so BLEU / ROUGE / BERTScore have something to score.
    per_class : cap per class, spread ACROSS EVENTS rather than taking the first N --
                taking the first N would hand one event the whole class.
    """
    root_p = Path(_to_local(root))
    if not root_p.is_dir():
        raise FileNotFoundError(f"not a directory: {root_p}")
    overrides = {k.lower(): v for k, v in (label_overrides or {}).items()}
    files = [p for p in (root_p.rglob("*") if recursive else root_p.glob("*"))
             if p.suffix.lower() in IMG_EXT]

    # group every file by what a pre and its post share
    groups: Dict[Tuple[Any, str], Dict[str, Path]] = defaultdict(dict)
    no_modality = []
    for p in sorted(files):
        kv = _pair_key(p.stem)
        if kv is None:
            no_modality.append(str(p))
            continue
        key, mod = kv
        groups[(p.parent, key)].setdefault(mod, p)

    pairs, unlabelled, unmatched, non_optical, out_of_scope = [], [], [], [], []
    for (parent, key), mods in sorted(groups.items(), key=lambda kv: str(kv[0])):
        if "sar_post" in mods:
            non_optical.append(str(mods["sar_post"]))
        pre, post = mods.get("pre"), mods.get("post")
        if pre is None or post is None:
            unmatched.append(str(pre or post or ""))
            continue

        event = infer_event(str(pre))
        if class_from == "folder":
            lab = next((normalize_disaster(x) for x in pre.relative_to(root_p).parts[:-1][::-1]
                        if normalize_disaster(x)), None)
        elif class_from == "name":
            lab = normalize_disaster(pre.stem) or event_to_class(event)
        else:
            lab = _label_from_path(pre, root_p)
        lab = overrides.get(event) or overrides.get(parent.name.lower()) or lab
        if lab is None:
            # A name the project deliberately excludes (explosion, tornado, conflict)
            # is NOT the same failure as a name nothing recognised. Reporting them
            # together hides a broken parser behind an expected exclusion.
            t = dm3_type_of_event(pre.stem) or dm3_type_of_event(event)
            (out_of_scope if t and DM3_TO_CLASS.get(t) is None else unlabelled
             ).append(f"{pre.name} ({t})" if t else str(pre))
            continue

        ref = None
        if captions:
            base = pre.stem
            for pre_s, _ in PAIR_SUFFIXES:
                if base.lower().endswith(pre_s.lower()):
                    base = base[: -len(pre_s)]
                    break
            for cand in (pre.with_suffix(".txt"), parent / f"{base}.txt"):
                if cand.is_file():
                    ref = cand.read_text(errors="ignore").strip() or None
                    break
        pairs.append(LabeledPair(pre=str(pre), post=str(post), disaster=lab,
                                 reference_caption=ref, event=event))

    if per_class:
        rng = np.random.RandomState(seed)
        kept = []
        for cls in sorted({p.disaster for p in pairs}):
            ev = defaultdict(list)
            for p in pairs:
                if p.disaster == cls:
                    ev[p.event].append(p)
            order = sorted(ev)
            rng.shuffle(order)
            for lst in ev.values():
                rng.shuffle(lst)
            take, i = [], 0
            while len(take) < per_class and any(ev[e] for e in order):   # round-robin
                e = order[i % len(order)]
                if ev[e]:
                    take.append(ev[e].pop())
                i += 1
            kept.extend(take)
        pairs = kept

    pairs.sort(key=lambda p: (p.disaster, p.event, p.pre))
    if verbose:
        preview_pairs(pairs, unlabelled, unmatched, root_p,
                      non_optical=non_optical, no_modality=no_modality,
                      out_of_scope=out_of_scope)
    return pairs


def preview_pairs(pairs: Sequence[LabeledPair], unlabelled: Sequence[str] = (),
                  unmatched: Sequence[str] = (), root: Optional[Path] = None,
                  non_optical: Sequence[str] = (),
                  no_modality: Sequence[str] = (),
                  out_of_scope: Sequence[str] = ()) -> dict:
    """What was found, and what will go wrong if you run on it as it stands."""
    by_cls = defaultdict(list)
    for p in pairs:
        by_cls[p.disaster].append(p)
    rep = {"n_pairs": len(pairs), "n_events": len({p.event for p in pairs}),
           "n_with_caption": sum(1 for p in pairs if p.reference_caption),
           "classes": {}}
    print("\n" + "=" * 86)
    print(f"PAIR SCAN{f' — {root}' if root else ''}")
    print("=" * 86)
    print(f"{'class':<20}{'pairs':>7}{'events':>8}{'captions':>10}  events")
    for cls in sorted(by_cls):
        ps = by_cls[cls]
        evs = sorted({p.event for p in ps})
        rep["classes"][cls] = {"pairs": len(ps), "events": len(evs),
                               "captions": sum(1 for p in ps if p.reference_caption),
                               "event_names": evs}
        print(f"{cls:<20}{len(ps):>7}{len(evs):>8}"
              f"{sum(1 for p in ps if p.reference_caption):>10}  "
              f"{', '.join(evs[:4])}{' ...' if len(evs) > 4 else ''}")
    print("-" * 86)
    print(f"{'TOTAL':<20}{len(pairs):>7}{rep['n_events']:>8}{rep['n_with_caption']:>10}")

    warn = []
    if out_of_scope:
        print(f"\n  {len(out_of_scope)} pair(s) excluded as out of scope by DM3_TO_CLASS "
              f"(expected, not an error): {out_of_scope[0]}"
              + (f" and {len(out_of_scope) - 1} more" if len(out_of_scope) > 1 else ""))
    if unlabelled:
        warn.append(f"{len(unlabelled)} pair(s) had no recognisable class and were DROPPED, "
                    f"e.g. {Path(unlabelled[0]).name}. This is a PARSER miss, not an "
                    f"exclusion -- pass label_overrides={{...}} or extend _TYPE_TOKENS.")
    if unmatched:
        warn.append(f"{len(unmatched)} file(s) had no partner of the other date, e.g. "
                    f"{Path(unmatched[0]).name}. Check PAIR_SUFFIXES / _pair_key.")
    if non_optical:
        warn.append(f"{len(non_optical)} SAR file(s) ignored (e.g. "
                    f"{Path(non_optical[0]).name}). ChangeOS, AnyChange and RemoteCLIP are "
                    f"optical-only, so a SAR post paired with an optical pre would produce "
                    f"confident nonsense.")
    if no_modality:
        warn.append(f"{len(no_modality)} image(s) name no pre/post modality and were "
                    f"ignored, e.g. {Path(no_modality[0]).name}.")
    if rep["classes"] and all(v["events"] < 2 for v in rep["classes"].values()):
        warn.append("EVERY class has a single event. cross_validate_leave_event_out() "
                    "cannot run meaningfully: each test fold holds the only event of its "
                    "class, so the head never sees that class in training. Use this sample "
                    "for the leak audit, calibration and the ablation; get a second event "
                    "per class before reporting any accuracy or macro-F1.")
    thin = [c for c, v in rep["classes"].items() if v["events"] < 2]
    if root is not None:
        # A folder named after ONE disaster, holding tiles of several, silently labels
        # them all with the folder. Your earlier path was .../earthquake/<all tiles>,
        # so this is the exact trap: compare the two readings and say so.
        clash = [p for p in pairs
                 if normalize_disaster(Path(p.pre).parent.name)
                 and normalize_disaster(Path(p.pre).parent.name) != p.disaster]
        if clash:
            warn.append(f"{len(clash)} pair(s) sit in a folder whose name is a DIFFERENT "
                        f"class from the file name, e.g. "
                        f"{Path(clash[0].pre).parent.name}/ vs {clash[0].disaster}. "
                        f"class_from='name' is what you want when one folder holds "
                        f"several disasters.")
    if thin:
        warn.append(f"{thin} have a single event each. Under leave-event-out their test "
                    f"fold has no training pair of that class, so their recall is a FLOOR, "
                    f"not a measurement -- say so when reporting, or drop them from "
                    f"CFG.eval_classes.")
    if rep["n_events"] and rep["n_events"] == len(pairs):
        warn.append("every pair is its own event -- the CV split degenerates to "
                    "leave-one-out. Check that infer_event() parses your file names.")
    if rep["n_with_caption"] == 0:
        warn.append("no reference captions found, so BLEU / ROUGE / BERTScore cannot be "
                    "computed. The caption-condition experiment still reports stated "
                    "label, pipeline agreement and echo rate, which is the part that "
                    "matters for the leakage claim.")
    small = [c for c, v in rep["classes"].items() if v["pairs"] < 5]
    if small:
        warn.append(f"{small} have under 5 pairs; their per-class F1 will be very noisy.")
    for w in warn:
        print("  ! " + w)
    print("=" * 86)
    rep["warnings"] = warn
    return rep


def dm3_labeled_pairs(per_class: Optional[int] = None, classes: Optional[Sequence[str]] = None,
                      download: bool = True, seed: Optional[int] = None) -> List[LabeledPair]:
    """Up to `per_class` Bench pairs per class, dealt round-robin across each class's
    EVENTS so leave-event-out has more than one event to hold out, plus the §13b focus
    pair. Reference captions come from `benchmark_release.json` where DisasterM3 has one.

    Images are fetched on demand: roughly 4 MB per pair, so 30 per class over 5 usable
    classes is about 600 MB. `download=False` returns the selection without fetching, so
    you can see the size and the event spread first. Defaults to `TARGET_CLASSES` (§13b),
    so DisasterM3's man-made types never enter the evaluation."""
    per_class = CFG.dm3_eval_per_class if per_class is None else per_class
    classes = TARGET_CLASSES if classes is None else classes
    picked = sample_across_events(PAIRS_INDEX, per_class, classes,
                                  CFG.dm3_seed if seed is None else seed,
                                  include=[DM3_FOCUS] if "DM3_FOCUS" in globals() else [])
    n_ev = len({p.event for p in picked})
    print(f"selected {len(picked)} pairs over {n_ev} events "
          f"(~{4.0 * len(picked):.0f} MB to fetch)" if download else
          f"selected {len(picked)} pairs over {n_ev} events (nothing fetched)")
    out = []
    for i, p in enumerate(picked, 1):
        if download and not (p.pre_local and p.post_local):
            try:
                materialize(DM3, p, masks=False)
            except Exception as e:
                log.warning("skipped %s (%s: %s)", p.key, type(e).__name__, e)
                continue
            if i % 20 == 0 or i == len(picked):
                log.info("fetched %d/%d pairs", i, len(picked))
        out.append(LabeledPair(p.pre_local or p.pre, p.post_local or p.post, p.disaster,
                               p.reference_caption(), p.event))
    n_ref = sum(1 for p in out if p.reference_caption)
    log.info("DisasterM3: %d pairs | classes %s | %d events | %d with a reference caption",
             len(out), dict(Counter(p.disaster for p in out)),
             len({p.event for p in out}), n_ref)
    _warn_single_event_classes(out)
    return out


def _warn_single_event_classes(pairs: Sequence[LabeledPair]):
    """Leave-event-out cannot learn a class that has only ONE event: when that event is
    the test fold, no pair of that class is in the training set. Say so up front rather
    than letting it look like a modelling failure."""
    ev = defaultdict(set)
    for p in pairs:
        ev[p.disaster].add(p.event)
    thin = sorted(c for c, e in ev.items() if len(e) < 2)
    if thin:
        log.warning("classes with a single event in this sample: %s — under leave-event-out "
                    "their test fold has no training pair of that class, so their recall is a "
                    "floor, not a measurement. Report it as such.", thin)


def build_dm3_manifest(pairs: Sequence[LabeledPair], path: Optional[str] = None) -> str:
    """Write the selection as the CSV `load_manifest` reads, so a run is reproducible
    without re-listing the repo."""
    import csv as _csv
    path = path or f"{CFG.out_dir}/dm3_manifest.csv"
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = _csv.DictWriter(f, fieldnames=["pre", "post", "disaster", "caption", "event"])
        w.writeheader()
        for p in pairs:
            w.writerow({"pre": p.pre, "post": p.post, "disaster": p.disaster,
                        "caption": p.reference_caption or "", "event": p.event or ""})
    log.info("wrote %s (%d pairs)", path, len(pairs))
    return path


def extract_dataset_features(pipeline: DisasterCaptioningPipeline, pairs: List[LabeledPair],
                             cache: Optional[str] = None, force: bool = False
                             ) -> Tuple[List[ClassifierFeatures], List[str]]:
    """Runs the frozen pipeline once per pair (no Qwen) and caches z/t to disk, so
    training and re-training cost one forward pass per pair in total."""
    if cache and Path(cache).is_file() and not force:
        d = np.load(cache, allow_pickle=True)
        if d["tab_dim"] == TAB_DIM and len(d["labels"]) == len(pairs):
            feats = [ClassifierFeatures(z, t, g, s, o) for z, t, g, s, o in
                     zip(d["z"], d["t"], d["g"], d["s"], d["o"])]
            log.info("Loaded cached features for %d pairs from %s", len(feats), cache)
            return feats, list(d["labels"])
        log.warning("Cache at %s does not match the current pairs/layout; recomputing.", cache)

    feats, labels = [], []
    for i, p in enumerate(pairs, 1):
        try:
            feats.append(pipeline.extract_features(p.pre, p.post))
            labels.append(p.disaster)
        except Exception as e:
            log.warning("[%d/%d] skipped %s (%s: %s)", i, len(pairs), p.pre,
                        type(e).__name__, e)
            continue
        if i % 10 == 0 or i == len(pairs):
            log.info("features %d/%d", i, len(pairs))
    if cache and feats:
        np.savez_compressed(
            cache, z=np.stack([f.z for f in feats]), t=np.stack([f.t for f in feats]),
            g=np.stack([f.global_post for f in feats]),
            s=np.stack([f.scene_repr for f in feats]),
            o=np.stack([f.object_repr for f in feats]),
            labels=np.array(labels), tab_dim=TAB_DIM)
        log.info("Cached %d feature vectors -> %s", len(feats), cache)
    return feats, labels


def train_head_on_pairs(pipeline: DisasterCaptioningPipeline, pairs: List[LabeledPair],
                        cache: Optional[str] = None, **fit_kwargs) -> dict:
    """Train on an in-memory pair list (what `dm3_labeled_pairs` returns).

    For a REPORTABLE number use `cross_validate_leave_event_out` instead: it retrains per
    fold on held-out events. This fits one head on everything you give it, which is only
    useful for a head you then run on pairs from events it never saw."""
    feats, labels = extract_dataset_features(pipeline, pairs,
                                             cache or f"{CFG.out_dir}/features_train.npz")
    hist = pipeline.classifier.fit(feats, labels, **fit_kwargs)
    print(f"best val accuracy {hist['best_val_acc']:.3f} "
          f"({hist['n_train']} train / {hist['n_val']} val), T={hist['temperature']:.2f}")
    return hist


def train_head(pipeline: DisasterCaptioningPipeline, manifest_csv: str,
               cache: Optional[str] = None, **fit_kwargs) -> dict:
    """End-to-end: manifest -> frozen features -> trained, temperature-calibrated head.

    Only `DisasterClassifierHead` receives gradients. ChangeOS, AnyChange, SAM2,
    RemoteCLIP and Qwen are all frozen and are used purely as feature extractors.
    """
    pairs = load_manifest(manifest_csv)
    feats, labels = extract_dataset_features(pipeline, pairs,
                                             cache or f"{CFG.out_dir}/features_train.npz")
    hist = pipeline.classifier.fit(feats, labels, **fit_kwargs)
    print(f"best val accuracy {hist['best_val_acc']:.3f} "
          f"({hist['n_train']} train / {hist['n_val']} val), T={hist['temperature']:.2f}")
    return hist


# ---------------------------------------------------------------------------
# The pipeline is fully functional untrained: `is_trained=False` just means the pool
# is DKG + the CLIP prior. To fit one head on a DisasterM3 sample (NOT the reportable
# number -- see §19):
# ---------------------------------------------------------------------------
# dm3_pairs = dm3_labeled_pairs(per_class=30)
# history = train_head_on_pairs(pipeline, dm3_pairs)
# plt.plot([h["train_acc"] for h in history["curve"]], label="train")
# plt.plot([h["val_acc"] for h in history["curve"]], label="val")
# plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.grid(alpha=.3); plt.show()
print("Classifier head:", "TRAINED" if classifier.is_trained else
      "zero-shot (text-initialised) — supply a manifest to train_head() to fine-tune")


## 19 · Evaluation and ablation (Part 8)

* `profile()` — parameter counts, per-stage timings, peak VRAM.
* `contribution_analysis()` — a real per-image probe: it re-runs the DKG with the scene sources
  dropped, re-runs the classifier on masked features, re-pools, and reports how far the posterior
  moved (JSD) and whether the label flipped. Then the same for the object sources. This measures a
  posterior shift, not an attention weight.
* `run_ablation()` / `ablation_table()` — ten configurations covering every switch plus
  `object_only` (the Stage 0 baseline), tabulated side by side.
All of these take the pair list `dm3_labeled_pairs()` returns (§13b), or any `load_manifest` CSV.

* `cross_validate_leave_event_out()` — **the headline table.** Event-grouped folds (no event in both train
  and test), the head re-initialised and retrained per fold without touching the checkpoint, then
  out-of-fold accuracy / macro-F1 / top-k for the prior, the DKG, the untrained and trained head, and both
  pools, with event-level bootstrap CIs and the measured prior-vs-head dependence. Macro-F1 averages only
  the classes present in the ground truth; predictions outside them count as errors and are tallied.
* `run_caption_conditions()` — **the leakage experiment.** C1 label-primed vs C2 measurements-only vs C3
  blind: each caption's *stated* class, its accuracy, its agreement with the pipeline, its echo rate where
  the pipeline is wrong, and caption metrics per condition when references exist.
* `evaluate_dataset()` — a quick in-sample pass (accuracy, macro-F1, top-k, confusion matrix, contribution
  summary, caption metrics). Not for the paper when the head is trained.
* `compare_pipelines()` — Stage 0 vs Stage 2 on one pair, captions included.

The cross-validation, the caption experiment and the A/B are left commented out because they fetch more
images and spend extra Qwen generations; the profile, contribution probe and ablation sweep run automatically on the current
pair.


In [ ]:
# ---------------------------------------------------------------------------
# Caption / localisation metrics (unchanged from Stage 1)
# ---------------------------------------------------------------------------
def evaluate_captions(hypotheses: List[str], references: List[str]) -> dict:
    if not hypotheses or len(hypotheses) != len(references):
        raise ValueError("hypotheses and references must be non-empty and equal length")
    import sacrebleu
    from rouge_score import rouge_scorer
    import bert_score
    out = {"BLEU-4": round(sacrebleu.corpus_bleu(hypotheses, [references]).score, 2)}
    rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    out["ROUGE-L"] = round(100 * float(np.mean(
        [rs.score(r, h)["rougeL"].fmeasure for h, r in zip(hypotheses, references)])), 2)
    _, _, f1 = bert_score.score(hypotheses, references, lang="en", verbose=False)
    out["BERTScore-F1"] = round(100 * float(f1.mean()), 2)
    return out


def evaluate_localization(pred_map, gt_mask, threshold=0.5) -> dict:
    p, g = (pred_map > threshold).astype(bool), gt_mask.astype(bool)
    tp = np.logical_and(p, g).sum(); fp = np.logical_and(p, ~g).sum()
    fn = np.logical_and(~p, g).sum()
    prec, rec = tp / (tp + fp + 1e-9), tp / (tp + fn + 1e-9)
    return {"IoU": round(float(tp / (tp + fp + fn + 1e-9)), 4),
            "F1": round(float(2 * prec * rec / (prec + rec + 1e-9)), 4),
            "precision": round(float(prec), 4), "recall": round(float(rec), 4)}


def sda_ablation(result: PredictionResult, gt_mask, threshold=0.5) -> dict:
    before = evaluate_localization(result.fused.fused, gt_mask, threshold)
    after = evaluate_localization(result.refined_map, gt_mask, threshold)
    return {"without_SDA": before, "with_SDA": after,
            "delta_recall": round(after["recall"] - before["recall"], 4),
            "delta_IoU": round(after["IoU"] - before["IoU"], 4)}


def profile(result, models: dict) -> dict:
    params = {}
    for name, m in models.items():
        inner = getattr(m, "model", getattr(m, "head", m))
        if isinstance(inner, nn.Module):
            params[name] = round(sum(p.numel() for p in inner.parameters()) / 1e6, 3)
    trainable = round(sum(p.numel() for p in classifier.head.parameters()
                          if p.requires_grad) / 1e6, 3)
    return {"timings_s": {k: round(v, 3) for k, v in result.timings.items()},
            "params_M": params, "trainable_params_M": trainable,
            "peak_vram_GB": (round(torch.cuda.max_memory_allocated() / 1e9, 3)
                             if torch.cuda.is_available() else 0.0)}


# ---------------------------------------------------------------------------
# Per-image evidence contribution (Part 8): scene vs object
# ---------------------------------------------------------------------------
def _jsd(p: np.ndarray, q: np.ndarray) -> float:
    p = np.clip(p, 1e-9, 1); q = np.clip(q, 1e-9, 1)
    m = 0.5 * (p + q)
    kl = lambda a, b: float((a * np.log(a / b)).sum())
    return round(0.5 * kl(p, m) + 0.5 * kl(q, m), 4)


def contribution_analysis(result: PredictionResult, pipeline: DisasterCaptioningPipeline,
                          cfg: Config) -> dict:
    """How much does each branch actually move the answer on THIS image?

    Both reasoners are re-run with one branch removed: the graph is re-inferred from its
    concept mass with that branch's evidence sources dropped, and the classifier is
    re-run on features with that branch's block zeroed and its embedding removed from
    `z`. The two are then re-pooled, so the number is a real posterior shift, not an
    attention weight reinterpreted as an explanation.
    """
    base = result.posterior.probs
    d = DISASTER_CLASSES.index(result.posterior.label)
    by_source = result.dkg.get("concept_by_source", {})
    w = dict(result.posterior.weights)
    out = {"baseline": {"label": result.posterior.label, "prob": round(float(base[d]), 4)}}

    for branch, drop_sources, block, emb in (
            ("scene", {"scene", "transition", "physical"}, "scene", "scene"),
            ("object", {"object"}, "object", "object")):
        sources = {}
        if "dkg" in w and by_source:
            concepts = {c: {s: m for s, m in bs.items() if s not in drop_sources}
                        for c, bs in by_source.items()}
            concepts = {c: v for c, v in concepts.items() if any(v.values())}
            sources["dkg"] = pipeline.dkg.infer(nx.DiGraph(), concepts)["prob_vector"]
        if "classifier" in w and result.classifier is not None:
            f = result.features.masked(block, "concepts", drop_embedding=emb)
            sources["classifier"] = pipeline.classifier.predict(f)["probs"]
        if "zeroshot" in w:
            sources["zeroshot"] = result.zeroshot["probs"]
        if not sources:
            continue
        post = log_linear_pool(sources, w)
        out[f"without_{branch}"] = {
            "label": post.label, "prob_of_baseline_label": round(float(post.probs[d]), 4),
            "delta_prob": round(float(base[d] - post.probs[d]), 4),
            "label_flips": post.label != result.posterior.label,
            "jsd_vs_full": _jsd(base, post.probs)}
    out["scene_contribution"] = out.get("without_scene", {}).get("delta_prob", 0.0)
    out["object_contribution"] = out.get("without_object", {}).get("delta_prob", 0.0)
    return out


# ---------------------------------------------------------------------------
# Ablation switches (Part 8)
# ---------------------------------------------------------------------------
ABLATION_CONFIGS: Dict[str, Dict[str, Any]] = {
    "full (Stage 2)":     {},
    "-AnyChange":         {"use_anychange": False},
    "-ChangeOS":          {"use_changeos": False},
    "-semantic transitions": {"use_transitions": False},
    "-SDA":               {"use_sda": False},
    "-cross-branch attention": {"use_attention": False},
    "-DKG":               {"use_dkg": False},
    "-learned classifier": {"use_classifier": False},
    "-CLIP prior":        {"use_zeroshot": False},
    "-RVG":               {"use_rvg": False},
    "object_only (Stage 0)": {"mode": "object_only"},
}


def run_ablation(pipeline: DisasterCaptioningPipeline, pre_path: str, post_path: str,
                 configs: Optional[Dict[str, Dict[str, Any]]] = None,
                 generate_caption: bool = False) -> List[dict]:
    """Every row is the SAME loaded weights on the SAME pair with one module switched
    off, so a difference is attributable to that module and nothing else. Captions are
    off by default: two Qwen passes per row dominates the runtime otherwise."""
    configs = configs or ABLATION_CONFIGS
    cfg = pipeline.cfg
    saved = {k: getattr(cfg, k) for k in
             ("mode", "use_changeos", "use_anychange", "use_transitions", "use_sda",
              "use_dkg", "use_classifier", "use_rvg", "use_attention", "use_blind_pass",
              "use_zeroshot", "label_in_prompt")}
    rows = []
    try:
        for name, over in configs.items():
            for k, v in saved.items():
                setattr(cfg, k, v)
            for k, v in over.items():
                setattr(cfg, k, v)
            if not generate_caption:
                cfg.use_blind_pass = False
            log.info("=" * 26 + f"  ABLATION: {name}  " + "=" * 26)
            t0 = time.perf_counter()
            r = pipeline.predict(pre_path, post_path, generate_caption=generate_caption)
            rows.append({
                "config": name,
                "posterior": r.disaster,
                "p": round(r.posterior.prob, 3),
                "margin": round(r.posterior.margin, 3),
                "entropy": round(r.posterior.entropy, 3),
                "confidence": round(r.rvg["confidence"], 3),
                "hedge": r.rvg["hedge"],
                "dkg": r.dkg.get("disaster"),
                "clf": r.classifier["label"] if r.classifier else "-",
                "zs": r.zeroshot["label"],
                "transitions": len(r.scene_evidence.transitions),
                "instances": len(r.instances),
                "scene_cov": round(r.fused.scalars["scene_coverage"], 3),
                "concepts": len(r.dkg.get("concept_mass", {})),
                "tokens": sum(r.tokens.counts().values()),
                "sda_gain%": r.sda_report["area_gain_pct"],
                "sec": round(time.perf_counter() - t0, 1),
                "_result": r,
            })
    finally:
        for k, v in saved.items():
            setattr(cfg, k, v)

    cols = [c for c in rows[0] if not c.startswith("_")]
    widths = {c: max(len(c), max(len(str(r[c])) for r in rows)) + 2 for c in cols}
    line = "".join(f"{c:<{widths[c]}}" for c in cols)
    print("\n" + "=" * len(line)); print(line); print("=" * len(line))
    for r in rows:
        print("".join(f"{str(r[c]):<{widths[c]}}" for c in cols))
    print("=" * len(line))
    return rows


def ablation_table(rows: List[dict]):
    """The same rows as a pandas DataFrame, for pasting into the report."""
    import pandas as pd
    return pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows])



# ---------------------------------------------------------------------------
# ISSUE 6: a one-image ablation cannot answer "does this module contribute?".
# The rev5 earthquake sweep is the proof: -ChangeOS cost 0.157 of posterior mass
# and flipped the DKG to landslide, while on the flood and volcano pairs the SAME
# row was byte-identical to the full pipeline. A module that is decisive on one
# class and inert on another has an effect that only a MULTI-PAIR sweep can state.
# ---------------------------------------------------------------------------
def run_ablation_multi(pipeline: DisasterCaptioningPipeline, pairs: List[LabeledPair],
                       configs: Optional[Dict[str, Dict[str, Any]]] = None,
                       n_boot: int = 1000, seed: int = 0,
                       save_prefix: Optional[str] = None) -> dict:
    """Every switch, over a pair set, reported as an effect with an interval.

    Per configuration: accuracy against ground truth, the mean change in the posterior
    probability of the FULL pipeline's label, how often the label flips, and an
    event-level bootstrap CI on accuracy. A module whose CI for the accuracy difference
    spans zero has not been shown to contribute, and should not be claimed as a
    contribution in the paper."""
    configs = configs or ABLATION_CONFIGS
    cfg = pipeline.cfg
    keys = ("mode", "use_changeos", "use_anychange", "use_transitions", "use_sda",
            "use_dkg", "use_classifier", "use_rvg", "use_attention", "use_blind_pass",
            "use_zeroshot", "label_in_prompt")
    saved = {k: getattr(cfg, k) for k in keys}
    truth = [p.disaster for p in pairs]
    events = [p.event or infer_event(p.pre) for p in pairs]
    preds: Dict[str, List[Optional[str]]] = {}
    probs: Dict[str, List[float]] = {}
    try:
        for name, over in configs.items():
            for k, v in saved.items():
                setattr(cfg, k, v)
            for k, v in over.items():
                setattr(cfg, k, v)
            cfg.use_blind_pass = False                    # no captions in a sweep
            lab, pr = [], []
            for i, p in enumerate(pairs, 1):
                try:
                    r = pipeline.predict(p.pre, p.post, generate_caption=False)
                    lab.append(r.posterior.label)
                    pr.append({c: float(r.posterior.probs[j])
                               for j, c in enumerate(DISASTER_CLASSES)})
                except Exception as e:
                    log.warning("[%s %d/%d] skipped (%s: %s)", name, i, len(pairs),
                                type(e).__name__, e)
                    lab.append(None); pr.append({})
            preds[name], probs[name] = lab, pr
            log.info("ablation row done: %s", name)
    finally:
        for k, v in saved.items():
            setattr(cfg, k, v)

    base_name = next(iter(configs))
    base_lab = preds[base_name]
    ok = [i for i in range(len(pairs)) if all(preds[n][i] is not None for n in configs)]
    if not ok:
        raise RuntimeError("no pair completed every configuration")

    rng = np.random.RandomState(seed)
    ev = sorted({events[i] for i in ok})
    ev_idx = {e: [i for i in ok if events[i] == e] for e in ev}
    draws = [[i for e in rng.choice(ev, size=len(ev), replace=True) for i in ev_idx[e]]
             for _ in range(n_boot)]

    rows = []
    base_acc_boot = [np.mean([preds[base_name][i] == truth[i] for i in d]) for d in draws]
    for name in configs:
        lab = preds[name]
        acc = float(np.mean([lab[i] == truth[i] for i in ok]))
        d_acc = [np.mean([lab[i] == truth[i] for i in d]) - b
                 for d, b in zip(draws, base_acc_boot)]
        rows.append({
            "config": name,
            "accuracy": round(acc, 4),
            "d_accuracy_vs_full": round(acc - float(np.mean(
                [base_lab[i] == truth[i] for i in ok])), 4),
            "d_accuracy_95ci": [round(float(np.percentile(d_acc, 2.5)), 4),
                                round(float(np.percentile(d_acc, 97.5)), 4)],
            "mean_d_prob_of_full_label": round(float(np.mean(
                [probs[name][i].get(base_lab[i], 0.0)
                 - probs[base_name][i].get(base_lab[i], 0.0) for i in ok])), 4),
            "label_flip_rate_vs_full": round(float(np.mean(
                [lab[i] != base_lab[i] for i in ok])), 4),
        })
    out = {"n_pairs": len(ok), "n_events": len(ev), "baseline": base_name, "rows": rows}

    print("\n" + "=" * 112)
    print(f"ABLATION over {len(ok)} pairs / {len(ev)} events — effect of removing each module")
    print("=" * 112)
    print(f"{'config':<26}{'acc':>7}{'d acc':>8}{'95% CI on d acc':>22}"
          f"{'d p(full label)':>17}{'flip rate':>11}")
    for r in rows:
        print(f"{r['config']:<26}{r['accuracy']:>7.3f}{r['d_accuracy_vs_full']:>+8.3f}"
              f"{str(r['d_accuracy_95ci']):>22}{r['mean_d_prob_of_full_label']:>+17.3f}"
              f"{r['label_flip_rate_vs_full']:>11.3f}")
    print("\nA row whose 95% CI on d-accuracy spans 0 has NOT been shown to contribute.")
    print("=" * 112)
    if save_prefix:
        Path(f"{save_prefix}_ablation_multi.json").write_text(json.dumps(out, indent=2,
                                                                        default=str))
    return out

# ---------------------------------------------------------------------------
# Dataset evaluation (Part 8)
# ---------------------------------------------------------------------------
def confusion_matrix(y_true: List[str], y_pred: List[str],
                     classes: Sequence[str] = DISASTER_CLASSES) -> np.ndarray:
    idx = {c: i for i, c in enumerate(classes)}
    M = np.zeros((len(classes), len(classes)), int)
    for t, p in zip(y_true, y_pred):
        if t in idx and p in idx:
            M[idx[t], idx[p]] += 1
    return M


def plot_confusion(M: np.ndarray, classes: Sequence[str] = DISASTER_CLASSES,
                   title: str = "Confusion matrix", save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(8.5, 7))
    norm = M / np.clip(M.sum(1, keepdims=True), 1, None)
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{M[i, j]}", ha="center", va="center", fontsize=9,
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, label="row-normalised")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()


def macro_f1(M: np.ndarray) -> float:
    f1 = []
    for i in range(M.shape[0]):
        tp = M[i, i]; fp = M[:, i].sum() - tp; fn = M[i, :].sum() - tp
        if tp + fp + fn == 0:
            continue
        p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
        f1.append(0.0 if p + r == 0 else 2 * p * r / (p + r))
    return round(float(np.mean(f1)) if f1 else 0.0, 4)


def evaluate_dataset(pipeline: DisasterCaptioningPipeline, pairs: List[LabeledPair],
                     generate_caption: bool = False, k: int = 3,
                     contributions: bool = True, save_prefix: Optional[str] = None) -> dict:
    """Classification accuracy, top-k accuracy, per-source accuracy, confusion matrix,
    the scene/object contribution summary and (with references) caption metrics.

    Scores the pipeline AS IT IS NOW. If the head is trained, those weights came from some
    split you chose; unless these pairs are held-out EVENTS the number is optimistic. For
    the reportable table use `cross_validate_leave_event_out`."""
    if pipeline.classifier.is_trained:
        log.warning("evaluate_dataset: the head is TRAINED; if it saw these events the "
                    "result is in-sample. Use cross_validate_leave_event_out for the paper.")
    y_true, per_source, topk_hits, rows = [], defaultdict(list), 0, []
    hyps, refs, contrib = [], [], []
    for i, p in enumerate(pairs, 1):
        try:
            r = pipeline.predict(p.pre, p.post, generate_caption=generate_caption)
        except Exception as e:
            log.warning("[%d/%d] skipped %s (%s: %s)", i, len(pairs), p.pre,
                        type(e).__name__, e)
            continue
        y_true.append(p.disaster)
        per_source["posterior"].append(r.posterior.label)
        per_source["dkg"].append(r.dkg.get("disaster", "undetermined"))
        per_source["classifier"].append(r.classifier["label"] if r.classifier else "undetermined")
        per_source["zeroshot"].append(r.zeroshot["label"])
        topk_hits += int(p.disaster in [t["disaster"] for t in r.posterior.top_k(k)])
        if contributions:
            contrib.append(contribution_analysis(r, pipeline, pipeline.cfg))
        if generate_caption and p.reference_caption:
            hyps.append(r.caption); refs.append(p.reference_caption)
        rows.append({"pre": p.pre, "true": p.disaster, "pred": r.posterior.label,
                     "p": round(r.posterior.prob, 3),
                     "confidence": round(r.rvg["confidence"], 3), "hedge": r.rvg["hedge"]})
        if i % 5 == 0 or i == len(pairs):
            log.info("evaluated %d/%d", i, len(pairs))

    n = len(y_true)
    if n == 0:
        raise RuntimeError("no pair evaluated successfully")
    acc = {s: round(float(np.mean([a == b for a, b in zip(y_true, preds)])), 4)
           for s, preds in per_source.items()}
    M = confusion_matrix(y_true, per_source["posterior"])
    cm = classification_metrics(y_true, per_source["posterior"])
    out = {"n": n, "accuracy": acc["posterior"], f"top{k}_accuracy": round(topk_hits / n, 4),
           "accuracy_by_source": acc, "macro_F1": cm["macro_F1"],
           "per_class": cm["per_class"],
           "out_of_scope_predictions": cm["out_of_scope_predictions"],
           "confusion_matrix": M.tolist(), "classes": list(DISASTER_CLASSES),
           "per_pair": rows}
    if contrib:
        out["mean_scene_contribution"] = round(
            float(np.mean([c["scene_contribution"] for c in contrib])), 4)
        out["mean_object_contribution"] = round(
            float(np.mean([c["object_contribution"] for c in contrib])), 4)
        out["scene_flip_rate"] = round(float(np.mean(
            [c.get("without_scene", {}).get("label_flips", False) for c in contrib])), 4)
        out["object_flip_rate"] = round(float(np.mean(
            [c.get("without_object", {}).get("label_flips", False) for c in contrib])), 4)
    if hyps:
        out["caption_metrics"] = evaluate_captions(hyps, refs)

    print("\n" + "=" * 78)
    print(f"DATASET EVALUATION — {n} pairs")
    print("=" * 78)
    print(f"{'source':<16}{'accuracy':>10}")
    for s, v in acc.items():
        print(f"{s:<16}{v:>10.3f}")
    print(f"\ntop-{k} accuracy (fused posterior): {out[f'top{k}_accuracy']:.3f}")
    print(f"macro F1                        : {out['macro_F1']:.3f}")
    if contrib:
        print(f"mean scene contribution         : {out['mean_scene_contribution']:+.3f} "
              f"(label flips without scene: {out['scene_flip_rate']:.2f})")
        print(f"mean object contribution        : {out['mean_object_contribution']:+.3f} "
              f"(label flips without object: {out['object_flip_rate']:.2f})")
    if "caption_metrics" in out:
        print("caption metrics                 :", out["caption_metrics"])
    print("=" * 78)
    plot_confusion(M, title=f"Fused posterior — {n} pairs, acc {acc['posterior']:.2f}",
                   save_path=f"{save_prefix}_confusion.png" if save_prefix else None)
    if save_prefix:
        Path(f"{save_prefix}_eval.json").write_text(json.dumps(out, indent=2, default=str))
    return out


# ---------------------------------------------------------------------------
# TIER-1 EVALUATION HARNESS: leave-event-out cross-validation + the caption-leakage
# experiment. These are the functions that produce the paper's headline tables.
# ---------------------------------------------------------------------------
import copy as _copy


def classification_metrics(y_true: List[str], y_pred: List[str],
                           classes: Optional[Sequence[str]] = None) -> dict:
    """Accuracy and macro-F1 where macro-F1 averages ONLY the classes present in the
    ground truth. The head predicts seven classes but a four-class benchmark has no
    support for the other three; predicting one of them is an error (it lowers recall)
    and is counted in `out_of_scope_predictions`, but an empty class is not averaged in
    as a spurious 0."""
    # ISSUE 9: the scored taxonomy is explicit. `CFG.eval_classes` pins it; otherwise it
    # is every class in DISASTER_CLASSES. `present` is the subset the TEST SET actually
    # supports -- macro-F1 averages only those, and a prediction outside them is an
    # error that is counted separately rather than quietly averaged in as a zero.
    if classes is None:
        classes = getattr(CFG, "eval_classes", None) or DISASTER_CLASSES
    present = [c for c in classes if c in set(y_true)]
    per = {}
    for c in present:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        per[c] = {"precision": round(pr, 4), "recall": round(rc, 4),
                  "f1": round(0.0 if pr + rc == 0 else 2 * pr * rc / (pr + rc), 4),
                  "support": tp + fn}
    n = max(len(y_true), 1)
    return {"n": len(y_true),
            "scored_classes": list(classes),
            "classes_with_support": present,
            "classes_without_support": [c for c in classes if c not in present],
            "accuracy": round(sum(t == p for t, p in zip(y_true, y_pred)) / n, 4),
            "macro_F1": round(float(np.mean([v["f1"] for v in per.values()])) if per else 0.0, 4),
            "per_class": per,
            "out_of_scope_predictions": int(sum(p not in present for p in y_pred)),
            "confusion_matrix": confusion_matrix(y_true, y_pred, classes).tolist()}


def cluster_bootstrap_ci(y_true: List[str], y_pred: List[str], events: List[str],
                         n_boot: int = 1000, seed: int = 0) -> dict:
    """95% CI that resamples whole EVENTS, not pairs. Pairs from one event are strongly
    correlated, so a per-pair bootstrap would report an interval that is too narrow."""
    rng = np.random.RandomState(seed)
    ev = sorted(set(events))
    idx = {e: [i for i, x in enumerate(events) if x == e] for e in ev}
    accs, f1s = [], []
    for _ in range(n_boot):
        pick = [i for e in rng.choice(ev, size=len(ev), replace=True) for i in idx[e]]
        yt, yp = [y_true[i] for i in pick], [y_pred[i] for i in pick]
        m = classification_metrics(yt, yp)
        accs.append(m["accuracy"]); f1s.append(m["macro_F1"])
    q = lambda a: [round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4)]
    return {"accuracy_95ci": q(accs), "macro_F1_95ci": q(f1s), "n_events": len(ev),
            "n_boot": n_boot}


def leave_event_out_folds(pairs: List[LabeledPair], n_folds: int = 5,
                          seed: int = 42) -> List[List[int]]:
    """Event-grouped folds: every pair of an event lands in the same fold, so the test
    fold contains only events the head never saw. Events are dealt out per class to the
    fold that currently holds the fewest events of that class."""
    ev2idx: Dict[str, List[int]] = defaultdict(list)
    for i, p in enumerate(pairs):
        ev2idx[p.event or infer_event(p.pre)].append(i)
    ev_cls = {e: Counter(pairs[i].disaster for i in ix).most_common(1)[0][0]
              for e, ix in ev2idx.items()}
    if len(ev2idx) < n_folds:
        log.warning("only %d events for %d folds; using %d folds", len(ev2idx), n_folds,
                    len(ev2idx))
        n_folds = len(ev2idx)
    rng = np.random.RandomState(seed)
    folds: List[List[int]] = [[] for _ in range(n_folds)]
    per_cls = [Counter() for _ in range(n_folds)]
    for cls in sorted(set(ev_cls.values())):
        evs = sorted(e for e in ev2idx if ev_cls[e] == cls)
        rng.shuffle(evs)
        for e in evs:
            k = min(range(n_folds), key=lambda j: (per_cls[j][cls], len(folds[j])))
            folds[k].extend(ev2idx[e]); per_cls[k][cls] += 1
    return [sorted(f) for f in folds if f]


def collect_pair_cache(pipeline: DisasterCaptioningPipeline, pairs: List[LabeledPair],
                       cache: Optional[str] = None) -> List[dict]:
    """One frozen forward pass per pair (no Qwen). Everything the pooling step needs is
    cached, so every fold, source subset and weight setting is re-scored without
    re-running ChangeOS / SAM2 / RemoteCLIP."""
    import pickle
    if cache and Path(cache).is_file():
        d = pickle.load(open(cache, "rb"))
        if d.get("paths") == [(p.pre, p.post) for p in pairs]:
            log.info("loaded pair cache %s (%d pairs)", cache, len(d["recs"]))
            return d["recs"]
    recs = []
    for i, p in enumerate(pairs, 1):
        r = pipeline.predict(p.pre, p.post, generate_caption=False)
        recs.append({"feats": r.features,
                     "dkg": (np.asarray(r.dkg["prob_vector"], np.float32)
                             if pipeline.cfg.use_dkg else None),
                     "zs": np.asarray(r.zeroshot["probs"], np.float32)})
        if i % 10 == 0 or i == len(pairs):
            log.info("pair cache %d/%d", i, len(pairs))
    if cache:
        Path(cache).parent.mkdir(parents=True, exist_ok=True)
        pickle.dump({"paths": [(p.pre, p.post) for p in pairs], "recs": recs},
                    open(cache, "wb"))
    return recs


def cross_validate_leave_event_out(pipeline: DisasterCaptioningPipeline,
                                   pairs: List[LabeledPair], n_folds: int = 5,
                                   cache: Optional[str] = None, k: int = 3,
                                   fit_kwargs: Optional[dict] = None,
                                   n_boot: int = 1000,
                                   save_prefix: Optional[str] = None) -> dict:
    """The headline classification table.

    For each fold the head is re-initialised, trained on the OTHER folds' events only,
    and scored on the held-out events. Reported side by side, all out-of-fold:
      zeroshot              the CLIP prior alone
      dkg                   the symbolic graph alone
      classifier_untrained  the text-initialised head (== prior; shown to prove it)
      pool_untrained        DKG + prior          (the training-free system)
      classifier_trained    the head trained on the other folds
      pool_trained          DKG + trained head   (the full system)
    with an event-level bootstrap CI on accuracy and macro-F1, and the measured
    dependence between the prior and the untrained head.

    Does not write the head checkpoint, and restores the head's prior state on exit."""
    cfg, clf = pipeline.cfg, pipeline.classifier
    labels = [p.disaster for p in pairs]
    events = [p.event or infer_event(p.pre) for p in pairs]
    prior_state = (_copy.deepcopy(clf.head.state_dict()), clf.is_trained, dict(clf.history))
    try:
        clf.reset()                                    # collect with the UNtrained head
        recs = collect_pair_cache(pipeline, pairs, cache)
        for r in recs:                                 # untrained-head opinion per pair
            r["clf_untrained"] = clf.predict(r["feats"])["probs"]
        folds = leave_event_out_folds(pairs, n_folds, cfg.seed)
        pred = {s: [None] * len(pairs) for s in
                ("zeroshot", "dkg", "classifier_untrained", "pool_untrained",
                 "classifier_trained", "pool_trained")}
        topk = {"pool_untrained": 0, "pool_trained": 0}
        fold_log = []
        for fi, test in enumerate(folds):
            tset = set(test)
            train = [i for i in range(len(pairs)) if i not in tset]
            assert not ({events[i] for i in train} & {events[i] for i in test}), \
                "event leaked across the train/test boundary"
            clf.reset()
            hist = clf.fit([recs[i]["feats"] for i in train], [labels[i] for i in train],
                           verbose=False, save_ckpt=False, **(fit_kwargs or {}))
            fold_log.append({"fold": fi, "n_test": len(test),
                             "n_train": len(train),
                             "test_events": sorted({events[i] for i in test}),
                             "head_val_acc": round(hist["best_val_acc"], 4)})
            for i in test:
                r = recs[i]
                C = DISASTER_CLASSES
                pred["zeroshot"][i] = C[int(r["zs"].argmax())]
                # a DKG that satisfied no rule returns a UNIFORM vector; argmax would
                # silently score it as class 0, so an abstention is recorded as one.
                dv = r["dkg"]
                pred["dkg"][i] = ("undetermined" if dv is None
                                  or float(dv.max() - dv.min()) < 1e-6
                                  else C[int(dv.argmax())])
                pred["classifier_untrained"][i] = C[int(r["clf_untrained"].argmax())]
                ct = clf.predict(r["feats"])
                pred["classifier_trained"][i] = ct["label"]
                src_u = {"zeroshot": r["zs"], "classifier": r["clf_untrained"], "dkg": r["dkg"]}
                src_t = {"zeroshot": r["zs"], "classifier": ct["probs"], "dkg": r["dkg"]}
                pu = pool_posterior(cfg, src_u, head_trained=False)
                pt = pool_posterior(cfg, src_t, head_trained=True)
                pred["pool_untrained"][i], pred["pool_trained"][i] = pu.label, pt.label
                topk["pool_untrained"] += int(labels[i] in [t["disaster"] for t in pu.top_k(k)])
                topk["pool_trained"] += int(labels[i] in [t["disaster"] for t in pt.top_k(k)])
            log.info("fold %d/%d done (%d test pairs, %d held-out events)",
                     fi + 1, len(folds), len(test), len({events[i] for i in test}))

        systems = {}
        for s, yp in pred.items():
            m = classification_metrics(labels, yp)
            m.update(cluster_bootstrap_ci(labels, yp, events, n_boot))
            if s in topk:
                m[f"top{k}_accuracy"] = round(topk[s] / len(pairs), 4)
            systems[s] = m
        dep = [source_dependence({"zeroshot": r["zs"], "classifier": r["clf_untrained"]})
               ["zeroshot~classifier"] for r in recs]
        dependence = {"zeroshot~untrained_head": {
            "mean_rank_corr": round(float(np.mean([d["rank_corr"] for d in dep])), 4),
            "mean_jsd": round(float(np.mean([d["jsd"] for d in dep])), 5),
            "top1_agreement": round(float(np.mean([d["same_top1"] for d in dep])), 4)}}
        out = {"n_pairs": len(pairs), "n_events": len(set(events)), "n_folds": len(folds),
               "classes_in_truth": sorted(set(labels)), "systems": systems,
               "source_dependence": dependence, "folds": fold_log,
               "predictions": [{"pre": p.pre, "event": e, "true": p.disaster,
                                **{s: pred[s][i] for s in pred}}
                               for i, (p, e) in enumerate(zip(pairs, events))]}
    finally:
        clf.reset()
        clf.head.load_state_dict(prior_state[0]); clf.head.eval()
        clf.is_trained, clf.history = prior_state[1], prior_state[2]

    print("\n" + "=" * 92)
    print(f"LEAVE-EVENT-OUT CV — {out['n_pairs']} pairs, {out['n_events']} events, "
          f"{out['n_folds']} folds, classes {out['classes_in_truth']}")
    print("=" * 92)
    print(f"{'system':<24}{'accuracy':>10}{'95% CI':>20}{'macro-F1':>10}{'95% CI':>20}"
          f"{'out-of-scope':>14}")
    for s, m in systems.items():
        print(f"{s:<24}{m['accuracy']:>10.3f}{str(m['accuracy_95ci']):>20}"
              f"{m['macro_F1']:>10.3f}{str(m['macro_F1_95ci']):>20}"
              f"{m['out_of_scope_predictions']:>14}")
    print(f"\nprior vs untrained head: rank corr {dependence['zeroshot~untrained_head']['mean_rank_corr']:.3f}, "
          f"top-1 agreement {dependence['zeroshot~untrained_head']['top1_agreement']:.3f} "
          f"(a duplicate source if both are ~1.0 — hence never pooled together)")
    print("=" * 92)
    plot_confusion(np.array(systems["pool_trained"]["confusion_matrix"]),
                   title=f"pool_trained, leave-event-out, acc {systems['pool_trained']['accuracy']:.2f}",
                   save_path=f"{save_prefix}_cv_confusion.png" if save_prefix else None)
    if save_prefix:
        Path(f"{save_prefix}_cv.json").write_text(json.dumps(out, indent=2, default=str))
    return out


CAPTION_CONDITIONS = ("C1_label_primed", "C2_measurements_only", "C3_blind")


def run_caption_conditions(pipeline: DisasterCaptioningPipeline, pairs: List[LabeledPair],
                           save_prefix: Optional[str] = None) -> dict:
    """The label-leakage experiment. SUPERSEDED by §21 (rev9) and no longer maintained:
    kept only so older LabeledPair runs still execute. Use §21 for every caption number.

    C1 Qwen sees images + checklist headed by the pipeline's verdict   (the SYSTEM caption, rev8)
    C2 Qwen sees images + the same checklist with no label              (no-label ablation)
    C3 Qwen sees images only                                            (blind / baseline)
    (§21 is the maintained harness for the RSCC comparison; this one is kept for the
    DisasterM3-style LabeledPair runs.)

    Per condition: the class the caption STATES (its 'Most likely disaster:' line),
    accuracy against ground truth, agreement with the pipeline's posterior, and the ECHO
    RATE -- among pairs where the pipeline is WRONG, how often the caption repeats the
    pipeline's wrong class. C1 echo >> C2 echo is the leakage signature. Caption metrics
    (BLEU / ROUGE-L / BERTScore) are reported per condition when references exist, which
    gives the cost of removing the labels."""
    cfg = pipeline.cfg
    saved = (cfg.label_in_prompt, cfg.use_blind_pass)
    caps = {c: [] for c in CAPTION_CONDITIONS}
    pipe_lab, truth, refs, used, leaks = [], [], [], [], []
    try:
        cfg.label_in_prompt, cfg.use_blind_pass = False, True
        for i, p in enumerate(pairs, 1):
            try:
                r = pipeline.predict(p.pre, p.post, generate_caption=True)       # C2 + C3
                pkg1 = package_from_result(r, label_in_prompt=True)   # rev8: same maps as predict()
                c1 = pipeline.captioner.caption(                                    # C1
                    Image.fromarray(r.pre), Image.fromarray(r.post),
                    pipeline._heat(r.refined_map), pkg1.to_prompt(),
                    hedge=bool(r.rvg_provisional["hedge"]), label_in_prompt=True)
            except Exception as e:
                log.warning("[%d/%d] skipped %s (%s: %s)", i, len(pairs), p.pre,
                            type(e).__name__, e)
                continue
            # The class lives in the FILE NAME of every pair, so prove it did not reach
            # the prompt. C1 is the control and is allowed to name classes; C2 and C3
            # must be clean, and a file-name or event token is a failure in all three.
            leak = assert_no_label_leak(
                r, p.pre, p.post, p.disaster, p.event,
                extra_prompts={"C1_prompt": pkg1.to_prompt()}, strict=True)
            leaks.append(leak)
            caps["C1_label_primed"].append(c1)
            caps["C2_measurements_only"].append(r.caption)
            caps["C3_blind"].append(r.blind_caption)
            pipe_lab.append(r.posterior.label); truth.append(p.disaster)
            refs.append(p.reference_caption); used.append(p.pre)
            if i % 5 == 0 or i == len(pairs):
                log.info("caption conditions %d/%d", i, len(pairs))
    finally:
        cfg.label_in_prompt, cfg.use_blind_pass = saved

    n = len(truth)
    if n == 0:
        raise RuntimeError("no pair produced captions")
    wrong = [i for i in range(n) if pipe_lab[i] != truth[i]]
    res = {"n": n, "pipeline_wrong": len(wrong), "conditions": {},
           "label_leak_audit": {"n_checked": len(leaks),
                                "n_leaking": sum(1 for l in leaks if not l["clean"])}}
    for c in CAPTION_CONDITIONS:
        lab = [stated_disaster(t) for t in caps[c]]
        row = {"label_accuracy": round(float(np.mean([a == b for a, b in zip(lab, truth)])), 4),
               "agrees_with_pipeline": round(float(np.mean([a == b for a, b in
                                                            zip(lab, pipe_lab)])), 4),
               "echo_rate_when_pipeline_wrong": (round(float(np.mean(
                   [lab[i] == pipe_lab[i] for i in wrong])), 4) if wrong else None),
               "unclear_rate": round(float(np.mean([x == "unclear" for x in lab])), 4)}
        idx = [i for i in range(n) if refs[i]]
        if idx:
            row["caption_metrics"] = evaluate_captions([caps[c][i] for i in idx],
                                                       [refs[i] for i in idx])
        res["conditions"][c] = row
    c1, c2 = res["conditions"]["C1_label_primed"], res["conditions"]["C2_measurements_only"]
    res["leakage_gap_agreement"] = round(c1["agrees_with_pipeline"] - c2["agrees_with_pipeline"], 4)
    res["label_removal_cost_label_accuracy"] = round(c1["label_accuracy"] - c2["label_accuracy"], 4)
    res["captions"] = [{"pre": used[i], "true": truth[i], "pipeline": pipe_lab[i],
                        **{c: caps[c][i] for c in CAPTION_CONDITIONS}} for i in range(n)]

    print("\n" + "=" * 92)
    print(f"CAPTION CONDITIONS — {n} pairs ({len(wrong)} where the pipeline is wrong)")
    print("=" * 92)
    print(f"{'condition':<24}{'label acc':>11}{'= pipeline':>12}{'echo (wrong)':>14}{'unclear':>10}")
    for c, row in res["conditions"].items():
        e = row["echo_rate_when_pipeline_wrong"]
        print(f"{c:<24}{row['label_accuracy']:>11.3f}{row['agrees_with_pipeline']:>12.3f}"
              f"{('n/a' if e is None else f'{e:.3f}'):>14}{row['unclear_rate']:>10.3f}")
        if "caption_metrics" in row:
            print(f"{'':<24}{row['caption_metrics']}")
    print(f"\nleakage gap (C1 - C2 agreement with pipeline): {res['leakage_gap_agreement']:+.3f}")
    a = res["label_leak_audit"]
    print(f"filename-leak audit: {a['n_leaking']}/{a['n_checked']} pairs leaked the class, "
          f"the event or a file-name token into a C2/C3 prompt")
    print("=" * 92)
    if save_prefix:
        Path(f"{save_prefix}_caption_conditions.json").write_text(
            json.dumps(res, indent=2, default=str))
    return res



# ---------------------------------------------------------------------------
# CALIBRATION (issues 5, 8, 12): the three places the pipeline used a constant
# someone picked by eye. Each of these MEASURES it instead, over real pairs.
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# FILENAME-LEAK AUDIT
# The ground-truth label lives in the file name. Nothing the models see may carry it.
#
# What each model actually receives (verified against the call sites, not assumed):
#   Qwen blind pass : Image.fromarray(pre_w), Image.fromarray(post_w), BLIND_PROMPT
#   Qwen grounded   : the same two arrays + the damage heat map + package.to_prompt()
#   RemoteCLIP      : numpy crops only
#   ChangeOS / SAM2 : numpy arrays only
# `Image.fromarray` builds a PIL image from a numpy buffer, so it carries no file name,
# no path and no EXIF. The path is read once in `_load` and never travels further.
#
# That is the design. `assert_no_label_leak` is what keeps it true: it reads the prompt
# strings a prediction actually produced and fails if the class, the event or any
# distinctive file-name token appears in them.
# ---------------------------------------------------------------------------
_LEAK_STOPWORDS = {"pre", "post", "disaster", "png", "jpg", "jpeg", "tif", "tiff",
                   "image", "images", "img", "data", "input", "kaggle", "datasets",
                   "train", "test", "val", "before", "after", "tile", "patch"}


def _class_aliases(cls: str) -> List[str]:
    return sorted({cls, cls.replace("_", " ")}
                  | {k for k, v in _STATED_ALIASES.items() if v == cls}
                  | {k for k, v in (globals().get("DM3_TO_CLASS") or {}).items() if v == cls})


def static_prompt_vocab() -> set:
    """Every word that is in EVERY prompt regardless of the pair: the instruction
    scaffolding, the list of answer options, the neutral measurement names. A term that
    lives here is not pair-specific and therefore cannot carry the answer."""
    parts = [BLIND_PROMPT, QwenCaptioner.build_prompt("", hedge=True, label_in_prompt=False),
             QwenCaptioner.build_prompt("", hedge=True, label_in_prompt=True)]
    for tbl in (NEUTRAL_SCENE_TERMS, NEUTRAL_CONCEPT_TERMS, NEUTRAL_GROUP_TERMS):
        parts.extend(tbl.values())
    return set(re.split(r"[^a-z0-9]+", " ".join(parts).lower())) - {""}


def _leak_terms(truth: Optional[str], pre_path: str, post_path: str,
                event: Optional[str] = None) -> Dict[str, List[str]]:
    """Every string that would betray the answer if it reached a prompt."""
    terms: Dict[str, List[str]] = {"class": [], "filename": [], "event": []}
    if truth:
        terms["class"] = _class_aliases(truth)
    static = static_prompt_vocab()
    for p in (pre_path, post_path):
        for tok in re.split(r"[^a-z0-9]+", Path(p).stem.lower()):
            if (len(tok) >= 4 and not tok.isdigit() and tok not in _LEAK_STOPWORDS
                    and tok not in static):
                terms["filename"].append(tok)
    terms["filename"] = sorted(set(terms["filename"]))
    if event:
        terms["event"] = sorted({event.lower(), event.lower().replace("_", " ")} - static)
    return terms


def _excess_class_mentions(text: str, truth: str) -> int:
    """How many times the TRUE class is named beyond the number of times the least-named
    class is. Both prompts enumerate all seven options, and an option list given
    identically for every pair carries no information about this pair -- so the test is
    not "is the class named" but "is it named MORE than its rivals"."""
    low = (text or "").lower()
    counts = {c: sum(low.count(a) for a in _class_aliases(c)) for c in DISASTER_CLASSES}
    return counts[truth] - min(counts.values())


def assert_no_label_leak(result: PredictionResult, pre_path: str, post_path: str,
                         truth: Optional[str] = None, event: Optional[str] = None,
                         extra_prompts: Optional[Dict[str, str]] = None,
                         strict: bool = True, verbose: bool = False) -> dict:
    """Fail if this pair's answer reached anything Qwen read.

    Audits the three strings the models actually receive: the blind prompt, the evidence
    package, and the full grounded prompt rebuilt from the same builder `caption()` uses.

    Two different tests, because the two kinds of leak look different:
      * CLASS  -- a leak is the true class named MORE OFTEN than the others, not merely
        named: every prompt lists all seven as the answer options.
      * FILE NAME / EVENT -- never allowed anywhere. Tokens that appear in the static
        scaffolding are excluded first, so a file called `..._water_12_pre.png` is not
        flagged for the prompt's own word "water".

    `label_in_prompt=True` is the C1 control and is SUPPOSED to name the predicted class,
    so excess class mentions there are recorded, not raised."""
    primed = bool(getattr(result.package, "label_in_prompt", False))
    evidence = result.package.to_prompt()
    prompts = {"blind_prompt": BLIND_PROMPT,
               "evidence_package": evidence,
               "grounded_prompt": QwenCaptioner.build_prompt(
                   evidence, hedge=bool(result.rvg.get("hedge")), label_in_prompt=primed)}
    prompts.update(extra_prompts or {})
    terms = _leak_terms(truth, pre_path, post_path, event)

    hits, allowed = [], []
    for where, text in prompts.items():
        low = (text or "").lower()
        if truth:
            excess = _excess_class_mentions(text, truth)
            if excess > 0:
                rec = {"where": where, "kind": "class", "term": truth, "excess": excess}
                (allowed if (primed and where != "blind_prompt") else hits).append(rec)
        for kind in ("filename", "event"):
            for w in terms[kind]:
                if w and w in low:
                    hits.append({"where": where, "kind": kind, "term": w})

    rep = {"pre": pre_path, "truth": truth, "label_in_prompt": primed,
           "checked": sorted(prompts), "terms": terms,
           "leaks": hits, "expected_C1_class_mentions": allowed, "clean": not hits}
    if verbose or hits:
        for h in hits:
            log.error("LABEL LEAK: %r (%s) in %s", h["term"], h["kind"], h["where"])
    if hits and strict:
        raise AssertionError(
            f"label leak into the prompt for {Path(pre_path).name}: "
            + ", ".join(f"{h['term']}({h['kind']}) in {h['where']}" for h in hits[:6]))
    return rep


def audit_label_leak(pipeline: DisasterCaptioningPipeline, pairs: Sequence[LabeledPair],
                     n: int = 10, strict: bool = False) -> dict:
    """Run the audit over a sample and report. Do this ONCE on your own data before
    trusting any classification number -- it is the evidence a reviewer will ask for."""
    rows, bad = [], 0
    for i, p in enumerate(list(pairs)[:n], 1):
        r = pipeline.predict(p.pre, p.post, generate_caption=False)
        rep = assert_no_label_leak(r, p.pre, p.post, p.disaster, p.event, strict=strict)
        rows.append(rep)
        bad += int(not rep["clean"])
        if i % 5 == 0 or i == min(n, len(pairs)):
            log.info("leak audit %d/%d", i, min(n, len(pairs)))
    n_terms = sum(len(r["terms"]["filename"]) for r in rows)
    print("\n" + "=" * 86)
    print(f"FILENAME-LEAK AUDIT — {len(rows)} pairs, {n_terms} file-name tokens checked")
    print("=" * 86)
    print(f"  prompts checked per pair : {', '.join(rows[0]['checked']) if rows else '-'}")
    print(f"  pairs with a leak        : {bad} / {len(rows)}")
    if rows:
        print(f"  example terms checked    : class={rows[0]['terms']['class'][:4]}, "
              f"filename={rows[0]['terms']['filename'][:4]}")
    if bad == 0:
        print("\n  CLEAN. The class name, the event name and every distinctive file-name")
        print("  token are absent from the blind prompt and the evidence package, so the")
        print("  classification came from the images and the measurements alone.")
    print("=" * 86)
    return {"n": len(rows), "n_leaking": bad, "rows": rows}


def collect_calibration_stats(pipeline: DisasterCaptioningPipeline,
                              pairs: Sequence[LabeledPair], n: Optional[int] = None,
                              cache: Optional[str] = None) -> List[dict]:
    """ONE forward pass per pair, holding everything the three calibrators need.

    Running them separately costs three passes over the same images -- about 40 s each
    on a T4, so an hour of GPU for 30 pairs. Collect once, calibrate three times."""
    import pickle
    sel = list(pairs)[: n] if n else list(pairs)
    if cache and Path(cache).is_file():
        d = pickle.load(open(cache, "rb"))
        if d.get("paths") == [(p.pre, p.post) for p in sel]:
            log.info("loaded calibration stats %s (%d pairs)", cache, len(d["rows"]))
            return d["rows"]
    rows = []
    for i, p in enumerate(sel, 1):
        try:
            r = pipeline.predict(p.pre, p.post, generate_caption=False)
        except Exception as e:
            log.warning("[%d/%d] skipped %s (%s: %s)", i, len(sel), p.pre,
                        type(e).__name__, e)
            continue
        by_src = r.dkg.get("concept_by_source", {})
        rows.append({
            "pre": p.pre, "truth": p.disaster, "event": p.event,
            "raw_phys": pipeline.dkg.raw_physical_concepts(r.fused.physical),
            "phys_mean": {k: float(v.mean()) for k, v in r.fused.physical.items()},
            "scalars": dict(r.fused.scalars),
            "soft_iou": ReliabilityVerificationGate._soft_iou(
                r.fused.anychange_map, r.fused.physical["composite"]),
            "margin": float(r.posterior.margin),
            "max_semantic_mass": max(
                [float(v.get("transition", 0.0)) + float(v.get("scene", 0.0))
                 for v in by_src.values()] or [0.0]),
            "concept_mass": dict(r.dkg.get("concept_mass", {})),
            "n_transitions": len(r.scene_evidence.transitions),
            "n_instances": len(r.instances),
            "rvg_confidence": float(r.rvg["confidence"]),
            "posterior": r.posterior.label,
        })
        if i % 5 == 0 or i == len(sel):
            log.info("calibration stats %d/%d", i, len(sel))
    if cache and rows:
        Path(cache).parent.mkdir(parents=True, exist_ok=True)
        pickle.dump({"paths": [(p.pre, p.post) for p in sel], "rows": rows},
                    open(cache, "wb"))
    return rows


def no_change_controls(pairs: List[LabeledPair], n: int = 20,
                       seed: int = 0) -> List[LabeledPair]:
    """ISSUE 5: pairs with NO disaster between them, built as (pre, pre).

    Every "coverage" the pipeline reports should be at or near zero on these. Whatever
    it reports instead is the floor that a real number has to clear, and it is the
    single thing missing before any scene-coverage figure can go in the paper. A (pre,
    pre) control bounds the floor from below -- it has no co-registration error, no
    seasonal difference and no illumination difference. Report it as a lower bound and
    say so; a same-season pre/pre pair from two dates is the stronger control if the
    dataset ever provides one."""
    rng = np.random.RandomState(seed)
    pick = rng.choice(len(pairs), size=min(n, len(pairs)), replace=False)
    return [LabeledPair(pre=pairs[i].pre, post=pairs[i].pre,
                        disaster=pairs[i].disaster, event=pairs[i].event) for i in pick]


def measure_change_floor(pipeline: DisasterCaptioningPipeline,
                         pairs: List[LabeledPair], n: int = 20) -> dict:
    """What the pipeline reports when nothing changed. Run this before quoting any
    coverage, concept mass or RVG confidence as evidence of anything."""
    ctrl = no_change_controls(pairs, n)
    stats = collect_calibration_stats(pipeline, ctrl)
    rows = [{"scene_coverage": s["scalars"]["scene_coverage"],
             "fused_mean": s["scalars"]["fused_mean"],
             "n_transitions": s["n_transitions"],
             "n_instances": s["n_instances"],
             "rvg_confidence": s["rvg_confidence"],
             **{f"phys_{k}": v for k, v in s["phys_mean"].items()},
             **{f"concept_{k}": v for k, v in s["concept_mass"].items()}}
            for s in stats]
    if not rows:
        raise RuntimeError("no control pair evaluated")
    keys = sorted({k for row in rows for k in row})
    out = {"n": len(rows),
           "median": {k: round(float(np.median([row.get(k, 0.0) for row in rows])), 5)
                      for k in keys},
           "p90": {k: round(float(np.percentile([row.get(k, 0.0) for row in rows], 90)), 5)
                   for k in keys}}
    print("\n" + "=" * 78)
    print(f"NO-CHANGE CONTROL — {out['n']} (pre, pre) pairs")
    print("=" * 78)
    print(f"{'quantity':<34}{'median':>12}{'p90':>12}")
    for k in keys:
        print(f"{k:<34}{out['median'][k]:>12.4f}{out['p90'][k]:>12.4f}")
    print("\nAnything a real pair reports below the p90 column is indistinguishable "
          "from no change.")
    print("=" * 78)
    return out


def calibrate_physical_baselines(pipeline: DisasterCaptioningPipeline,
                                 pairs: Optional[List[LabeledPair]] = None, n: int = 30,
                                 apply: bool = True,
                                 stats: Optional[List[dict]] = None) -> Dict[str, float]:
    """ISSUE 8: measure `Config.phys_concept_baseline` instead of guessing it.

    The six RGB priors are not comparable as they stand -- `inundation` multiplies one
    mid-sized map by `(1 - texture)`, whose mean is about 0.85, while `ashfall` and
    `burned` multiply two small maps -- so `flood` won the DKG's `required` term on
    every pair tested, including a volcano pair where 99% of the mass was this prior.
    The baseline is the MEDIAN of each prior over a pair sample; dividing by it turns
    each prior into "excess over typical", which is comparable across concepts."""
    stats = stats if stats is not None else collect_calibration_stats(pipeline, pairs, n)
    vals: Dict[str, List[float]] = defaultdict(list)
    for s in stats:
        for k, v in s["raw_phys"].items():
            vals[k].append(float(v))
    if not vals:
        raise RuntimeError("no pair evaluated")
    base = {k: round(float(np.median(v)), 5) for k, v in vals.items()}
    print("\nphys_concept_baseline (median over %d pairs):" % len(stats))
    for k, v in sorted(base.items(), key=lambda kv: -kv[1]):
        print(f"  {k:<18}{v:>10.5f}")
    print("\nThe spread between the largest and the smallest IS the bias that favoured "
          "whichever rule they satisfy.")
    if apply:
        pipeline.cfg.phys_concept_baseline = base
        log.info("applied phys_concept_baseline to CFG")
    return base


def fast_physical_baselines(pipeline: DisasterCaptioningPipeline,
                            pairs: Sequence[Tuple[str, str]],
                            apply: bool = True) -> Dict[str, float]:
    """rev9: `calibrate_physical_baselines` without the full forward pass.

    The DKG's physical priors read only the RGB physical channels, which need no deep
    model, so the per-concept MEDIAN can be measured in seconds per pair instead of one
    `predict()` (~2 min) each. It uses no labels. `pairs` are (pre_path, post_path)."""
    vals: Dict[str, List[float]] = defaultdict(list)
    for pre_p, post_p in pairs:
        try:
            # rev10: the same preparation as predict() -- aligned, clouds excluded
            prep = pipeline._prepare(pre_p, post_p)
            raw = pipeline.dkg.raw_physical_concepts(
                pipeline.physical(prep["pre_w"], prep["post_w"], prep["invalid"]))
        except Exception as e:
            log.warning("calibration skipped %s (%s: %s)", pre_p, type(e).__name__, e)
            continue
        for k, v in raw.items():
            vals[k].append(float(v))
    if not vals:
        raise RuntimeError("no pair could be read for calibration")
    base = {k: round(float(np.median(v)), 5) for k, v in vals.items()}
    n = len(next(iter(vals.values())))
    print(f"\nphys_concept_baseline (median over {n} pairs, no labels used):")
    for k, v in sorted(base.items(), key=lambda kv: -kv[1]):
        print(f"  {k:<18}{v:>10.5f}")
    if apply:
        pipeline.cfg.phys_concept_baseline = base
        log.info("applied phys_concept_baseline to CFG (DKG physical priors calibrated)")
    return base


def calibrate_rvg_denominators(pipeline: DisasterCaptioningPipeline,
                               pairs: Optional[List[LabeledPair]] = None, n: int = 30,
                               q: float = 75.0, apply: bool = True,
                               stats: Optional[List[dict]] = None) -> Dict[str, float]:
    """ISSUE 12: five of the nine checks pinned at 1.00 because each divides by a
    constant chosen by eye. Each denominator is set to the q-th percentile of the
    quantity it scores, so a check reads 1.00 on a pair that is genuinely in the top
    quartile and spreads out below that."""
    stats = stats if stats is not None else collect_calibration_stats(pipeline, pairs, n)
    obs: Dict[str, List[float]] = defaultdict(list)
    for s in stats:
        obs["rvg_scene_cov_ref"].append(s["scalars"]["scene_coverage"])
        obs["rvg_obj_dmg_ref"].append(s["scalars"]["object_damaged_area_frac"])
        obs["rvg_iou_ref"].append(s["soft_iou"])
        obs["rvg_margin_ref"].append(s["margin"])
        obs["rvg_transition_mass_ref"].append(s["max_semantic_mass"])
    if not obs:
        raise RuntimeError("no pair evaluated")
    out = {k: round(float(np.percentile(v, q)), 5) for k, v in obs.items()}
    print(f"\nRVG denominators at the {q:.0f}th percentile of {n} pairs:")
    for k, v in out.items():
        print(f"  {k:<28}{getattr(pipeline.cfg, k):>10.4f}  ->{v:>10.4f}")
    if apply:
        for k, v in out.items():
            setattr(pipeline.cfg, k, max(v, 1e-4))
        log.info("applied RVG denominators to CFG")
    return out


# ---------------------------------------------------------------------------
# Stage 0 vs Stage 2 A/B on one pair (kept from Stage 1)
# ---------------------------------------------------------------------------
def compare_pipelines(pre_path, post_path, generate_caption: bool = True
                      ) -> Dict[str, PredictionResult]:
    results, cfg = {}, pipeline.cfg
    original_mode = cfg.mode
    try:
        for m in ("object_only", "dual"):
            cfg.mode = m
            log.info("=" * 30 + f"  RUN: mode={m}  " + "=" * 30)
            results[m] = pipeline.predict(pre_path, post_path, generate_caption=generate_caption)
    finally:
        cfg.mode = original_mode

    old, new = results["object_only"], results["dual"]
    rows = [
        ("scene coverage (frac of image)", old.fused.scalars["scene_coverage"],
         new.fused.scalars["scene_coverage"]),
        ("AnyChange coverage", old.fused.scalars["anychange_coverage"],
         new.fused.scalars["anychange_coverage"]),
        ("semantic transitions found", len(old.scene_evidence.transitions),
         len(new.scene_evidence.transitions)),
        ("evidence tokens", sum(old.tokens.counts().values()),
         sum(new.tokens.counts().values())),
        ("SAM2 instances", len(old.instances), len(new.instances)),
        ("distinct DKG concepts", len(old.dkg["concept_mass"]), len(new.dkg["concept_mass"])),
        ("SDA area gain %", old.sda_report["area_gain_pct"], new.sda_report["area_gain_pct"]),
        ("fused mean severity", old.fused.scalars["fused_mean"], new.fused.scalars["fused_mean"]),
        ("posterior probability", old.posterior.prob, new.posterior.prob),
        ("posterior margin", old.posterior.margin, new.posterior.margin),
        ("RVG confidence", old.rvg["confidence"], new.rvg["confidence"]),
    ]
    print("\n" + "=" * 92)
    print(f"{'METRIC':<34}{'OBJECT-ONLY (Stage 0)':>28}{'DUAL + STAGE 2':>28}")
    print("=" * 92)
    for name, a, b in rows:
        fa = f"{a:.4f}" if isinstance(a, float) else str(a)
        fb = f"{b:.4f}" if isinstance(b, float) else str(b)
        print(f"{name:<34}{fa:>28}{fb:>28}")
    print(f"{'inferred disaster':<34}{old.disaster:>28}{new.disaster:>28}")
    print("=" * 92)
    for tag, r in (("OBJECT-ONLY (Stage 0)", old), ("DUAL + STAGE 2", new)):
        print(f"\n----- CAPTION: {tag} -----\n{r.caption}")
    return results


# [rev7] demo disabled: print(json.dumps(profile(result, {"changeos": changeos_det, "anychange": anychange_det,
# [rev7] demo disabled:                                   "sam2": refiner.predictor, "remoteclip": clip,
# [rev7] demo disabled:                                   "head": classifier, "qwen": captioner}),
# [rev7] demo disabled:                  indent=2, default=str))

# [rev7] demo disabled: print("\n--- PER-IMAGE EVIDENCE CONTRIBUTION ---")
# [rev7] demo disabled: print(json.dumps(contribution_analysis(result, pipeline, CFG), indent=2, default=str))

# Ablation over every switch on the current pair (no captions -> ~1 min per row):
# [rev7] demo disabled: ablation_rows = run_ablation(pipeline, pre_path, post_path, generate_caption=False)

# ---- Tier-1 evaluation on DisasterM3 -- no manifest file needed (§13b indexed it) -----
# 0) See the selection and its download size before fetching anything:
# dm3_pairs = dm3_labeled_pairs(per_class=30, download=False)
#
# 0b) CALIBRATION FIRST (issues 5, 8, 12). None of the numbers below mean anything
#     until these have been run once; each writes its result into CFG.
# floor = measure_change_floor(pipeline, dm3_pairs, n=20)      # what "no change" reads
# ISSUE 6: the module-contribution table, with intervals, over the pair set:
# abl   = run_ablation_multi(pipeline, dm3_pairs, save_prefix=f"{CFG.out_dir}/dm3")
# base  = calibrate_physical_baselines(pipeline, dm3_pairs, n=30)   # ISSUE 8
# refs  = calibrate_rvg_denominators(pipeline, dm3_pairs, n=30)     # ISSUE 12
#
# 1) Headline classification table. Fetches the images, then one frozen forward pass per
#    pair (cached), then leave-event-out CV: CLIP prior / DKG / untrained head / trained
#    head / both pools, out-of-fold, with event-level bootstrap CIs.
# dm3_pairs = dm3_labeled_pairs(per_class=30)          # ~150 pairs, ~600 MB, no Qwen
# build_dm3_manifest(dm3_pairs)                        # for the record
# cv = cross_validate_leave_event_out(pipeline, dm3_pairs, n_folds=5,
#                                     cache=f"{CFG.out_dir}/pair_cache.pkl",
#                                     save_prefix=f"{CFG.out_dir}/dm3")
#
# 2) The leakage experiment: C1 label-primed vs C2 measurements-only vs C3 blind.
#    Three Qwen calls per pair, so keep it small; captions are only scored against a
#    reference where DisasterM3 has one (earthquake and flood in Bench).
# with_refs = [p for p in dm3_pairs if p.reference_caption]
# cap = run_caption_conditions(pipeline, with_refs[:40], save_prefix=f"{CFG.out_dir}/dm3")
#
# Quick in-sample check of the current pipeline state (NOT for the paper):
# metrics = evaluate_dataset(pipeline, dm3_pairs, generate_caption=False,
#                            save_prefix=f"{CFG.out_dir}/test")

# Stage 0 vs Stage 2 A/B (adds 4 Qwen generations):
# ab = compare_pipelines(pre_path, post_path)


print('§19 definitions loaded; single-pair demos disabled.')


## 21 · Caption comparison — baseline Qwen vs our pipeline, against RSCC (rev12)

Five captions per pair, scored against the RSCC change captions: **C3_baseline** — Qwen with the images only; **C1_system** — exactly what `pipeline.predict()` returns (rev12: evidence → close-up looks → write from scratch → verify coverage); **C2_no_label** — the same looks written up without the verdict; **C4_checklist** — rev8's checklist caption; **C5_revise** — the rev9–rev11 draft → check → revise caption (`CC_RUN_REVISE`), so rev12 is measured against what it replaces.

Before captioning, §21 **calibrates the DKG physical priors** on the pairs (per-concept median, no labels; set `CC_CAL_DIR` to calibrate on pairs outside the test set) and uses the **water-agreement rule**.

Sections: **A** classification, water signals and detector firing per event · **B** disaster named by each caption · **C** BLEU / METEOR / ROUGE-L / CIDEr-D / BERTScore (all, verdict-right, without Tonga, per class) · **D** paired comparison with the baseline · **E** consistency with the pipeline's measurements (not ground truth) · **F** examples with Qwen's notes per region · **G** copying vs describing, and **coverage: how many visible measured changes reach the caption**, by kind of change, plus what was excluded and why · **H** the paper tables: **METEOR and ROUGE-1/2/L per class and overall, for ALL pairs and for pairs whose VERDICT IS RIGHT** · **I** grounding tests on `CC_GT_N` pairs (verdict flipped on the same looks; the whole system caption on grey images).

Metrics follow RSICC practice: PTB tokens, METEOR 1.5 (Java) as a set-level aggregate, COCO ROUGE-L (β = 1.2), ROUGE-1/2 from rouge-score with stemming, BLEU and CIDEr-D from pycocoevalcap. If Java is unavailable METEOR falls back to NLTK and every header says so.

Outputs in `/kaggle/working/caption_compare`: `caption_pairs_rev12.jsonl` (resumable), `caption_summary_rev12.json`, `caption_metrics_rev12.csv`, `phys_concept_baseline_rev11.json` (the channels did not change in rev12), and `visual_check/*.png` — BEFORE | AFTER | overlay for tiles where ChangeOS or the water tests disagree with the reference. **Look at those sheets**: they settle whether ChangeOS or the references are right about collapsed buildings.

Cost per pair: the blind caption, 4–10 short close-up looks and 1–3 writes for C1, 1–3 writes for C2, one call for C4 and up to one for C5; the `CC_GT_N` pairs add one write and a full grey-image run.

**rev10:** section A compares classification with the rev9 run on the same pairs (43/57) and marks any class that REGRESSED. That is the check that the rev10 changes are not biased toward one class.


In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # §21 · CAPTION COMPARISON (rev12) — baseline Qwen vs our pipeline, scored against RSCC
# #
# #   C3_baseline   Qwen2.5-VL with the two images only (no pipeline; the blind pass).
# #   C1_system     THE SYSTEM CAPTION, exactly what pipeline.predict() returns (rev12):
# #                 every valid measured change -> Qwen describes a close-up of each region
# #                 -> one caption written from scratch from the images, the verdict and those
# #                 notes -> verified to hold every visible change (else rewritten from scratch).
# #   C2_no_label   the same looks written up without the verdict (label ablation; no extra looks)
# #   C4_checklist  rev8's system caption: Qwen writes from the evidence checklist. Kept as
# #                 a comparison because it copies the checklist (copy rate 0.147).
# #   C5_revise     rev9-rev11's system caption: the blind draft patched with at most 4
# #                 corrections. Kept so rev12 is measured against what it replaces.
# #
# # What each section answers:
# #   A  classification (after DKG calibration and the water-agreement rule)
# #   B  does the caption name the right disaster?
# #   C  BLEU / METEOR / ROUGE-L / CIDEr-D / BERTScore against RSCC
# #   D  paired comparison with the baseline on the same pair
# #   E  consistency with the pipeline's own measurements (NOT ground truth)
# #   F  examples: Qwen's notes per region, the caption, and the rev9 revision beside it
# #   G  copying vs describing; how many measured changes reach the caption (coverage)
# #   H  METEOR and ROUGE per class and overall: ALL pairs, and pairs whose VERDICT IS RIGHT
# #   I  grounding tests: does the caption follow the measurements, and the images?
# #
# # REFERENCES are RSCC change captions matched by tile name. Only 7 of the 39 Tonga
# # references in the rev8 run mention a volcano or ash, so every table is also reported
# # without Tonga. Resumable: each pair is appended to the JSONL as soon as it finishes.
# # ═══════════════════════════════════════════════════════════════════════════════
# import json as _cj, re as _cre, os as _cos, gc as _cgc, math as _cm, collections as _cc
# import statistics as _cst
# import numpy as np, torch
# from PIL import Image as _CImage

# CC_IMG_DIR    = "/kaggle/input/datasets/vaidurya13/rscc-52/rscc_captioning_52"   # <-- EDIT
# CC_RSCC_JSONL = "/kaggle/input/datasets/vaidurya13/rscc-gt/RSCC_qvq.jsonl"    # <-- EDIT
# CC_OUT        = "/kaggle/working/caption_compare"
# CC_REF_ONLY   = False          # True = skip pairs that have no RSCC reference
# CC_BERTSCORE  = True
# CC_RUN_CHECKLIST = True        # C4: rev8's checklist caption, for comparison (+1 Qwen call/pair)
# CC_RUN_REVISE = True           # C5: rev9-rev11's draft -> check -> revise caption (+0-1 Qwen call/pair)
# CC_CALIBRATE  = True           # measure the DKG physical baselines first (no labels used)
# CC_CAL_DIR    = None           # folder of OTHER pairs to calibrate on; None = the evaluated pairs
# CC_GT_N       = 15             # pairs (stratified by event) that get the grounding tests (+1 write, +1 full grey-image run)
# CC_VISUAL_CHECK = True         # save image sheets where ChangeOS / water disagree with the reference
# CONDS = (("C3_baseline", "C1_system", "C2_no_label") + (("C4_checklist",) if CC_RUN_CHECKLIST else ())
#          + (("C5_revise",) if CC_RUN_REVISE else ()))
# SYSTEM = "C1_system"
# OTHERS = tuple(c for c in CONDS if c != "C3_baseline")
# _cos.makedirs(CC_OUT, exist_ok=True)
# CC_LOG = f"{CC_OUT}/caption_pairs_rev12.jsonl"     # rev12 records never mix with earlier ones

# CC_CLASS_OF = {"turkey_earthquake": "earthquake", "earthquake_turkey": "earthquake",
#                "mexico_earthquake": "earthquake", "pakistan_flooding": "flood",
#                "nepal_flooding": "flood", "midwest_flooding": "flood",
#                "tonga_volcano": "volcanic_eruption", "guatemala_volcano": "volcanic_eruption",
#                "lower_puna_volcano": "volcanic_eruption",
#                "stvincent_volcano": "volcanic_eruption"}

# def _cc_norm(s): return s.lower().replace("-", "_")
# def _cc_stem(f):
#     return _cre.sub(r"_(pre|post)(_disaster)?(_part\d+)?\.(png|jpg|jpeg|tif)$", "",
#                     _cos.path.basename(f))
# # rev11: classes are open. A pair's class is read from its event name -- the known
# # RSCC events below first, then any disaster word (wildfire, hurricane, landslide, ...),
# # so RSCC wildfire or hurricane pairs can be evaluated too.
# _CC_WORDS = [("tsunami", r"tsunami"), ("volcanic_eruption", r"volcan|erupt|lava"),
#              ("earthquake", r"earthquake|quake|seism"), ("flood", r"flood|inundat"),
#              ("cyclone", r"hurricane|cyclone|typhoon|storm"), ("landslide", r"landslide|mudslide"),
#              ("wildfire", r"wildfire|fire|burn")]
# def _cc_cls(k):
#     for p, c in CC_CLASS_OF.items():
#         if _cc_norm(k).startswith(p): return c
#     for c, pat in _CC_WORDS:
#         if _cre.search(pat, _cc_norm(k)): return c
#     return None
# def _cc_event(k): return _cre.sub(r"_?\d+$", "", _cc_norm(k))

# # ── references ────────────────────────────────────────────────────────────────
# CC_REFS = _cc.defaultdict(list)
# with open(CC_RSCC_JSONL) as fh:
#     for line in fh:
#         r = _cj.loads(line)
#         CC_REFS[_cc_norm(_cc_stem(r["pre_image"]))].append(r["change_caption"])

# # ── pairs ────────────────────────────────────────────────────────────────────
# _g = _cc.defaultdict(dict)
# for f in sorted(_cos.listdir(CC_IMG_DIR)):
#     m = _cre.match(r"(.+?)_(pre|post)(_disaster)?\.(png|jpg|jpeg|tif)$", f)
#     if m: _g[m.group(1)][m.group(2)] = _cos.path.join(CC_IMG_DIR, f)
# CC_PAIRS = []
# for k, v in sorted(_g.items()):
#     if "pre" in v and "post" in v and _cc_cls(k):
#         refs = CC_REFS.get(_cc_norm(k), [])
#         if CC_REF_ONLY and not refs: continue
#         CC_PAIRS.append({"key": k, "cls": _cc_cls(k), "event": _cc_event(k),
#                          "pre": v["pre"], "post": v["post"], "refs": refs})
# print(f"{len(CC_PAIRS)} pairs | with RSCC reference: {sum(bool(p['refs']) for p in CC_PAIRS)} | "
#       f"{dict(_cc.Counter(p['cls'] for p in CC_PAIRS))}")
# print("  references per pair:", dict(_cc.Counter(len(p["refs"]) for p in CC_PAIRS)),
#       "(4 = quadrant tiles, scored as multiple references)")
# _noref = [p["key"] for p in CC_PAIRS if not p["refs"]]
# if _noref:
#     print(f"  {len(_noref)} pair(s) have NO RSCC reference (naming mismatch?), e.g. {_noref[:5]}")
# _unpaired = sorted(k for k, v in _g.items() if not ("pre" in v and "post" in v))
# _nocls = sorted(k for k, v in _g.items() if "pre" in v and "post" in v and not _cc_cls(k))
# if _unpaired: print(f"  {len(_unpaired)} file stem(s) without both pre and post, e.g. {_unpaired[:3]}")
# if _nocls: print(f"  {len(_nocls)} pair(s) skipped: event not in CC_CLASS_OF, e.g. {_nocls[:3]}")
# assert CC_PAIRS, "no pairs — check CC_IMG_DIR"


# # ── DKG calibration (review item 17) ─────────────────────────────────────────
# # The DKG's physical priors are not comparable until each is divided by its typical
# # value; uncalibrated, `flood` wins by default -- 17 of the 19 rev8 misclassifications
# # were "flood". The baseline is a per-concept MEDIAN over pairs; no labels are used.
# def _pairs_in(folder):
#     g = _cc.defaultdict(dict)
#     for f in sorted(_cos.listdir(folder)):
#         m = _cre.match(r"(.+?)_(pre|post)(_disaster)?\.(png|jpg|jpeg|tif)$", f)
#         if m: g[m.group(1)][m.group(2)] = _cos.path.join(folder, f)
#     return [(v["pre"], v["post"]) for _, v in sorted(g.items()) if "pre" in v and "post" in v]

# CC_CAL_FILE = f"{CC_OUT}/phys_concept_baseline_rev11.json"   # rev11: new baseline file
# if CC_CALIBRATE:
#     if _cos.path.exists(CC_CAL_FILE):
#         _base = _cj.load(open(CC_CAL_FILE))["baseline"]
#         CFG.phys_concept_baseline = _base; pipeline.cfg.phys_concept_baseline = _base
#         print(f"DKG physical baselines loaded from {CC_CAL_FILE}")
#     else:
#         _cal_pairs = _pairs_in(CC_CAL_DIR) if CC_CAL_DIR else [(p["pre"], p["post"]) for p in CC_PAIRS]
#         _base = fast_physical_baselines(pipeline, _cal_pairs, apply=True)
#         CFG.phys_concept_baseline = _base
#         _cj.dump({"baseline": _base, "source": CC_CAL_DIR or CC_IMG_DIR, "n_pairs": len(_cal_pairs),
#                   "labels_used": False}, open(CC_CAL_FILE, "w"), indent=1)
#     if not CC_CAL_DIR:
#         print("  note: calibrated on the evaluated pairs themselves. No labels are used (a\n"
#               "  per-concept median), but for the paper set CC_CAL_DIR to pairs outside the test set.")
# else:
#     print("WARNING: DKG physical priors left UNCALIBRATED (CC_CALIBRATE=False)")
# print(f"water counts only where both tests agree: {bool(CFG.water_requires_agreement)} | "
#       f"building checks: {CFG.caption_check_buildings} | caption mode: revise")

# # grounding-test pairs: round-robin over events, so every event is represented
# _by_ev = _cc.defaultdict(list)
# for p in CC_PAIRS: _by_ev[p["event"]].append(p["key"])
# CC_GT_KEYS = []
# while len(CC_GT_KEYS) < min(CC_GT_N, len(CC_PAIRS)):
#     for ev in sorted(_by_ev):
#         if _by_ev[ev] and len(CC_GT_KEYS) < CC_GT_N:
#             CC_GT_KEYS.append(_by_ev[ev].pop(0))
# CC_GT_KEYS = set(CC_GT_KEYS)

# # ── helpers ──────────────────────────────────────────────────────────────────
# def _cc_split(raw: str):
#     """(caption without the verdict line, stated label)."""
#     t = (raw or "").replace("*", "")
#     cap = _cre.split(r"\n?\s*most likely disaster\s*[:\-]", t, flags=_cre.I)[0].strip()
#     return cap, stated_disaster(t)

# def _cc_safe(fn, *a, **k):
#     try:
#         return (fn(*a, **k) or "").strip()
#     except Exception as e:
#         return f"[generation failed: {type(e).__name__}]"

# _alias_set = {k for k in _STATED_ALIASES if k != "unclear"}
# if "DM3_TO_CLASS" in globals():
#     _alias_set |= {k for k, v in DM3_TO_CLASS.items() if v}
# _ALIASES = sorted(_alias_set, key=len, reverse=True)
# def _cc_leaks(text):
#     low = (text or "").lower()
#     return sorted({a for a in _ALIASES if _cre.search(r"\b" + _cre.escape(a) + r"\b", low)})

# def _rl(a: str, b: str, beta: float = 1.2) -> float:
#     """ROUGE-L F between two texts (used for image dependence)."""
#     A, B = _cre.findall(r"[a-z0-9]+", (a or "").lower()), _cre.findall(r"[a-z0-9]+", (b or "").lower())
#     if not A or not B: return 0.0
#     prev = [0] * (len(B) + 1)
#     for i in range(1, len(A) + 1):
#         cur = [0] * (len(B) + 1)
#         for j in range(1, len(B) + 1):
#             cur[j] = prev[j-1] + 1 if A[i-1] == B[j-1] else max(prev[j], cur[j-1])
#         prev = cur
#     l = prev[-1]
#     if not l: return 0.0
#     p, r = l / len(A), l / len(B)
#     return (1 + beta**2) * p * r / (r + beta**2 * p)

# def _water_said(raw: str) -> bool:
#     return any(asserts(s, "water") for s in _sentences(caption_body(raw)))

# def grounding_tests(r, pre_im, post_im, heat, ev_c1, raw) -> dict:
#     """I · Is the evidence USED, and are the images? Two tests on the same pair.
#     (a) CORRUPTION: cycle the verdict to another class and write the caption again from
#         the SAME looks. Does the caption name the false event? Following = the verdict
#         decides what the caption says. (rev12 has no water flip: water is not a line in a
#         prompt any more, it is a region Qwen looked at.)
#     (b) IMAGE ABLATION: replace BEFORE/AFTER with flat grey and run the WHOLE system
#         caption again (looks included). 1 - ROUGE-L between the normal and the grey-image
#         caption = how much the caption depends on the images; with grey close-ups Qwen
#         sees no change, so a caption that still describes changes came from elsewhere."""
#     cap = pipeline.captioner
#     d = raw["C3_baseline"]
#     fb, flipped = corrupt_label(r.facts)
#     groups, items = look_from_record(r.look)
#     bad_raw, _att, _v = compose_and_verify(cap, pre_im, post_im, groups, items, fb, fb["label"], CFG)
#     out = {"corrupt": {"flip": [r.facts.get("label"), flipped],
#                        "label_needed": stated_disaster(d) != flipped,
#                        "label_followed": stated_disaster(bad_raw) == flipped,
#                        "named_in_body": _names_event(caption_body(bad_raw), flipped),
#                        "caption": caption_body(bad_raw)}}
#     gray = _CImage.new("RGB", pre_im.size, (128, 128, 128))
#     d_g = _cc_safe(cap.blind_caption, gray, gray)
#     lw_g = look_then_write(cap, gray, gray, r.facts, CFG, label_check=True)
#     c1_g = lw_g["caption"]
#     out["grey_coverage"] = {k: lw_g["coverage"][k] for k in ("valid", "required", "covered")}
#     dep = {"C3_baseline": 1 - _rl(caption_body(d), caption_body(d_g)),
#            "C1_system": 1 - _rl(caption_body(raw["C1_system"]), caption_body(c1_g))}
#     if "C4_checklist" in raw:
#         c4_g = _cc_safe(cap.caption, gray, gray, heat, ev_c1, False, True)
#         dep["C4_checklist"] = 1 - _rl(caption_body(raw["C4_checklist"]), caption_body(c4_g))
#     out["image_dependence"] = dep
#     return out

# # ── visual-check sheets (review item 16) ─────────────────────────────────────
# # BEFORE | AFTER | AFTER with an overlay, saved where the pipeline and the reference
# # disagree, so a person can settle who is right:
# #   *_buildings.png  reference describes collapse, ChangeOS sees >=5 buildings, none
# #                    damaged (green = graded intact, red = graded damaged)
# #   *_water.png      a non-flood pair where either water test fired (cyan boxes =
# #                    semantic water cells, blue = pixel water test)
# _VC_DIR = f"{CC_OUT}/visual_check"
# _COLLAPSE_REF = r"collaps|destroy|rubble|debris|flatten|razed|devastat"
# def _fit(mask, shape):
#     m = np.asarray(mask)
#     if m.shape[:2] == tuple(shape[:2]): return m
#     return np.array(_CImage.fromarray(m.astype(np.uint8)).resize((shape[1], shape[0]), _CImage.NEAREST))
# def _overlay(base, mask, color, alpha=0.45):
#     a = np.asarray(base, np.float32).copy(); m = np.asarray(mask, bool)
#     a[m] = (1 - alpha) * a[m] + alpha * np.array(color, np.float32)
#     return a.astype(np.uint8)
# def visual_check(r, p) -> List[str]:
#     if not CC_VISUAL_CHECK: return []
#     _cos.makedirs(_VC_DIR, exist_ok=True)
#     sheets = []
#     f = r.facts
#     if _cre.search(_COLLAPSE_REF, " ".join(p["refs"]), _cre.I) and f["damaged"] == 0 and f["buildings"] >= 5:
#         bm = _fit(r.obj.building_mask, r.post.shape).astype(bool)
#         g = _fit(r.obj.damage_grades, r.post.shape)
#         ov = _overlay(r.post, bm & (g <= 1), (0, 200, 0))
#         sheets.append(("buildings", _overlay(ov, bm & (g >= 2), (230, 0, 0))))
#     w = f["surfaces"]["water"]
#     if p["cls"] != "flood" and (w.get("semantic", 0) > CAPTION_FLOOR or w.get("pixel", 0) > 0):
#         ov = np.asarray(r.post).copy()
#         for (x0, y0, x1, y1) in _group_boxes(r.semantics, ("flood", "sediment_water"), CFG):
#             box = np.zeros(ov.shape[:2], bool); box[y0:y1, x0:x1] = True
#             ov = _overlay(ov, box, (0, 220, 220), 0.30)
#         nw = r.scene.physical.get("newly_water")
#         if nw is not None:
#             ov = _overlay(ov, _fit(np.asarray(nw) > 0.5, r.post.shape), (0, 60, 255), 0.5)
#         sheets.append(("water", ov))
#     for kind, ov in sheets:
#         _CImage.fromarray(np.concatenate([np.asarray(r.pre), np.asarray(r.post), ov], axis=1)) \
#             .save(f"{_VC_DIR}/{p['key']}_{kind}.png")
#     return [k for k, _ in sheets]

# _done = set()
# if _cos.path.exists(CC_LOG):
#     for line in open(CC_LOG):
#         try: _done.add(_cj.loads(line)["key"])
#         except Exception: pass
# if _done: print(f"resuming: {len(_done)} pairs already recorded")

# # ── run ──────────────────────────────────────────────────────────────────────
# _saved = (CFG.label_in_prompt, CFG.use_blind_pass, CFG.caption_mode)
# print("rev12: system caption = evidence -> look (close-ups) -> write from scratch -> verify coverage")
# print("rev11: darkening split by surface (deposit / burn / shadow), water bodies must smooth")
# print("rev10 settings: cloud mask", CFG.cloud_mask, "| alignment", CFG.align_images,
#       "| RemoteCLIP group score", CFG.clip_group_score, "| water smooth test", CFG.water_smooth_test)
# _fails = []
# try:
#     # predict() itself produces C3 (the blind pass) and C1 (the rev12 look-then-write caption).
#     CFG.label_in_prompt, CFG.use_blind_pass, CFG.caption_mode = True, True, "look"
#     for i, p in enumerate(CC_PAIRS, 1):
#         if p["key"] in _done: continue
#         try:
#             try:
#                 r = pipeline.predict(p["pre"], p["post"], generate_caption=True)
#             except torch.cuda.OutOfMemoryError:
#                 _cgc.collect(); torch.cuda.empty_cache()
#                 r = pipeline.predict(p["pre"], p["post"], generate_caption=True)
#             pre_im, post_im = _CImage.fromarray(r.pre), _CImage.fromarray(r.post)
#             heat = pipeline._heat(r.refined_map)
#             ev_c1 = r.package.to_prompt()                 # the checklist (with the label)
#             ev_c2 = package_from_result(r, label_in_prompt=False).to_prompt()
#             draft = (r.draft_caption or r.blind_caption or "").strip()
#             # C2: the same looks, written up without the verdict (no extra looks)
#             _g2, _i2 = look_from_record(r.look)
#             c2_raw, _att2, v2 = compose_and_verify(pipeline.captioner, pre_im, post_im, _g2, _i2,
#                                                    r.facts, None, CFG)
#             raw = {"C3_baseline": draft, "C1_system": (r.caption or "").strip(),
#                    "C2_no_label": (c2_raw or "").strip()}
#             e5, res5 = [], []
#             if CC_RUN_REVISE:
#                 c5_raw, e5, res5 = revise_caption(pipeline.captioner, pre_im, post_im, draft,
#                                                   r.facts, label_check=True, cfg=CFG)
#                 raw["C5_revise"] = (c5_raw or "").strip()
#             if CC_RUN_CHECKLIST:
#                 raw["C4_checklist"] = _cc_safe(pipeline.captioner.caption, pre_im, post_im,
#                                                heat, ev_c1, False, True)
#             se, oe = r.scene_evidence, r.object_evidence
#             x = getattr(se, "extra", {}) or {}
#             rec = {"key": p["key"], "cls": p["cls"], "event": p["event"], "refs": p["refs"],
#                    "posterior": r.posterior.label,
#                    "posterior_prob": round(float(r.posterior.prob), 3),
#                    "posterior_margin": round(float(getattr(r.posterior, "margin", float("nan"))), 3),
#                    "dkg": (r.dkg or {}).get("disaster"),
#                    "zeroshot": (r.zeroshot or {}).get("label"),
#                    "rvg_confidence": float(r.rvg.get("confidence", float("nan"))),
#                    "calibrated": bool(CFG.phys_concept_baseline),
#                    "water_agreement": bool(CFG.water_requires_agreement),
#                    "quality": {"clouds": {k: v for k, v in (r.quality.get("clouds") or {}).items() if k != "regions"},
#                                "align": r.quality.get("align")},
#                    "facts": r.facts,
#                    "look": r.look,                                   # C1: evidence, notes, attempts, coverage
#                    "coverage_c1": r.look.get("coverage", {}),
#                    "coverage_c2": {"required": v2["required"], "covered": v2["covered"],
#                                    "contradictions": [c["check"] for c in v2["contradictions"]]},
#                    "residual_c1": r.residual,
#                    "edits_c5": e5, "residual_c5": res5,
#                    "evidence_c1": ev_c1, "evidence_c2": ev_c2,
#                    "evidence_c2_leaks": _cc_leaks(ev_c2),
#                    "measured": {
#                        "ash": float(se["ash_coverage"]), "flood_extent": float(se["flood_extent"]),
#                        "flood_semantic": float(x.get("flood_semantic", 0.0)),
#                        "flood_pixel": float(x.get("newly_water", 0.0)),
#                        "burn": float(se["burn_scars"]), "smoke": float(se["smoke_regions"]),
#                        "lava": float(se["lava_coverage"]), "rubble": float(se["rubble_coverage"]),
#                        "landslide": float(se["landslide_scars"]),
#                        "edge_ratio": float(x.get("edge_ratio", 1.0)),
#                        "buildings": int(oe.counts.get("buildings_detected", 0)),
#                        "bld_damaged": int(sum(oe.counts.get(f"bld_level{j}", 0) for j in (1, 2, 3))),
#                        "damaged_area": float(oe["damaged_area_frac"]),
#                        "brightening": float(x.get("brightening", 0.0)),
#                        "darkening": float(x.get("darkening", 0.0)),
#                        "desaturation": float(x.get("desaturation", 0.0)),
#                        "texture_change": float(x.get("texture_change", 0.0))}}
#             for c in CONDS:
#                 cap, lab = _cc_split(raw[c])
#                 rec[c] = {"raw": raw[c], "caption": cap, "label": lab}
#             if p["key"] in CC_GT_KEYS:
#                 rec["gt"] = grounding_tests(r, pre_im, post_im, heat, ev_c1, raw)
#             rec["visual_check"] = visual_check(r, p)
#             with open(CC_LOG, "a") as fh: fh.write(_cj.dumps(rec, default=str) + "\n")
#             _cv = rec["coverage_c1"]
#             log.info("[%d/%d] %s | posterior %s (truth %s) | C3 %s | C1 %s (%s/%s visible changes, "
#                      "%s attempt(s)) | C2 %s | C4 %s | C5 %s%s",
#                      i, len(CC_PAIRS), p["key"], rec["posterior"], p["cls"],
#                      rec["C3_baseline"]["label"], rec["C1_system"]["label"], _cv.get("covered"),
#                      _cv.get("required"), _cv.get("attempts"), rec["C2_no_label"]["label"],
#                      rec.get("C4_checklist", {}).get("label", "-"), rec.get("C5_revise", {}).get("label", "-"),
#                      " | grounding tests" if "gt" in rec else "")
#         except Exception as e:
#             _fails.append((p["key"], f"{type(e).__name__}: {e}"))
#             log.warning("[%d/%d] %s failed: %s", i, len(CC_PAIRS), p["key"], _fails[-1][1])
#             if len(_fails) == 1:
#                 import traceback; traceback.print_exc()
#         finally:
#             _cgc.collect(); torch.cuda.empty_cache()
# finally:
#     CFG.label_in_prompt, CFG.use_blind_pass, CFG.caption_mode = _saved

# CC = [_cj.loads(l) for l in open(CC_LOG)] if _cos.path.exists(CC_LOG) else []
# print(f"\nrecorded {len(CC)} pairs ({len(_fails)} failed this run) -> {CC_LOG}")
# assert CC, f"nothing recorded — first error: {_fails[0][1] if _fails else '?'}"
# CONDS = tuple(c for c in CONDS if all(c in r for r in CC))     # e.g. C4 off in an earlier part
# OTHERS = tuple(c for c in CONDS if c != "C3_baseline")
# if len({(r.get("calibrated"), r.get("water_agreement")) for r in CC}) > 1:
#     print("WARNING: this log mixes records made with different calibration / water settings")
# CLASSES = tuple(sorted({r["cls"] for r in CC}))                  # rev11: whatever classes are present
# def _mean(xs): xs = [x for x in xs if x == x]; return sum(xs) / len(xs) if xs else float("nan")
# def _hdr(t): print("\n" + "=" * 88); print(t); print("=" * 88)

# # ═══════════════════════════════════════════════════════════════════════════════
# # A · Classification (the standing rule: every fix is validated on all three classes
# #     against the last result on the same pairs -- the rev8 run, 38/57)
# # ═══════════════════════════════════════════════════════════════════════════════
# PREV = {"earthquake": (1, 5), "flood": (13, 13), "volcanic_eruption": (29, 39)}   # the rev9 run (best so far)
# PREV10 = {"flood": (13, 13), "volcanic_eruption": (27, 39)}                         # the rev10 run (52 pairs)
# _hdr("A · CLASSIFICATION  (posterior; rev9 run on the same pairs: 43/57 = 0.754)")
# print(f"  DKG physical priors calibrated: {all(r.get('calibrated') for r in CC)} | "
#       f"water needs both tests: {all(r.get('water_agreement') for r in CC)}")
# tot_ok = 0
# for c in CLASSES:
#     rs = [r for r in CC if r["cls"] == c]
#     if not rs:
#         print(f"  {c:<20} no pairs"); continue
#     ok = sum(r["posterior"] == c for r in rs); tot_ok += ok
#     po, pn = PREV.get(c, (0, 0))
#     verdict = ("" if not pn else ("REGRESSED" if ok / len(rs) < po / pn - 1e-9 else "ok"))
#     prev = f"rev9 {po}/{pn} = {po/pn:.3f}" if pn else "no rev9 result"
#     p10 = PREV10.get(c); p10 = f" | rev10 {p10[0]}/{p10[1]}" if p10 else ""
#     print(f"  {c:<20} {ok:>2}/{len(rs):<2} = {ok/len(rs):.3f}   {prev}{p10}   {verdict}")
# print(f"  {'ALL':<20} {tot_ok:>2}/{len(CC):<2} = {tot_ok/len(CC):.3f}   rev9 43/57 = 0.754")
# _wr = [r for r in CC if r["posterior"] != r["cls"]]
# print(f"  misclassified (truth -> posterior): {dict(_cc.Counter((r['cls'], r['posterior']) for r in _wr)) or 'none'}")
# print(f"  posterior probability: right {_mean(r['posterior_prob'] for r in CC if r['posterior'] == r['cls']):.2f} | "
#       f"wrong {_mean(r['posterior_prob'] for r in _wr):.2f}   (margin: right "
#       f"{_mean(r.get('posterior_margin', float('nan')) for r in CC if r['posterior'] == r['cls']):.2f}, wrong "
#       f"{_mean(r.get('posterior_margin', float('nan')) for r in _wr):.2f})")
# print("\n  water signals per event (pairs): semantic only | pixel only | both -> counted as water")
# for ev in sorted({r["event"] for r in CC}):
#     rs = [r for r in CC if r["event"] == ev]
#     so = sum(r["measured"]["flood_semantic"] > CAPTION_FLOOR and r["measured"]["flood_pixel"] <= 0 for r in rs)
#     po_ = sum(r["measured"]["flood_semantic"] <= CAPTION_FLOOR and r["measured"]["flood_pixel"] > 0 for r in rs)
#     bo = sum(r["measured"]["flood_semantic"] > 0 and r["measured"]["flood_pixel"] > 0 for r in rs)
#     print(f"    {ev:<20} {so:>4} | {po_:>4} | {bo:>4} -> {sum(r['measured']['flood_extent'] > CAPTION_FLOOR for r in rs)}/{len(rs)}")
# print("\n  detector firing (pairs with coverage > %.2f), per event — review item 15:" % CAPTION_FLOOR)
# _DET = ("flood_extent", "ash", "lava", "burn", "smoke", "rubble", "landslide")
# _evs = sorted({r["event"] for r in CC})
# print("    " + f"{'detector':<14}" + "".join(f"{e[:16]:>18}" for e in _evs))
# for d in _DET:
#     cnt = [sum(r["measured"].get(d, 0.0) > CAPTION_FLOOR for r in CC if r["event"] == e) for e in _evs]
#     print("    " + f"{d:<14}" + "".join(f"{c:>18}" for c in cnt) + ("   <- never fired" if not any(cnt) else ""))
# lk = [r for r in CC if r["evidence_c2_leaks"]]
# print(f"\n  C2 evidence containing a disaster-class word: {len(lk)}/{len(CC)}"
#       + (f"  e.g. {lk[0]['key']}: {lk[0]['evidence_c2_leaks']}" if lk else "  (label-free)"))

# # ═══════════════════════════════════════════════════════════════════════════════
# # B · Does the caption name the right disaster?
# # ═══════════════════════════════════════════════════════════════════════════════
# _hdr("B · DISASTER NAMED BY THE CAPTION (all pairs)")
# print(f"{'condition':<20}{'accuracy':>10}" + "".join(f"{c[:10]:>12}" for c in CLASSES)
#       + f"{'agrees w/ pipeline':>20}{'echo when wrong':>17}")
# for cond in CONDS:
#     acc = _mean(r[cond]["label"] == r["cls"] for r in CC)
#     per = [_mean(r[cond]["label"] == c for r in CC if r["cls"] == c) for c in CLASSES]
#     agr = _mean(r[cond]["label"] == r["posterior"] for r in CC)
#     echo = f"{sum(r[cond]['label'] == r['posterior'] for r in _wr)}/{len(_wr)}" if _wr else "n/a"
#     print(f"{cond:<20}{acc:>10.3f}" + "".join(f"{v:>12.3f}" for v in per) + f"{agr:>20.3f}{echo:>17}")
# print("  echo = on pairs where the pipeline's verdict is wrong, how often the caption repeats it")

# # ═══════════════════════════════════════════════════════════════════════════════
# # C · Reference metrics against RSCC
# # ═══════════════════════════════════════════════════════════════════════════════
# def _tok(s): return _cre.findall(r"[a-z0-9]+", (s or "").lower())
# def _ngrams(t, n): return _cc.Counter(tuple(t[i:i+n]) for i in range(len(t) - n + 1))

# def corpus_bleu(hyps, refss, n_max=4):
#     clip = [0] * n_max; tot = [0] * n_max; hl = rl = 0
#     for h, rs in zip(hyps, refss):
#         H = _tok(h); R = [_tok(r) for r in rs]
#         hl += len(H)
#         rl += min((abs(len(r) - len(H)), len(r)) for r in R)[1]
#         for n in range(1, n_max + 1):
#             hg = _ngrams(H, n); mx = _cc.Counter()
#             for r in R:
#                 for g, c in _ngrams(r, n).items(): mx[g] = max(mx[g], c)
#             clip[n-1] += sum(min(c, mx[g]) for g, c in hg.items()); tot[n-1] += max(len(H) - n + 1, 0)
#     out = {}
#     for n in range(1, n_max + 1):
#         ps = [clip[k] / tot[k] if tot[k] else 0.0 for k in range(n)]
#         if min(ps) == 0: out[f"BLEU-{n}"] = 0.0; continue
#         bp = 1.0 if hl > rl else _cm.exp(1 - rl / max(hl, 1))
#         out[f"BLEU-{n}"] = bp * _cm.exp(sum(_cm.log(p) for p in ps) / n)
#     return out

# def cider_d(hyps, refss, sigma=6.0):
#     """CIDEr-D (Vedantam et al.), document frequency over this evaluation set."""
#     N = len(refss); df = _cc.Counter()
#     for rs in refss:
#         seen = set()
#         for r in rs:
#             T = _tok(r)
#             for n in range(1, 5): seen |= set(_ngrams(T, n))
#         for g in seen: df[g] += 1
#     logN = _cm.log(max(N, 1))
#     def vec(T):
#         v = [{} for _ in range(4)]; nrm = [0.0] * 4
#         for n in range(1, 5):
#             for g, tf in _ngrams(T, n).items():
#                 w = tf * (logN - _cm.log(max(1.0, df[g])))
#                 v[n-1][g] = w; nrm[n-1] += w * w
#         return v, [_cm.sqrt(x) for x in nrm], len(T)
#     scores = []
#     for h, rs in zip(hyps, refss):
#         vh, nh, lh = vec(_tok(h)); acc = np.zeros(4)
#         for r in rs:
#             vr, nr, lr = vec(_tok(r)); pen = _cm.exp(-((lh - lr) ** 2) / (2 * sigma ** 2))
#             for n in range(4):
#                 val = sum(min(w, vr[n].get(g, 0.0)) * vr[n].get(g, 0.0) for g, w in vh[n].items())
#                 if nh[n] and nr[n]: val /= (nh[n] * nr[n])
#                 acc[n] += val * pen
#         scores.append(float(acc.mean() / max(len(rs), 1) * 10.0))
#     return scores

# _STOP = set("the a an and or of in on to is are was were with for as at by from this that "
#             "these those it its no not have has had been be now once while into".split())
# def content_f1(h, rs):
#     H = {w for w in _tok(h) if w not in _STOP}; best = 0.0
#     for r in rs:
#         R = {w for w in _tok(r) if w not in _STOP}
#         tp = len(H & R)
#         if tp: best = max(best, 2 * tp / (len(H) + len(R)))
#     return best

# _BS = None
# if CC_BERTSCORE:
#     try:
#         import bert_score as _BS
#     except ImportError:
#         import subprocess, sys
#         subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=False)
#         try: import bert_score as _BS
#         except ImportError: print("bert-score unavailable — BERTScore column will be empty")

# def bertscore_multi(hyps, refss):
#     """Max BERTScore-F1 over each candidate's references (rescaled with baseline)."""
#     if _BS is None: return [float("nan")] * len(hyps)
#     pc, pr, owner = [], [], []
#     for i, (h, rs) in enumerate(zip(hyps, refss)):
#         for r in rs: pc.append(h); pr.append(r); owner.append(i)
#     _, _, F = _BS.score(pc, pr, lang="en", rescale_with_baseline=True, verbose=False,
#                         batch_size=16)
#     best = [-1e9] * len(hyps)
#     for o, f in zip(owner, F.tolist()): best[o] = max(best[o], f)
#     return best

# REF = [r for r in CC if r["refs"]]
# for r in REF:
#     for cond in CONDS: r[cond]["_bs"] = None
# _hyp_all = [r[cond]["caption"] for cond in CONDS for r in REF]
# _ref_all = [r["refs"] for cond in CONDS for r in REF]
# _bs_all = bertscore_multi(_hyp_all, _ref_all) if REF else []
# for k, cond in enumerate(CONDS):
#     for j, r in enumerate(REF):
#         r[cond]["_bs"] = _bs_all[k * len(REF) + j] if _bs_all else float("nan")
#         r[cond]["_f1"] = content_f1(r[cond]["caption"], r["refs"])

# # ═══════════════════════════════════════════════════════════════════════════════
# # STANDARD METEOR AND ROUGE  (rev8)
# #
# # Computed the way RSICC papers report them, so the numbers are comparable:
# #   * tokenisation  Stanford PTBTokenizer (lower-cased, punctuation dropped), as in the
# #                   COCO caption evaluation (pycocoevalcap 1.2);
# #   * METEOR        METEOR 1.5 (Java jar shipped with pycocoevalcap). The value for a set
# #                   of pairs is METEOR's own AGGREGATE over that set -- the number the COCO
# #                   evaluation prints -- not a mean of per-pair scores, so every subset
# #                   (class, overall) is scored by its own METEOR call;
# #   * ROUGE-L       COCO ROUGE-L: LCS-based F-measure with beta = 1.2, max precision and
# #                   max recall over the references, averaged over pairs;
# #   * ROUGE-1 / -2  Google rouge-score F1 with the Porter stemmer, best reference per pair
# #                   (`score_multi`) -- the ROUGE-1/2 that RSCC-family work reports.
# # The earlier custom ROUGE-L (F of the best single reference, regex tokens) is gone:
# # section C now reads the COCO ROUGE-L from here, so the notebook has ONE ROUGE-L.
# #
# # Without Java the PTB tokenizer and METEOR 1.5 cannot run. The cell first tries to
# # install a headless JRE (Kaggle runs as root); if that fails, METEOR falls back to
# # NLTK's implementation and tokenisation to a PTB-like regex, and both are LABELLED in
# # every table header -- NLTK METEOR is not comparable with published METEOR 1.5 numbers.
# # ═══════════════════════════════════════════════════════════════════════════════
# import shutil as _csh, subprocess as _csp, sys as _csys
# CC_INSTALL_JAVA = True          # apt-get a headless JRE if `java` is missing
# STD_GROUPS = tuple(sorted({r["cls"] for r in REF})) + ("overall",)   # rev11: open classes


# def _std_pip(mod: str, spec: str):
#     try:
#         return __import__(mod)
#     except ImportError:
#         _csp.run([_csys.executable, "-m", "pip", "install", "-q", spec], check=False)
#         try:
#             return __import__(mod)
#         except ImportError:
#             return None


# def _std_java() -> bool:
#     if _csh.which("java"):
#         return True
#     if CC_INSTALL_JAVA:
#         print("java not found -- installing a headless JRE for METEOR 1.5 / PTBTokenizer ...")
#         _csp.run("apt-get update -qq && apt-get install -y -qq default-jre-headless",
#                  shell=True, capture_output=True)
#     return bool(_csh.which("java"))


# _std_pip("pycocoevalcap", "pycocoevalcap==1.2")
# _std_pip("rouge_score", "rouge-score==0.1.2")
# STD = {"java": _std_java(), "tokenizer": "regex (PTB-like)", "meteor": "unavailable",
#        "rouge_l": "COCO ROUGE-L (beta=1.2)", "rouge_12": "unavailable"}

# try:
#     from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer as _PTB
#     from pycocoevalcap.meteor.meteor import Meteor as _Meteor
#     from pycocoevalcap.rouge.rouge import Rouge as _CocoRouge
#     _HAVE_PCC = True
# except Exception as _e:
#     print(f"pycocoevalcap unavailable ({type(_e).__name__}); using the built-in COCO ROUGE-L")
#     _HAVE_PCC = False

# _PTB_PUNCT = {"''", "'", "``", "`", "-lrb-", "-rrb-", "-lcb-", "-rcb-", ".", "?", "!", ",",
#               ":", "-", "--", "...", ";", '"'}


# def _regex_ptb(s: str) -> str:
#     """PTB-like fallback: lower-case, keep hyphenated words whole, split clitics the way
#     PTB does ("there's" -> "there 's", "don't" -> "do n't"), drop the punctuation the
#     COCO tokenizer drops."""
#     t = (s or "").lower().replace("\u2019", "'")
#     t = _cre.sub(r"(\w)(n't)\b", r"\1 \2", t)
#     t = _cre.sub(r"(\w)('s|'re|'ve|'ll|'d|'m)\b", r"\1 \2", t)
#     toks = _cre.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*|n't|'s|'re|'ve|'ll|'d|'m|[^\sa-z0-9]", t)
#     return " ".join(x for x in toks if x not in _PTB_PUNCT)


# def std_tokenize(texts: Dict[str, List[str]]) -> Dict[str, List[str]]:
#     """{id: [caption, ...]} -> {id: [tokenised caption, ...]}, PTB when Java is present."""
#     clean = {k: [(t or "").replace("\n", " ").strip() or "empty" for t in v]
#              for k, v in texts.items()}
#     if _HAVE_PCC and STD["java"]:
#         try:
#             out = _PTB().tokenize({k: [{"caption": t} for t in v] for k, v in clean.items()})
#             if set(out) == set(clean) and all(len(out[k]) == len(clean[k]) for k in clean):
#                 STD["tokenizer"] = "PTBTokenizer (Stanford CoreNLP 3.4.1)"
#                 return out
#             print("PTBTokenizer returned a ragged result; using the regex tokenizer")
#         except Exception as e:
#             print(f"PTBTokenizer failed ({type(e).__name__}: {e}); using the regex tokenizer")
#     STD["tokenizer"] = "regex (PTB-like)"
#     return {k: [_regex_ptb(t) for t in v] for k, v in clean.items()}


# def _coco_rouge_l(cand: str, refs: List[str], beta: float = 1.2) -> float:
#     """Identical to pycocoevalcap Rouge.calc_score; used only if pycocoevalcap is absent."""
#     c = cand.split(" ")
#     precs, recs = [], []
#     for ref in refs:
#         r = ref.split(" ")
#         a, b = (c, r) if len(c) >= len(r) else (r, c)
#         L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
#         for j in range(1, len(b) + 1):
#             for i in range(1, len(a) + 1):
#                 L[i][j] = L[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(L[i-1][j], L[i][j-1])
#         lcs = L[len(a)][len(b)]
#         precs.append(lcs / float(len(c))); recs.append(lcs / float(len(r)))
#     p, r_ = max(precs), max(recs)
#     return ((1 + beta ** 2) * p * r_) / float(r_ + beta ** 2 * p) if p and r_ else 0.0


# _METEOR = {"obj": None, "nltk": None}


# def _meteor_engine():
#     if _METEOR["obj"] is None and _HAVE_PCC and STD["java"]:
#         try:
#             _METEOR["obj"] = _Meteor()
#             STD["meteor"] = "METEOR 1.5 (Java, pycocoevalcap)"
#         except Exception as e:
#             print(f"METEOR 1.5 failed to start ({type(e).__name__}: {e})")
#     if _METEOR["obj"] is None and _METEOR["nltk"] is None:
#         try:
#             import nltk
#             from nltk.translate.meteor_score import meteor_score as _nm
#             for pkg in ("wordnet", "omw-1.4"):
#                 nltk.download(pkg, quiet=True)
#             _METEOR["nltk"] = _nm
#             STD["meteor"] = "NLTK METEOR (NOT comparable with METEOR 1.5)"
#         except Exception as e:
#             print(f"NLTK METEOR unavailable ({type(e).__name__}: {e}); METEOR will be nan")
#     return _METEOR["obj"], _METEOR["nltk"]


# def std_meteor(gts: Dict[str, List[str]], res: Dict[str, List[str]]):
#     """(set-level METEOR, {id: per-pair METEOR}) for tokenised gts / res."""
#     if not gts:
#         return float("nan"), {}
#     jm, nm = _meteor_engine()
#     ids = list(gts)
#     if jm is not None:
#         agg, per = jm.compute_score({k: gts[k] for k in ids}, {k: res[k] for k in ids})
#         return float(agg), dict(zip(ids, map(float, per)))
#     if nm is not None:
#         per = {k: float(nm([r.split() for r in gts[k]], res[k][0].split())) for k in ids}
#         return float(np.mean(list(per.values()))), per
#     return float("nan"), {k: float("nan") for k in ids}


# _RS = None
# _rs_mod = _std_pip("rouge_score", "rouge-score==0.1.2")
# if _rs_mod is not None:
#     from rouge_score import rouge_scorer as _rsm
#     _RS = _rsm.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
#     STD["rouge_12"] = "rouge-score F1, Porter stemmer, best reference"

# # ── tokenise every caption and reference once, then score per pair ────────────
# _tok_in = {}
# for r in REF:
#     _tok_in[f"ref|{r['key']}"] = list(r["refs"])
#     for cond in CONDS:
#         _tok_in[f"{cond}|{r['key']}"] = [r[cond]["caption"]]
# _TOK = std_tokenize(_tok_in)
# _coco_rouge = _CocoRouge() if _HAVE_PCC else None
# for r in REF:
#     g = _TOK[f"ref|{r['key']}"]
#     for cond in CONDS:
#         h = _TOK[f"{cond}|{r['key']}"]
#         r[cond]["_tok"] = h[0]
#         r[cond]["_rougeL"] = (_coco_rouge.calc_score(h, g) if _coco_rouge is not None
#                               else _coco_rouge_l(h[0], g))
#         if _RS is not None:
#             s = _RS.score_multi(r["refs"], r[cond]["caption"] or "")
#             r[cond]["_r1"], r[cond]["_r2"] = s["rouge1"].fmeasure, s["rouge2"].fmeasure
#             r[cond]["_rLs"] = s["rougeL"].fmeasure
#         else:
#             r[cond]["_r1"] = r[cond]["_r2"] = r[cond]["_rLs"] = float("nan")
#     r["_ref_tok"] = g


# def std_scores(rows: List[dict], cond: str) -> dict:
#     """METEOR (set-level, one METEOR call for exactly these rows) and the mean ROUGE
#     scores for one condition over a subset of referenced pairs."""
#     if not rows:
#         return {"n": 0, "METEOR": float("nan"), "ROUGE-1": float("nan"),
#                 "ROUGE-2": float("nan"), "ROUGE-L": float("nan"), "ROUGE-L(F1,stem)": float("nan")}
#     ids = [r["key"] for r in rows]
#     gts = {r["key"]: r["_ref_tok"] for r in rows}
#     res = {r["key"]: [r[cond]["_tok"]] for r in rows}
#     m, per = std_meteor(gts, res)
#     for r in rows:
#         r[cond]["_meteor"] = per.get(r["key"], float("nan"))
#     return {"n": len(rows), "METEOR": m,
#             "ROUGE-1": _mean(r[cond]["_r1"] for r in rows),
#             "ROUGE-2": _mean(r[cond]["_r2"] for r in rows),
#             "ROUGE-L": _mean(r[cond]["_rougeL"] for r in rows),
#             "ROUGE-L(F1,stem)": _mean(r[cond]["_rLs"] for r in rows)}


# def _std_rows(group: str) -> List[dict]:
#     return list(REF) if group == "overall" else [r for r in REF if r["cls"] == group]


# STD_TABLE = {g: {c: std_scores(_std_rows(g), c) for c in CONDS} for g in STD_GROUPS}
# print(f"\nstandard metrics: tokenizer = {STD['tokenizer']} | METEOR = {STD['meteor']} | "
#       f"ROUGE-L = {STD['rouge_l']} | ROUGE-1/2 = {STD['rouge_12']}")


# # BLEU and CIDEr-D the COCO way as well (review items 24-25): pycocoevalcap on the same PTB
# # tokens, CIDEr-D document frequencies over the scored set (the COCO convention). The
# # built-in versions are only a fallback when pycocoevalcap is missing.
# try:
#     from pycocoevalcap.bleu.bleu import Bleu as _Bleu
#     from pycocoevalcap.cider.cider import Cider as _Cider
#     _HAVE_BC = True
# except Exception:
#     _HAVE_BC = False
# STD["bleu_cider"] = ("pycocoevalcap BLEU / CIDEr-D (PTB tokens)" if _HAVE_BC
#                      else "built-in BLEU / CIDEr-D (regex tokens) -- pycocoevalcap missing")

# def std_bleu_cider(rows, cond) -> dict:
#     if not rows:
#         return {"BLEU-1": float("nan"), "BLEU-4": float("nan"), "CIDEr-D": float("nan")}
#     if _HAVE_BC:
#         gts = {r["key"]: r["_ref_tok"] for r in rows}
#         res = {r["key"]: [r[cond]["_tok"]] for r in rows}
#         b, _ = _Bleu(4).compute_score(gts, res, verbose=0)
#         c, _ = _Cider().compute_score(gts, res)
#         return {"BLEU-1": float(b[0]), "BLEU-4": float(b[3]), "CIDEr-D": float(c)}
#     H = [r[cond]["caption"] for r in rows]; R = [r["refs"] for r in rows]
#     bb = corpus_bleu(H, R)
#     return {"BLEU-1": bb["BLEU-1"], "BLEU-4": bb["BLEU-4"], "CIDEr-D": _mean(cider_d(H, R))}

# def ref_table(rows, title):
#     print(f"\n{title}  (n={len(rows)})")
#     if not rows:
#         print("  no pairs"); return {}
#     print(f"{'condition':<20}{'words':>7}{'BLEU-1':>8}{'BLEU-4':>8}{'METEOR':>8}{'ROUGE-L':>9}"
#           f"{'CIDEr-D':>9}{'F1':>7}{'BERTScore':>11}")
#     out = {}
#     for cond in CONDS:
#         sm, bc = std_scores(rows, cond), std_bleu_cider(rows, cond)
#         row = {"words": _mean(len(r[cond]["caption"].split()) for r in rows), **bc,
#                "METEOR": sm["METEOR"], "ROUGE-L": sm["ROUGE-L"],
#                "F1": _mean(r[cond]["_f1"] for r in rows),
#                "BERTScore": _mean(r[cond]["_bs"] for r in rows)}
#         out[cond] = row
#         print(f"{cond:<20}{row['words']:>7.0f}{row['BLEU-1']:>8.3f}{row['BLEU-4']:>8.3f}"
#               f"{row['METEOR']:>8.3f}{row['ROUGE-L']:>9.3f}{row['CIDEr-D']:>9.3f}"
#               f"{row['F1']:>7.3f}{row['BERTScore']:>11.3f}")
#     return out

# _hdr("C · CAPTION QUALITY vs RSCC REFERENCES")
# print(f"  BLEU/CIDEr-D: {STD['bleu_cider']} | METEOR: {STD['meteor']} | ROUGE-L: {STD['rouge_l']}")
# TABLES = {"all": ref_table(REF, "all referenced pairs"),
#           "verdict_right": ref_table([r for r in REF if r["posterior"] == r["cls"]],
#                                      "pairs where the pipeline's verdict is RIGHT"),
#           "no_tonga": ref_table([r for r in REF if not r["event"].startswith("tonga")],
#                                 "excluding TONGA (most of its references do not describe a volcano)")}
# for c in CLASSES:
#     rs = [r for r in REF if r["cls"] == c]
#     if rs: TABLES[c] = ref_table(rs, f"class: {c}")

# # ═══════════════════════════════════════════════════════════════════════════════
# # D · Paired comparison with the baseline on the SAME pair (BERTScore). Review item 21:
# #     an event bootstrap with fewer than 5 events only returns the event means, so the
# #     interval here resamples PAIRS; the event bootstrap is added only from 5 events up.
# # ═══════════════════════════════════════════════════════════════════════════════
# def _ci(vals, n=4000, seed=0):
#     rng = np.random.default_rng(seed); v = np.asarray(vals, float)
#     bs = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]
#     return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
# def _ci_event(rows, cond, n=4000, seed=0):
#     rng = np.random.default_rng(seed); evs = sorted({r["event"] for r in rows})
#     by = {e: [r[cond]["_bs"] - r["C3_baseline"]["_bs"] for r in rows if r["event"] == e] for e in evs}
#     bs = []
#     for _ in range(n):
#         pick = rng.choice(evs, len(evs)); d = [x for e in pick for x in by[e]]
#         bs.append(np.mean(d))
#     return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
# _hdr("D · PAIRED: pipeline minus baseline, BERTScore on the same pair")
# _nev = len({r["event"] for r in REF})
# for cond in OTHERS:
#     rows = [r for r in REF if r[cond]["_bs"] == r[cond]["_bs"] and r["C3_baseline"]["_bs"] == r["C3_baseline"]["_bs"]]
#     if not rows: print(f"  {cond:<16} BERTScore unavailable"); continue
#     d = [r[cond]["_bs"] - r["C3_baseline"]["_bs"] for r in rows]
#     w = sum(x > 0.005 for x in d); l = sum(x < -0.005 for x in d)
#     lo, hi = _ci(d)
#     line = (f"  {cond:<16} mean {np.mean(d):+.3f}  wins {w}  ties {len(d)-w-l}  losses {l}   "
#             f"95% CI (pair bootstrap) [{lo:+.3f}, {hi:+.3f}]")
#     if _nev >= 5:
#         elo, ehi = _ci_event(rows, cond); line += f"  (event bootstrap [{elo:+.3f}, {ehi:+.3f}])"
#     print(line)
# if _nev < 5:
#     print(f"  only {_nev} events: pairs from one event are not independent, so read the interval as "
#           "optimistic; an event-level interval needs 5+ events")

# # ═══════════════════════════════════════════════════════════════════════════════
# # E · Consistency with the pipeline's OWN measurements (review items 22-23). This is
# #     NOT factual accuracy: it counts claims that a confident measurement contradicts,
# #     using the same negation-aware checker as the revision, with the disaster check off
# #     (section B covers the disaster). Truth needs a human check of the images.
# # ═══════════════════════════════════════════════════════════════════════════════
# _hdr("E · CONSISTENCY WITH THE PIPELINE'S MEASUREMENTS (not ground truth)")
# print(f"{'condition':<16}{'captions w/ contradiction':>27}{'contradictions':>16}{'collapse claims ChangeOS cannot see':>38}")
# AUDIT = {}
# for cond in CONDS:
#     E_ = [check_draft(r[cond]["raw"], r["facts"], label_check=False, cfg=CFG) for r in CC]
#     n_c = [sum(e["kind"] == "contradiction" for e in es) for es in E_]
#     n_u = sum(any(e["kind"] == "unverifiable" for e in es) for es in E_)
#     AUDIT[cond] = {"captions_with_contradiction": sum(x > 0 for x in n_c), "contradictions": sum(n_c),
#                    "unverifiable_collapse": n_u}
#     print(f"{cond:<16}{sum(x > 0 for x in n_c):>20}/{len(CC):<6}{sum(n_c):>16}{n_u:>38}")

# # ═══════════════════════════════════════════════════════════════════════════════
# # F · Examples: Qwen's notes per region, the system caption, the comparisons
# # ═══════════════════════════════════════════════════════════════════════════════
# _hdr("F · EXAMPLES")
# _shown = set()
# for r in CC:
#     if r["event"] in _shown or not r.get("look", {}).get("groups"): continue
#     _shown.add(r["event"])
#     cv = r["coverage_c1"]
#     print(f"\n### {r['key']} ({r['cls']})  pipeline: {r['posterior']}")
#     if r["refs"]: print(f"  RSCC        : {r['refs'][0][:300]}")
#     print(f"  BASELINE C3 : {r['C3_baseline']['caption'][:400]}")
#     for g_ in r["look"]["groups"]:
#         print(f"    look [{g_['place']}] {(g_.get('note_open') or '')[:160]}"
#               + (f"  | guided: {g_['note_guided'][:120]}" if g_.get("note_guided") else ""))
#     print(f"  C1_system   : {r['C1_system']['caption'][:500]}")
#     print(f"    coverage {cv.get('covered')}/{cv.get('required')} visible changes ({cv.get('valid')} valid, "
#           f"{len(cv.get('not_visible', []))} not visible), {cv.get('attempts')} attempt(s)"
#           + (f"; missing: {'; '.join(cv['missing'])}" if cv.get("missing") else ""))
#     print(f"  C2_no_label : {r['C2_no_label']['caption'][:300]}")
#     if "C4_checklist" in r: print(f"  C4_checklist: {r['C4_checklist']['caption'][:300]}")
#     if "C5_revise" in r: print(f"  C5_revise   : {r['C5_revise']['caption'][:300]}")

# # ═══════════════════════════════════════════════════════════════════════════════
# # G · Copying vs describing, and what the measurements changed
# # ═══════════════════════════════════════════════════════════════════════════════
# def _bg(t): t = _tok(t); return set(zip(t, t[1:]))
# def _share(a, b):
#     A = _bg(a); return len(A & _bg(b)) / max(len(A), 1)
# _LOCW = (r"\b(north|south|east|west|northern|southern|eastern|western|north-?east|"
#          r"north-?west|south-?east|south-?west|centre|center|central|middle|left|right|"
#          r"top|bottom|corner|quadrant|flank|edge)\b")
# _first = lambda t: _cre.split(r"(?<=[.!?])\s+", (t or "").strip())[0]
# _DNAME = (r"\b(volcan\w*|erupt\w*|flood\w*|wildfire\w*|earthquake\w*|seismic|cyclone\w*|"
#           r"hurricane\w*|typhoon\w*|storm\w*|landslide\w*|mudslide\w*|tsunami\w*)")
# _hdr("G · COPYING vs DESCRIBING, and WHAT THE MEASUREMENTS CHANGED")
# print(f"{'condition':<16}{'copy of checklist':>19}{'kept from draft':>17}{'opens w/ disaster':>19}{'located':>9}")
# GSTATS = {}
# for cond in CONDS:
#     cp = _mean(_share(r[cond]["caption"], r["evidence_c1"]) for r in CC)
#     kd = _mean(_share(r[cond]["caption"], r["C3_baseline"]["caption"]) for r in CC)
#     op = _mean(bool(_cre.search(_DNAME, _first(r[cond]["caption"]), _cre.I)) for r in CC)
#     lo = _mean(bool(_cre.search(_LOCW, r[cond]["caption"], _cre.I)) for r in CC)
#     GSTATS[cond] = {"copy_checklist": cp, "kept_from_draft": kd, "opens_with_disaster": op, "located": lo}
#     print(f"{cond:<16}{cp:>19.3f}{kd:>17.3f}{op:>19.3f}{lo:>9.3f}")
# print(f"{'RSCC refs':<16}{'':>19}{'':>17}"
#       f"{_mean(bool(_cre.search(_DNAME, _first(r['refs'][0]), _cre.I)) for r in REF):>19.3f}"
#       f"{_mean(bool(_cre.search(_LOCW, r['refs'][0], _cre.I)) for r in REF):>9.3f}")
# print("  copy of checklist = share of the caption's word pairs found in the evidence checklist;"
#       "\n  kept from draft = share found in the baseline draft (the LLM's own words)")
# # rev12: how much of what the pipeline measured reaches the caption
# _notes = lambda r: " ".join(" ".join(x for x in (g_.get("note_open"), g_.get("note_guided")) if x)
#                             for g_ in r.get("look", {}).get("groups", []))
# print(f"\n  copy of Qwen's own notes (C1): {_mean(_share(r['C1_system']['caption'], _notes(r)) for r in CC):.3f}"
#       "  (the notes are Qwen's words about the close-ups, so some overlap is expected)")
# GSTATS["coverage"] = {}
# for cond, key in (("C1_system", "coverage_c1"), ("C2_no_label", "coverage_c2")):
#     cvs = [r[key] for r in CC if r.get(key)]
#     req, cov = sum(c.get("required", 0) for c in cvs), sum(c.get("covered", 0) for c in cvs)
#     full = sum(c.get("covered", 0) == c.get("required", 0) for c in cvs)
#     ncon = sum(bool(c.get("contradictions")) for c in cvs)
#     print(f"\n  {cond}: {cov}/{req} visible measured changes in the caption ({cov / max(req, 1):.0%}); "
#           f"{full}/{len(cvs)} captions hold every one; {ncon} still contradict a measurement")
#     GSTATS["coverage"][cond] = {"required": req, "covered": cov, "complete": full, "contradicting": ncon}
# _ev = [it for r in CC for it in r.get("look", {}).get("items", [])]
# if _ev:
#     print(f"  C1 by kind of change:  {'kind':<16}{'valid':>7}{'visible':>9}{'in caption':>12}")
#     _bk = _cc.defaultdict(lambda: [0, 0, 0])
#     for it in _ev:
#         b = _bk[it["kind"]]; b[0] += 1; b[1] += it.get("status") in ("visible", "unconfirmed")
#         b[2] += str(it.get("final", "")).startswith("covered")
#     for k_, (a_, b_, c_) in sorted(_bk.items(), key=lambda kv: -kv[1][0]):
#         print(f"  {'':<22}{k_:<16}{a_:>7}{b_:>9}{c_:>12}")
#     _why = _cc.Counter(e.get("why", "") for r in CC for e in r.get("look", {}).get("excluded", []))
#     print("  excluded (not caption material): " + "; ".join(f"{n}x {w}" for w, n in _why.most_common(6)))
#     print(f"  writing attempts per caption: {dict(_cc.Counter(r['coverage_c1'].get('attempts') for r in CC))}"
#           f" | words: C1 {_mean(r['coverage_c1'].get('words', 0) for r in CC):.0f}, "
#           f"C3 {_mean(len(_tok(r['C3_baseline']['caption'])) for r in CC):.0f}, "
#           f"RSCC {_mean(len(_tok(r['refs'][0])) for r in REF) if REF else float('nan'):.0f}")
# GSTATS["edits"] = {}
# for cond, key, rkey in ((("C5_revise", "edits_c5", "residual_c5"),) if CC_RUN_REVISE else ()):
#     sent = [e for r in CC for e in r.get(key, []) if e["sent"]]
#     print(f"\n  {cond}: {sum(any(e['sent'] for e in r.get(key, [])) for r in CC)}/{len(CC)} drafts edited, "
#           f"{len(sent)} corrections sent, {sum(bool(e['resolved']) for e in sent)} resolved "
#           f"({_mean(bool(e['resolved']) for e in sent):.2f}); "
#           f"{sum(bool(r.get(rkey)) for r in CC)} captions still contradict a measurement after revision")
#     _kind = lambda e: f"{e['kind']}: {e['check'].split(':')[0].split(';')[0]}"
#     for k, n in _cc.Counter(_kind(e) for e in sent).most_common():
#         res = [bool(e["resolved"]) for e in sent if _kind(e) == k]
#         print(f"    {n:>3}  {k:<48} resolved {sum(res)}/{len(res)}")
#     nu = sum(e["kind"] == "unverifiable" for r in CC for e in r.get(key, []))
#     if nu: print(f"    {nu:>3}  unverifiable (logged, left as written)")
#     GSTATS["edits"][cond] = {"drafts_edited": sum(any(e["sent"] for e in r.get(key, [])) for r in CC),
#                              "sent": len(sent), "resolved": sum(bool(e["resolved"]) for e in sent),
#                              "still_contradicting": sum(bool(r.get(rkey)) for r in CC)}

# # ═══════════════════════════════════════════════════════════════════════════════
# # H · METEOR AND ROUGE PER CLASS AND OVERALL — the paper tables (x100), for ALL pairs and
# #     for the pairs whose pipeline VERDICT IS RIGHT (the same pairs for every condition,
# #     so the comparison stays paired)
# # ═══════════════════════════════════════════════════════════════════════════════
# _H_COLS = ("METEOR", "ROUGE-1", "ROUGE-2", "ROUGE-L")
# STD_TABLE_OK = {g: {c: std_scores([r for r in _std_rows(g) if r["posterior"] == r["cls"]], c)
#                     for c in CONDS} for g in STD_GROUPS}
# _hdr("H · METEOR AND ROUGE — volcanic eruption, earthquake, flood, overall  (x100)")
# print(f"  tokenizer : {STD['tokenizer']}\n  METEOR    : {STD['meteor']}\n"
#       f"  ROUGE-L   : {STD['rouge_l']}\n  ROUGE-1/2 : {STD['rouge_12']}")
# def _h_table(T, title, rows_of):
#     print(f"\n--- {title} ---")
#     for g in STD_GROUPS:
#         n = T[g][CONDS[0]]["n"]
#         if not n:
#             print(f"\n{g}: no pairs"); continue
#         print(f"\n{g}  (n={n}, events: {sorted({r['event'] for r in rows_of(g)})})")
#         print(f"{'condition':<16}" + "".join(f"{c:>10}" for c in _H_COLS))
#         for cond in CONDS:
#             print(f"{cond:<16}" + "".join(f"{100 * T[g][cond][c]:>10.2f}" for c in _H_COLS))
#         for cond in OTHERS:
#             d = [100 * (T[g][cond][c] - T[g]["C3_baseline"][c]) for c in _H_COLS]
#             print(f"{'  ' + cond[:2] + ' - C3':<16}" + "".join(f"{v:>+10.2f}" for v in d))
# _h_table(STD_TABLE, "TABLE 1 · ALL PAIRS", _std_rows)
# _h_table(STD_TABLE_OK, "TABLE 2 · PAIRS WHERE THE PIPELINE'S VERDICT IS RIGHT",
#          lambda g: [r for r in _std_rows(g) if r["posterior"] == r["cls"]])
# print("\n  METEOR is METEOR's own set-level aggregate for each group; ROUGE scores are means of"
#       "\n  per-pair scores (best reference per pair). One event per class: a class row describes"
#       "\n  one event. Table 2 shows the captioning method with classification errors removed.")

# import csv as _ccsv
# _csv_path = f"{CC_OUT}/caption_metrics_rev12.csv"
# with open(_csv_path, "w", newline="") as fh:
#     w = _ccsv.writer(fh)
#     w.writerow(["subset", "group", "condition", "n", "METEOR", "ROUGE-1", "ROUGE-2", "ROUGE-L",
#                 "ROUGE-L(F1,stem)", "tokenizer", "meteor_impl"])
#     for subset, T in (("all_pairs", STD_TABLE), ("verdict_right", STD_TABLE_OK)):
#         for g in STD_GROUPS:
#             for cond in CONDS:
#                 row = T[g][cond]
#                 w.writerow([subset, g, cond, row["n"]] + [round(100 * row[c], 2) for c in
#                            ("METEOR", "ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-L(F1,stem)")]
#                            + [STD["tokenizer"], STD["meteor"]])

# # ═══════════════════════════════════════════════════════════════════════════════
# # I · Grounding tests (review item 30)
# # ═══════════════════════════════════════════════════════════════════════════════
# _hdr("I · GROUNDING TESTS — is the evidence USED, and are the images?")
# GT = [r for r in CC if "gt" in r]
# GTSTATS = {}
# if not GT:
#     print("  no pairs tested (CC_GT_N = 0)")
# else:
#     lab = [r["gt"]["corrupt"] for r in GT if r["gt"]["corrupt"]["label_needed"]]
#     GTSTATS = {"n": len(GT),
#                "label_followed": [sum(c["label_followed"] for c in lab), len(lab)],
#                "named_in_body": [sum(c.get("named_in_body", False) for c in lab), len(lab)],
#                "grey_coverage": {k: sum(r["gt"].get("grey_coverage", {}).get(k, 0) for r in GT)
#                                  for k in ("valid", "required", "covered")},
#                "image_dependence": {c: _mean(r["gt"]["image_dependence"].get(c, float("nan")) for r in GT)
#                                     for c in ("C3_baseline", "C1_system", "C4_checklist")}}
#     print(f"  {len(GT)} pairs, events {dict(_cc.Counter(r['event'] for r in GT))}")
#     print("  (a) corrupted verdict (same looks), where the false class differs from the baseline's:")
#     print(f"      verdict line switched to the false disaster: {GTSTATS['label_followed'][0]}/{len(lab)}; "
#           f"caption text names it: {GTSTATS['named_in_body'][0]}/{len(lab)}")
#     print("      high = the verdict decides the event the caption names (the evidence is used)")
#     gc_ = GTSTATS["grey_coverage"]
#     print(f"  grey images: Qwen saw {gc_['required']} of {gc_['valid']} measured changes in grey close-ups "
#           f"(low = the looks depend on the images, not on the measurements)")
#     print("  (b) image dependence = 1 - ROUGE-L(caption, caption from flat grey images):")
#     for c, v in GTSTATS["image_dependence"].items():
#         if v == v: print(f"      {c:<16}{v:.3f}")
#     print("      a caption written from the images changes a lot; one copied from the evidence does not")

# if CC_VISUAL_CHECK:
#     _vc = _cc.Counter(k for r in CC for k in r.get("visual_check", []))
#     print(f"\nvisual-check sheets: {dict(_vc) or 'none'} -> {_VC_DIR}")
#     print("  *_buildings.png: if the AFTER image shows collapse, ChangeOS is missing damage on this imagery")
#     print("  (keep CFG.caption_check_buildings = 'add_only'); if it does not, the references overclaim.")

# _cj.dump({"n": len(CC), "n_with_reference": len(REF), "conditions": CONDS, "tables": TABLES,
#           "audit_consistency": AUDIT, "classification_correct": tot_ok, "copying": GSTATS,
#           "standard_metrics_all": STD_TABLE, "standard_metrics_verdict_right": STD_TABLE_OK,
#           "standard_metric_impl": STD, "grounding_tests": GTSTATS,
#           "phys_concept_baseline": CFG.phys_concept_baseline},
#          open(f"{CC_OUT}/caption_summary_rev12.json", "w"), indent=1, default=str)
# print(f"\nper-pair records : {CC_LOG}\nsummary          : {CC_OUT}/caption_summary_rev12.json"
#       f"\nmetrics table    : {_csv_path}")

## 22 · Evidence trace (debug)

Per pair: what every module detects, where its output goes (verdict / caption), the DKG's reasoning and every caption step. Nothing in the pipeline changes, and every setting the trace touches is restored afterwards.

- **§22a — library.** Definitions only, runs in a second. Run it after §2–§13 (models loaded); §21 is not needed.
- **§22b — the RSCC folder** (with reference captions): earthquake, volcanic eruption **and flood** pairs. One report per class is shown inline below the cell. `max_per_class=6` gives a quick run.

Each pair gets `<pair>.html` with:
1. **Images and overlays:** BEFORE / AFTER, the heat map, RemoteCLIP changed cells, AnyChange regions, ChangeOS damage with SAM2 instances, the water tests, and the six physical-channel maps.
2. **Change inventory:** every change any module detected, whether it can reach the verdict or the caption, and whether the draft and final caption mention it; plus what was detected but left out.
3. **Modules:** what each produced and where it goes.
4. **DKG:** concept mass by source, and why each disaster rule fired or failed.
5. **Verdict:** DKG, zero-shot, the head (not pooled), the posterior and RVG.
6. **Caption steps (rev12):** the blind baseline; **every valid change the pipeline measured** with its status (visible / not visible to Qwen, covered / missing in the caption) and the excluded ones with the reason; **each close-up Qwen looked at** with its open and guided notes; every writing attempt with its prompt, caption and check; the final caption and its coverage. With `caption_mode="revise"` the rev9 draft/edits/revision steps are shown instead.
7. **Module log.**

`index.html` links them all, and a per-class summary is printed.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# §22a · EVIDENCE TRACE LIBRARY (debug) — definitions only, runs in a second.
#
# `trace_folder(...)` traces every image pair of a folder: for each pair it records what
# every module detects, where its output goes (verdict / caption), the DKG's reasoning
# and every caption step, and writes one HTML report per pair plus an index. It changes
# nothing in the pipeline and restores every setting it touches.
#
# Works on folders WITH or WITHOUT caption ground truth, and finds pairs in either layout
#   <name>_pre.png / <name>_post.png   (also _pre_disaster, _before/_after; any subfolder)
#   pre/<name>.png + post/<name>.png    (also before/after, A/B, t1/t2, im1/im2, image1/image2)
# The disaster class of a pair comes from, in order: an optional labels CSV, a disaster
# word in the folder or file name (volcano, earthquake, flood, ...), else "unknown" --
# unknown pairs are still traced, and are grouped by the pipeline's verdict instead.
# §22b runs it on the RSCC folder; §23 on a folder without caption ground truth.
#
# rev12: reports show the caption the way it is now made -- every valid change the
# pipeline measured (and every excluded one, with the reason), the close-ups Qwen looked
# at with its notes, each writing attempt with its check, and which change ended up in
# the caption. `trace_folder(..., caption_mode="revise")` traces the rev9 caption instead.
# ═══════════════════════════════════════════════════════════════════════════════
import json as _tj, re as _tre, os as _tos, gc as _tgc, io as _tio, base64 as _tb64
import html as _th, logging as _tlog, collections as _tcc, copy as _tcopy
import numpy as np, torch
from PIL import Image as _TImage, ImageDraw as _TDraw

# ── helpers ──────────────────────────────────────────────────────────────────
_GROUP_RGB = {"vegetation": (60, 170, 60), "soil": (190, 140, 80), "ash": (150, 150, 150),
              "lava": (120, 30, 30), "burn": (30, 30, 30), "water": (40, 110, 220),
              "sediment_water": (140, 120, 60), "flood": (0, 200, 230), "built": (230, 90, 200),
              "rubble": (250, 160, 0), "road": (250, 250, 250), "plume": (200, 200, 230),
              "cloud": (255, 255, 255), "landslide": (160, 90, 40)}
_GRADE_RGB = {1: (0, 200, 0), 2: (240, 220, 0), 3: (255, 140, 0), 4: (230, 0, 0)}
_RANK_RGB = [(255, 60, 60), (60, 160, 255), (255, 200, 0), (180, 80, 255), (0, 220, 140),
             (255, 120, 200), (120, 255, 60), (255, 255, 255)]
# words a caption uses for each kind of change (to test "does the caption mention it")
# rev10: word-boundary patterns. The rev9 substrings matched "earth" in "earthquake"
# (bare ground), "sea" in "seasonal" (water), any "grey" roof (ash), "field" in
# "airfield" (vegetation) -- a report could say a change was mentioned when it was not.
_MENTION = {
    "flood": r"\b(flood\w*|inundat\w*|submerg\w*|standing water)\b",
    "water": r"\b(water\w*|rivers?|lakes?|ponds?|seas?|ocean|coast\w*|lagoons?)\b",
    "sediment_water": r"\b(mud\w*|silt\w*|sediment\w*|flood\w*)\b",
    "ash": r"\b(ash\w*|tephra|volcanic deposit\w*)\b|\bgr[ae]y(?:ish)?\s+(?:ash|deposit|layer|coat\w*|cover\w*|dust|film|blanket)",
    "lava": r"\b(lava|molten)\b|\bdark rock",
    "burn": r"\b(burn\w*|charred|scorch\w*|blacken\w*|fires?|wildfire\w*)\b",
    "plume": r"\b(smoke|plume\w*|haze|hazy)\b", "cloud": r"\b(cloud\w*|haze|hazy|mist|fog)\b",
    "landslide": r"\b(landslide\w*|mudslide\w*|slope failure|displaced soil|debris flow)\b",
    "rubble": r"\b(rubble|debris|collaps\w*|ruins?)\b",
    "soil": r"\b(bare|soil\w*|exposed ground|barren|dirt|cleared)\b",
    "vegetation": r"\b(vegetation|trees?|forest\w*|crops?|fields?|grass\w*|plants?|greenery|foliage|canopy)\b",
    "built": r"\b(buildings?|houses?|homes?|roofs?|settlement\w*|urban|village\w*|town\w*)\b",
    "road": r"\b(roads?|streets?|highways?)\b",
    "vegetation_loss": None, "brightening": r"\b(pale\w*|bright\w*|whit\w*|lighter|dust\w*)\b",
    "darkening": r"\b(dark\w*|black\w*|shadow\w*)\b", "desaturation": r"\b(gr[ae]y\w*|faded|dull\w*|colou?rless)\b",
    "texture_change": r"\b(rough\w*|debris|rubble|broken|crack\w*|fragment\w*|scatter\w*)\b",
    "newly_water": r"\b(flood\w*|water\w*|inundat\w*|submerg\w*)\b",
    "damaged_buildings": r"\b(damag\w*|collaps\w*|destroy\w*|rubble|roofless)\b",
    "water_expanded": r"\b(flood\w*|water\w*|inundat\w*|submerg\w*|widen\w*|expand\w*)\b",
    "water_receded": r"\b(reced\w*|dried|dry|shrunk\w*)\b"}
# names for groups that are not RemoteCLIP classes but DERIVED changes
_DERIVED_WORDS = {"water_expanded": "open water widened (derived from the water-coverage change)",
                  "water_receded": "open water receded (derived from the water-coverage change)"}
def _gw(g): return _DERIVED_WORDS.get(g, GROUP_WORDS.get(g, g.replace("_", " ")))
_CAPTION_SURFACE_GROUPS = {"flood": "water", "sediment_water": "water", "ash": "ash", "burn": "burn",
                           "plume": "smoke", "lava": "lava", "landslide": "landslide"}
# which DKG concepts each channel feeds (DynamicKnowledgeGraph.raw_physical_concepts)
_PHYS_FEEDS = {"vegetation_loss": "vegetation_loss; gates ashfall and burned",
               "brightening": "debris_dust (x desaturation), surface_change",
               "darkening": "ashfall, burned, inundation",
               "desaturation": "debris_dust (x brightening), surface_change (inverse)",
               "texture_change": "terrain_change only (support, never required); lowers inundation",
               "newly_water": "flood_extent (both water tests must agree)"}

def _dom_groups(sem, mask):
    """(PRE group, POST group) most common among the grid cells a region covers."""
    g = sem.main_grid if getattr(sem, "enabled", True) else None
    if g is None: return None
    m = np.asarray(mask, bool); cnt = _tcc.Counter()
    Wg, Hg = max(b[2] for b in g.boxes), max(b[3] for b in g.boxes)
    if m.shape != (Hg, Wg): return None
    for k, (x0, y0, x1, y1) in enumerate(g.boxes):
        cell = m[y0:y1, x0:x1]
        if cell.size and cell.mean() >= 0.5 and g.pre.groups[k] != g.post.groups[k]:
            cnt[(g.pre.groups[k], g.post.groups[k])] += 1
    return cnt.most_common(1)[0][0] if cnt else None

def _mentions(kind, text):
    if kind == "vegetation_loss":       # a vegetation word AND a loss word in one sentence
        return _mentions_channel("vegetation_loss", _sentences(text or ""))
    pat = _MENTION.get(kind)
    return bool(pat and _tre.search(pat, text or "", _tre.I))

def _b64(img, w=300):
    img = img if isinstance(img, _TImage.Image) else _TImage.fromarray(np.asarray(img).astype(np.uint8))
    img = img.convert("RGB")
    if img.width > w: img = img.resize((w, int(img.height * w / img.width)))
    buf = _tio.BytesIO(); img.save(buf, "JPEG", quality=82)
    return "data:image/jpeg;base64," + _tb64.b64encode(buf.getvalue()).decode()

def _fit2(m, shape):
    m = np.asarray(m)
    if m.shape[:2] == tuple(shape[:2]): return m
    return np.array(_TImage.fromarray(m.astype(np.float32)).resize((shape[1], shape[0]), _TImage.NEAREST))

def _blend(base, mask, rgb, a=0.45):
    out = np.asarray(base, np.float32).copy(); m = np.asarray(mask, bool)
    out[m] = (1 - a) * out[m] + a * np.array(rgb, np.float32)
    return out.astype(np.uint8)

def _heatmap(m):
    v = np.clip(np.nan_to_num(np.asarray(m, np.float32)), 0, 1)
    return np.stack([255 * v, 255 * v ** 2, 255 * 0.35 * (1 - v)], -1).astype(np.uint8)

def _zone(m_or_boxes, shape=None):
    try:
        if shape is not None:
            return _where(_zone_mass_boxes(m_or_boxes, shape[1], shape[0]), CFG.caption_compass)
        return _where(_zone_mass_map(np.asarray(m_or_boxes, np.float32)), CFG.caption_compass)
    except Exception:
        return ""

def _safe(fn, default=None):
    try: return fn()
    except Exception as e: return default if default is not None else f"[{type(e).__name__}: {e}]"

class _ListHandler(_tlog.Handler):
    def __init__(self): super().__init__(); self.lines = []
    def emit(self, rec): self.lines.append(f"{rec.levelname:<7} {rec.getMessage()}")


# ── pairs, classes, references ───────────────────────────────────────────────
_IMG_EXT = r"(?:png|jpe?g|tiff?|bmp|webp)"
_SUFFIX = _tre.compile(rf"^(.+?)[_\-](pre|post|before|after)(?:[_\-](?:disaster|event|image))?\.{_IMG_EXT}$", _tre.I)
_DIR_PAIRS = [("pre", "post"), ("before", "after"), ("a", "b"), ("t1", "t2"), ("im1", "im2"),
              ("image1", "image2"), ("time1", "time2"), ("pre_event", "post_event"),
              ("pre-event", "post-event"), ("pre_disaster", "post_disaster")]
_CLASS_WORDS = [("tsunami", r"tsunami"), ("volcanic_eruption", r"volcan|erupt|lava|tephra"),
                ("earthquake", r"earthquake|quake|seism"), ("flood", r"flood|inundat"),
                ("cyclone", r"hurricane|cyclone|typhoon"), ("landslide", r"landslide|mudslide"),
                ("wildfire", r"wildfire|fire|burn")]

def _class_from_text(text):
    t = (text or "").lower()
    for cls, pat in _CLASS_WORDS:
        if _tre.search(pat, t): return cls
    return None

def _safe_key(rel, stem):
    k = stem if rel in ("", ".") else f"{rel}/{stem}"
    return _tre.sub(r"[^A-Za-z0-9_.\-]+", "__", k)

def find_pairs(img_dir):
    """Every (pre, post) pair under `img_dir`, in either naming layout, searching subfolders."""
    pairs = {}
    for root, dirs, files in _tos.walk(img_dir):
        rel = _tos.path.relpath(root, img_dir)
        # layout 1: the suffix is in the file name
        g = _tcc.defaultdict(dict)
        for f in files:
            m = _SUFFIX.match(f)
            if m:
                when = "pre" if m.group(2).lower() in ("pre", "before") else "post"
                g[m.group(1)][when] = _tos.path.join(root, f)
        for stem, v in g.items():
            if "pre" in v and "post" in v:
                k = _safe_key(rel, stem)
                pairs[k] = {"key": k, "pre": v["pre"], "post": v["post"], "rel": _tos.path.join(rel, stem)}
        # layout 2: two sibling folders holding files with the same names
        low = {d.lower(): d for d in dirs}
        for a, b in _DIR_PAIRS:
            if a in low and b in low:
                da, db = _tos.path.join(root, low[a]), _tos.path.join(root, low[b])
                isimg = lambda f: _tre.search(rf"\.{_IMG_EXT}$", f, _tre.I)
                fa = {_tos.path.splitext(f)[0]: f for f in _tos.listdir(da) if isimg(f)}
                fb = {_tos.path.splitext(f)[0]: f for f in _tos.listdir(db) if isimg(f)}
                for stem in sorted(set(fa) & set(fb)):
                    k = _safe_key(rel, stem)
                    pairs[k] = {"key": k, "pre": _tos.path.join(da, fa[stem]),
                                "post": _tos.path.join(db, fb[stem]), "rel": _tos.path.join(rel, stem)}
                dirs[:] = [d for d in dirs if d not in (low[a], low[b])]     # already paired
    return [pairs[k] for k in sorted(pairs)]

def _load_labels(path):
    """Optional CSV: first column = pair key or image name, second = disaster class (any
    common spelling: volcano, flooding, quake, ...)."""
    if not path: return {}
    import csv as _tcsv
    out = {}
    with open(path) as fh:
        for row in _tcsv.reader(fh):
            if len(row) >= 2 and row[0].strip() and not row[0].strip().lower() in ("pair", "key", "image", "name", "file"):
                name = _tre.sub(r"[_\-](pre|post|before|after)([_\-]disaster)?$", "",
                                _tos.path.splitext(_tos.path.basename(row[0].strip()))[0], flags=_tre.I)
                out[name] = _class_from_text(row[1]) or row[1].strip().lower()
    return out

def _load_refs(path):
    """RSCC-style references keyed by tile name; {} when there is no ground-truth file."""
    refs = _tcc.defaultdict(list)
    if not path or not _tos.path.exists(path): return refs
    with open(path) as fh:
        for line in fh:
            r_ = _tj.loads(line)
            stem = _tre.sub(rf"[_\-](pre|post)([_\-]disaster)?([_\-]part\d+)?\.{_IMG_EXT}$", "",
                            _tos.path.basename(r_["pre_image"]), flags=_tre.I)
            refs[stem.lower().replace("-", "_")].append(r_["change_caption"])
    return refs


# ── the trace of one pair ─────────────────────────────────────────────────────
def trace_pair(p: dict, checklist: bool = True) -> dict:
    h = _ListHandler(); log.addHandler(h)
    try:
        r = pipeline.predict(p["pre"], p["post"], generate_caption=True)
    finally:
        log.removeHandler(h)
    pre, post = np.asarray(r.pre), np.asarray(r.post)
    H, W = post.shape[:2]
    sem, se, oe, f = r.semantics, r.scene_evidence, r.object_evidence, r.facts
    x = getattr(se, "extra", {}) or {}
    draft, final = caption_body(r.draft_caption or r.blind_caption), caption_body(r.caption)
    T = {"key": p["key"], "truth": p["cls"], "refs": p["refs"], "log": h.lines,
         "timings": {k: round(float(v), 2) for k, v in (r.timings or {}).items()}}
    images, inventory, modules = {}, [], []

    def item(change, extent, where, by, value, verdict, caption, kind):
        inventory.append({"change": change, "extent": extent, "where": where, "detected_by": by,
                          "value": round(float(value), 4), "to_verdict": verdict, "to_caption": caption,
                          "in_draft": _mentions(kind, draft), "in_final": _mentions(kind, final),
                          "kind": kind})

    images["PRE"], images["POST"] = _b64(pre), _b64(post)
    images["change heat map (SDA-refined)"] = _b64(pipeline._heat(r.refined_map))

    # 1 · RemoteCLIP semantic grid: PRE -> POST transitions (main grid, area-weighted)
    g = sem.main_grid if getattr(sem, "enabled", True) else None
    if g is not None:
        ov = post.copy(); present = set()
        Wg, Hg = max(b[2] for b in g.boxes), max(b[3] for b in g.boxes)
        agg = _tcc.defaultdict(lambda: {"area": 0.0, "conf": [], "boxes": []})
        for t in sem.transitions:
            if not t.changed or t.level != "scene" or (t.grid and t.grid != g.g):
                continue
            a = agg[(t.src_group, t.dst_group)]
            # rev10: a patch transition's weight is its area share / number of grids; the
            # derived water transition's weight is already a true area
            _w = float(t.weight) * (max(len(getattr(sem, "grids", {}) or {}), 1) if t.kind == "patch" else 1)
            a["area"] += _w; a["conf"].append(float(t.confidence)); a["boxes"].append(t.where)
        for k, (x0, y0, x1, y1) in enumerate(g.boxes):
            if g.pre.groups[k] != g.post.groups[k]:
                box = np.zeros((Hg, Wg), bool); box[y0:y1, x0:x1] = True
                ov = _blend(ov, _fit2(box, post.shape).astype(bool), _GROUP_RGB.get(g.post.groups[k], (255, 0, 255)), 0.45)
                present.add(g.post.groups[k])
        images["RemoteCLIP: cells whose class changed (colour = AFTER class)"] = _b64(ov)
        T["semantic_legend"] = sorted(present)
        for (src, dst), a in sorted(agg.items(), key=lambda kv: -kv[1]["area"]):
            if a["area"] < 0.01: continue
            concepts = [c for c, _w in transition_concepts(src, dst)]
            cap_route = (f"checker fact ({_CAPTION_SURFACE_GROUPS[dst]} coverage)"
                         if dst in _CAPTION_SURFACE_GROUPS else "no")
            _dst_txt = (f"{_gw(dst)} (RemoteCLIP's 'cloud' label; the cloud test found no cloud here, "
                        "so likely pale ash, dust or haze)" if dst == "cloud" else _gw(dst))
            item(f"{_gw(src)} -> {_dst_txt}", _extent(a["area"]),
                 _zone(a["boxes"], (Hg, Wg)), "RemoteCLIP grid", a["area"],
                 ("DKG: " + ", ".join(concepts)) if concepts else "no (maps to no concept)", cap_route, dst)
        modules.append({"module": "RemoteCLIP scene grid + transitions",
                        "found": f"{sum(1 for t in sem.transitions if t.changed)} changed cells/patches; "
                                 f"water {sem.water_pre:.2f}->{sem.water_post:.2f}, vegetation {sem.veg_pre:.2f}->{sem.veg_post:.2f}",
                        "verdict": "yes - transitions and coverage fields feed DKG concepts",
                        "caption": "yes - water / ash / burn / smoke / lava / landslide coverage only"})
        cov = {k: (round(sem.group_cov_pre.get(k, 0), 3), round(sem.group_cov_post.get(k, 0), 3))
               for k in sorted(set(sem.group_cov_pre) | set(sem.group_cov_post))}
        T["group_coverage_pre_post"] = {k: v for k, v in cov.items() if max(v) > 0.005}

    # 2 · physical RGB channels
    phys = r.scene.physical or {}
    pthr = dict(CAPTION_PHYS); pthr.setdefault("vegetation_loss", 0.25); pthr["newly_water"] = 0.15
    for name in ("vegetation_loss", "brightening", "darkening", "desaturation", "texture_change", "newly_water"):
        mp = phys.get(name)
        if mp is None: continue
        mean = float(np.nan_to_num(np.asarray(mp, np.float32)).mean())
        fired = mean >= pthr[name]
        images[f"physical: {name}  mean {mean:.2f}{'  FIRED' if fired else ''}"] = _b64(_heatmap(mp), 150)
        if fired and name == "newly_water":
            item("new water (pixel test)", f"mean {mean:.2f} (threshold {pthr[name]:.2f})",
                 _zone(np.where(np.asarray(mp) >= pthr[name], mp, 0)), "physical RGB channel", mean,
                 "DKG: " + _PHYS_FEEDS[name], "checker fact", name)
    # rev10: physical changes as LOCATED REGIONS (whole-image means hide local changes)
    for reg in (f.get("regions") or []):
        item(f"{reg['kind'].replace('_', ' ')} (strong, located)", f"{100 * reg['area']:.1f}% of observable ground",
             reg["where"], "physical RGB channel", reg["area"], "DKG: " + _PHYS_FEEDS.get(reg["kind"], "-"),
             "checker fact (added if the draft misses it)" if reg["kind"] in REGION_CHANNELS else "no", reg["kind"])
    # rev11: what KIND of darkening, by surface
    dt = (getattr(se, "extra", {}) or {}).get("darkening_types", {}) or {}
    for kind_, lab_, where_, verdict_ in (("deposit", "dark deposit on open ground AND vegetation (a layer settled on everything)", "where_deposit", "DKG: ashfall (0.8), surface_change"),
                                          ("burn", "blackened vegetation, open ground untouched (burn-like)", "where_burn", "DKG: burned, vegetation_loss"),
                                          ("ambiguous", "darkened vegetation, too little open ground nearby to tell deposit from burn", "", "no"),
                                          ("shadow", "darker but same colours (shadow-like, ignored)", "", "no")):
        v_ = float(dt.get(kind_, 0.0))
        if v_ >= 0.01:
            item(lab_, f"{100 * v_:.1f}% of observable ground", dt.get(where_, "") if where_ else "",
                 "darkening by surface (rev11)", v_, verdict_,
                 "checker fact (added if the draft misses it)" if kind_ in ("deposit", "burn") else "no",
                 "darkening")
    if dt.get("zones"):
        modules.append({"module": "darkening by surface (deposit / burn / shadow)",
                        "found": "; ".join(f"{z['zone']}: {z['kind']} (open ground {z['open_ground_hit']}, "
                                           f"vegetation {z['vegetation_hit']})" for z in dt["zones"] if z["kind"] != "none") or "no strong darkening",
                        "verdict": "yes - deposit -> ashfall, burn -> burned",
                        "caption": "yes - typed darkening is passed to the checker"})
    q = getattr(r, "quality", {}) or {}
    cl, al = q.get("clouds") or {}, q.get("align") or {}
    if q.get("cloud_pre") is not None:
        ov = _blend(pre, _fit2(q["cloud_pre"], pre.shape).astype(bool), (255, 0, 255), 0.5)
        ov2 = _blend(post, _fit2(q["cloud_post"], post.shape).astype(bool), (255, 0, 255), 0.5)
        images[f"clouds excluded (magenta): BEFORE {100 * cl.get('pre', 0):.0f}% | AFTER "
               f"{100 * cl.get('post', 0):.0f}%"] = _b64(np.concatenate([ov, ov2], axis=1), 600)
        for side, key in (("BEFORE", "cloud_pre"), ("AFTER", "cloud_post")):
            fr = float(np.asarray(q[key]).mean())
            if fr > 0:
                item(f"cloud or haze in the {side} image (not ground)", f"{100 * fr:.1f}% of the tile",
                     _zone(np.asarray(q[key], np.float32)), "cloud test", fr,
                     "excluded from every measurement", "checker fact (caption must not describe it as ground)", "cloud")
    modules.append({"module": "image quality: clouds and alignment",
                    "found": f"clouds BEFORE {100 * cl.get('pre', 0):.1f}%, AFTER {100 * cl.get('post', 0):.1f}%; "
                             f"rejected bright regions {[x for x in cl.get('regions', []) if not x.get('cloud')]}; "
                             f"alignment {al}",
                    "verdict": "yes - cloud pixels and clouded cells are excluded from every measurement",
                    "caption": "yes - cloud edits"})
    modules.append({"module": "physical RGB channels",
                    "found": ", ".join(f"{k} {float(np.mean(v)):.2f}" for k, v in phys.items()
                                       if k in pthr and np.asarray(v).ndim == 2),
                    "verdict": "yes - DKG physical priors (calibrated)",
                    "caption": "only as the second check before vetoing a claim"})

    # 3 · water: the two tests and what counted
    w_sem, w_px = float(x.get("flood_semantic", 0.0)), float(x.get("newly_water", 0.0))
    ov = post.copy()
    if g is not None:
        for (x0, y0, x1, y1) in _group_boxes(sem, ("flood", "sediment_water"), CFG):
            box = np.zeros((Hg, Wg), bool); box[y0:y1, x0:x1] = True
            ov = _blend(ov, _fit2(box, post.shape).astype(bool), (0, 220, 220), 0.30)
    if phys.get("newly_water") is not None:
        ov = _blend(ov, _fit2(np.asarray(phys["newly_water"]) > 0.5, post.shape).astype(bool), (0, 60, 255), 0.5)
    images[f"water: semantic {w_sem:.2f} (cyan) | pixel {w_px:.2f} (blue) | counted {float(se['flood_extent']):.2f}"] = _b64(ov)
    modules.append({"module": "water tests (semantic grid + pixel test + override)",
                    "found": f"semantic {w_sem:.3f}, pixel {w_px:.3f}, counted {float(se['flood_extent']):.3f}; "
                             + "; ".join(l for l in h.lines if "water override" in l),
                    "verdict": "yes - flood_extent, and semantic flood transitions (not gated in rev9)",
                    "caption": "yes - water facts and absences"})

    # 4 · ChangeOS buildings + SAM2 instances
    bm = _fit2(r.obj.building_mask, post.shape).astype(bool)
    gr = _fit2(r.obj.damage_grades, post.shape).astype(int)
    ov = post.copy()
    for grade, rgb in _GRADE_RGB.items():
        ov = _blend(ov, bm & (gr == grade), rgb, 0.5)
    img = _TImage.fromarray(ov); d = _TDraw.Draw(img)
    for i, inst in enumerate(r.instances or []):
        x0, y0, x1, y1 = inst.bbox
        d.rectangle([x0, y0, x1, y1], outline=(255, 255, 255), width=2); d.text((x0 + 2, y0 + 1), str(i), fill=(255, 255, 255))
    images["ChangeOS damage (green none, yellow minor, orange major, red destroyed) + SAM2 instances"] = _b64(img)
    c = oe.counts
    L = [int(c.get(f"bld_level{i}", 0)) for i in range(4)]
    nd = L[1] + L[2] + L[3]
    T["changeos"] = {"buildings": int(c.get("buildings_detected", 0)), "levels_0_to_3": L,
                     "damaged_area": round(float(r.obj.damaged_area_frac), 4),
                     "damaged_where": f.get("damaged_where", "")}
    sd = _safe(lambda: r.dkg["concept_by_source"].get("structural_damage", {}), {})
    if nd:
        item(f"{nd} damaged buildings (levels 1-3: {L[1]}/{L[2]}/{L[3]})", f"of {T['changeos']['buildings']} detected",
             f.get("damaged_where", ""), "ChangeOS", nd,
             f"DKG: structural_damage (object mass {sd.get('object', 0):.3f})", "checker fact", "damaged_buildings")
    modules.append({"module": "ChangeOS (xBD building damage)",
                    "found": f"{T['changeos']['buildings']} buildings, levels 0-3 = {L}, damaged area {T['changeos']['damaged_area']}",
                    "verdict": f"yes - structural_damage (object mass {sd.get('object', 0):.3f})",
                    "caption": "yes - building checks (damage it detects)"})
    inst_rows = []
    for i, inst in enumerate(r.instances or []):
        tr = inst.transition
        inst_rows.append({"#": i, "label": inst.label, "conf": round(float(inst.label_conf), 2),
                          "change": round(float(inst.change_score), 2),
                          "transition": tr.name if tr is not None else "",
                          "damage": XBD_DAMAGE_CLASSES.get(inst.damage_grade, "") if inst.damage_grade is not None else "",
                          "where": _zone([inst.bbox], (H, W))})
        if tr is not None and tr.changed and inst.change_score >= CFG.damage_change_thr:
            item(f"object {i}: {tr.src_group} -> {tr.dst_group}", f"{int(inst.area)} px",
                 _zone([inst.bbox], (H, W)), "SAM2 instance + RemoteCLIP", inst.change_score,
                 "DKG instance transition (small mass)", "no", tr.dst_group)
    T["instances"] = inst_rows
    modules.append({"module": "SAM2 instances", "found": f"{len(inst_rows)} instances",
                    "verdict": "marginal - instance transitions, divided by the count", "caption": "no"})

    # 5 · AnyChange regions
    regs = sorted(r.scene.regions or [], key=lambda q: -(float(q.prob) * float(np.asarray(q.mask).mean())))
    ov = post.copy(); img_rows = []
    for k, q in enumerate(regs[:8]):
        m = _fit2(q.mask, post.shape).astype(bool)
        ov = _blend(ov, m, _RANK_RGB[k % len(_RANK_RGB)], 0.45)
        frac = float(m.mean())
        dg = _safe(lambda: _dom_groups(sem, q.mask), None)
        dg = dg if isinstance(dg, tuple) else None
        tr = f"{_gw(dg[0])} -> {_gw(dg[1])}" if dg else ""
        img_rows.append({"rank": k, "prob": round(float(q.prob), 2), "angle": round(float(q.angle), 1),
                         "area": round(frac, 4), "where": _zone(m), "measured inside": tr})
        if q.prob >= 0.5 and frac >= 0.01:
            item(f"AnyChange region {k}: " + (tr or "change of unknown kind"), f"{100 * frac:.1f}% of tile",
                 _zone(m), "AnyChange (SAM)", frac, "no", "no", dg[1] if dg else "")
    img = _TImage.fromarray(ov); d = _TDraw.Draw(img)
    for k, q in enumerate(regs[:8]):
        x0, y0, x1, y1 = q.box; d.text((x0 + 2, y0 + 1), str(k), fill=(255, 255, 255))
    images[f"AnyChange: top {min(8, len(regs))} of {len(regs)} regions (numbered by rank)"] = _b64(img)
    T["anychange"] = img_rows
    modules.append({"module": "AnyChange (SAM ViT-B)", "found": f"{len(regs)} regions, coverage "
                    f"{float(se.values.get('anychange_coverage', 0.0)):.3f}",
                    "verdict": "no - seeds SAM2, one RVG check", "caption": "no (only the rev8 checklist used it)"})

    # 6 · SDA refined map, tokens + attention, head
    rm = np.asarray(r.refined_map, np.float32)
    hot = rm >= CFG.change_threshold
    modules.append({"module": "SDA refinement", "found": f"{100 * float(hot.mean()):.1f}% above {CFG.change_threshold}, "
                    f"where: {_zone(np.where(hot, rm, 0)) or '-'}",
                    "verdict": "no", "caption": "no (only the rev8 heat map)"})
    modules.append({"module": "evidence tokens + cross-attention", "found": _safe(lambda: f"{r.tokens.counts()}, align s->o "
                    f"{r.tokens.align_scene_to_object:.2f}, o->s {r.tokens.align_object_to_scene:.2f}"),
                    "verdict": "no - feeds only the classifier head", "caption": "no"})
    clf = r.classifier or {}
    modules.append({"module": "classifier head", "found": f"trained={clf.get('is_trained')}, top: "
                    + ", ".join(f"{t['disaster']} {t['prob']:.2f}" for t in clf.get("top_k", [])[:3]),
                    "verdict": "no - not pooled while untrained", "caption": "no"})

    # 7 · scene-evidence fields (what the DKG reads from the scene branch)
    T["scene_fields"] = {k: round(float(se[k]), 4) for k in SCENE_EVIDENCE_CONCEPTS
                         if _safe(lambda: float(se[k]), 0.0) > 1e-4}

    # 8 · DKG, verdict, RVG
    dk = r.dkg or {}
    T["dkg"] = {"verdict": dk.get("disaster"), "score": dk.get("score"), "margin": dk.get("margin"),
                "concept_mass": dk.get("concept_mass", {}), "by_source": dk.get("concept_by_source", {}),
                "rules": [{"disaster": q["disaster"], "score": q.get("score"), "required": q.get("required"),
                           "support": q.get("support"), "veto": q.get("veto"), "reason": q.get("reason", ""),
                           "conflicting": q.get("conflicting", [])} for q in dk.get("ranking", [])]}
    zs = r.zeroshot or {}
    T["verdict"] = {"posterior": r.posterior.label, "prob": round(float(r.posterior.prob), 3),
                    "weights": r.posterior.weights,
                    "posterior_top": _safe(lambda: r.posterior.top_k(), []),
                    "posterior_all": _safe(lambda: {c: round(float(q), 4) for c, q in
                                                    zip(r.posterior.classes, r.posterior.probs)}, {}),
                    "sources_all": _safe(lambda: {s_: {c: round(float(q), 4) for c, q in
                                                       zip(r.posterior.classes, np.asarray(v_).ravel())}
                                                  for s_, v_ in (r.posterior.sources or {}).items()}, {}),
                    "head_all": _safe(lambda: {c: round(float(q), 4) for c, q in
                                               zip(r.posterior.classes, np.asarray(clf["probs"]).ravel())}
                                      if clf.get("probs") is not None else {}, {}),
                    "dkg_probs": dk.get("probs", {}),
                    "zeroshot_top": zs.get("top_k", [])[:3], "head_top": clf.get("top_k", [])[:3]}
    T["rvg"] = {k: r.rvg.get(k) for k in ("confidence", "checks", "weakest_check", "uncertainty_reason")}
    modules.append({"module": "DKG (symbolic reasoner)", "found": f"{dk.get('disaster')} ({dk.get('score')})",
                    "verdict": "yes - decides almost every verdict", "caption": "via the label"})
    modules.append({"module": "zero-shot RemoteCLIP prior", "found": ", ".join(
                    f"{t['disaster']} {t['prob']:.2f}" for t in zs.get("top_k", [])[:3]),
                    "verdict": "pooled with the DKG", "caption": "via the label"})
    modules.append({"module": "RVG", "found": f"confidence {r.rvg.get('confidence')}",
                    "verdict": "no - confidence only", "caption": "no"})

    # 9 · captions: prompts, draft, facts, then the system caption's own steps
    sent = [e for e in (r.edits or []) if e.get("sent")]
    T["caption"] = {"mode": CFG.caption_mode, "draft_prompt": BLIND_PROMPT, "draft": draft, "facts": f,
                    "edits": r.edits, "revision_prompt": revision_prompt(draft, sent) if sent else "(no edits: the draft is the caption)",
                    "final": final, "final_raw": (r.caption or "").strip(),
                    "final_verdict_line": stated_disaster(r.caption), "residual": r.residual}
    lk = getattr(r, "look", None) or {}
    if lk:
        # rev12: evidence -> looks -> attempts -> coverage
        T["caption"].update({
            "evidence": [{k: it.get(k) for k in ("id", "kind", "label", "where", "area", "source", "status",
                                                  "seen_by", "final", "covered_by", "group")} for it in lk.get("items", [])],
            "excluded": [{k: e.get(k) for k in ("kind", "label", "where", "area", "source", "why")}
                         for e in lk.get("excluded", [])],
            "groups": [{"id": g_["id"], "place": g_["place"], "box": g_.get("box"), "whole": g_.get("whole"),
                        "items": g_["items"], "note_open": g_.get("note_open"), "note_guided": g_.get("note_guided")}
                       for g_ in lk.get("groups", [])],
            "look_prompt": look_prompt(False), "attempts": [{"prompt": a_["prompt"], "caption": a_["caption"],
                                                              "check": {k: a_["check"].get(k) for k in
                                                                        ("ok", "event_ok", "cloud_ok", "required", "covered",
                                                                         "missing_items", "contradictions", "rules")}}
                                                             for a_ in lk.get("attempts", [])],
            "coverage": lk.get("coverage", {})})
        for k_, (a_, b_) in enumerate(getattr(r, "look_crops", []) or []):
            g_ = lk["groups"][k_] if k_ < len(lk.get("groups", [])) else {"place": "?"}
            images[f"look:{k_}: {g_['place']} (BEFORE | AFTER close-up)"] = _b64(
                np.concatenate([np.asarray(a_.convert("RGB")), np.asarray(b_.convert("RGB").resize(a_.size))], axis=1), 420)
    if checklist:
        ev = r.package.to_prompt()
        T["caption"]["checklist_prompt"] = QwenCaptioner.build_prompt(ev, False, True)
        T["caption"]["checklist_caption"] = caption_body(_safe(lambda: pipeline.captioner.caption(
            _TImage.fromarray(pre), _TImage.fromarray(post), pipeline._heat(r.refined_map), ev, False, True), ""))
    if lk:
        # rev12: every module's located changes are caption evidence; section 5 lists each
        # one with its status (visible / not visible / excluded with the reason / covered)
        for it_ in inventory:
            it_["to_caption"] = ("no - a shadow, ignored" if "shadow" in it_["change"]
                                 else "rev12 evidence (status in section 5)")
        for m_ in modules:
            if m_["module"].startswith(("AnyChange", "SAM2 instances", "physical RGB", "RemoteCLIP",
                                        "darkening by surface", "ChangeOS", "water tests")):
                m_["caption"] = "yes - rev12 caption evidence (located, looked at, verified)"
    T["inventory"], T["modules"] = inventory, modules
    T["unmentioned"] = [i for i in inventory if not i["in_final"] and i["kind"]]
    return T, images


# ── HTML ─────────────────────────────────────────────────────────────────────
_CSS = """body{font-family:system-ui,sans-serif;margin:18px;color:#222} h1{font-size:20px} h2{font-size:16px;margin-top:26px;border-bottom:1px solid #ccc}
table{border-collapse:collapse;font-size:12px;margin:6px 0} td,th{border:1px solid #ccc;padding:3px 6px;vertical-align:top;text-align:left}
th{background:#f2f2f2} .imgs figure{display:inline-block;vertical-align:top;margin:0 10px 12px 0;width:300px} .imgs.small figure{width:150px}
figcaption{font-size:11px;color:#444} .yes{color:#0a7d2a;font-weight:600} .no{color:#b00020} pre{white-space:pre-wrap;background:#f7f7f7;padding:8px;font-size:12px}
.sw{display:inline-block;width:12px;height:12px;margin:0 3px 0 10px;border:1px solid #888;vertical-align:middle}"""

def _tbl(rows, cols=None):
    if not rows: return "<p><i>none</i></p>"
    cols = cols or list(rows[0].keys())
    def cell(v):
        if isinstance(v, bool): return f"<td class='{'yes' if v else 'no'}'>{'yes' if v else 'no'}</td>"
        if isinstance(v, (dict, list)): v = _tj.dumps(v, default=str)
        s = _th.escape(str(v))
        cls = "yes" if s.startswith(("yes", "DKG:", "checker")) else ("no" if s.startswith("no") else "")
        return f"<td class='{cls}'>{s}</td>"
    return ("<table><tr>" + "".join(f"<th>{_th.escape(c)}</th>" for c in cols) + "</tr>"
            + "".join("<tr>" + "".join(cell(r.get(c, "")) for c in cols) + "</tr>" for r in rows) + "</table>")

def _figs(images, small=False):
    return (f"<div class='imgs{' small' if small else ''}'>" + "".join(
        f"<figure><img src='{src}' width='100%'><figcaption>{_th.escape(t)}</figcaption></figure>"
        for t, src in images) + "</div>")

_SRC_NAME = {"dkg": "DKG", "zeroshot": "zero-shot", "zero_shot": "zero-shot", "classifier": "head"}

def _all_types_rows(T):
    """One row per disaster type the pipeline can name (DISASTER_CLASSES)."""
    v = T["verdict"]; post = v.get("posterior_all", {}); srcs = dict(v.get("sources_all", {}))
    if v.get("head_all") and "classifier" not in srcs:
        srcs["classifier"] = v["head_all"]          # rev10: shown, not pooled
    rules = {q["disaster"]: q for q in T["dkg"]["rules"]}
    rows = []
    for c in sorted(DISASTER_CLASSES, key=lambda c: -post.get(c, 0.0)):
        q = rules.get(c, {})
        rows.append({"disaster type": c, "posterior": post.get(c, ""),
                     **{_SRC_NAME.get(s_, s_) + ("" if s_ in v.get("weights", {}) else " (not pooled)"): srcs[s_].get(c, "")
                        for s_ in srcs},
                     "DKG rule": (f"score {q.get('score')}" if not q.get("reason") else f"0 - {q['reason']}") if q else "-"})
    return rows

def render_html(T, images):
    main = [(k, v) for k, v in images.items() if not k.startswith("physical") and not k.startswith("look:")]
    looks = [(k[5:], v) for k, v in images.items() if k.startswith("look:")]
    phys = [(k, v) for k, v in images.items() if k.startswith("physical")]
    v, cap = T["verdict"], T["caption"]
    legend = "".join(f"<span class='sw' style='background:rgb{_GROUP_RGB.get(g, (255, 0, 255))}'></span>{_th.escape(GROUP_WORDS.get(g, g))}"
                     for g in T.get("semantic_legend", []))
    _srcs = sorted({s_ for v_ in T["dkg"]["by_source"].values() for s_ in v_})
    concept_rows = [{"concept": c, "total mass": m, **{s_: T['dkg']['by_source'].get(c, {}).get(s_, '')
                     for s_ in _srcs}} for c, m in T["dkg"]["concept_mass"].items()]
    edit_rows = [{"kind": e["kind"], "check": e["check"], "sent": e["sent"], "resolved": e.get("resolved"),
                  "instruction / reason": e.get("instruction") or e.get("why", "")} for e in cap["edits"]]
    known = T["truth"] != "unknown"
    ok = v["posterior"] == T["truth"]
    vcls = ("yes" if ok else "no") if known else ""
    truth = T["truth"] if known else "unknown (no label for this pair)"
    ref = (_th.escape(" || ".join(T["refs"])) if T["refs"] else
           "<i>none — no caption ground truth for this pair</i>" if T.get("no_refs_file") else "<i>none found for this pair</i>")
    return f"""<html><head><meta charset='utf-8'><style>{_CSS}</style><title>{T['key']}</title></head><body>
<h1>{_th.escape(T['key'])} — truth: {truth} | pipeline: <span class='{vcls}'>{v['posterior']} ({v['prob']})</span></h1>
<p><b>Folder:</b> {_th.escape(T.get('folder', ''))} &nbsp; <b>Reference caption:</b> {ref}
{('&nbsp; <b>DKG calibration:</b> ' + _th.escape(T['calibration'])) if T.get('calibration') else ''}</p>
{_figs(main)}<p>RemoteCLIP AFTER classes: {legend}</p>
<h2>Physical channels (bright = strong change)</h2>{_figs(phys, small=True)}
<h2>1 · Change inventory — every change any module detected</h2>
<p>to_verdict / to_caption = where that module's output can go; in_draft / in_final = whether Qwen's draft / the final caption mention that kind of change.</p>
{_tbl(T['inventory'], ['change', 'extent', 'where', 'detected_by', 'to_verdict', 'to_caption', 'in_draft', 'in_final'])}
<h2>Detected but NOT in the final caption</h2>{_tbl(T['unmentioned'], ['change', 'extent', 'where', 'detected_by', 'to_caption'])}
<h2>2 · Modules — what each produced and where it goes</h2>{_tbl(T['modules'])}
<p><b>Scene-evidence fields read by the DKG:</b> {_th.escape(_tj.dumps(T.get('scene_fields', {})))}</p>
<p><b>RemoteCLIP class coverage BEFORE -> AFTER:</b> {_th.escape(_tj.dumps(T.get('group_coverage_pre_post', {})))}</p>
<p><b>ChangeOS:</b> {_th.escape(_tj.dumps(T['changeos']))}</p>
<h3>SAM2 instances</h3>{_tbl(T['instances'])}<h3>AnyChange regions</h3>{_tbl(T['anychange'])}
<h2>3 · DKG — concept mass by source</h2>{_tbl(concept_rows)}
<h3>Disaster rules (score 0 = a required concept is missing; reason says which)</h3>{_tbl(T['dkg']['rules'])}
<h2>4 · Verdict — all seven disaster types</h2>{_tbl(_all_types_rows(T))}
<p>posterior = the pooled verdict; one column per source; "pooled" sources make the posterior, the
head is shown but not pooled while untrained. DKG rule = the symbolic rule for that type: its score,
or which required evidence was missing.</p>{_tbl([{'source': 'DKG', 'output': v['dkg_probs']}, {'source': 'zero-shot', 'output': v['zeroshot_top']},
     {'source': 'head (not pooled)', 'output': v['head_top']}, {'source': 'POSTERIOR', 'output': v['posterior_top']},
     {'source': 'pool weights', 'output': v['weights']}, {'source': 'RVG', 'output': T['rvg']}])}
<h2>5 · Caption generation ({_th.escape(str(cap.get('mode', 'revise')))})</h2>
{_caption_html(cap, looks)}
{('<p><b>rev8 checklist caption (C4), for comparison</b></p><details><summary>its prompt</summary><pre>' + _th.escape(cap.get('checklist_prompt', '')) + '</pre></details><pre>' + _th.escape(cap.get('checklist_caption', '')) + '</pre>') if 'checklist_caption' in cap else ''}
<h2>Module log for this pair</h2><pre>{_th.escape(chr(10).join(T['log']))}</pre>
<p>timings (s): {_th.escape(_tj.dumps(T['timings']))}</p></body></html>"""


def _caption_html(cap, looks):
    """Section 5 of a report: the rev12 look-then-write steps, or the rev9 draft/revise steps."""
    base = (f"<p><b>Baseline — blind caption (Qwen, images only; C3 and the RVG blind check).</b></p>"
            f"<details><summary>prompt</summary><pre>{_th.escape(cap['draft_prompt'])}</pre></details>"
            f"<pre>{_th.escape(cap['draft'])}</pre>")
    if "evidence" not in cap:
        edit_rows = [{"kind": e["kind"], "check": e["check"], "sent": e["sent"], "resolved": e.get("resolved"),
                      "instruction / reason": e.get("instruction") or e.get("why", "")} for e in cap["edits"]]
        return (base + f"<p><b>Facts the checker compared the draft with</b></p><pre>{_th.escape(_tj.dumps(cap['facts'], indent=1, default=str))}</pre>"
                f"<p><b>Edits</b></p>{_tbl(edit_rows)}<p><b>Revision prompt</b></p><pre>{_th.escape(cap['revision_prompt'])}</pre>"
                f"<p><b>Final caption</b> (verdict line: {cap['final_verdict_line']})</p><pre>{_th.escape(cap['final'])}</pre>"
                f"<p><b>Still contradicting a measurement:</b> {_th.escape(_tj.dumps([e['check'] for e in cap['residual']]))}</p>")
    cov = cap.get("coverage", {})
    grp = []
    for k, g_ in enumerate(cap["groups"]):
        img = next((v for kk, v in looks if kk.startswith(f"{k}:")), "")
        grp.append(f"<div style='margin:8px 0;padding:6px;border:1px solid #ddd'>"
                   f"<b>Region {k} — {_th.escape(g_['place'])}</b> (evidence {', '.join(g_['items'])}; crop {g_.get('box')})<br>"
                   + (f"<img src='{img}' style='max-width:420px'><br>" if img else "")
                   + f"<b>open look:</b> {_th.escape(str(g_.get('note_open') or ''))}<br>"
                   + (f"<b>guided look:</b> {_th.escape(str(g_['note_guided']))}" if g_.get("note_guided") else "")
                   + "</div>")
    att = []
    for k, a_ in enumerate(cap["attempts"]):
        c_ = a_["check"]
        att.append(f"<details><summary>attempt {k + 1}: covered {c_['covered']}/{c_['required']}, event "
                   f"{'named' if c_['event_ok'] else 'NOT named'}, clouds {'ok' if c_['cloud_ok'] else 'NOT mentioned'}, "
                   f"{len(c_['contradictions'])} contradiction(s){' — kept' if a_['caption'] == cap['final_raw'] else ''}"
                   f"</summary><pre>{_th.escape(a_['prompt'])}</pre><pre>{_th.escape(a_['caption'])}</pre>"
                   f"<p>missing: {_th.escape(', '.join(c_['missing_items']) or '-')} | rules for the next attempt: "
                   f"{_th.escape(' / '.join(c_['rules']) or '-')}</p></details>")
    return (base
            + "<p><b>Step 1 — evidence: every valid change the pipeline measured</b> (status = what Qwen saw in the "
              "close-up; final = whether the caption carries it)</p>"
            + _tbl(cap["evidence"], ["id", "kind", "label", "where", "area", "source", "group", "status", "seen_by", "final", "covered_by"])
            + "<p><b>Excluded, with the reason</b></p>" + _tbl(cap["excluded"], ["kind", "label", "where", "area", "source", "why"])
            + f"<p><b>Step 2 — looks</b> (close-up of each region; the open-look prompt holds no evidence)</p>"
              f"<details><summary>open-look prompt</summary><pre>{_th.escape(cap['look_prompt'])}</pre></details>"
            + "".join(grp)
            + "<p><b>Step 3 — writing (each attempt from scratch) and Step 4 — verification</b></p>" + "".join(att)
            + f"<p><b>Final caption</b> (verdict line: {cap['final_verdict_line']})</p><pre>{_th.escape(cap['final'])}</pre>"
            + f"<p><b>Coverage:</b> {cov.get('covered')}/{cov.get('required')} visible changes in the caption "
              f"({cov.get('valid')} valid, {len(cov.get('not_visible', []))} not visible to Qwen, {cov.get('excluded')} excluded); "
              f"missing: {_th.escape('; '.join(cov.get('missing', [])) or 'none')}; still contradicting: "
              f"{_th.escape('; '.join(cov.get('contradictions', [])) or 'nothing')}</p>")


# ── running a folder ─────────────────────────────────────────────────────────
def _pick_inline(traces, cls):
    """One pair to display for `cls`: a right verdict first, then any pair of that class,
    then (in an unlabelled folder) a pair the pipeline calls `cls`."""
    known = [T for T in traces if T["truth"] == cls]
    for group in ([T for T in known if T["verdict"]["posterior"] == cls], known,
                  [T for T in traces if T["truth"] == "unknown" and T["verdict"]["posterior"] == cls]):
        if group: return group[0]
    return None

def trace_folder(img_dir, out_dir, refs_file=None, classes=None, max_per_class=None, labels_csv=None,
                 calibration="this_folder", checklist=True,
                 inline_classes=("earthquake", "volcanic_eruption", "flood"), caption_mode="look"):
    """Trace every pair of `img_dir` (see the §22a header).
      refs_file       RSCC-style JSONL of reference captions, or None (no ground truth)
      classes         only trace these classes; None = every pair, including unknown ones
      max_per_class   an int, or {class: int}; None = no limit
      labels_csv      optional CSV giving each pair's disaster class
      calibration     "this_folder" (median over this folder's pairs, no labels; cached in
                      out_dir), a path to a phys_concept_baseline JSON, or "keep"
      inline_classes  display one report per class inside the notebook
      caption_mode    "look" (rev12 system caption, default) or "revise" (rev9-rev11)
    """
    _tos.makedirs(f"{out_dir}/json", exist_ok=True)
    labels, refs = _load_labels(labels_csv), _load_refs(refs_file)
    has_refs = bool(refs_file) and _tos.path.exists(str(refs_file))
    found = find_pairs(img_dir)
    for p in found:
        stem = _tos.path.basename(p["rel"])
        p["cls"] = labels.get(p["key"]) or labels.get(stem) or _class_from_text(p["rel"]) or "unknown"
        p["refs"] = refs.get(stem.lower().replace("-", "_"), [])
        p["folder"], p["no_refs_file"] = img_dir, not has_refs
    limit = lambda c: max_per_class.get(c) if isinstance(max_per_class, dict) else max_per_class
    todo, per = [], _tcc.Counter()
    for p in found:
        if classes is not None and p["cls"] not in classes: continue
        if limit(p["cls"]) is not None and per[p["cls"]] >= limit(p["cls"]): continue
        todo.append(p); per[p["cls"]] += 1
    print(f"{img_dir}\n  {len(found)} pairs found | tracing {len(todo)}: {dict(per)} | "
          f"reference captions: {sum(bool(p['refs']) for p in todo)}/{len(todo)} -> {out_dir}")
    if not todo:
        print("  nothing to trace — check the folder layout (see the §22a header)"); return []

    saved = (CFG.label_in_prompt, CFG.use_blind_pass, CFG.caption_mode, dict(CFG.phys_concept_baseline))
    traces, fails = [], []
    try:
        # DKG calibration: a label-free per-concept median
        if calibration == "this_folder":
            cal = f"{out_dir}/phys_concept_baseline_rev11.json"
            if _tos.path.exists(cal):
                base = _tj.load(open(cal))["baseline"]
            else:
                base = fast_physical_baselines(pipeline, [(p["pre"], p["post"]) for p in found], apply=False)
                _tj.dump({"baseline": base, "source": img_dir, "n_pairs": len(found), "labels_used": False},
                         open(cal, "w"), indent=1)
            print(f"  DKG calibrated on this folder ({len(found)} pairs, no labels used)")
        elif calibration not in ("keep", None) and _tos.path.exists(str(calibration)):
            base = _tj.load(open(calibration))["baseline"]
            print(f"  DKG calibration loaded from {calibration}")
        else:
            base = dict(CFG.phys_concept_baseline)
            print("  DKG calibration: " + ("kept as currently set" if base else "NONE (uncalibrated)"))
        CFG.phys_concept_baseline = base; pipeline.cfg.phys_concept_baseline = base
        CFG.label_in_prompt, CFG.use_blind_pass, CFG.caption_mode = True, True, caption_mode

        for i, p in enumerate(todo, 1):
            jpath = f"{out_dir}/json/{p['key']}.json"
            if _tos.path.exists(jpath):
                traces.append(_tj.load(open(jpath))); continue
            try:
                T, images = trace_pair(p, checklist=checklist)
                T["folder"], T["no_refs_file"] = p["folder"], p["no_refs_file"]
                open(f"{out_dir}/{p['key']}.html", "w").write(render_html(T, images))
                _tj.dump(T, open(jpath, "w"), indent=1, default=str)
                traces.append(T)
                _cv = T["caption"].get("coverage") or {}
                log.info("[trace %d/%d] %s | %s -> %s | %d changes detected, %d in the final caption%s",
                         i, len(todo), p["key"], p["cls"], T["verdict"]["posterior"],
                         len(T["inventory"]), sum(x["in_final"] for x in T["inventory"]),
                         (f" | caption evidence: {_cv.get('covered')}/{_cv.get('required')} visible changes "
                          f"in the caption ({_cv.get('valid')} valid, {len(_cv.get('not_visible', []))} not visible, "
                          f"{_cv.get('attempts')} attempt(s))") if _cv else "")
            except Exception as e:
                fails.append((p["key"], f"{type(e).__name__}: {e}"))
                log.warning("[trace %d/%d] %s failed: %s", i, len(todo), p["key"], fails[-1][1])
                if len(fails) == 1:
                    import traceback; traceback.print_exc()
            finally:
                _tgc.collect(); torch.cuda.empty_cache()
    finally:
        CFG.label_in_prompt, CFG.use_blind_pass, CFG.caption_mode, base0 = saved
        CFG.phys_concept_baseline = base0; pipeline.cfg.phys_concept_baseline = base0

    # index page
    rows = [{"pair": f"<a href='{T['key']}.html'>{T['key']}</a>", "truth": T["truth"],
             "pipeline": T["verdict"]["posterior"], "DKG": T["dkg"]["verdict"],
             "zero-shot": (T["verdict"]["zeroshot_top"] or [{}])[0].get("disaster", ""),
             "changes detected": len(T["inventory"]),
             "in final caption": sum(x["in_final"] for x in T["inventory"]),
             "valid evidence": (T["caption"].get("coverage") or {}).get("valid", ""),
             "visible": (T["caption"].get("coverage") or {}).get("required", ""),
             "visible in caption": (T["caption"].get("coverage") or {}).get("covered", ""),
             "attempts": (T["caption"].get("coverage") or {}).get("attempts", ""),
             "edits sent": sum(bool(e.get("sent")) for e in T["caption"]["edits"])} for T in traces]
    idx = "<table><tr>" + "".join(f"<th>{c}</th>" for c in (rows[0] if rows else {})) + "</tr>" + "".join(
        "<tr>" + "".join(f"<td>{v}</td>" for v in r_.values()) + "</tr>" for r_ in rows) + "</table>"
    open(f"{out_dir}/index.html", "w").write(
        f"<html><head><meta charset='utf-8'><style>{_CSS}</style></head><body>"
        f"<h1>Evidence trace — {_th.escape(img_dir)} — {len(traces)} pairs</h1>{idx}</body></html>")

    # summary, grouped by the true class where known, else by the pipeline's verdict
    print("\n" + "=" * 88); print(f"EVIDENCE TRACE SUMMARY — {img_dir}"); print("=" * 88)
    groups = _tcc.defaultdict(list)
    for T in traces:
        groups[T["truth"] if T["truth"] != "unknown"
               else f"unlabelled, pipeline says {T['verdict']['posterior']}"].append(T)
    for gname, Ts in sorted(groups.items()):
        line = f"\n{gname}: {len(Ts)} pairs"
        if not gname.startswith("unlabelled"):
            line += f" | verdict right {sum(T['verdict']['posterior'] == gname for T in Ts)}"
        print(line + f" | verdicts {dict(_tcc.Counter(T['verdict']['posterior'] for T in Ts))}")
        by = _tcc.defaultdict(lambda: [0, 0, 0])            # pairs with a detection, detections, in caption
        for T in Ts:
            seen = set()
            for it in T["inventory"]:
                b = by[it["detected_by"]]; b[1] += 1; b[2] += it["in_final"]
                if it["detected_by"] not in seen: b[0] += 1; seen.add(it["detected_by"])
        print(f"  {'module':<30}{'pairs with a change':>20}{'changes':>9}{'in final caption':>18}")
        for mod, (np_, nc, ni) in sorted(by.items(), key=lambda kv: -kv[1][1]):
            print(f"  {mod:<30}{np_:>14}/{len(Ts):<5}{nc:>9}{ni:>12} ({ni / max(nc, 1):.0%})")
        # rev12: the caption's own evidence ledger
        CV = [T["caption"]["coverage"] for T in Ts if T["caption"].get("coverage")]
        if CV:
            ev = [e for T in Ts for e in T["caption"].get("evidence", [])]
            print(f"  caption evidence (rev12): {sum(c['valid'] for c in CV)} valid changes, "
                  f"{sum(c['required'] for c in CV)} visible to Qwen, {sum(c['covered'] for c in CV)} in the caption "
                  f"({sum(c['covered'] for c in CV) / max(sum(c['required'] for c in CV), 1):.0%}); "
                  f"{sum(c['complete'] for c in CV)}/{len(CV)} captions complete; "
                  f"{sum(bool(c['contradictions']) for c in CV)} still contradicting a measurement; "
                  f"attempts {_tcc.Counter(c['attempts'] for c in CV)}")
            byk = _tcc.defaultdict(lambda: [0, 0, 0])          # valid, visible, covered
            for e in ev:
                b = byk[e["kind"]]; b[0] += 1
                b[1] += e.get("status") in ("visible", "unconfirmed")
                b[2] += str(e.get("final", "")).startswith("covered")
            print(f"    {'kind':<18}{'valid':>7}{'visible':>9}{'in caption':>12}")
            for k_, (a_, b_, c_) in sorted(byk.items(), key=lambda kv: -kv[1][0]):
                print(f"    {k_:<18}{a_:>7}{b_:>9}{c_:>12}")
            why = _tcc.Counter(e.get("why", "") for T in Ts for e in T["caption"].get("excluded", []))
            if why:
                print("    excluded: " + "; ".join(f"{n}x {w}" for w, n in why.most_common(5)))
        miss = _tcc.Counter(_tre.sub(r"^(AnyChange region \d+|object \d+): ", "", it["change"])
                            for T in Ts for it in T["unmentioned"])
        print("  most common changes detected but not named in the caption (word test, any module):")
        for ch, n in miss.most_common(8): print(f"    {n:>3}  {ch}")
        target = gname if not gname.startswith("unlabelled") else None
        why = _tcc.Counter(f"{q['disaster']}: {q['reason'].split(':')[-1].strip()}" for T in Ts
                           for q in T["dkg"]["rules"] if q.get("reason")
                           and (q["disaster"] == target or (target is None and
                                q["disaster"] in ("earthquake", "volcanic_eruption", "flood"))))
        if why: print(f"  DKG rules that scored 0 (missing required concept): {dict(why)}")
    print(f"\nreports: {out_dir}/index.html  (+ one .html per pair)" + (f" | failed: {fails}" if fails else ""))

    # inline: one report per class, displayed in the notebook
    try:
        from IPython.display import HTML as _THTML, display as _tdisplay
        for cls in inline_classes or ():
            T = _pick_inline(traces, cls)
            if T is None:
                print(f"\n(no {cls} pair among the traced pairs to display)"); continue
            print(f"\n{'─' * 24} {cls}: {T['key']}  (truth {T['truth']}, pipeline {T['verdict']['posterior']}) {'─' * 24}")
            _tdisplay(_THTML(open(f"{out_dir}/{T['key']}.html").read()))
    except Exception as e:
        print(f"(inline display unavailable: {e})")
    return traces


def preview_pairs(img_dir, labels_csv=None, n=8):
    """Check a folder before tracing it: how many pairs, which classes, and a few examples."""
    labels = _load_labels(labels_csv)
    found = find_pairs(img_dir)
    for p in found:
        stem = _tos.path.basename(p["rel"])
        p["cls"] = labels.get(p["key"]) or labels.get(stem) or _class_from_text(p["rel"]) or "unknown"
    print(f"{img_dir}: {len(found)} pairs | classes {dict(_tcc.Counter(p['cls'] for p in found))}")
    for p in found[:n]:
        print(f"  {p['key']:<40} {p['cls']:<18} {_tos.path.basename(p['pre'])}  |  {_tos.path.basename(p['post'])}")
    if not found:
        print("  no pairs found. Expected either <name>_pre.<ext> + <name>_post.<ext>, or pre/ and post/ "
              "folders (also before/after, A/B, t1/t2) holding files with the same names.")
    elif len(found) < 10:
        print("  note: fewer than 10 pairs -- a calibration on this folder alone is noisy; consider "
              "calibration=<path to another phys_concept_baseline.json>")
    return found


In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # §22b · Evidence trace on the RSCC folder (with reference captions)
# #   Shows one earthquake, one volcanic eruption and one flood report inline below.
# #   Same calibration as §21: its phys_concept_baseline.json if §21 ran in this session,
# #   otherwise the same label-free median computed here on the same pairs.
# # ═══════════════════════════════════════════════════════════════════════════════
# _cal21 = globals().get("CC_CAL_FILE", "")
# RSCC_TRACES = trace_folder(
#     img_dir       = globals().get("CC_IMG_DIR", "/kaggle/input/datasets/vaidurya13/rscc-52/rscc_captioning_52"),
#     out_dir       = "/kaggle/working/evidence_trace_rev12",
#     refs_file     = globals().get("CC_RSCC_JSONL", "/kaggle/input/datasets/vaidurya13/rscc-gt/RSCC_qvq.jsonl"),
#     classes       = ("earthquake", "volcanic_eruption", "flood"),
#     max_per_class = None,          # e.g. 6, or {"flood": 3}, for a quicker run
#     calibration   = _cal21 if _cal21 and _tos.path.exists(_cal21) else "this_folder",
#     inline_classes = ("earthquake", "volcanic_eruption", "flood"))

## 23 · Evidence trace on a folder without caption ground truth

Same reports as §22b for any folder of image pairs. rev12: reports go to `evidence_trace_rev12_<folder>`, and the summary prints how many visible measured changes reached each caption, by kind. Set `NEW_IMG_DIR`, run once with `NEW_PREVIEW_ONLY = True` to see the pairs found and the class read for each, then set it to `False` to trace.

**Pair layouts recognised (subfolders are searched):**
- `<name>_pre.png` + `<name>_post.png`, also `_pre_disaster`, `_before` / `_after`, any capitalisation; png, jpg, tif, bmp, webp.
- `pre/<name>.png` + `post/<name>.png`, also `before/after`, `A/B`, `t1/t2`, `im1/im2`, `image1/image2`.

**Disaster class**, in this order:
1. `NEW_LABELS_CSV` if given: one row per pair, `name,class`, where name is the pair or image name and class is any spelling (volcano, flooding, quake…).
2. A disaster word in the folder or file name.
3. Otherwise `unknown`. Unknown pairs are still traced and summarised by the pipeline's verdict, and the inline earthquake / volcanic eruption / flood reports are then chosen by verdict.

**Calibration:** the DKG is calibrated on this folder's own pairs (label-free median, cached in the output folder). With fewer than about 10 pairs that is noisy; pass `calibration=` the path of another `phys_concept_baseline.json` instead.

**Without ground truth there is no METEOR / ROUGE.** Everything else in the trace works. Classification accuracy needs class labels, from names or the CSV.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# §23 · Evidence trace on a folder WITHOUT caption ground truth
#   1. set NEW_IMG_DIR, keep NEW_PREVIEW_ONLY = True and run: it lists the pairs and the
#      class read from each name, in a second;
#   2. if that looks right, set NEW_PREVIEW_ONLY = False and run again to trace.
#   The disaster class comes from NEW_LABELS_CSV if given, else from a disaster word in the
#   folder / file names; pairs with neither are traced as "unknown" and summarised by the
#   pipeline's verdict. One earthquake, one volcanic eruption and one flood report are
#   shown inline (chosen by the pipeline's verdict when the folder has no labels).
# ═══════════════════════════════════════════════════════════════════════════════
NEW_IMG_DIR      = "/kaggle/input/datasets/vaidurya13/sample2/sample2"     # <-- EDIT
NEW_LABELS_CSV   = None        # optional CSV: "pair_or_image_name,class" (volcano / flood / earthquake ...)
NEW_PREVIEW_ONLY = False       # True: just list the pairs found; False: trace them

preview_pairs(NEW_IMG_DIR, NEW_LABELS_CSV)
if not NEW_PREVIEW_ONLY:
    NEW_TRACES = trace_folder(
        img_dir       = NEW_IMG_DIR,
        out_dir       = "/kaggle/working/evidence_trace_rev12_"      # rev12: never reuses rev11 reports
                        + _tre.sub(r"[^A-Za-z0-9]+", "_", _tos.path.basename(NEW_IMG_DIR.rstrip("/"))),
        refs_file     = None,          # no caption ground truth
        classes       = None,          # every pair, unknown classes included
        max_per_class = None,          # e.g. 5 for a quick look
        labels_csv    = NEW_LABELS_CSV,
        calibration   = "this_folder", # label-free; or a path to another phys_concept_baseline.json
        inline_classes = ("earthquake", "volcanic_eruption", "flood"),
        caption_mode  = "look")        # rev12 system caption; "revise" traces the rev9-rev11 caption


In [ ]:
import os, shutil, time
from IPython.display import FileLink, display
W = "/kaggle/working"
for d in sorted(os.listdir(W)):
    if os.path.isdir(f"{W}/{d}") and d.startswith(("caption_compare", "evidence_trace", "dashboard_runs", "single_trace")):
        z = shutil.make_archive(f"{W}/{d}_{time.strftime('%Y%m%d_%H%M')}", "zip", W, d)
        print(f"{d}: {os.path.getsize(z) / 1e6:.1f} MB"); display(FileLink(os.path.relpath(z, W)))